In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


# New Section

In [ ]:
# This cell loads the prepared AMI baseline dataset from the existing project structure.

import os
import pandas as pd

PROJECT_DIR = "/content/drive/MyDrive/MTechIndProj/MoM_Project"
BASELINE_PATH = os.path.join(
    PROJECT_DIR,
    "data",
    "processed",
    "ami_baseline_dataset.csv"
)

print("Project directory:", PROJECT_DIR)
print("Baseline dataset:", BASELINE_PATH)
print("Dataset exists:", os.path.exists(BASELINE_PATH))

if not os.path.exists(BASELINE_PATH):
    raise FileNotFoundError(f"Dataset not found: {BASELINE_PATH}")

baseline_df = pd.read_csv(
    BASELINE_PATH,
    encoding="utf-8"
)

print("\nDataset loaded successfully.")
print("Shape:", baseline_df.shape)
print("Columns:", baseline_df.columns.tolist())
print("Meetings:", baseline_df["meeting_id"].tolist())

Project directory: /content/drive/MyDrive/MTechIndProj/MoM_Project
Baseline dataset: /content/drive/MyDrive/MTechIndProj/MoM_Project/data/processed/ami_baseline_dataset.csv
Dataset exists: True

Dataset loaded successfully.
Shape: (10, 6)
Columns: ['meeting_id', 'transcript', 'reference_summary', 'clean_transcript', 'timestamped_words', 'clean_words']
Meetings: ['ES2004a', 'ES2004b', 'ES2004c', 'ES2004d', 'ES2005a', 'ES2005b', 'ES2005c', 'ES2006a', 'ES2006b', 'ES2008a']


In [ ]:
# This cell compares the raw and existing cleaned transcript for ES2004a before we send it to the summarization model.

meeting_id = "ES2004a"

row = baseline_df.loc[
    baseline_df["meeting_id"] == meeting_id
].iloc[0]

print("=" * 70)
print("RAW TRANSCRIPT — FIRST 1500 CHARACTERS")
print("=" * 70)
print(row["transcript"][:1500])

print("\n" + "=" * 70)
print("CLEAN TRANSCRIPT — FIRST 1500 CHARACTERS")
print("=" * 70)
print(row["clean_transcript"][:1500])

print("\n" + "=" * 70)
print("CLEAN WORDS — FIRST 1000 CHARACTERS")
print("=" * 70)
print(str(row["clean_words"])[:1000])

RAW TRANSCRIPT — FIRST 1500 CHARACTERS
[0.37 - 0.95] A: Hmm
[0.95 - 1.53] A: hmm
[1.53 - 1.76] A: hmm
[1.76 - 1.76] A: .
[10.99 - 11.02] B: Are
[11.02 - 12.13] B: we
[12.13 - 12.29] B: we're
[12.29 - 12.42] B: not
[12.42 - 12.62] B: allowed
[12.62 - 12.70] B: to
[12.70 - 12.84] B: dim
[12.84 - 12.91] B: the
[12.91 - 13.18] B: lights
[13.18 - 13.31] B: so
[13.31 - 13.53] B: people
[13.53 - 13.71] B: can
[13.71 - 13.81] B: see
[13.81 - 13.96] B: that
[13.96 - 13.99] B: a
[13.99 - 14.15] B: bit
[14.15 - 14.53] B: better
[14.53 - 14.53] B: ?
[17.88 - 18.15] A: Yeah
[18.15 - 18.15] A: .
[18.87 - 19.70] B: Okay
[19.70 - 19.70] B: ,
[19.70 - 19.99] B: that's
[19.99 - 20.29] B: fine
[20.29 - 20.29] B: .
[22.37 - 22.50] B: Am
[22.50 - 22.56] B: I
[22.56 - 22.78] B: supposed
[22.78 - 22.84] B: to
[22.84 - 22.90] B: be
[22.90 - 23.28] B: standing
[23.28 - 23.44] B: up
[23.44 - 23.81] B: there
[23.81 - 23.81] B: ?
[25.15 - 25.23] D: So
[25.18 - 25.60] B: Okay
[25.23 - 25.33] D: we've
[25.33 - 25.4

In [ ]:
# This cell reconstructs the word-level AMI transcript into readable speaker utterances for BART input.

import re
import pandas as pd

def reconstruct_transcript(clean_transcript):
    """
    Convert AMI word-level speaker annotations into readable text.

    Example:
        B: Hello
        B: everybody
        A: Hi
        A: ,
        A: good
        A: morning
        B: .

    becomes:
        B: Hello everybody.
        A: Hi, good morning.
    """

    if not isinstance(clean_transcript, str):
        return ""

    lines = clean_transcript.splitlines()

    utterances = []
    current_speaker = None
    current_words = []

    punctuation = {".", ",", "?", "!", ";", ":"}

    def flush_utterance():
        nonlocal current_speaker, current_words

        if not current_words:
            return

        text = ""

        for word in current_words:
            word = word.strip()

            if not word:
                continue

            if word in punctuation:
                text = text.rstrip() + word
            else:
                if text:
                    text += " "
                text += word

        text = re.sub(r"\s+", " ", text).strip()

        if text:
            utterances.append(f"{current_speaker}: {text}")

        current_words = []

    for line in lines:

        line = line.strip()

        if not line:
            continue

        match = re.match(r"^([A-Z]):\s*(.*)$", line)

        if not match:
            continue

        speaker = match.group(1)
        word = match.group(2).strip()

        # Speaker changed → finish previous utterance
        if current_speaker is not None and speaker != current_speaker:
            flush_utterance()

        current_speaker = speaker

        if word:
            current_words.append(word)

        # Sentence-ending punctuation → finish utterance
        if word in {".", "?", "!"}:
            flush_utterance()

    # Add final unfinished utterance
    flush_utterance()

    return "\n".join(utterances)


# Test on ES2004a
test_clean = baseline_df.loc[
    baseline_df["meeting_id"] == "ES2004a",
    "clean_transcript"
].iloc[0]

bart_test = reconstruct_transcript(test_clean)

print("=" * 70)
print("BART-READY TRANSCRIPT — ES2004a")
print("=" * 70)
print(bart_test[:3000])

BART-READY TRANSCRIPT — ES2004a
A: Hmm hmm hmm.
B: Are we we're not allowed to dim the lights so people can see that a bit better?
A: Yeah.
B: Okay, that's fine.
B: Am I supposed to be standing up there?
D: So
B: Okay
D: we've got both
B: .
D: of these clipped on?
D: She gonna answer me
B: Yeah
D: or not
B: , I've got
D: ?
D: Right, both of them, okay.
B: Yes.
D: God.
D: Jesus, it's gonna fall off.
A: Okay.
A: Yep, yep.
A: Okay.
B: Okay
A: Tu tu tu tu
B: .
B: Hello everybody
A: Hi, good morning
B: .
B: Um
A: .
B: I'm Sarah, the Project Manager and this is our first meeting, surprisingly enough.
B: Okay, this is our agenda, um we will do some stuff, get to know each other a bit better to feel more comfortable with each other.
B: Um then we'll go do tool training, talk about the project plan, discuss our own ideas and everything um and we've got twenty five minutes to do that, as far as I can understand.
B: Now, we're developing a remote control which you probably already know.
B: Um, we

In [ ]:
# This cell creates a speaker-neutral BART input while preserving the chronological content of the AMI transcript.

def create_bart_input(reconstructed_transcript):
    """
    Remove AMI speaker labels while preserving the reconstructed
    chronological transcript.
    """

    if not isinstance(reconstructed_transcript, str):
        return ""

    lines = reconstructed_transcript.splitlines()
    cleaned_lines = []

    for line in lines:
        line = line.strip()

        if not line:
            continue

        # Remove speaker label such as A:, B:, C:, D:
        line = re.sub(r"^[A-Z]:\s*", "", line)

        # Normalize whitespace
        line = re.sub(r"\s+", " ", line).strip()

        if line:
            cleaned_lines.append(line)

    # Join utterances into paragraphs
    text = " ".join(cleaned_lines)

    # Normalize spaces before punctuation
    text = re.sub(r"\s+([,.!?;:])", r"\1", text)

    # Normalize repeated spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text


bart_input_test = create_bart_input(bart_test)

print("=" * 70)
print("SPEAKER-NEUTRAL BART INPUT — ES2004a")
print("=" * 70)
print(bart_input_test[:3000])

print("\n" + "=" * 70)
print("WORD COUNT")
print("=" * 70)
print(len(bart_input_test.split()))

SPEAKER-NEUTRAL BART INPUT — ES2004a
Hmm hmm hmm. Are we we're not allowed to dim the lights so people can see that a bit better? Yeah. Okay, that's fine. Am I supposed to be standing up there? So Okay we've got both. of these clipped on? She gonna answer me Yeah or not, I've got? Right, both of them, okay. Yes. God. Jesus, it's gonna fall off. Okay. Yep, yep. Okay. Okay Tu tu tu tu. Hello everybody Hi, good morning. Um. I'm Sarah, the Project Manager and this is our first meeting, surprisingly enough. Okay, this is our agenda, um we will do some stuff, get to know each other a bit better to feel more comfortable with each other. Um then we'll go do tool training, talk about the project plan, discuss our own ideas and everything um and we've got twenty five minutes to do that, as far as I can understand. Now, we're developing a remote control which you probably already know. Um, we want it to be original, something that's uh people haven't thought of, that's not out in the shops, um, t

In [ ]:
# This cell creates BART-ready inputs for all meetings and checks their lengths before chunking.

baseline_df["bart_input"] = baseline_df["clean_transcript"].apply(
    lambda x: create_bart_input(reconstruct_transcript(x))
)

baseline_df["bart_input_words"] = baseline_df["bart_input"].apply(
    lambda x: len(x.split())
)

print("=" * 70)
print("BART INPUT LENGTHS")
print("=" * 70)

print(
    baseline_df[
        ["meeting_id", "bart_input_words"]
    ].to_string(index=False)
)

print("\nAverage BART input words:",
      round(baseline_df["bart_input_words"].mean()))

print("Maximum BART input words:",
      baseline_df["bart_input_words"].max())

print("Minimum BART input words:",
      baseline_df["bart_input_words"].min())

BART INPUT LENGTHS
meeting_id  bart_input_words
   ES2004a              2614
   ES2004b              6763
   ES2004c              6968
   ES2004d              6134
   ES2005a               747
   ES2005b              6190
   ES2005c              6694
   ES2006a              2777
   ES2006b              6201
   ES2008a              2506

Average BART input words: 4759
Maximum BART input words: 6968
Minimum BART input words: 747


In [ ]:
# This cell splits each BART-ready meeting transcript into overlapping chunks that can be processed safely by BART.

CHUNK_WORDS = 900
OVERLAP_WORDS = 100


def chunk_text(text, chunk_words=CHUNK_WORDS, overlap_words=OVERLAP_WORDS):
    """Split text into overlapping word-based chunks."""

    words = text.split()

    if not words:
        return []

    chunks = []
    start = 0

    while start < len(words):
        end = min(start + chunk_words, len(words))

        chunk = " ".join(words[start:end])
        chunks.append(chunk)

        if end >= len(words):
            break

        start = end - overlap_words

    return chunks


# Test chunking on ES2004a
test_chunks = chunk_text(
    baseline_df.loc[
        baseline_df["meeting_id"] == "ES2004a",
        "bart_input"
    ].iloc[0]
)

print("=" * 70)
print("CHUNKING TEST — ES2004a")
print("=" * 70)

print("Total words:", len(
    baseline_df.loc[
        baseline_df["meeting_id"] == "ES2004a",
        "bart_input"
    ].iloc[0].split()
))

print("Number of chunks:", len(test_chunks))

for i, chunk in enumerate(test_chunks, 1):
    print(f"\nChunk {i}: {len(chunk.split())} words")
    print(chunk[:300] + "...")

CHUNKING TEST — ES2004a
Total words: 2614
Number of chunks: 4

Chunk 1: 900 words
Hmm hmm hmm. Are we we're not allowed to dim the lights so people can see that a bit better? Yeah. Okay, that's fine. Am I supposed to be standing up there? So Okay we've got both. of these clipped on? She gonna answer me Yeah or not, I've got? Right, both of them, okay. Yes. God. Jesus, it's gonna ...

Chunk 2: 900 words
Uh. It's not a vampire bat honestly Okay, yeah.. Uh and somewhere there's a body behind Okay. That's, some my dreadful sort of that's the worst yet bird, that's. it's meant to be an eagle A seagu Ah Eagle right eagle, okay, right., not you okay can a seagull tell. it's a flying animal could. have be...

Chunk 3: 900 words
they're. when you've got the main things on the front of it and a section opens up or something to the other functions where you can do sound or options Oh yeah kind. of recording, things like that inside it Mm-hmm.. 'Cause it doesn't make when you pick it up it doesn't

In [ ]:
# This cell loads the BART summarization model on the available Tesla T4 GPU for a single-chunk baseline test.

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = "facebook/bart-large-cnn"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Model:", MODEL_NAME)
print("Device:", DEVICE)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

bart_model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME
).to(DEVICE)

bart_model.eval()

print("BART loaded successfully.")

Model: facebook/bart-large-cnn
Device: cuda


config.json:   0%|          | 0.00/1.58k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

BART loaded successfully.


In [ ]:
# This cell generates a first BART summary from one ES2004a transcript chunk to verify the model output before processing all meetings.

import torch

test_chunk = test_chunks[0]

inputs = tokenizer(
    test_chunk,
    return_tensors="pt",
    max_length=1024,
    truncation=True
)

inputs = {
    key: value.to(DEVICE)
    for key, value in inputs.items()
}

with torch.no_grad():
    summary_ids = bart_model.generate(
        **inputs,
        max_length=180,
        min_length=60,
        num_beams=4,
        length_penalty=2.0,
        no_repeat_ngram_size=3,
        early_stopping=True
    )

test_summary = tokenizer.decode(
    summary_ids[0],
    skip_special_tokens=True
)

print("=" * 70)
print("BART TEST SUMMARY — ES2004a CHUNK 1")
print("=" * 70)
print(test_summary)

BART TEST SUMMARY — ES2004a CHUNK 1
Sarah, the Project Manager, introduces the team. They will work on a remote control that can be controlled by a dog. The team then draw their favourite animal on a white board. The group then discuss their ideas and work on the design. The project is due to be completed by the end of the year.


In [ ]:
# This cell creates stable BART token chunks without altering BPE spacing during decoding.

BART_CHUNK_TOKENS = 900
BART_OVERLAP_TOKENS = 100


def chunk_text_by_tokens(
    text,
    tokenizer,
    chunk_tokens=BART_CHUNK_TOKENS,
    overlap_tokens=BART_OVERLAP_TOKENS
):
    """Create overlapping chunks directly from BART token IDs."""

    token_ids = tokenizer.encode(
        text,
        add_special_tokens=False
    )

    if not token_ids:
        return []

    chunks = []
    start = 0

    while start < len(token_ids):

        end = min(
            start + chunk_tokens,
            len(token_ids)
        )

        chunk_ids = token_ids[start:end]

        chunk_text = tokenizer.decode(
            chunk_ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False
        ).strip()

        chunks.append({
            "text": chunk_text,
            "token_count": len(chunk_ids),
            "start_token": start,
            "end_token": end
        })

        if end >= len(token_ids):
            break

        start = end - overlap_tokens

    return chunks


# Test the corrected token-aware chunking on ES2004a.

es2004a_bart_input = baseline_df.loc[
    baseline_df["meeting_id"] == "ES2004a",
    "bart_input"
].iloc[0]

token_chunks = chunk_text_by_tokens(
    es2004a_bart_input,
    tokenizer
)

print("=" * 70)
print("CORRECTED TOKEN-AWARE CHUNKING — ES2004a")
print("=" * 70)

total_tokens = len(
    tokenizer.encode(
        es2004a_bart_input,
        add_special_tokens=False
    )
)

print("Total input tokens:", total_tokens)
print("Number of chunks:", len(token_chunks))

for i, chunk in enumerate(token_chunks, 1):
    print(
        f"\nChunk {i}: "
        f"{chunk['token_count']} tokens "
        f"({chunk['start_token']} → {chunk['end_token']})"
    )
    print(chunk["text"][:300] + "...")

CORRECTED TOKEN-AWARE CHUNKING — ES2004a
Total input tokens: 3463
Number of chunks: 5

Chunk 1: 900 tokens (0 → 900)
Hmm hmm hmm. Are we we're not allowed to dim the lights so people can see that a bit better? Yeah. Okay, that's fine. Am I supposed to be standing up there? So Okay we've got both. of these clipped on? She gonna answer me Yeah or not, I've got? Right, both of them, okay. Yes. God. Jesus, it's gonna ...

Chunk 2: 900 tokens (800 → 1700)
gonna be because that looks like a beak now, so. Crocodile? Gonna be Yeah a, it bird can be. a crocodile, it can be Is a it crocodile gonna be. Well it was it was it's an gonna at be first a bird firstly. it was an attempt at a T_ Rex and then it sort O of changed into a pelican but it can be a croc...

Chunk 3: 900 tokens (1600 → 2500)
mm.. Um, and profit aim is fifty million Euros, which is uh In our first year? Yi yes, um yeah, I presume so Mm-hmm. So. Um then You've got market range international and you did say earlier it's got to be 

In [ ]:
# This cell generates an abstractive BART summary for each token-aware ES2004a chunk.

def summarize_chunk(chunk_text):
    inputs = tokenizer(
        chunk_text,
        return_tensors="pt",
        max_length=1024,
        truncation=False
    )

    inputs = {
        key: value.to(DEVICE)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        summary_ids = bart_model.generate(
            **inputs,
            max_length=180,
            min_length=40,
            num_beams=4,
            length_penalty=2.0,
            no_repeat_ngram_size=3,
            early_stopping=True
        )

    return tokenizer.decode(
        summary_ids[0],
        skip_special_tokens=True
    )


chunk_summaries = []

print("=" * 70)
print("BART CHUNK SUMMARIZATION — ES2004a")
print("=" * 70)

for i, chunk in enumerate(token_chunks, 1):

    print(f"\nProcessing chunk {i}/{len(token_chunks)}...")

    summary = summarize_chunk(chunk["text"])

    chunk_summaries.append(summary)

    print(f"\nCHUNK {i} SUMMARY:")
    print(summary)

print("\n" + "=" * 70)
print("ALL CHUNK SUMMARIES GENERATED")
print("=" * 70)
print("Number of summaries:", len(chunk_summaries))

BART CHUNK SUMMARIZATION — ES2004a

Processing chunk 1/5...

CHUNK 1 SUMMARY:
Sarah, the Project Manager, introduces the team. They will work on a remote control that can be controlled by a dog. The team then draw their favourite animal on the white board. The group then discuss their ideas and work on the design.

Processing chunk 2/5...

CHUNK 2 SUMMARY:
It was an attempt at a T_ Rex and then it sort O of changed into a pelican but it can be a crocodile now actually. We've got a selling price at twenty five Euros, which I don't actually know what that is in Pounds, at all. The profit aim is fifty million Euros.

Processing chunk 3/5...

CHUNK 3 SUMMARY:
The company is aiming for a profit of fifty million Euros in its first year. It is aimed at the international market, not the business market. The company is also targeting the older generation.

Processing chunk 4/5...

CHUNK 4 SUMMARY:
I don't know how for twenty fi, or twelve Euros fifty how much of a excellent screen you could get

In [ ]:
# This cell combines the five independently generated BART chunk summaries into one baseline meeting-level summary input.

combined_chunk_summary = " ".join(chunk_summaries)

print("=" * 70)
print("COMBINED CHUNK SUMMARIES — ES2004a")
print("=" * 70)

print(combined_chunk_summary)

print("\n" + "=" * 70)
print("COMBINED SUMMARY TOKEN COUNT")
print("=" * 70)

combined_tokens = tokenizer.encode(
    combined_chunk_summary,
    add_special_tokens=False
)

print("Tokens:", len(combined_tokens))

COMBINED CHUNK SUMMARIES — ES2004a
Sarah, the Project Manager, introduces the team. They will work on a remote control that can be controlled by a dog. The team then draw their favourite animal on the white board. The group then discuss their ideas and work on the design. It was an attempt at a T_ Rex and then it sort O of changed into a pelican but it can be a crocodile now actually. We've got a selling price at twenty five Euros, which I don't actually know what that is in Pounds, at all. The profit aim is fifty million Euros. The company is aiming for a profit of fifty million Euros in its first year. It is aimed at the international market, not the business market. The company is also targeting the older generation. I don't know how for twenty fi, or twelve Euros fifty how much of a excellent screen you could get Yeah, you'd you'd have to keep it down to a black and white L_C_D_ thing anyway. The other thing is, just ch chucking into mobile phone f design features again, it could h

In [ ]:
# This cell performs the second-stage BART summarization to create the final meeting-level baseline MoM for ES2004a.

final_inputs = tokenizer(
    combined_chunk_summary,
    return_tensors="pt",
    max_length=1024,
    truncation=True
)

final_inputs = {
    key: value.to(DEVICE)
    for key, value in final_inputs.items()
}

with torch.no_grad():
    final_summary_ids = bart_model.generate(
        **final_inputs,
        max_length=220,
        min_length=80,
        num_beams=4,
        length_penalty=2.0,
        no_repeat_ngram_size=3,
        early_stopping=True
    )

es2004a_baseline_summary = tokenizer.decode(
    final_summary_ids[0],
    skip_special_tokens=True
)

print("=" * 70)
print("FINAL BART BASELINE — ES2004a")
print("=" * 70)
print(es2004a_baseline_summary)

FINAL BART BASELINE — ES2004a
The company is aiming for a profit of fifty million Euros in its first year. It is aimed at the international market, not the business market. The company is also targeting the older generation. It could have a flip top remote control so that when you flip over the top, your screen is you can have a bigger screen in Mm-hmm the. flip over. . Just. just a quick thing about Sure the. um about what you're saying about the uh does does it need to be fashionable?


Then we'll compare it with the AMI reference

The reference for ES2004a is:

The Project Manager gave an introduction to the goal of the project,
to create a trendy yet user-friendly remote.

She presented a long-range agenda for the whole project.

The group introduced themselves to each other and practiced with
the meeting room tools by drawing on the board.

The Project Manager presented the project budget, the projected
price point, and the projected profit aim for the project.

Then the group began a discussion about their own experiences with
remote controls to generate initial design ideas for making the
product user-friendly.

They discussed grouping features into a menu and adding an LCD display.

They also discussed the look of various materials that may be used
in the design, in keeping with the company's goal to create
fashionable electronics.

The important baseline question is now:

Can plain BART recover the important meeting-level information from the transcript?

In [ ]:
# This cell records the exact baseline BART configuration so later improvements can be compared fairly.

BASELINE_CONFIG = {
    "model": "facebook/bart-large-cnn",
    "device": str(DEVICE),
    "chunk_tokens": BART_CHUNK_TOKENS,
    "overlap_tokens": BART_OVERLAP_TOKENS,
    "chunk_max_length": 180,
    "chunk_min_length": 40,
    "final_max_length": 220,
    "final_min_length": 80,
    "num_beams": 4,
    "length_penalty": 2.0,
    "no_repeat_ngram_size": 3,
    "dataset": "AMI",
    "meetings": len(baseline_df)
}

print("=" * 70)
print("BART BASELINE CONFIGURATION")
print("=" * 70)

for key, value in BASELINE_CONFIG.items():
    print(f"{key:25}: {value}")

BART BASELINE CONFIGURATION
model                    : facebook/bart-large-cnn
device                   : cuda
chunk_tokens             : 900
overlap_tokens           : 100
chunk_max_length         : 180
chunk_min_length         : 40
final_max_length         : 220
final_min_length         : 80
num_beams                : 4
length_penalty           : 2.0
no_repeat_ngram_size     : 3
dataset                  : AMI
meetings                 : 10


In [ ]:
# This cell runs the same BART baseline pipeline on all 10 AMI meetings and saves each generated MoM.

import os
import json
import time

MOM_OUTPUT_DIR = os.path.join(
    PROJECT_DIR,
    "outputs",
    "mom"
)

os.makedirs(MOM_OUTPUT_DIR, exist_ok=True)

all_baseline_results = []

print("=" * 70)
print("RUNNING BART BASELINE — ALL AMI MEETINGS")
print("=" * 70)

for idx, row in baseline_df.iterrows():

    meeting_id = row["meeting_id"]
    bart_input = row["bart_input"]

    print(
        f"\n[{idx + 1}/{len(baseline_df)}] "
        f"Processing {meeting_id}..."
    )

    start_time = time.time()

    # Create token-aware chunks
    meeting_chunks = chunk_text_by_tokens(
        bart_input,
        tokenizer
    )

    meeting_chunk_summaries = []

    # Generate summary for each chunk
    for chunk_idx, chunk in enumerate(meeting_chunks, 1):

        summary = summarize_chunk(
            chunk["text"]
        )

        meeting_chunk_summaries.append(summary)

    # Combine chunk summaries
    combined_summary = " ".join(
        meeting_chunk_summaries
    )

    # Token count of combined summaries
    combined_token_count = len(
        tokenizer.encode(
            combined_summary,
            add_special_tokens=False
        )
    )

    # Second-stage summarization
    final_inputs = tokenizer(
        combined_summary,
        return_tensors="pt",
        max_length=1024,
        truncation=True
    )

    final_inputs = {
        key: value.to(DEVICE)
        for key, value in final_inputs.items()
    }

    with torch.no_grad():

        final_summary_ids = bart_model.generate(
            **final_inputs,
            max_length=220,
            min_length=80,
            num_beams=4,
            length_penalty=2.0,
            no_repeat_ngram_size=3,
            early_stopping=True
        )

    final_summary = tokenizer.decode(
        final_summary_ids[0],
        skip_special_tokens=True
    )

    elapsed = time.time() - start_time

    # Store complete result
    result = {
        "meeting_id": meeting_id,
        "model": "facebook/bart-large-cnn",
        "num_chunks": len(meeting_chunks),
        "combined_summary_tokens": combined_token_count,
        "chunk_summaries": meeting_chunk_summaries,
        "final_summary": final_summary,
        "processing_time_seconds": round(elapsed, 2)
    }

    all_baseline_results.append(result)

    # Save individual meeting result
    output_path = os.path.join(
        MOM_OUTPUT_DIR,
        f"{meeting_id}_bart_baseline.json"
    )

    with open(
        output_path,
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            result,
            f,
            indent=2,
            ensure_ascii=False
        )

    print(
        f"  Chunks: {len(meeting_chunks)}"
    )

    print(
        f"  Combined tokens: {combined_token_count}"
    )

    print(
        f"  Time: {elapsed:.1f} sec"
    )

    print(
        f"  Saved: {output_path}"
    )

print("\n" + "=" * 70)
print("BART BASELINE RUN COMPLETE")
print("=" * 70)

print(
    "Meetings processed:",
    len(all_baseline_results)
)

RUNNING BART BASELINE — ALL AMI MEETINGS

[1/10] Processing ES2004a...
  Chunks: 5
  Combined tokens: 327
  Time: 14.1 sec
  Saved: /content/drive/MyDrive/MTechIndProj/MoM_Project/outputs/mom/ES2004a_bart_baseline.json

[2/10] Processing ES2004b...
  Chunks: 11
  Combined tokens: 794
  Time: 19.2 sec
  Saved: /content/drive/MyDrive/MTechIndProj/MoM_Project/outputs/mom/ES2004b_bart_baseline.json

[3/10] Processing ES2004c...
  Chunks: 11
  Combined tokens: 823
  Time: 18.9 sec
  Saved: /content/drive/MyDrive/MTechIndProj/MoM_Project/outputs/mom/ES2004c_bart_baseline.json

[4/10] Processing ES2004d...
  Chunks: 10
  Combined tokens: 604
  Time: 15.0 sec
  Saved: /content/drive/MyDrive/MTechIndProj/MoM_Project/outputs/mom/ES2004d_bart_baseline.json

[5/10] Processing ES2005a...
  Chunks: 2
  Combined tokens: 158
  Time: 4.4 sec
  Saved: /content/drive/MyDrive/MTechIndProj/MoM_Project/outputs/mom/ES2005a_bart_baseline.json

[6/10] Processing ES2005b...
  Chunks: 10
  Combined tokens: 661
 

In [ ]:
# This cell checks whether the Hugging Face Evaluate library is available for computing ROUGE scores.

import importlib.util

evaluate_available = (
    importlib.util.find_spec("evaluate") is not None
)

print("=" * 70)
print("EVALUATION LIBRARY CHECK")
print("=" * 70)

print("Evaluate installed:", evaluate_available)

EVALUATION LIBRARY CHECK
Evaluate installed: False


In [ ]:
# This cell installs the Hugging Face evaluation package and ROUGE dependency required for baseline evaluation.

!pip install -q evaluate rouge_score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.7 MB/s eta 0:00:00


In [ ]:
# This cell verifies that the evaluation library and ROUGE metric can now be imported successfully.

import evaluate

print("=" * 70)
print("EVALUATION LIBRARY READY")
print("=" * 70)

print("Evaluate version:", evaluate.__version__)

rouge = evaluate.load("rouge")

print("ROUGE metric loaded successfully.")

EVALUATION LIBRARY READY
Evaluate version: 0.4.6


ROUGE metric loaded successfully.


In [ ]:
# This cell computes ROUGE-1, ROUGE-2, and ROUGE-L for all 10 BART-generated MoMs against the AMI reference summaries.

import os
import json
import pandas as pd

rouge_results = []

print("=" * 70)
print("BART BASELINE — ROUGE EVALUATION")
print("=" * 70)

for _, row in baseline_df.iterrows():

    meeting_id = row["meeting_id"]

    output_path = os.path.join(
        MOM_OUTPUT_DIR,
        f"{meeting_id}_bart_baseline.json"
    )

    with open(
        output_path,
        "r",
        encoding="utf-8"
    ) as f:
        result = json.load(f)

    generated_summary = result["final_summary"]
    reference_summary = row["reference_summary"]

    scores = rouge.compute(
        predictions=[generated_summary],
        references=[reference_summary],
        use_stemmer=True
    )

    rouge_results.append({
        "meeting_id": meeting_id,
        "rouge1": scores["rouge1"],
        "rouge2": scores["rouge2"],
        "rougeL": scores["rougeL"]
    })

    print(
        f"{meeting_id}: "
        f"R1={scores['rouge1']:.4f} | "
        f"R2={scores['rouge2']:.4f} | "
        f"RL={scores['rougeL']:.4f}"
    )


rouge_df = pd.DataFrame(rouge_results)

print("\n" + "=" * 70)
print("AVERAGE BASELINE ROUGE")
print("=" * 70)

print(
    f"ROUGE-1: {rouge_df['rouge1'].mean():.4f}"
)

print(
    f"ROUGE-2: {rouge_df['rouge2'].mean():.4f}"
)

print(
    f"ROUGE-L: {rouge_df['rougeL'].mean():.4f}"
)

BART BASELINE — ROUGE EVALUATION
ES2004a: R1=0.2247 | R2=0.0226 | RL=0.1273
ES2004b: R1=0.1671 | R2=0.0148 | RL=0.1032
ES2004c: R1=0.2143 | R2=0.0240 | RL=0.1310
ES2004d: R1=0.1758 | R2=0.0239 | RL=0.0998
ES2005a: R1=0.1949 | R2=0.0104 | RL=0.1333
ES2005b: R1=0.2382 | R2=0.0891 | RL=0.1496
ES2005c: R1=0.2135 | R2=0.0524 | RL=0.1406
ES2006a: R1=0.2126 | R2=0.0462 | RL=0.1264
ES2006b: R1=0.1818 | R2=0.0400 | RL=0.0966
ES2008a: R1=0.1720 | R2=0.0432 | RL=0.1075

AVERAGE BASELINE ROUGE
ROUGE-1: 0.1995
ROUGE-2: 0.0367
ROUGE-L: 0.1215


In [ ]:
# This cell saves the BART baseline ROUGE scores for use in later comparisons and the final project evaluation.

EVALUATION_DIR = os.path.join(
    PROJECT_DIR,
    "evaluation_results"
)

os.makedirs(
    EVALUATION_DIR,
    exist_ok=True
)

ROUGE_OUTPUT_PATH = os.path.join(
    EVALUATION_DIR,
    "bart_baseline_rouge.csv"
)

rouge_df.to_csv(
    ROUGE_OUTPUT_PATH,
    index=False
)

print("=" * 70)
print("BASELINE EVALUATION SAVED")
print("=" * 70)
print(ROUGE_OUTPUT_PATH)

BASELINE EVALUATION SAVED
/content/drive/MyDrive/MTechIndProj/MoM_Project/evaluation_results/bart_baseline_rouge.csv


08-09-2026

In [ ]:
# ============================================================
# PROJECT PATH SETUP
# Purpose:
# Define the project directories and verify access to the
# saved transcript and MoM artifacts.
# ============================================================

from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive/MyDrive/MTechIndProj/MoM_Project"
)

DATA_DIR = PROJECT_DIR / "data"
TRANSCRIPTS_DIR = DATA_DIR / "transcripts"

SPEAKER_TRANSCRIPT_PATH = (
    TRANSCRIPTS_DIR /
    "ES2004a_speaker_transcript.json"
)

MOM_CANDIDATES_PATH = (
    TRANSCRIPTS_DIR /
    "ES2004a_mom_candidates_rule_based.json"
)

MOM_WORTHY_PATH = (
    TRANSCRIPTS_DIR /
    "ES2004a_mom_worthy_candidates.json"
)

MOM_CLAIMS_PATH = (
    TRANSCRIPTS_DIR /
    "ES2004a_structured_mom_claims.json"
)

print("Project directory:", PROJECT_DIR)
print("Transcripts directory:", TRANSCRIPTS_DIR)

print("\nSaved artifacts:")

for path in [
    SPEAKER_TRANSCRIPT_PATH,
    MOM_CANDIDATES_PATH,
    MOM_WORTHY_PATH,
    MOM_CLAIMS_PATH
]:
    print(
        f"{'✓' if path.exists() else '✗'} "
        f"{path.name}"
    )

Project directory: /content/drive/MyDrive/MTechIndProj/MoM_Project
Transcripts directory: /content/drive/MyDrive/MTechIndProj/MoM_Project/data/transcripts

Saved artifacts:
✓ ES2004a_speaker_transcript.json
✓ ES2004a_mom_candidates_rule_based.json
✓ ES2004a_mom_worthy_candidates.json
✓ ES2004a_structured_mom_claims.json


In [ ]:
# ============================================================
# LOAD STRUCTURED MoM CLAIMS
# Purpose:
# Load the previously generated MoM claims so that the
# evidence retrieval stage can use them.
# ============================================================

import json

with open(MOM_CLAIMS_PATH, "r", encoding="utf-8") as f:
    mom_claims_data = json.load(f)

mom_claims = mom_claims_data["claims"]

print("=" * 80)
print("STRUCTURED MoM CLAIMS LOADED")
print("=" * 80)

print("\nMeeting ID:", mom_claims_data["meeting_id"])
print("Candidate count:", mom_claims_data["candidate_count"])
print("Topic groups:", mom_claims_data["topic_group_count"])
print("Claim count:", mom_claims_data["claim_count"])

print("\nFirst 3 claims:")

for claim in mom_claims[:3]:

    print(
        f"\nClaim {claim['claim_id']:02d} | "
        f"{claim['event_type']}"
    )

    print("Topic:", claim["topic"])
    print("Text:", claim["claim_text"])
    print("Sources:", claim["source_candidate_ids"])
    print(
        f"Time: {claim['start']:.3f}s → "
        f"{claim['end']:.3f}s"
    )
    print("Speaker:", claim["speaker"])

print("\n" + "=" * 80)

STRUCTURED MoM CLAIMS LOADED

Meeting ID: ES2004a
Candidate count: 42
Topic groups: 11
Claim count: 12

First 3 claims:

Claim 01 | INFORMATION
Topic: meeting_introduction_and_agenda
Text: The meeting was introduced as the first meeting, with an agenda covering team interaction, tool training, the project plan, and discussion of ideas.
Sources: [5, 6, 7, 8]
Time: 84.658s → 109.823s
Speaker: SPEAKER_02

Claim 02 | INFORMATION
Topic: product_objective_and_requirements
Text: The team is developing a remote control intended to be original, trendy, appealing to a wide market, and user-friendly for a broad range of users.
Sources: [10, 11, 12, 13]
Time: 117.699s → 141.016s
Speaker: SPEAKER_02

Claim 03 | INFORMATION
Topic: design_process
Text: The design process includes functional design, conceptual design, and detailed design, with individual work addressing product requirements and implementation.
Sources: [14, 15]
Time: 142.933s → 167.740s
Speaker: SPEAKER_02



In [ ]:
# ============================================================
# LOAD SPEAKER-ATTRIBUTED TRANSCRIPT
# Purpose:
# Load the speaker-attributed transcript that will serve as
# the evidence corpus for BGE + FAISS retrieval.
#
# Each utterance retains:
# - speaker
# - start timestamp
# - end timestamp
# - transcript text
# ============================================================

with open(
    SPEAKER_TRANSCRIPT_PATH,
    "r",
    encoding="utf-8"
) as f:
    transcript_data = json.load(f)

speaker_utterances = transcript_data["utterances"]

print("=" * 80)
print("SPEAKER-ATTRIBUTED TRANSCRIPT LOADED")
print("=" * 80)

print("\nMeeting ID:", transcript_data["meeting_id"])
print("Number of speakers:", transcript_data["num_speakers"])
print("Number of utterances:", transcript_data["num_utterances"])

print("\nFirst 3 evidence units:")

for item in speaker_utterances[:3]:

    print(
        f"\nUtterance {item.get('utterance_id', 'N/A')}"
    )

    print("Speaker:", item["speaker"])
    print(
        f"Time: {item['start']:.3f}s → "
        f"{item['end']:.3f}s"
    )
    print("Text:", item["text"])

print("\n" + "=" * 80)

SPEAKER-ATTRIBUTED TRANSCRIPT LOADED

Meeting ID: ES2004a
Number of speakers: 4
Number of utterances: 148

First 3 evidence units:

Utterance N/A
Speaker: SPEAKER_02
Time: 10.998s → 14.521s
Text: Are we, we're not like the dim lights, so we can see that a bit better.

Utterance N/A
Speaker: SPEAKER_01
Time: 17.943s → 18.163s
Text: Yeah.

Utterance N/A
Speaker: SPEAKER_02
Time: 18.944s → 20.945s
Text: Okay, that's fine.



In [ ]:
# ============================================================
# CREATE EVIDENCE DOCUMENTS
# Purpose:
# Convert the speaker-attributed transcript into standardized
# evidence units for semantic retrieval.
#
# Each evidence document contains:
# - evidence_id
# - speaker
# - start timestamp
# - end timestamp
# - duration
# - transcript text
#
# These IDs will later be used to link verified MoM claims
# back to their supporting transcript evidence.
# ============================================================

evidence_documents = []

for idx, utterance in enumerate(speaker_utterances, start=1):

    evidence_documents.append({
        "evidence_id": idx,
        "speaker": utterance["speaker"],
        "start": utterance["start"],
        "end": utterance["end"],
        "duration": utterance["end"] - utterance["start"],
        "text": utterance["text"]
    })


print("=" * 80)
print("EVIDENCE DOCUMENTS CREATED")
print("=" * 80)

print("\nNumber of evidence documents:",
      len(evidence_documents))

print("\nFirst 5 evidence documents:")

for evidence in evidence_documents[:5]:

    print(
        f"\nEvidence {evidence['evidence_id']:03d}"
    )

    print("Speaker:", evidence["speaker"])

    print(
        f"Time: {evidence['start']:.3f}s → "
        f"{evidence['end']:.3f}s"
    )

    print(
        f"Duration: "
        f"{evidence['duration']:.3f}s"
    )

    print("Text:", evidence["text"])

print("\n" + "=" * 80)

EVIDENCE DOCUMENTS CREATED

Number of evidence documents: 148

First 5 evidence documents:

Evidence 001
Speaker: SPEAKER_02
Time: 10.998s → 14.521s
Duration: 3.523s
Text: Are we, we're not like the dim lights, so we can see that a bit better.

Evidence 002
Speaker: SPEAKER_01
Time: 17.943s → 18.163s
Duration: 0.220s
Text: Yeah.

Evidence 003
Speaker: SPEAKER_02
Time: 18.944s → 20.945s
Duration: 2.001s
Text: Okay, that's fine.

Evidence 004
Speaker: SPEAKER_02
Time: 22.406s → 23.707s
Duration: 1.301s
Text: Am I supposed to be standing up there?

Evidence 005
Speaker: SPEAKER_03
Time: 25.128s → 26.509s
Duration: 1.381s
Text: So we've got both of these clipped on.



In [ ]:
# ============================================================
# LOAD BGE EMBEDDING MODEL
# Purpose:
# Load BGE-small to create semantic embeddings for the
# transcript evidence units and MoM claims.
# ============================================================

import torch
from sentence_transformers import SentenceTransformer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

BGE_MODEL_NAME = "BAAI/bge-small-en-v1.5"

print("=" * 80)
print("LOADING BGE EMBEDDING MODEL")
print("=" * 80)

print("\nDevice:", DEVICE)
print("Model:", BGE_MODEL_NAME)

bge_model = SentenceTransformer(
    BGE_MODEL_NAME,
    device=DEVICE
)

print("\nBGE model loaded successfully.")

print("=" * 80)

LOADING BGE EMBEDDING MODEL

Device: cuda
Model: BAAI/bge-small-en-v1.5


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


BGE model loaded successfully.


In [ ]:
# ============================================================
# GENERATE EVIDENCE EMBEDDINGS
# Purpose:
# Convert each transcript evidence unit into a normalized
# BGE semantic embedding.
#
# Embedding dimension:
# 384
#
# Normalization allows FAISS inner-product similarity
# to behave as cosine similarity.
# ============================================================

evidence_texts = [
    evidence["text"]
    for evidence in evidence_documents
]

print("=" * 80)
print("GENERATING EVIDENCE EMBEDDINGS")
print("=" * 80)

print("\nEvidence documents:", len(evidence_texts))

evidence_embeddings = bge_model.encode(
    evidence_texts,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True
)

print("\nEmbedding generation complete.")

print("Embedding shape:", evidence_embeddings.shape)
print("Embedding dtype:", evidence_embeddings.dtype)

print("\nExpected shape:")
print(f"({len(evidence_documents)}, 384)")

print("\n" + "=" * 80)

GENERATING EVIDENCE EMBEDDINGS

Evidence documents: 148


Batches:   0%|          | 0/5 [00:00<?, ?it/s]


Embedding generation complete.
Embedding shape: (148, 384)
Embedding dtype: float32

Expected shape:
(148, 384)



In [ ]:
# ============================================================
# BUILD FAISS EVIDENCE INDEX
# Purpose:
# Build a FAISS vector index over the 148 normalized BGE
# evidence embeddings.
#
# Because the embeddings are normalized, inner-product
# similarity is equivalent to cosine similarity.
# ============================================================

import faiss
import numpy as np

print("=" * 80)
print("BUILDING FAISS EVIDENCE INDEX")
print("=" * 80)

# Ensure embeddings are float32 for FAISS
evidence_embeddings = np.asarray(
    evidence_embeddings,
    dtype="float32"
)

embedding_dimension = evidence_embeddings.shape[1]

# Inner-product index
faiss_index = faiss.IndexFlatIP(
    embedding_dimension
)

# Add evidence embeddings
faiss_index.add(evidence_embeddings)

print("\nEmbedding dimension:", embedding_dimension)
print("Evidence vectors added:", faiss_index.ntotal)
print("FAISS index type:", type(faiss_index).__name__)

print("\nExpected vectors:", len(evidence_documents))

print(
    "\nIndex successfully built:",
    faiss_index.ntotal == len(evidence_documents)
)

print("\n" + "=" * 80)

ModuleNotFoundError: No module named 'faiss'

In [ ]:
# ============================================================
# INSTALL FAISS
# Purpose:
# Install FAISS for vector similarity search.
# ============================================================

!pip install -q faiss-cpu

print("FAISS installation completed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 71.0 MB/s eta 0:00:00
FAISS installation completed.


In [ ]:
# ============================================================
# BUILD FAISS EVIDENCE INDEX
# Purpose:
# Build a FAISS vector index over the 148 normalized BGE
# evidence embeddings.
# ============================================================

import faiss
import numpy as np

print("=" * 80)
print("BUILDING FAISS EVIDENCE INDEX")
print("=" * 80)

evidence_embeddings = np.asarray(
    evidence_embeddings,
    dtype="float32"
)

embedding_dimension = evidence_embeddings.shape[1]

faiss_index = faiss.IndexFlatIP(
    embedding_dimension
)

faiss_index.add(evidence_embeddings)

print("\nEmbedding dimension:", embedding_dimension)
print("Evidence vectors added:", faiss_index.ntotal)
print("FAISS index type:", type(faiss_index).__name__)

print("\nExpected vectors:", len(evidence_documents))

print(
    "\nIndex successfully built:",
    faiss_index.ntotal == len(evidence_documents)
)

print("\n" + "=" * 80)

BUILDING FAISS EVIDENCE INDEX

Embedding dimension: 384
Evidence vectors added: 148
FAISS index type: IndexFlatIP

Expected vectors: 148

Index successfully built: True



In [ ]:
# ============================================================
# EVIDENCE RETRIEVAL FUNCTION
# Purpose:
# Retrieve the most semantically relevant transcript evidence
# for a given MoM claim using BGE + FAISS.
#
# Important:
# Retrieval similarity identifies candidate evidence.
# It does NOT by itself prove that the evidence supports
# the claim. Verification will be performed in the next stage.
# ============================================================

def retrieve_evidence(claim_text, top_k=5):

    # Generate normalized BGE embedding for the claim
    claim_embedding = bge_model.encode(
        [claim_text],
        normalize_embeddings=True,
        convert_to_numpy=True
    ).astype("float32")

    # Search FAISS index
    similarities, indices = faiss_index.search(
        claim_embedding,
        top_k
    )

    retrieved = []

    for similarity, index in zip(
        similarities[0],
        indices[0]
    ):

        evidence = evidence_documents[index].copy()

        evidence["retrieval_similarity"] = float(
            similarity
        )

        retrieved.append(evidence)

    return retrieved


print("=" * 80)
print("EVIDENCE RETRIEVAL FUNCTION CREATED")
print("=" * 80)

print("\nFunction: retrieve_evidence()")
print("Default top-K:", 5)

print("\nRetrieval pipeline:")
print("Claim")
print("  ↓")
print("BGE embedding")
print("  ↓")
print("FAISS similarity search")
print("  ↓")
print("Top-K evidence candidates")

print("\n" + "=" * 80)

EVIDENCE RETRIEVAL FUNCTION CREATED

Function: retrieve_evidence()
Default top-K: 5

Retrieval pipeline:
Claim
  ↓
BGE embedding
  ↓
FAISS similarity search
  ↓
Top-K evidence candidates



In [ ]:
# ============================================================
# TEST EVIDENCE RETRIEVAL
# Purpose:
# Test BGE + FAISS retrieval on one structured MoM claim.
#
# Claim 2 discusses the remote control's objectives and
# usability requirements.
# ============================================================

TEST_CLAIM = mom_claims[1]

print("=" * 90)
print("EVIDENCE RETRIEVAL TEST")
print("=" * 90)

print("\nClaim ID:", TEST_CLAIM["claim_id"])
print("Topic:", TEST_CLAIM["topic"])
print("Claim:")
print(TEST_CLAIM["claim_text"])

print(
    f"\nClaim timestamp: "
    f"{TEST_CLAIM['start']:.3f}s → "
    f"{TEST_CLAIM['end']:.3f}s"
)

print("\n" + "-" * 90)
print("TOP 5 RETRIEVED EVIDENCE")
print("-" * 90)

retrieved_test_evidence = retrieve_evidence(
    TEST_CLAIM["claim_text"],
    top_k=5
)

for rank, evidence in enumerate(
    retrieved_test_evidence,
    start=1
):

    print(
        f"\nRank {rank}"
    )

    print(
        f"Evidence ID: "
        f"{evidence['evidence_id']:03d}"
    )

    print(
        f"Similarity: "
        f"{evidence['retrieval_similarity']:.4f}"
    )

    print(
        f"Speaker: "
        f"{evidence['speaker']}"
    )

    print(
        f"Time: "
        f"{evidence['start']:.3f}s → "
        f"{evidence['end']:.3f}s"
    )

    print(
        f"Text: "
        f"{evidence['text']}"
    )

print("\n" + "=" * 90)

EVIDENCE RETRIEVAL TEST

Claim ID: 2
Topic: product_objective_and_requirements
Claim:
The team is developing a remote control intended to be original, trendy, appealing to a wide market, and user-friendly for a broad range of users.

Claim timestamp: 117.699s → 141.016s

------------------------------------------------------------------------------------------
TOP 5 RETRIEVED EVIDENCE
------------------------------------------------------------------------------------------

Rank 1
Evidence ID: 014
Similarity: 0.7713
Speaker: SPEAKER_02
Time: 117.699s → 120.921s
Text: Now, we're developing a remote control, which you probably already know.

Rank 2
Evidence ID: 105
Similarity: 0.7575
Speaker: SPEAKER_01
Time: 707.248s → 710.889s
Text: remote controls. You want to integrate everything into one.

Rank 3
Evidence ID: 135
Similarity: 0.6684
Speaker: SPEAKER_03
Time: 935.208s → 946.993s
Text: The other thing is, just tacking into mobile phone design features again, you could have a flip top 

In [ ]:
# ============================================================
# IMPROVED EVIDENCE RETRIEVAL FUNCTION
# Purpose:
# Retrieve top-K evidence candidates while preserving their
# retrieval rank.
#
# Retrieval similarity is used only to find candidate evidence.
# It is NOT treated as proof of claim correctness.
# ============================================================

def retrieve_evidence(claim_text, top_k=10):

    # Generate normalized BGE embedding for the claim
    claim_embedding = bge_model.encode(
        [claim_text],
        normalize_embeddings=True,
        convert_to_numpy=True
    ).astype("float32")

    # Search FAISS
    similarities, indices = faiss_index.search(
        claim_embedding,
        top_k
    )

    retrieved = []

    for rank, (similarity, index) in enumerate(
        zip(similarities[0], indices[0]),
        start=1
    ):

        evidence = evidence_documents[index].copy()

        evidence["retrieval_rank"] = rank

        evidence["retrieval_similarity"] = float(
            similarity
        )

        retrieved.append(evidence)

    return retrieved


print("=" * 80)
print("IMPROVED EVIDENCE RETRIEVAL FUNCTION")
print("=" * 80)

print("\nDefault top-K: 10")
print("Retrieval rank: preserved")
print("Similarity: preserved")

print(
    "\nImportant:",
    "Similarity is used for retrieval, not verification."
)

print("=" * 80)

IMPROVED EVIDENCE RETRIEVAL FUNCTION

Default top-K: 10
Retrieval rank: preserved
Similarity: preserved

Important: Similarity is used for retrieval, not verification.


In [ ]:
# ============================================================
# RETRIEVE EVIDENCE FOR ALL MoM CLAIMS
# Purpose:
# Retrieve the top-10 semantically relevant transcript
# evidence units for every structured MoM claim.
#
# These are candidate evidence units only.
# They have NOT yet been verified.
# ============================================================

all_claim_retrievals = []

print("=" * 100)
print("EVIDENCE RETRIEVAL FOR ALL MoM CLAIMS")
print("=" * 100)

for claim in mom_claims:

    retrieved = retrieve_evidence(
        claim["claim_text"],
        top_k=10
    )

    claim_result = {
        "claim_id": claim["claim_id"],
        "topic": claim["topic"],
        "claim_text": claim["claim_text"],
        "claim_speaker": claim["speaker"],
        "claim_start": claim["start"],
        "claim_end": claim["end"],
        "event_type": claim["event_type"],
        "source_candidate_ids": claim[
            "source_candidate_ids"
        ],
        "retrieved_evidence": retrieved
    }

    all_claim_retrievals.append(claim_result)


print("\nTotal claims processed:",
      len(all_claim_retrievals))

print(
    "Expected claims:",
    len(mom_claims)
)

print(
    "\nRetrieval completed successfully:",
    len(all_claim_retrievals) == len(mom_claims)
)

print("\n" + "=" * 100)

EVIDENCE RETRIEVAL FOR ALL MoM CLAIMS

Total claims processed: 12
Expected claims: 12

Retrieval completed successfully: True



In [ ]:
# ============================================================
# COMPACT RETRIEVAL INSPECTION
# Purpose:
# Inspect the top-3 retrieved evidence candidates for each
# MoM claim before moving to evidence verification.
#
# This is a qualitative retrieval checkpoint.
# ============================================================

print("=" * 110)
print("TOP-3 RETRIEVED EVIDENCE FOR EACH MoM CLAIM")
print("=" * 110)

for result in all_claim_retrievals:

    print(
        f"\nCLAIM {result['claim_id']:02d} | "
        f"{result['topic']}"
    )

    print("Claim:")
    print(" ", result["claim_text"])

    print("\nTop retrieved evidence:")

    for evidence in result["retrieved_evidence"][:3]:

        print(
            f"  Rank {evidence['retrieval_rank']} | "
            f"ID {evidence['evidence_id']:03d} | "
            f"Sim {evidence['retrieval_similarity']:.4f} | "
            f"{evidence['speaker']} | "
            f"{evidence['start']:.1f}s–"
            f"{evidence['end']:.1f}s"
        )

        print(
            f"    {evidence['text']}"
        )

    print("-" * 110)

print("\n" + "=" * 110)
print("RETRIEVAL INSPECTION COMPLETE")
print("=" * 110)

TOP-3 RETRIEVED EVIDENCE FOR EACH MoM CLAIM

CLAIM 01 | meeting_introduction_and_agenda
Claim:
  The meeting was introduced as the first meeting, with an agenda covering team interaction, tool training, the project plan, and discussion of ideas.

Top retrieved evidence:
  Rank 1 | ID 009 | Sim 0.7391 | SPEAKER_02 | 84.7s–93.1s
    I'm Sarah, the project manager, and this is our first meeting, surprisingly enough. Okay, this is our agenda.
  Rank 2 | ID 012 | Sim 0.6854 | SPEAKER_02 | 106.3s–109.8s
    talk about the project plan, discuss our own ideas and everything.
  Rank 3 | ID 019 | Sim 0.6029 | SPEAKER_02 | 162.4s–167.7s
    what we're thinking, how it's going to go, and then the detailed design, how we're actually going to put it into practice and make it work.
--------------------------------------------------------------------------------------------------------------

CLAIM 02 | product_objective_and_requirements
Claim:
  The team is developing a remote control intended to be 

In [ ]:
# ============================================================
# LOAD NLI MODEL
# Purpose:
# Load a Natural Language Inference model to determine whether
# retrieved transcript evidence supports a generated MoM claim.
#
# NLI is used for CONTENT CONSISTENCY.
#
# BGE similarity is NOT treated as proof of entailment.
# ============================================================

from transformers import AutoTokenizer, AutoModelForSequenceClassification

NLI_MODEL_NAME = "cross-encoder/nli-deberta-v3-small"

print("=" * 80)
print("LOADING NLI MODEL")
print("=" * 80)

print("\nModel:", NLI_MODEL_NAME)
print("Device:", DEVICE)

nli_tokenizer = AutoTokenizer.from_pretrained(
    NLI_MODEL_NAME
)

nli_model = AutoModelForSequenceClassification.from_pretrained(
    NLI_MODEL_NAME
)

nli_model = nli_model.to(DEVICE)
nli_model.eval()

print("\nNLI model loaded successfully.")

print("=" * 80)

LOADING NLI MODEL

Model: cross-encoder/nli-deberta-v3-small
Device: cuda


config.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.35k [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

...

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  568MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]


NLI model loaded successfully.


In [ ]:
# ============================================================
# INSPECT NLI LABEL MAPPING
# Purpose:
# Check how the NLI model maps its output indices to
# ENTAILMENT, NEUTRAL, and CONTRADICTION.
#
# This is important because different NLI models can use
# different label-index orders.
# ============================================================

print("=" * 80)
print("NLI LABEL MAPPING")
print("=" * 80)

print("\nModel label mapping:")

for label_id, label_name in nli_model.config.id2label.items():
    print(f"  {label_id} -> {label_name}")

print("\nNumber of labels:", nli_model.config.num_labels)

print("=" * 80)

NLI LABEL MAPPING

Model label mapping:
  0 -> contradiction
  1 -> entailment
  2 -> neutral

Number of labels: 3


In [ ]:
# ============================================================
# NLI SANITY TEST
# Purpose:
# Verify that the NLI model correctly distinguishes:
#   1. Supporting evidence  -> ENTAILMENT
#   2. Unrelated evidence   -> NEUTRAL
#   3. Conflicting evidence -> CONTRADICTION
#
# IMPORTANT:
# Premise    = transcript evidence
# Hypothesis = generated MoM claim
# ============================================================

import torch

def nli_predict(premise, hypothesis):
    """
    Run NLI with:
        premise    = evidence
        hypothesis = MoM claim
    """

    inputs = nli_tokenizer(
        premise,
        hypothesis,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    inputs = {
        key: value.to(DEVICE)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        outputs = nli_model(**inputs)

    probabilities = torch.softmax(outputs.logits, dim=-1)[0]

    label_id = torch.argmax(probabilities).item()
    label = nli_model.config.id2label[label_id].upper()
    confidence = probabilities[label_id].item()

    return label_id, label, confidence


# ------------------------------------------------------------
# Test 1: Supporting evidence
# ------------------------------------------------------------

evidence_1 = (
    "It's got to be accessible and usable by all age groups."
)

claim_1 = (
    "The product should be accessible and usable by all age groups."
)


# ------------------------------------------------------------
# Test 2: Unrelated evidence
# ------------------------------------------------------------

evidence_2 = (
    "We discussed the design process and the detailed design stage."
)

claim_2 = (
    "The product should be accessible and usable by all age groups."
)


# ------------------------------------------------------------
# Test 3: Contradicting evidence
# ------------------------------------------------------------

evidence_3 = (
    "The product is intended only for young adults and is not "
    "designed for older users."
)

claim_3 = (
    "The product should be accessible and usable by all age groups."
)


# ------------------------------------------------------------
# Run tests
# ------------------------------------------------------------

tests = [
    ("SUPPORTING", evidence_1, claim_1),
    ("UNRELATED", evidence_2, claim_2),
    ("CONTRADICTING", evidence_3, claim_3),
]

print("=" * 80)
print("NLI SANITY TEST RESULTS")
print("=" * 80)

for test_name, evidence, claim in tests:

    label_id, label, confidence = nli_predict(
        evidence,
        claim
    )

    print(f"\nTest: {test_name}")
    print("-" * 80)

    print("Evidence:")
    print(evidence)

    print("\nClaim:")
    print(claim)

    print("\nPrediction:")
    print(f"  Label ID   : {label_id}")
    print(f"  Label      : {label}")
    print(f"  Confidence : {confidence:.4f}")

print("\n" + "=" * 80)

NLI SANITY TEST RESULTS

Test: SUPPORTING
--------------------------------------------------------------------------------
Evidence:
It's got to be accessible and usable by all age groups.

Claim:
The product should be accessible and usable by all age groups.

Prediction:
  Label ID   : 2
  Label      : NEUTRAL
  Confidence : 0.9719

Test: UNRELATED
--------------------------------------------------------------------------------
Evidence:
We discussed the design process and the detailed design stage.

Claim:
The product should be accessible and usable by all age groups.

Prediction:
  Label ID   : 2
  Label      : NEUTRAL
  Confidence : 0.9996

Test: CONTRADICTING
--------------------------------------------------------------------------------
Evidence:
The product is intended only for young adults and is not designed for older users.

Claim:
The product should be accessible and usable by all age groups.

Prediction:
  Label ID   : 0
  Label      : CONTRADICTION
  Confidence : 0.9987



In [ ]:
# ============================================================
# REAL PROJECT NLI TEST
# Purpose:
# Test NLI using an actual transcript evidence segment and
# its corresponding structured MoM claim.
#
# This is more meaningful than a manually created example
# because it reflects the actual language produced by the
# meeting transcript and our MoM claim-generation stage.
# ============================================================

# ------------------------------------------------------------
# Select Claim 2
# ------------------------------------------------------------

claim_data = mom_claims_data["claims"][1]

claim_text = claim_data["claim"]

print("=" * 80)
print("REAL PROJECT NLI TEST")
print("=" * 80)

print("\nClaim ID:")
print(claim_data["claim_id"])

print("\nMoM Claim:")
print(claim_text)


# ------------------------------------------------------------
# Retrieve the evidence segments already associated with
# this claim during structured claim construction.
# ------------------------------------------------------------

source_ids = claim_data["source_candidate_ids"]

print("\nSource Candidate IDs:")
print(source_ids)


# ------------------------------------------------------------
# Find the corresponding evidence/utterance text
# ------------------------------------------------------------

print("\n" + "-" * 80)
print("SOURCE EVIDENCE")
print("-" * 80)

for source_id in source_ids:

    # Candidate IDs are stored as integers
    candidate = next(
        (
            item
            for item in mom_candidates_data["candidates"]
            if item["candidate_id"] == source_id
        ),
        None
    )

    if candidate is None:
        print(f"\nCandidate {source_id}: NOT FOUND")
        continue

    evidence_text = candidate["text"]

    print(f"\nCandidate {source_id}")
    print("Evidence:")
    print(evidence_text)

    # --------------------------------------------------------
    # Run NLI
    # --------------------------------------------------------

    label_id, label, confidence = nli_predict(
        evidence_text,
        claim_text
    )

    print("\nNLI Prediction:")
    print(f"  Label ID   : {label_id}")
    print(f"  Label      : {label}")
    print(f"  Confidence : {confidence:.4f}")


print("\n" + "=" * 80)

KeyError: 'claim'

In [ ]:
# ============================================================
# INSPECT STRUCTURED MOM CLAIM FORMAT
# Purpose:
# Check the exact field names used in the structured MoM
# claims JSON before building the NLI verification code.
# ============================================================

print("=" * 80)
print("STRUCTURED MOM CLAIM FORMAT")
print("=" * 80)

print("\nNumber of claims:", len(mom_claims_data["claims"]))

print("\nKeys in first claim:")
print(mom_claims_data["claims"][0].keys())

print("\nFirst claim:")
print(mom_claims_data["claims"][0])

print("\n" + "=" * 80)

STRUCTURED MOM CLAIM FORMAT

Number of claims: 12

Keys in first claim:
dict_keys(['claim_id', 'topic', 'claim_text', 'source_candidate_ids', 'speaker', 'start', 'end', 'event_type'])

First claim:
{'claim_id': 1, 'topic': 'meeting_introduction_and_agenda', 'claim_text': 'The meeting was introduced as the first meeting, with an agenda covering team interaction, tool training, the project plan, and discussion of ideas.', 'source_candidate_ids': [5, 6, 7, 8], 'speaker': 'SPEAKER_02', 'start': 84.658, 'end': 109.823, 'event_type': 'INFORMATION'}



In [ ]:
# ============================================================
# REAL PROJECT NLI TEST
# Purpose:
# Test NLI using an actual structured MoM claim and the
# transcript evidence used to construct that claim.
#
# Premise    = Evidence
# Hypothesis = MoM claim
# ============================================================

print("=" * 80)
print("REAL PROJECT NLI TEST")
print("=" * 80)

# ------------------------------------------------------------
# Select Claim 2
# ------------------------------------------------------------

claim_data = mom_claims_data["claims"][1]

claim_text = claim_data["claim_text"]

print("\nClaim ID:")
print(claim_data["claim_id"])

print("\nTopic:")
print(claim_data["topic"])

print("\nMoM Claim:")
print(claim_text)

print("\nSource Candidate IDs:")
print(claim_data["source_candidate_ids"])


# ------------------------------------------------------------
# Find and test each source evidence
# ------------------------------------------------------------

print("\n" + "-" * 80)
print("SOURCE EVIDENCE + NLI RESULTS")
print("-" * 80)

for source_id in claim_data["source_candidate_ids"]:

    candidate = next(
        (
            item
            for item in mom_candidates_data["candidates"]
            if item["candidate_id"] == source_id
        ),
        None
    )

    if candidate is None:
        print(f"\nCandidate {source_id}: NOT FOUND")
        continue

    evidence_text = candidate["text"]

    print(f"\nCandidate {source_id}")
    print("Evidence:")
    print(evidence_text)

    # --------------------------------------------------------
    # Run NLI
    # --------------------------------------------------------

    label_id, label, confidence = nli_predict(
        evidence_text,
        claim_text
    )

    print("\nNLI Prediction:")
    print(f"  Label ID   : {label_id}")
    print(f"  Label      : {label}")
    print(f"  Confidence : {confidence:.4f}")


print("\n" + "=" * 80)

REAL PROJECT NLI TEST

Claim ID:
2

Topic:
product_objective_and_requirements

MoM Claim:
The team is developing a remote control intended to be original, trendy, appealing to a wide market, and user-friendly for a broad range of users.

Source Candidate IDs:
[10, 11, 12, 13]

--------------------------------------------------------------------------------
SOURCE EVIDENCE + NLI RESULTS
--------------------------------------------------------------------------------


NameError: name 'mom_candidates_data' is not defined

In [ ]:
# ============================================================
# LOAD RULE-BASED MOM CANDIDATES
# Purpose:
# Load the candidate utterances used as source evidence for
# the structured MoM claims.
# ============================================================

import json

print("=" * 80)
print("LOADING MOM CANDIDATES")
print("=" * 80)

with open(MOM_CANDIDATES_PATH, "r", encoding="utf-8") as f:
    mom_candidates_data = json.load(f)

print("\nFile loaded successfully:")
print(MOM_CANDIDATES_PATH)

print("\nNumber of candidates:",
      len(mom_candidates_data["candidates"]))

print("\nAvailable fields:")
print(mom_candidates_data["candidates"][0].keys())

print("=" * 80)

LOADING MOM CANDIDATES

File loaded successfully:
/content/drive/MyDrive/MTechIndProj/MoM_Project/data/transcripts/ES2004a_mom_candidates_rule_based.json

Number of candidates: 50

Available fields:
dict_keys(['candidate_id', 'source_utterance_id', 'speaker', 'start', 'end', 'text'])


In [ ]:
# ============================================================
# REAL PROJECT NLI TEST — CLAIM 2
# Purpose:
# Test whether the actual transcript evidence supports the
# structured MoM claim.
#
# Premise    = Transcript evidence
# Hypothesis = MoM claim
# ============================================================

print("=" * 80)
print("REAL PROJECT NLI TEST — CLAIM 2")
print("=" * 80)

# ------------------------------------------------------------
# Select Claim 2
# ------------------------------------------------------------

claim_data = mom_claims_data["claims"][1]

claim_text = claim_data["claim_text"]

print("\nClaim ID:")
print(claim_data["claim_id"])

print("\nTopic:")
print(claim_data["topic"])

print("\nMoM Claim:")
print(claim_text)

print("\nSource Candidate IDs:")
print(claim_data["source_candidate_ids"])


# ------------------------------------------------------------
# Create candidate lookup
# ------------------------------------------------------------

candidate_lookup = {
    item["candidate_id"]: item
    for item in mom_candidates_data["candidates"]
}


# ------------------------------------------------------------
# Test each source candidate
# ------------------------------------------------------------

print("\n" + "-" * 80)
print("SOURCE EVIDENCE + NLI RESULTS")
print("-" * 80)

for source_id in claim_data["source_candidate_ids"]:

    candidate = candidate_lookup.get(source_id)

    if candidate is None:
        print(f"\nCandidate {source_id}: NOT FOUND")
        continue

    evidence_text = candidate["text"]

    print(f"\nCandidate {source_id}")
    print(
        f"Speaker   : {candidate['speaker']}"
    )
    print(
        f"Timestamp : {candidate['start']:.3f} - "
        f"{candidate['end']:.3f}"
    )

    print("\nEvidence:")
    print(evidence_text)

    # --------------------------------------------------------
    # Run NLI
    # --------------------------------------------------------

    label_id, label, confidence = nli_predict(
        evidence_text,
        claim_text
    )

    print("\nNLI Prediction:")
    print(f"  Label ID   : {label_id}")
    print(f"  Label      : {label}")
    print(f"  Confidence : {confidence:.4f}")


print("\n" + "=" * 80)

REAL PROJECT NLI TEST — CLAIM 2

Claim ID:
2

Topic:
product_objective_and_requirements

MoM Claim:
The team is developing a remote control intended to be original, trendy, appealing to a wide market, and user-friendly for a broad range of users.

Source Candidate IDs:
[10, 11, 12, 13]

--------------------------------------------------------------------------------
SOURCE EVIDENCE + NLI RESULTS
--------------------------------------------------------------------------------

Candidate 10
Speaker   : SPEAKER_02
Timestamp : 117.699 - 120.921

Evidence:
Now, we're developing a remote control, which you probably already know.

NLI Prediction:
  Label ID   : 2
  Label      : NEUTRAL
  Confidence : 0.9900

Candidate 11
Speaker   : SPEAKER_02
Timestamp : 122.503 - 132.290

Evidence:
We want it to be original, something that people haven't thought of. It's not out in the shops. Trendy, appealing to a wide market, but,

NLI Prediction:
  Label ID   : 2
  Label      : NEUTRAL
  Confidence : 0.9

In [ ]:
# ============================================================
# SINGLE-PROPOSITION NLI TEST
# Purpose:
# Test NLI on a claim containing only ONE proposition.
#
# This helps determine whether the NLI model can correctly
# verify individual attributes before we build the
# multi-attribute verification stage.
# ============================================================

evidence = (
    "Now, we're developing a remote control, "
    "which you probably already know."
)

claim = (
    "The team is developing a remote control."
)

label_id, label, confidence = nli_predict(
    evidence,
    claim
)

print("=" * 80)
print("SINGLE-PROPOSITION NLI TEST")
print("=" * 80)

print("\nEvidence:")
print(evidence)

print("\nClaim:")
print(claim)

print("\nPrediction:")
print(f"  Label ID   : {label_id}")
print(f"  Label      : {label}")
print(f"  Confidence : {confidence:.4f}")

print("\n" + "=" * 80)

SINGLE-PROPOSITION NLI TEST

Evidence:
Now, we're developing a remote control, which you probably already know.

Claim:
The team is developing a remote control.

Prediction:
  Label ID   : 1
  Label      : ENTAILMENT
  Confidence : 0.7902



In [ ]:
# ============================================================
# REUSABLE NLI PREDICTION FUNCTION
# Purpose:
# Run NLI between one evidence segment and one proposition.
#
# Output:
#   - contradiction probability
#   - entailment probability
#   - neutral probability
#   - predicted label
#   - confidence
#
# Premise    = Evidence
# Hypothesis = Proposition
# ============================================================

def run_nli(evidence, proposition):
    """
    Perform NLI between transcript evidence and a proposition.

    Parameters
    ----------
    evidence : str
        Transcript evidence.

    proposition : str
        Single factual proposition derived from a MoM claim.

    Returns
    -------
    dict
        NLI probabilities and prediction.
    """

    inputs = nli_tokenizer(
        evidence,
        proposition,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    inputs = {
        key: value.to(DEVICE)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        outputs = nli_model(**inputs)

    probabilities = torch.softmax(
        outputs.logits,
        dim=-1
    )[0]

    # --------------------------------------------------------
    # Model mapping confirmed earlier:
    # 0 = contradiction
    # 1 = entailment
    # 2 = neutral
    # --------------------------------------------------------

    contradiction_prob = probabilities[0].item()
    entailment_prob = probabilities[1].item()
    neutral_prob = probabilities[2].item()

    predicted_id = torch.argmax(probabilities).item()

    predicted_label = (
        nli_model.config.id2label[predicted_id].upper()
    )

    confidence = probabilities[predicted_id].item()

    return {
        "predicted_label": predicted_label,
        "confidence": confidence,
        "contradiction_probability": contradiction_prob,
        "entailment_probability": entailment_prob,
        "neutral_probability": neutral_prob
    }


print("=" * 80)
print("NLI FUNCTION CREATED")
print("=" * 80)

print("\nFunction: run_nli(evidence, proposition)")
print("\nModel mapping:")
print("  0 -> CONTRADICTION")
print("  1 -> ENTAILMENT")
print("  2 -> NEUTRAL")

print("\nReady for attribute-level verification.")

print("=" * 80)

NLI FUNCTION CREATED

Function: run_nli(evidence, proposition)

Model mapping:
  0 -> CONTRADICTION
  1 -> ENTAILMENT
  2 -> NEUTRAL

Ready for attribute-level verification.


In [ ]:
# ============================================================
# TEST REAL PROPOSITION-LEVEL VERIFICATION
# Purpose:
# Verify one individual proposition from Claim 2 against
# its actual transcript evidence.
# ============================================================

evidence = candidate_lookup[10]["text"]

proposition = "The team is developing a remote control."

result = run_nli(
    evidence,
    proposition
)

print("=" * 80)
print("REAL PROPOSITION NLI TEST")
print("=" * 80)

print("\nEvidence:")
print(evidence)

print("\nProposition:")
print(proposition)

print("\nNLI Result:")
print(f"  Predicted Label       : {result['predicted_label']}")
print(f"  Confidence            : {result['confidence']:.4f}")
print(f"  Entailment Probability: {result['entailment_probability']:.4f}")
print(f"  Neutral Probability   : {result['neutral_probability']:.4f}")
print(f"  Contradiction Prob.   : {result['contradiction_probability']:.4f}")

print("\n" + "=" * 80)

REAL PROPOSITION NLI TEST

Evidence:
Now, we're developing a remote control, which you probably already know.

Proposition:
The team is developing a remote control.

NLI Result:
  Predicted Label       : ENTAILMENT
  Confidence            : 0.7902
  Entailment Probability: 0.7902
  Neutral Probability   : 0.2093
  Contradiction Prob.   : 0.0005



In [ ]:
# ============================================================
# CLAIM 2 — ATTRIBUTE / PROPOSITION REPRESENTATION
# Purpose:
# Break the multi-part MoM claim into individual factual
# propositions for evidence verification.
#
# This is the core idea behind the proposed
# Multi-Attribute Evidence Consistency approach.
# ============================================================

claim_2_attributes = [
    {
        "attribute_id": 1,
        "attribute": "product_development",
        "proposition": "The team is developing a remote control.",
        "source_candidate_ids": [10]
    },
    {
        "attribute_id": 2,
        "attribute": "originality",
        "proposition": "The remote control is intended to be original.",
        "source_candidate_ids": [11]
    },
    {
        "attribute_id": 3,
        "attribute": "trendiness",
        "proposition": "The remote control is intended to be trendy.",
        "source_candidate_ids": [11]
    },
    {
        "attribute_id": 4,
        "attribute": "market_appeal",
        "proposition": "The remote control is intended to appeal to a wide market.",
        "source_candidate_ids": [11]
    },
    {
        "attribute_id": 5,
        "attribute": "user_friendliness",
        "proposition": "The remote control is intended to be user-friendly.",
        "source_candidate_ids": [12]
    },
    {
        "attribute_id": 6,
        "attribute": "broad_usability",
        "proposition": "The remote control should be usable by a broad range of users.",
        "source_candidate_ids": [12, 13]
    }
]


print("=" * 80)
print("CLAIM 2 — ATTRIBUTE REPRESENTATION")
print("=" * 80)

print("\nNumber of attributes:", len(claim_2_attributes))

for item in claim_2_attributes:

    print("\nAttribute ID:", item["attribute_id"])
    print("Attribute   :", item["attribute"])
    print("Proposition :", item["proposition"])
    print("Evidence IDs:", item["source_candidate_ids"])

print("\n" + "=" * 80)

CLAIM 2 — ATTRIBUTE REPRESENTATION

Number of attributes: 6

Attribute ID: 1
Attribute   : product_development
Proposition : The team is developing a remote control.
Evidence IDs: [10]

Attribute ID: 2
Attribute   : originality
Proposition : The remote control is intended to be original.
Evidence IDs: [11]

Attribute ID: 3
Attribute   : trendiness
Proposition : The remote control is intended to be trendy.
Evidence IDs: [11]

Attribute ID: 4
Attribute   : market_appeal
Proposition : The remote control is intended to appeal to a wide market.
Evidence IDs: [11]

Attribute ID: 5
Attribute   : user_friendliness
Proposition : The remote control is intended to be user-friendly.
Evidence IDs: [12]

Attribute ID: 6
Attribute   : broad_usability
Proposition : The remote control should be usable by a broad range of users.
Evidence IDs: [12, 13]



In [ ]:
# ============================================================
# CLAIM 2 — ATTRIBUTE-LEVEL NLI VERIFICATION
# Purpose:
# Run NLI separately for each factual proposition in Claim 2.
#
# This demonstrates the core verification mechanism:
#
#   MoM Claim
#       ↓
#   Individual propositions
#       ↓
#   Evidence
#       ↓
#   NLI
# ============================================================

print("=" * 80)
print("CLAIM 2 — ATTRIBUTE-LEVEL NLI RESULTS")
print("=" * 80)

claim2_nli_results = []

for item in claim_2_attributes:

    attribute_id = item["attribute_id"]
    attribute = item["attribute"]
    proposition = item["proposition"]
    source_ids = item["source_candidate_ids"]

    print("\n" + "-" * 80)
    print(f"Attribute {attribute_id}: {attribute}")
    print("-" * 80)

    print("\nProposition:")
    print(proposition)

    best_result = None

    # --------------------------------------------------------
    # Test all designated evidence for this attribute
    # --------------------------------------------------------

    for source_id in source_ids:

        candidate = candidate_lookup.get(source_id)

        if candidate is None:
            print(f"\nCandidate {source_id}: NOT FOUND")
            continue

        evidence = candidate["text"]

        result = run_nli(
            evidence,
            proposition
        )

        print(f"\nEvidence Candidate: {source_id}")
        print(f"Speaker   : {candidate['speaker']}")
        print(
            f"Timestamp : {candidate['start']:.3f} - "
            f"{candidate['end']:.3f}"
        )
        print(f"Evidence  : {evidence}")

        print("\nNLI:")
        print(f"  Label      : {result['predicted_label']}")
        print(f"  Confidence : {result['confidence']:.4f}")
        print(
            f"  Entailment : "
            f"{result['entailment_probability']:.4f}"
        )

        # ----------------------------------------------------
        # Select the evidence with the highest entailment
        # probability.
        # ----------------------------------------------------

        if (
            best_result is None
            or result["entailment_probability"]
            > best_result["entailment_probability"]
        ):
            best_result = {
                "candidate_id": source_id,
                **result
            }

    # --------------------------------------------------------
    # Store best evidence for this attribute
    # --------------------------------------------------------

    if best_result is not None:

        claim2_nli_results.append({
            "attribute_id": attribute_id,
            "attribute": attribute,
            "proposition": proposition,
            "best_evidence_candidate_id":
                best_result["candidate_id"],
            "predicted_label":
                best_result["predicted_label"],
            "confidence":
                best_result["confidence"],
            "entailment_probability":
                best_result["entailment_probability"],
            "neutral_probability":
                best_result["neutral_probability"],
            "contradiction_probability":
                best_result["contradiction_probability"]
        })


print("\n" + "=" * 80)
print("CLAIM 2 ATTRIBUTE-LEVEL NLI COMPLETE")
print("=" * 80)

print("\nAttributes evaluated:",
      len(claim2_nli_results))

print("\nSummary:")

for result in claim2_nli_results:

    print(
        f"\nAttribute {result['attribute_id']} "
        f"({result['attribute']}):"
    )

    print(
        f"  Best Evidence : "
        f"{result['best_evidence_candidate_id']}"
    )

    print(
        f"  Label         : "
        f"{result['predicted_label']}"
    )

    print(
        f"  Entailment    : "
        f"{result['entailment_probability']:.4f}"
    )

print("\n" + "=" * 80)

CLAIM 2 — ATTRIBUTE-LEVEL NLI RESULTS

--------------------------------------------------------------------------------
Attribute 1: product_development
--------------------------------------------------------------------------------

Proposition:
The team is developing a remote control.

Evidence Candidate: 10
Speaker   : SPEAKER_02
Timestamp : 117.699 - 120.921
Evidence  : Now, we're developing a remote control, which you probably already know.

NLI:
  Label      : ENTAILMENT
  Confidence : 0.7902
  Entailment : 0.7902

--------------------------------------------------------------------------------
Attribute 2: originality
--------------------------------------------------------------------------------

Proposition:
The remote control is intended to be original.

Evidence Candidate: 11
Speaker   : SPEAKER_02
Timestamp : 122.503 - 132.290
Evidence  : We want it to be original, something that people haven't thought of. It's not out in the shops. Trendy, appealing to a wide market, but

In [ ]:
# ============================================================
# CONTEXT-EXPANDED NLI TEST
# Purpose:
# Test whether adding nearby transcript context improves
# NLI recognition of conversational/fragmented evidence.
#
# We test the same four attributes from Claim 2 that were
# incorrectly classified as NEUTRAL.
# ============================================================

# ------------------------------------------------------------
# Helper function:
# Get nearby candidates around a source candidate
# ------------------------------------------------------------

def get_context_text(candidate_id, window=1):

    # Sort candidates by candidate ID
    sorted_candidates = sorted(
        mom_candidates_data["candidates"],
        key=lambda x: x["candidate_id"]
    )

    # Find position of target candidate
    target_index = next(
        (
            i
            for i, item in enumerate(sorted_candidates)
            if item["candidate_id"] == candidate_id
        ),
        None
    )

    if target_index is None:
        return None

    start_index = max(0, target_index - window)
    end_index = min(
        len(sorted_candidates),
        target_index + window + 1
    )

    context_items = sorted_candidates[start_index:end_index]

    context_text = " ".join(
        item["text"]
        for item in context_items
    )

    return context_text


# ------------------------------------------------------------
# Test attributes
# ------------------------------------------------------------

context_tests = [
    (
        "originality",
        "The remote control is intended to be original.",
        11
    ),
    (
        "trendiness",
        "The remote control is intended to be trendy.",
        11
    ),
    (
        "market_appeal",
        "The remote control is intended to appeal to a wide market.",
        11
    ),
    (
        "user_friendliness",
        "The remote control is intended to be user-friendly.",
        12
    )
]


print("=" * 80)
print("CONTEXT-EXPANDED NLI TEST")
print("=" * 80)

for attribute, proposition, candidate_id in context_tests:

    context = get_context_text(
        candidate_id,
        window=1
    )

    result = run_nli(
        context,
        proposition
    )

    print("\n" + "-" * 80)
    print("Attribute:", attribute)
    print("Evidence candidate:", candidate_id)

    print("\nExpanded Evidence:")
    print(context)

    print("\nProposition:")
    print(proposition)

    print("\nNLI Result:")
    print(
        f"  Label       : {result['predicted_label']}"
    )
    print(
        f"  Entailment  : "
        f"{result['entailment_probability']:.4f}"
    )
    print(
        f"  Neutral     : "
        f"{result['neutral_probability']:.4f}"
    )
    print(
        f"  Contradiction: "
        f"{result['contradiction_probability']:.4f}"
    )

print("\n" + "=" * 80)

CONTEXT-EXPANDED NLI TEST

--------------------------------------------------------------------------------
Attribute: originality
Evidence candidate: 11

Expanded Evidence:
Now, we're developing a remote control, which you probably already know. We want it to be original, something that people haven't thought of. It's not out in the shops. Trendy, appealing to a wide market, but, you know, not a hunk of metal. And user-friendly, grannies to kids,

Proposition:
The remote control is intended to be original.

NLI Result:
  Label       : ENTAILMENT
  Entailment  : 0.9865
  Neutral     : 0.0124
  Contradiction: 0.0011

--------------------------------------------------------------------------------
Attribute: trendiness
Evidence candidate: 11

Expanded Evidence:
Now, we're developing a remote control, which you probably already know. We want it to be original, something that people haven't thought of. It's not out in the shops. Trendy, appealing to a wide market, but, you know, not a hunk

In [ ]:
# ============================================================
# M7 — EXPLICIT EVIDENCE MATCHING TEST
# Purpose:
# Test whether important words/phrases from a proposition
# are explicitly present in the retrieved transcript evidence.
#
# This is NOT used as proof by itself.
# It is a supporting signal for cases where NLI may fail on
# conversational or fragmented meeting transcripts.
# ============================================================

import re


def normalize_text(text):
    """
    Convert text to lowercase and remove punctuation.
    """
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s-]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def explicit_phrase_match(evidence, phrases):
    """
    Check whether important evidence phrases occur explicitly
    in the transcript evidence.

    Returns:
        matched_phrases
        match_ratio
    """

    normalized_evidence = normalize_text(evidence)

    matched = []

    for phrase in phrases:

        normalized_phrase = normalize_text(phrase)

        if normalized_phrase in normalized_evidence:
            matched.append(phrase)

    match_ratio = (
        len(matched) / len(phrases)
        if phrases
        else 0.0
    )

    return matched, match_ratio


# ------------------------------------------------------------
# Test cases
# ------------------------------------------------------------

tests = [
    {
        "name": "User friendliness",
        "evidence": (
            "We want it to be original, something that people "
            "haven't thought of. It's not out in the shops. "
            "Trendy, appealing to a wide market, but, you know, "
            "not a hunk of metal. And user-friendly, grannies "
            "to kids, maybe even pooches, should be able to use it."
        ),
        "phrases": [
            "user-friendly"
        ]
    },
    {
        "name": "Broad usability",
        "evidence": (
            "And user-friendly, grannies to kids, maybe even "
            "pooches, should be able to use it."
        ),
        "phrases": [
            "grannies",
            "kids",
            "should be able to use it"
        ]
    },
    {
        "name": "Unrelated example",
        "evidence": (
            "We discussed the design process and the detailed "
            "design stage."
        ),
        "phrases": [
            "user-friendly",
            "wide market"
        ]
    }
]


print("=" * 80)
print("M7 — EXPLICIT EVIDENCE MATCHING TEST")
print("=" * 80)

for test in tests:

    matched, ratio = explicit_phrase_match(
        test["evidence"],
        test["phrases"]
    )

    print("\n" + "-" * 80)
    print("Test:", test["name"])

    print("\nEvidence:")
    print(test["evidence"])

    print("\nRequired phrases:")
    print(test["phrases"])

    print("\nMatched phrases:")
    print(matched)

    print(f"\nMatch ratio: {ratio:.4f}")

print("\n" + "=" * 80)

M7 — EXPLICIT EVIDENCE MATCHING TEST

--------------------------------------------------------------------------------
Test: User friendliness

Evidence:
We want it to be original, something that people haven't thought of. It's not out in the shops. Trendy, appealing to a wide market, but, you know, not a hunk of metal. And user-friendly, grannies to kids, maybe even pooches, should be able to use it.

Required phrases:
['user-friendly']

Matched phrases:
['user-friendly']

Match ratio: 1.0000

--------------------------------------------------------------------------------
Test: Broad usability

Evidence:
And user-friendly, grannies to kids, maybe even pooches, should be able to use it.

Required phrases:
['grannies', 'kids', 'should be able to use it']

Matched phrases:
['grannies', 'kids', 'should be able to use it']

Match ratio: 1.0000

--------------------------------------------------------------------------------
Test: Unrelated example

Evidence:
We discussed the design proces

In [ ]:
# ============================================================
# M7 — HYBRID EVIDENCE SUPPORT TEST
# Purpose:
# Combine:
#   1. NLI entailment probability
#   2. Explicit evidence phrase matching
#
# This is an experimental verification signal.
# We will NOT finalize thresholds yet.
# ============================================================

def hybrid_evidence_score(
    evidence,
    proposition,
    required_phrases
):
    """
    Calculate a hybrid evidence-support score.

    Components:
        NLI entailment probability
        Explicit phrase match ratio

    The two components are kept separate so that we can
    inspect their behavior before defining the final rule.
    """

    # --------------------------------------------------------
    # NLI
    # --------------------------------------------------------

    nli_result = run_nli(
        evidence,
        proposition
    )

    # --------------------------------------------------------
    # Explicit phrase matching
    # --------------------------------------------------------

    matched_phrases, match_ratio = explicit_phrase_match(
        evidence,
        required_phrases
    )

    # --------------------------------------------------------
    # Experimental combined score
    #
    # 70% NLI + 30% explicit evidence
    #
    # IMPORTANT:
    # This weighting is temporary for experimentation only.
    # We will NOT use it as the final research threshold.
    # --------------------------------------------------------

    hybrid_score = (
        0.70 * nli_result["entailment_probability"]
        +
        0.30 * match_ratio
    )

    return {
        "nli_label": nli_result["predicted_label"],
        "nli_entailment": nli_result["entailment_probability"],
        "nli_neutral": nli_result["neutral_probability"],
        "nli_contradiction": nli_result["contradiction_probability"],
        "matched_phrases": matched_phrases,
        "phrase_match_ratio": match_ratio,
        "hybrid_score": hybrid_score
    }


# ------------------------------------------------------------
# Test cases
# ------------------------------------------------------------

tests = [
    {
        "name": "Strong semantic + explicit evidence",
        "evidence": (
            "Now, we're developing a remote control, "
            "which you probably already know."
        ),
        "proposition": (
            "The team is developing a remote control."
        ),
        "phrases": [
            "developing a remote control"
        ]
    },
    {
        "name": "Explicit evidence but weak NLI",
        "evidence": (
            "We want it to be original, something that people "
            "haven't thought of. It's not out in the shops. "
            "Trendy, appealing to a wide market, but, you know, "
            "not a hunk of metal. And user-friendly, grannies "
            "to kids, maybe even pooches, should be able to use it."
        ),
        "proposition": (
            "The remote control is intended to be user-friendly."
        ),
        "phrases": [
            "user-friendly"
        ]
    },
    {
        "name": "No supporting evidence",
        "evidence": (
            "We discussed the design process and the detailed "
            "design stage."
        ),
        "proposition": (
            "The remote control is intended to be user-friendly."
        ),
        "phrases": [
            "user-friendly"
        ]
    },
    {
        "name": "Potential contradiction",
        "evidence": (
            "The product is intended only for young adults and "
            "is not designed for older users."
        ),
        "proposition": (
            "The product should be usable by all age groups."
        ),
        "phrases": [
            "all age groups"
        ]
    }
]


print("=" * 80)
print("M7 — HYBRID EVIDENCE SUPPORT TEST")
print("=" * 80)

for test in tests:

    result = hybrid_evidence_score(
        test["evidence"],
        test["proposition"],
        test["phrases"]
    )

    print("\n" + "-" * 80)
    print("Test:", test["name"])

    print("\nEvidence:")
    print(test["evidence"])

    print("\nProposition:")
    print(test["proposition"])

    print("\nResults:")
    print(
        f"  NLI Label          : "
        f"{result['nli_label']}"
    )

    print(
        f"  NLI Entailment     : "
        f"{result['nli_entailment']:.4f}"
    )

    print(
        f"  NLI Neutral        : "
        f"{result['nli_neutral']:.4f}"
    )

    print(
        f"  NLI Contradiction  : "
        f"{result['nli_contradiction']:.4f}"
    )

    print(
        f"  Matched Phrases    : "
        f"{result['matched_phrases']}"
    )

    print(
        f"  Phrase Match Ratio : "
        f"{result['phrase_match_ratio']:.4f}"
    )

    print(
        f"  Experimental Score : "
        f"{result['hybrid_score']:.4f}"
    )

print("\n" + "=" * 80)


M7 — HYBRID EVIDENCE SUPPORT TEST


NameError: name 'run_nli' is not defined

In [ ]:
# ============================================================
# M7 — RESTORE NLI FUNCTION
# Purpose:
# Recreate the reusable NLI function in the current runtime.
#
# The NLI model was already loaded and validated earlier.
# No model re-download or reinstallation is required.
# ============================================================

import torch

def run_nli(evidence, proposition):
    """
    Perform NLI between transcript evidence and a proposition.

    Premise    = evidence
    Hypothesis = proposition
    """

    inputs = nli_tokenizer(
        evidence,
        proposition,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    inputs = {
        key: value.to(DEVICE)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        outputs = nli_model(**inputs)

    probabilities = torch.softmax(
        outputs.logits,
        dim=-1
    )[0]

    # Confirmed model mapping:
    # 0 = contradiction
    # 1 = entailment
    # 2 = neutral

    contradiction_prob = probabilities[0].item()
    entailment_prob = probabilities[1].item()
    neutral_prob = probabilities[2].item()

    predicted_id = torch.argmax(probabilities).item()

    predicted_label = (
        nli_model.config.id2label[predicted_id].upper()
    )

    confidence = probabilities[predicted_id].item()

    return {
        "predicted_label": predicted_label,
        "confidence": confidence,
        "contradiction_probability": contradiction_prob,
        "entailment_probability": entailment_prob,
        "neutral_probability": neutral_prob
    }


print("=" * 80)
print("NLI FUNCTION RESTORED")
print("=" * 80)

print("\nFunction: run_nli()")
print("Premise    = Evidence")
print("Hypothesis = Proposition")

print("\nModel mapping:")
print("  0 -> CONTRADICTION")
print("  1 -> ENTAILMENT")
print("  2 -> NEUTRAL")

print("\nReady for M7 hybrid verification.")

print("=" * 80)

NLI FUNCTION RESTORED

Function: run_nli()
Premise    = Evidence
Hypothesis = Proposition

Model mapping:
  0 -> CONTRADICTION
  1 -> ENTAILMENT
  2 -> NEUTRAL

Ready for M7 hybrid verification.


In [ ]:
# ============================================================
# M7 — QUICK HYBRID FUNCTION CHECK
# Purpose:
# Confirm that both NLI and explicit phrase matching work
# together before running the complete test set.
# ============================================================

evidence = (
    "And user-friendly, grannies to kids, "
    "maybe even pooches, should be able to use it."
)

proposition = (
    "The remote control is intended to be user-friendly."
)

required_phrases = [
    "user-friendly"
]

# ------------------------------------------------------------
# NLI
# ------------------------------------------------------------

nli_result = run_nli(
    evidence,
    proposition
)

# ------------------------------------------------------------
# Explicit phrase matching
# ------------------------------------------------------------

matched_phrases, match_ratio = explicit_phrase_match(
    evidence,
    required_phrases
)

# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print("=" * 80)
print("M7 — QUICK HYBRID FUNCTION CHECK")
print("=" * 80)

print("\nNLI:")
print(f"  Label       : {nli_result['predicted_label']}")
print(
    f"  Entailment  : "
    f"{nli_result['entailment_probability']:.4f}"
)
print(
    f"  Neutral     : "
    f"{nli_result['neutral_probability']:.4f}"
)
print(
    f"  Contradiction: "
    f"{nli_result['contradiction_probability']:.4f}"
)

print("\nExplicit Evidence:")
print(f"  Matched     : {matched_phrases}")
print(f"  Match Ratio : {match_ratio:.4f}")

print("\n" + "=" * 80)

NameError: name 'nli_tokenizer' is not defined

In [ ]:
# ============================================================
# M7 — RESTORE NLI TOKENIZER
# Purpose:
# Recreate the tokenizer object in the current runtime.
#
# The NLI model is already loaded. No model reinstallation
# is required.
# ============================================================

from transformers import AutoTokenizer

NLI_MODEL_NAME = "cross-encoder/nli-deberta-v3-small"

print("=" * 80)
print("RESTORING NLI TOKENIZER")
print("=" * 80)

nli_tokenizer = AutoTokenizer.from_pretrained(
    NLI_MODEL_NAME
)

print("\nNLI tokenizer restored successfully.")
print("Model:", NLI_MODEL_NAME)

print("=" * 80)

RESTORING NLI TOKENIZER


config.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.35k [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/8.66M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]


NLI tokenizer restored successfully.
Model: cross-encoder/nli-deberta-v3-small


In [ ]:
# ============================================================
# M7 — QUICK HYBRID FUNCTION CHECK
# Purpose:
# Confirm that NLI and explicit phrase matching work together.
# ============================================================

evidence = (
    "And user-friendly, grannies to kids, "
    "maybe even pooches, should be able to use it."
)

proposition = (
    "The remote control is intended to be user-friendly."
)

required_phrases = [
    "user-friendly"
]

# ------------------------------------------------------------
# NLI
# ------------------------------------------------------------

nli_result = run_nli(
    evidence,
    proposition
)

# ------------------------------------------------------------
# Explicit phrase matching
# ------------------------------------------------------------

matched_phrases, match_ratio = explicit_phrase_match(
    evidence,
    required_phrases
)

# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print("=" * 80)
print("M7 — QUICK HYBRID FUNCTION CHECK")
print("=" * 80)

print("\nNLI:")
print(f"  Label        : {nli_result['predicted_label']}")
print(
    f"  Entailment   : "
    f"{nli_result['entailment_probability']:.4f}"
)
print(
    f"  Neutral      : "
    f"{nli_result['neutral_probability']:.4f}"
)
print(
    f"  Contradiction: "
    f"{nli_result['contradiction_probability']:.4f}"
)

print("\nExplicit Evidence:")
print(f"  Matched      : {matched_phrases}")
print(f"  Match Ratio  : {match_ratio:.4f}")

print("\n" + "=" * 80)

NameError: name 'DEVICE' is not defined

In [ ]:
# ============================================================
# M7 — RESTORE DEVICE
# Purpose:
# Recreate the DEVICE variable used by the NLI verification
# functions.
#
# Automatically uses CUDA when available; otherwise CPU.
# ============================================================

import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("=" * 80)
print("M7 — DEVICE RESTORED")
print("=" * 80)

print("\nDevice:", DEVICE)

if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("CUDA is not available. Using CPU.")

print("=" * 80)

M7 — DEVICE RESTORED

Device: cpu
CUDA is not available. Using CPU.


M7 initialization/setup cell that:

Mounts Drive
Sets project paths
Loads the structured MoM claims
Loads the candidate data
Loads the NLI model/tokenizer
Sets DEVICE = "cuda"
Recreates the verification functions

In [1]:
# ============================================================
# M7 — RESTORE DEVICE
# Purpose:
# Recreate the DEVICE variable used by the NLI verification
# functions.
#
# Automatically uses CUDA when available; otherwise CPU.
# ============================================================

import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("=" * 80)
print("M7 — DEVICE RESTORED")
print("=" * 80)

print("\nDevice:", DEVICE)

if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("CUDA is not available. Using CPU.")

print("=" * 80)

M7 — DEVICE RESTORED

Device: cuda
GPU: Tesla T4


In [2]:
# ============================================================
# M7 — EVIDENCE VERIFICATION INITIALIZATION
# Purpose:
# Initialize the M7 verification environment after a runtime
# restart.
#
# This cell:
#   1. Mounts Google Drive
#   2. Defines project paths
#   3. Loads MoM claims and candidates
#   4. Sets the computation device
#   5. Loads the NLI model and tokenizer
#
# No earlier project stages are re-run.
# ============================================================

import os
import json
import torch

from google.colab import drive
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

# ------------------------------------------------------------
# 1. Mount Google Drive
# ------------------------------------------------------------

drive.mount("/content/drive")


# ------------------------------------------------------------
# 2. Project paths
# ------------------------------------------------------------

PROJECT_DIR = "/content/drive/MyDrive/MTechIndProj/MoM_Project"

DATA_DIR = os.path.join(
    PROJECT_DIR,
    "data"
)

TRANSCRIPTS_DIR = os.path.join(
    DATA_DIR,
    "transcripts"
)

MOM_CANDIDATES_PATH = os.path.join(
    TRANSCRIPTS_DIR,
    "ES2004a_mom_candidates_rule_based.json"
)

MOM_CLAIMS_PATH = os.path.join(
    TRANSCRIPTS_DIR,
    "ES2004a_structured_mom_claims.json"
)


# ------------------------------------------------------------
# 3. Device
# ------------------------------------------------------------

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


# ------------------------------------------------------------
# 4. Load MoM claims
# ------------------------------------------------------------

with open(
    MOM_CLAIMS_PATH,
    "r",
    encoding="utf-8"
) as f:

    mom_claims_data = json.load(f)


# ------------------------------------------------------------
# 5. Load MoM candidates
# ------------------------------------------------------------

with open(
    MOM_CANDIDATES_PATH,
    "r",
    encoding="utf-8"
) as f:

    mom_candidates_data = json.load(f)


# ------------------------------------------------------------
# 6. Create candidate lookup
# ------------------------------------------------------------

candidate_lookup = {
    item["candidate_id"]: item
    for item in mom_candidates_data["candidates"]
}


# ------------------------------------------------------------
# 7. Load NLI model and tokenizer
# ------------------------------------------------------------

NLI_MODEL_NAME = (
    "cross-encoder/nli-deberta-v3-small"
)

print("=" * 80)
print("M7 — EVIDENCE VERIFICATION INITIALIZATION")
print("=" * 80)

print("\nDevice:", DEVICE)

if DEVICE == "cuda":
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

print("\nLoading NLI tokenizer...")

nli_tokenizer = AutoTokenizer.from_pretrained(
    NLI_MODEL_NAME
)

print("Tokenizer loaded.")

print("\nLoading NLI model...")

nli_model = AutoModelForSequenceClassification.from_pretrained(
    NLI_MODEL_NAME
)

nli_model = nli_model.to(DEVICE)
nli_model.eval()

print("NLI model loaded.")


# ------------------------------------------------------------
# 8. Verify loaded project data
# ------------------------------------------------------------

print("\nProject data:")
print(
    "  MoM claims     :",
    len(mom_claims_data["claims"])
)
print(
    "  MoM candidates :",
    len(mom_candidates_data["candidates"])
)

print("\nNLI label mapping:")

for label_id, label_name in nli_model.config.id2label.items():
    print(
        f"  {label_id} -> {label_name}"
    )

print("\n" + "=" * 80)
print("M7 INITIALIZATION COMPLETE")
print("=" * 80)

Mounted at /content/drive
M7 — EVIDENCE VERIFICATION INITIALIZATION

Device: cuda
GPU: Tesla T4

Loading NLI tokenizer...


config.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.35k [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

...

Tokenizer loaded.

Loading NLI model...


model.safetensors: reconstructing file:   0%|          |  0.00B /  568MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

NLI model loaded.

Project data:
  MoM claims     : 12
  MoM candidates : 50

NLI label mapping:
  0 -> contradiction
  1 -> entailment
  2 -> neutral

M7 INITIALIZATION COMPLETE


In [3]:
# ============================================================
# M7 — NLI VERIFICATION FUNCTION
# Purpose:
# Provide a reusable function for checking whether a transcript
# evidence segment supports a single MoM proposition.
#
# Premise    = Transcript evidence
# Hypothesis = MoM proposition
#
# Model:
# cross-encoder/nli-deberta-v3-small
#
# Label mapping:
#   0 = CONTRADICTION
#   1 = ENTAILMENT
#   2 = NEUTRAL
# ============================================================

def run_nli(evidence, proposition):
    """
    Perform Natural Language Inference between evidence
    and a single factual proposition.
    """

    inputs = nli_tokenizer(
        evidence,
        proposition,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    inputs = {
        key: value.to(DEVICE)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        outputs = nli_model(**inputs)

    probabilities = torch.softmax(
        outputs.logits,
        dim=-1
    )[0]

    # --------------------------------------------------------
    # Extract probabilities
    # --------------------------------------------------------

    contradiction_probability = probabilities[0].item()
    entailment_probability = probabilities[1].item()
    neutral_probability = probabilities[2].item()

    # --------------------------------------------------------
    # Determine predicted label
    # --------------------------------------------------------

    predicted_id = torch.argmax(probabilities).item()

    predicted_label = (
        nli_model.config.id2label[predicted_id].upper()
    )

    confidence = probabilities[predicted_id].item()

    return {
        "predicted_label": predicted_label,
        "confidence": confidence,
        "contradiction_probability":
            contradiction_probability,
        "entailment_probability":
            entailment_probability,
        "neutral_probability":
            neutral_probability
    }


print("=" * 80)
print("M7 — NLI VERIFICATION FUNCTION READY")
print("=" * 80)

print("\nFunction: run_nli()")
print("Premise    = Evidence")
print("Hypothesis = Proposition")

print("\nOutput:")
print("  Predicted label")
print("  Confidence")
print("  Entailment probability")
print("  Neutral probability")
print("  Contradiction probability")

print("\n" + "=" * 80)

M7 — NLI VERIFICATION FUNCTION READY

Function: run_nli()
Premise    = Evidence
Hypothesis = Proposition

Output:
  Predicted label
  Confidence
  Entailment probability
  Neutral probability
  Contradiction probability



In [4]:
# ============================================================
# M7 — NLI SANITY TEST
# Purpose:
# Test NLI on one real transcript evidence segment
# and one simple MoM proposition.
# ============================================================

evidence = (
    "Now, we're developing a remote control, "
    "which you probably already know."
)

proposition = (
    "The team is developing a remote control."
)

result = run_nli(evidence, proposition)

print("=" * 80)
print("M7 — NLI SANITY TEST")
print("=" * 80)

print("\nEvidence:")
print(evidence)

print("\nProposition:")
print(proposition)

print("\nNLI Result:")
for key, value in result.items():
    print(f"  {key}: {value:.4f}" if isinstance(value, float)
          else f"  {key}: {value}")

print("\n" + "=" * 80)

M7 — NLI SANITY TEST

Evidence:
Now, we're developing a remote control, which you probably already know.

Proposition:
The team is developing a remote control.

NLI Result:
  predicted_label: ENTAILMENT
  confidence: 0.7902
  contradiction_probability: 0.0005
  entailment_probability: 0.7902
  neutral_probability: 0.2093



In [5]:
# ============================================================
# M7 — NLI CONTROLLED TEST
# Purpose:
# Test NLI behavior for:
#   1. Supporting evidence
#   2. Contradictory evidence
#   3. Unrelated evidence
#
# This helps validate the NLI component before integrating it
# into the full evidence verification pipeline.
# ============================================================

test_cases = [
    {
        "name": "SUPPORTING",
        "evidence": (
            "Now, we're developing a remote control, "
            "which you probably already know."
        ),
        "proposition": (
            "The team is developing a remote control."
        )
    },
    {
        "name": "CONTRADICTORY",
        "evidence": (
            "We're not developing a remote control. "
            "The project is focused on a different product."
        ),
        "proposition": (
            "The team is developing a remote control."
        )
    },
    {
        "name": "UNRELATED",
        "evidence": (
            "The team discussed the meeting schedule "
            "and the next meeting date."
        ),
        "proposition": (
            "The team is developing a remote control."
        )
    }
]

print("=" * 80)
print("M7 — NLI CONTROLLED TEST")
print("=" * 80)

for case in test_cases:

    result = run_nli(
        case["evidence"],
        case["proposition"]
    )

    print(f"\n[{case['name']}]")

    print("Evidence:")
    print(case["evidence"])

    print("\nProposition:")
    print(case["proposition"])

    print("\nResult:")
    print("  Predicted label :", result["predicted_label"])
    print("  Confidence      :", round(result["confidence"], 4))
    print("  Entailment      :", round(result["entailment_probability"], 4))
    print("  Neutral         :", round(result["neutral_probability"], 4))
    print("  Contradiction   :", round(result["contradiction_probability"], 4))

print("\n" + "=" * 80)

M7 — NLI CONTROLLED TEST

[SUPPORTING]
Evidence:
Now, we're developing a remote control, which you probably already know.

Proposition:
The team is developing a remote control.

Result:
  Predicted label : ENTAILMENT
  Confidence      : 0.7902
  Entailment      : 0.7902
  Neutral         : 0.2093
  Contradiction   : 0.0005

[CONTRADICTORY]
Evidence:
We're not developing a remote control. The project is focused on a different product.

Proposition:
The team is developing a remote control.

Result:
  Predicted label : CONTRADICTION
  Confidence      : 0.9967
  Entailment      : 0.0012
  Neutral         : 0.0021
  Contradiction   : 0.9967

[UNRELATED]
Evidence:
The team discussed the meeting schedule and the next meeting date.

Proposition:
The team is developing a remote control.

Result:
  Predicted label : CONTRADICTION
  Confidence      : 0.9975
  Entailment      : 0.0
  Neutral         : 0.0025
  Contradiction   : 0.9975



In [6]:
# ============================================================
# M7 — REAL MEETING NLI TEST
# Purpose:
# Test NLI using actual meeting evidence and a simple
# proposition derived from that evidence.
# ============================================================

evidence = (
    "We want it to be original. "
    "Trendy, appealing to a wide market."
)

proposition = (
    "The remote control is intended to be original."
)

result = run_nli(
    evidence,
    proposition
)

print("=" * 80)
print("M7 — REAL MEETING NLI TEST")
print("=" * 80)

print("\nEvidence:")
print(evidence)

print("\nProposition:")
print(proposition)

print("\nNLI Result:")
print("  Predicted label :", result["predicted_label"])
print("  Confidence      :", round(result["confidence"], 4))
print("  Entailment      :", round(result["entailment_probability"], 4))
print("  Neutral         :", round(result["neutral_probability"], 4))
print("  Contradiction   :", round(result["contradiction_probability"], 4))

print("\n" + "=" * 80)

M7 — REAL MEETING NLI TEST

Evidence:
We want it to be original. Trendy, appealing to a wide market.

Proposition:
The remote control is intended to be original.

NLI Result:
  Predicted label : NEUTRAL
  Confidence      : 0.9996
  Entailment      : 0.0002
  Neutral         : 0.9996
  Contradiction   : 0.0002



In [7]:
# ============================================================
# M7 — EVIDENCE RETRIEVAL INITIALIZATION
# Purpose:
# Rebuild the speaker-level evidence collection and BGE + FAISS
# retrieval resources after a Colab runtime restart.
#
# Source:
#   ES2004a speaker-level transcript
#
# Retrieval model:
#   BAAI/bge-small-en-v1.5
#
# FAISS:
#   IndexFlatIP on normalized embeddings
#   → inner product approximates cosine similarity
# ============================================================

import os
import json
import numpy as np
import torch

from sentence_transformers import SentenceTransformer
import faiss

# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

SPEAKER_TRANSCRIPT_PATH = os.path.join(
    TRANSCRIPTS_DIR,
    "ES2004a_speaker_transcript.json"
)

# ------------------------------------------------------------
# 2. Load speaker-level transcript
# ------------------------------------------------------------

with open(SPEAKER_TRANSCRIPT_PATH, "r", encoding="utf-8") as f:
    speaker_transcript_data = json.load(f)

speaker_utterances = speaker_transcript_data["speaker_utterances"]

# ------------------------------------------------------------
# 3. Create evidence documents
# ------------------------------------------------------------

evidence_documents = []

for i, utterance in enumerate(speaker_utterances):

    evidence_documents.append({
        "evidence_id": i + 1,
        "speaker": utterance["speaker"],
        "start": utterance["start"],
        "end": utterance["end"],
        "duration": utterance["end"] - utterance["start"],
        "text": utterance["text"]
    })

# ------------------------------------------------------------
# 4. Load BGE embedding model
# ------------------------------------------------------------

BGE_MODEL_NAME = "BAAI/bge-small-en-v1.5"

print("=" * 80)
print("M7 — EVIDENCE RETRIEVAL INITIALIZATION")
print("=" * 80)

print("\nDevice:", DEVICE)

print("\nLoading BGE model...")
bge_model = SentenceTransformer(
    BGE_MODEL_NAME,
    device=DEVICE
)

print("BGE model loaded.")

# ------------------------------------------------------------
# 5. Generate normalized evidence embeddings
# ------------------------------------------------------------

evidence_texts = [
    item["text"]
    for item in evidence_documents
]

print("\nGenerating evidence embeddings...")

evidence_embeddings = bge_model.encode(
    evidence_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

evidence_embeddings = evidence_embeddings.astype(
    np.float32
)

# ------------------------------------------------------------
# 6. Build FAISS index
# ------------------------------------------------------------

embedding_dimension = evidence_embeddings.shape[1]

faiss_index = faiss.IndexFlatIP(
    embedding_dimension
)

faiss_index.add(evidence_embeddings)

# ------------------------------------------------------------
# 7. Retrieval function
# ------------------------------------------------------------

def retrieve_evidence(claim_text, top_k=10):

    query_embedding = bge_model.encode(
        [claim_text],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype(np.float32)

    similarities, indices = faiss_index.search(
        query_embedding,
        top_k
    )

    results = []

    for rank, (index, similarity) in enumerate(
        zip(indices[0], similarities[0]),
        start=1
    ):

        evidence = evidence_documents[index]

        results.append({
            "retrieval_rank": rank,
            "retrieval_similarity": float(similarity),
            **evidence
        })

    return results

# ------------------------------------------------------------
# 8. Summary
# ------------------------------------------------------------

print("\nEvidence documents :", len(evidence_documents))
print("Embedding shape    :", evidence_embeddings.shape)
print("FAISS vectors      :", faiss_index.ntotal)
print("Embedding dimension:", embedding_dimension)

print("\n" + "=" * 80)
print("M7 EVIDENCE RETRIEVAL INITIALIZATION COMPLETE")
print("=" * 80)

ModuleNotFoundError: No module named 'faiss'

In [8]:
# ============================================================
# M7 — INSTALL FAISS
# Purpose:
# Install FAISS for vector similarity search.
# ============================================================

!pip install -q faiss-cpu

print("FAISS installation complete.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 54.3 MB/s eta 0:00:00
FAISS installation complete.


In [9]:
# ============================================================
# M7 — EVIDENCE RETRIEVAL INITIALIZATION
# Purpose:
# Rebuild the speaker-level evidence collection and BGE + FAISS
# retrieval resources after a Colab runtime restart.
#
# Source:
#   ES2004a speaker-level transcript
#
# Retrieval model:
#   BAAI/bge-small-en-v1.5
#
# FAISS:
#   IndexFlatIP on normalized embeddings
#   → inner product approximates cosine similarity
# ============================================================

import os
import json
import numpy as np
import torch

from sentence_transformers import SentenceTransformer
import faiss

# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

SPEAKER_TRANSCRIPT_PATH = os.path.join(
    TRANSCRIPTS_DIR,
    "ES2004a_speaker_transcript.json"
)

# ------------------------------------------------------------
# 2. Load speaker-level transcript
# ------------------------------------------------------------

with open(SPEAKER_TRANSCRIPT_PATH, "r", encoding="utf-8") as f:
    speaker_transcript_data = json.load(f)

speaker_utterances = speaker_transcript_data["speaker_utterances"]

# ------------------------------------------------------------
# 3. Create evidence documents
# ------------------------------------------------------------

evidence_documents = []

for i, utterance in enumerate(speaker_utterances):

    evidence_documents.append({
        "evidence_id": i + 1,
        "speaker": utterance["speaker"],
        "start": utterance["start"],
        "end": utterance["end"],
        "duration": utterance["end"] - utterance["start"],
        "text": utterance["text"]
    })

# ------------------------------------------------------------
# 4. Load BGE embedding model
# ------------------------------------------------------------

BGE_MODEL_NAME = "BAAI/bge-small-en-v1.5"

print("=" * 80)
print("M7 — EVIDENCE RETRIEVAL INITIALIZATION")
print("=" * 80)

print("\nDevice:", DEVICE)

print("\nLoading BGE model...")
bge_model = SentenceTransformer(
    BGE_MODEL_NAME,
    device=DEVICE
)

print("BGE model loaded.")

# ------------------------------------------------------------
# 5. Generate normalized evidence embeddings
# ------------------------------------------------------------

evidence_texts = [
    item["text"]
    for item in evidence_documents
]

print("\nGenerating evidence embeddings...")

evidence_embeddings = bge_model.encode(
    evidence_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

evidence_embeddings = evidence_embeddings.astype(
    np.float32
)

# ------------------------------------------------------------
# 6. Build FAISS index
# ------------------------------------------------------------

embedding_dimension = evidence_embeddings.shape[1]

faiss_index = faiss.IndexFlatIP(
    embedding_dimension
)

faiss_index.add(evidence_embeddings)

# ------------------------------------------------------------
# 7. Retrieval function
# ------------------------------------------------------------

def retrieve_evidence(claim_text, top_k=10):

    query_embedding = bge_model.encode(
        [claim_text],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype(np.float32)

    similarities, indices = faiss_index.search(
        query_embedding,
        top_k
    )

    results = []

    for rank, (index, similarity) in enumerate(
        zip(indices[0], similarities[0]),
        start=1
    ):

        evidence = evidence_documents[index]

        results.append({
            "retrieval_rank": rank,
            "retrieval_similarity": float(similarity),
            **evidence
        })

    return results

# ------------------------------------------------------------
# 8. Summary
# ------------------------------------------------------------

print("\nEvidence documents :", len(evidence_documents))
print("Embedding shape    :", evidence_embeddings.shape)
print("FAISS vectors      :", faiss_index.ntotal)
print("Embedding dimension:", embedding_dimension)

print("\n" + "=" * 80)
print("M7 EVIDENCE RETRIEVAL INITIALIZATION COMPLETE")
print("=" * 80)

KeyError: 'speaker_utterances'

In [10]:
# ============================================================
# M7 — INSPECT SPEAKER TRANSCRIPT JSON
# Purpose:
# Identify the actual structure of the saved transcript JSON.
# No data is modified.
# ============================================================

print("=" * 80)
print("M7 — SPEAKER TRANSCRIPT JSON STRUCTURE")
print("=" * 80)

print("\nTop-level keys:")
for key in speaker_transcript_data.keys():
    print(" ", key)

print("\nData types:")
for key, value in speaker_transcript_data.items():
    print(f"  {key}: {type(value).__name__}")

print("\n" + "=" * 80)

M7 — SPEAKER TRANSCRIPT JSON STRUCTURE

Top-level keys:
  meeting_id
  source
  gap_threshold_seconds
  num_speakers
  speakers
  num_utterances
  utterances

Data types:
  meeting_id: str
  source: str
  gap_threshold_seconds: float
  num_speakers: int
  speakers: list
  num_utterances: int
  utterances: list



In [11]:
# ============================================================
# M7 — EVIDENCE RETRIEVAL INITIALIZATION
# Purpose:
# Build speaker-level evidence documents and the BGE + FAISS
# retrieval index from the saved speaker transcript.
#
# Source:
#   ES2004a_speaker_transcript.json
#
# Retrieval model:
#   BAAI/bge-small-en-v1.5
#
# FAISS:
#   IndexFlatIP with normalized embeddings
# ============================================================

import numpy as np
from sentence_transformers import SentenceTransformer
import faiss

# ------------------------------------------------------------
# 1. Use the correct transcript structure
# ------------------------------------------------------------

speaker_utterances = speaker_transcript_data["utterances"]

# ------------------------------------------------------------
# 2. Create evidence documents
# ------------------------------------------------------------

evidence_documents = []

for i, utterance in enumerate(speaker_utterances):

    evidence_documents.append({
        "evidence_id": i + 1,
        "speaker": utterance["speaker"],
        "start": utterance["start"],
        "end": utterance["end"],
        "duration": utterance["end"] - utterance["start"],
        "text": utterance["text"]
    })

# ------------------------------------------------------------
# 3. Load BGE embedding model
# ------------------------------------------------------------

BGE_MODEL_NAME = "BAAI/bge-small-en-v1.5"

print("=" * 80)
print("M7 — EVIDENCE RETRIEVAL INITIALIZATION")
print("=" * 80)

print("\nDevice:", DEVICE)

print("\nLoading BGE model...")

bge_model = SentenceTransformer(
    BGE_MODEL_NAME,
    device=DEVICE
)

print("BGE model loaded.")

# ------------------------------------------------------------
# 4. Generate evidence embeddings
# ------------------------------------------------------------

evidence_texts = [
    item["text"]
    for item in evidence_documents
]

print("\nGenerating evidence embeddings...")

evidence_embeddings = bge_model.encode(
    evidence_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

evidence_embeddings = evidence_embeddings.astype(
    np.float32
)

# ------------------------------------------------------------
# 5. Build FAISS index
# ------------------------------------------------------------

embedding_dimension = evidence_embeddings.shape[1]

faiss_index = faiss.IndexFlatIP(
    embedding_dimension
)

faiss_index.add(evidence_embeddings)

# ------------------------------------------------------------
# 6. Retrieval function
# ------------------------------------------------------------

def retrieve_evidence(claim_text, top_k=10):

    query_embedding = bge_model.encode(
        [claim_text],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype(np.float32)

    similarities, indices = faiss_index.search(
        query_embedding,
        top_k
    )

    results = []

    for rank, (index, similarity) in enumerate(
        zip(indices[0], similarities[0]),
        start=1
    ):

        evidence = evidence_documents[index]

        results.append({
            "retrieval_rank": rank,
            "retrieval_similarity": float(similarity),
            **evidence
        })

    return results

# ------------------------------------------------------------
# 7. Summary
# ------------------------------------------------------------

print("\nEvidence documents :", len(evidence_documents))
print("Embedding shape    :", evidence_embeddings.shape)
print("FAISS vectors      :", faiss_index.ntotal)
print("Embedding dimension:", embedding_dimension)

print("\n" + "=" * 80)
print("M7 EVIDENCE RETRIEVAL INITIALIZATION COMPLETE")
print("=" * 80)

M7 — EVIDENCE RETRIEVAL INITIALIZATION

Device: cuda

Loading BGE model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

BGE model loaded.

Generating evidence embeddings...


Batches:   0%|          | 0/5 [00:00<?, ?it/s]


Evidence documents : 148
Embedding shape    : (148, 384)
FAISS vectors      : 148
Embedding dimension: 384

M7 EVIDENCE RETRIEVAL INITIALIZATION COMPLETE


In [12]:
# ============================================================
# M7 — BGE RETRIEVAL TEST
# Purpose:
# Test semantic retrieval for one factual MoM proposition.
#
# Note:
# BGE similarity indicates semantic relevance.
# It does NOT by itself prove that the evidence supports
# the proposition.
# ============================================================

proposition = (
    "The remote control is intended to be original."
)

results = retrieve_evidence(
    proposition,
    top_k=5
)

print("=" * 80)
print("M7 — BGE RETRIEVAL TEST")
print("=" * 80)

print("\nProposition:")
print(proposition)

print("\nTop retrieved evidence:")

for item in results:

    print("\n" + "-" * 80)
    print(
        f"Rank       : {item['retrieval_rank']}"
    )
    print(
        f"Similarity : {item['retrieval_similarity']:.4f}"
    )
    print(
        f"Evidence ID: {item['evidence_id']}"
    )
    print(
        f"Speaker    : {item['speaker']}"
    )
    print(
        f"Time       : {item['start']:.3f} - {item['end']:.3f}"
    )
    print(
        f"Text       : {item['text']}"
    )

print("\n" + "=" * 80)

M7 — BGE RETRIEVAL TEST

Proposition:
The remote control is intended to be original.

Top retrieved evidence:

--------------------------------------------------------------------------------
Rank       : 1
Similarity : 0.7917
Evidence ID: 105
Speaker    : SPEAKER_01
Time       : 707.248 - 710.889
Text       : remote controls. You want to integrate everything into one.

--------------------------------------------------------------------------------
Rank       : 2
Similarity : 0.7810
Evidence ID: 14
Speaker    : SPEAKER_02
Time       : 117.699 - 120.921
Text       : Now, we're developing a remote control, which you probably already know.

--------------------------------------------------------------------------------
Rank       : 3
Similarity : 0.7408
Evidence ID: 107
Speaker    : SPEAKER_03
Time       : 713.150 - 729.357
Text       : experience has only been given the remote control with the object I buy, not doing any tampering with it and programming, using it to program TV videos 

In [13]:
# ============================================================
# M7 — CONTEXT EXPANSION TEST
# Purpose:
# Check whether neighboring transcript segments around the
# retrieved evidence contain the specific information needed
# to verify the proposition.
#
# This is an experimental inspection step.
# No verification decision is made yet.
# ============================================================

proposition = (
    "The remote control is intended to be original."
)

# Retrieve the top evidence
results = retrieve_evidence(
    proposition,
    top_k=5
)

# Select the highest-ranked evidence
top_evidence_id = results[0]["evidence_id"]

# Convert to zero-based list position
top_index = top_evidence_id - 1

# Number of neighboring evidence segments to inspect
CONTEXT_WINDOW = 3

start_index = max(
    0,
    top_index - CONTEXT_WINDOW
)

end_index = min(
    len(evidence_documents),
    top_index + CONTEXT_WINDOW + 1
)

context_documents = evidence_documents[
    start_index:end_index
]

print("=" * 80)
print("M7 — CONTEXT EXPANSION TEST")
print("=" * 80)

print("\nProposition:")
print(proposition)

print("\nTop retrieved evidence:")
print(
    f"Evidence ID {results[0]['evidence_id']} "
    f"(similarity={results[0]['retrieval_similarity']:.4f})"
)

print("\nContext window:")
print(
    f"Evidence IDs {context_documents[0]['evidence_id']} "
    f"to {context_documents[-1]['evidence_id']}"
)

for item in context_documents:

    marker = (
        " <-- TOP RETRIEVED"
        if item["evidence_id"] == top_evidence_id
        else ""
    )

    print("\n" + "-" * 80)
    print(
        f"Evidence ID: {item['evidence_id']}{marker}"
    )
    print(
        f"Speaker    : {item['speaker']}"
    )
    print(
        f"Time       : {item['start']:.3f} - "
        f"{item['end']:.3f}"
    )
    print(
        f"Text       : {item['text']}"
    )

print("\n" + "=" * 80)

M7 — CONTEXT EXPANSION TEST

Proposition:
The remote control is intended to be original.

Top retrieved evidence:
Evidence ID 105 (similarity=0.7917)

Context window:
Evidence IDs 102 to 108

--------------------------------------------------------------------------------
Evidence ID: 102
Speaker    : SPEAKER_02
Time       : 682.011 - 704.502
Text       : we had three videos, a TV and a sort of amp thing all set up. So we got one of the universal remote controls that you program each of your things into. But that kept losing the signals, so we'd have to reprogram it every now and again. I think it was quite cheapy as well. So that might have had something to do with it. But that was quite good, the fact that you could

--------------------------------------------------------------------------------
Evidence ID: 103
Speaker    : SPEAKER_00
Time       : 705.387 - 706.668
Text       : use all the ones. You didn't have

----------------------------------------------------------------------

In [14]:
# ============================================================
# M7 — TOP-20 BGE RETRIEVAL TEST
# Purpose:
# Determine whether the correct evidence for an attribute
# appears within a larger semantic retrieval candidate pool.
# ============================================================

proposition = (
    "The remote control is intended to be original."
)

results = retrieve_evidence(
    proposition,
    top_k=20
)

print("=" * 80)
print("M7 — TOP-20 BGE RETRIEVAL TEST")
print("=" * 80)

print("\nProposition:")
print(proposition)

print("\nTop 20 retrieved evidence segments:")

for item in results:

    print("\n" + "-" * 80)

    print(
        f"Rank       : {item['retrieval_rank']}"
    )

    print(
        f"Similarity : {item['retrieval_similarity']:.4f}"
    )

    print(
        f"Evidence ID: {item['evidence_id']}"
    )

    print(
        f"Speaker    : {item['speaker']}"
    )

    print(
        f"Time       : {item['start']:.3f} - "
        f"{item['end']:.3f}"
    )

    print(
        f"Text       : {item['text']}"
    )

print("\n" + "=" * 80)

M7 — TOP-20 BGE RETRIEVAL TEST

Proposition:
The remote control is intended to be original.

Top 20 retrieved evidence segments:

--------------------------------------------------------------------------------
Rank       : 1
Similarity : 0.7917
Evidence ID: 105
Speaker    : SPEAKER_01
Time       : 707.248 - 710.889
Text       : remote controls. You want to integrate everything into one.

--------------------------------------------------------------------------------
Rank       : 2
Similarity : 0.7810
Evidence ID: 14
Speaker    : SPEAKER_02
Time       : 117.699 - 120.921
Text       : Now, we're developing a remote control, which you probably already know.

--------------------------------------------------------------------------------
Rank       : 3
Similarity : 0.7408
Evidence ID: 107
Speaker    : SPEAKER_03
Time       : 713.150 - 729.357
Text       : experience has only been given the remote control with the object I buy, not doing any tampering with it and programming, using it to

In [15]:
# ============================================================
# M7.4 — Attribute-Level Lexical Evidence Support
# ============================================================
#
# Purpose:
#   Identify whether important content words from a proposition
#   are explicitly present in retrieved evidence.
#
# Important:
#   This is NOT the final verification decision.
#   It is one signal that will later be combined with:
#       1. BGE retrieval
#       2. NLI
#       3. Speaker consistency
#       4. Timestamp consistency
#       5. Event-type consistency
#
# The function is intentionally lightweight and restart-safe.
# No model is loaded and no existing files are modified.
# ============================================================

import re

# Common English stopwords.
# These are ignored when calculating meaningful lexical overlap.
STOPWORDS = {
    "the", "a", "an", "and", "or", "but", "if", "then",
    "is", "are", "was", "were", "be", "been", "being",
    "to", "of", "in", "on", "for", "with", "by", "from",
    "at", "as", "into", "about", "over", "after", "before",
    "this", "that", "these", "those", "it", "its",
    "they", "them", "their", "there", "here",
    "he", "she", "we", "you", "i",
    "will", "would", "should", "could", "can", "may", "might",
    "do", "does", "did", "done",
    "have", "has", "had",
    "not", "no",
    "be", "being"
}


def normalize_tokens(text):
    """
    Convert text into lowercase content-word tokens.
    """
    text = text.lower()

    # Keep alphabetic words.
    tokens = re.findall(r"[a-z]+", text)

    # Remove stopwords and very short tokens.
    tokens = [
        token for token in tokens
        if token not in STOPWORDS and len(token) > 2
    ]

    return tokens


def lexical_support(proposition, evidence):
    """
    Calculate simple lexical support between a proposition
    and an evidence sentence.

    Returns:
        proposition_terms
        matched_terms
        missing_terms
        overlap_ratio
    """

    proposition_terms = list(dict.fromkeys(normalize_tokens(proposition)))
    evidence_terms = set(normalize_tokens(evidence))

    matched_terms = [
        term for term in proposition_terms
        if term in evidence_terms
    ]

    missing_terms = [
        term for term in proposition_terms
        if term not in evidence_terms
    ]

    if len(proposition_terms) == 0:
        overlap_ratio = 0.0
    else:
        overlap_ratio = len(matched_terms) / len(proposition_terms)

    return {
        "proposition_terms": proposition_terms,
        "matched_terms": matched_terms,
        "missing_terms": missing_terms,
        "overlap_ratio": round(overlap_ratio, 4)
    }


# ------------------------------------------------------------
# Controlled test using the known "original" proposition
# ------------------------------------------------------------

test_proposition = "The remote control is intended to be original."

test_evidence = (
    "We want it to be original. Trendy, appealing to a wide market."
)

result = lexical_support(test_proposition, test_evidence)

print("PROPOSITION:")
print(test_proposition)

print("\nEVIDENCE:")
print(test_evidence)

print("\nLEXICAL SUPPORT:")
print("Proposition terms :", result["proposition_terms"])
print("Matched terms     :", result["matched_terms"])
print("Missing terms     :", result["missing_terms"])
print("Overlap ratio     :", result["overlap_ratio"])

PROPOSITION:
The remote control is intended to be original.

EVIDENCE:
We want it to be original. Trendy, appealing to a wide market.

LEXICAL SUPPORT:
Proposition terms : ['remote', 'control', 'intended', 'original']
Matched terms     : ['original']
Missing terms     : ['remote', 'control', 'intended']
Overlap ratio     : 0.25


In [16]:
# ============================================================
# M7.4 — Lexical Support Diagnostic Tests
# ============================================================
#
# Purpose:
#   Test lexical overlap on:
#       1. Direct supporting evidence
#       2. Conversational/paraphrased evidence
#       3. Contradictory evidence
#       4. Unrelated evidence
#
# This is a diagnostic only.
# No verification threshold is introduced here.
# ============================================================

test_cases = [
    {
        "name": "Direct support",
        "proposition": "The remote control is intended to be original.",
        "evidence": "The remote control is intended to be original."
    },
    {
        "name": "Conversational support",
        "proposition": "The remote control is intended to be original.",
        "evidence": "We want it to be original. Trendy, appealing to a wide market."
    },
    {
        "name": "Contradictory evidence",
        "proposition": "The remote control is intended to be original.",
        "evidence": "We don't want it to be original. We want something similar to existing products."
    },
    {
        "name": "Unrelated evidence",
        "proposition": "The remote control is intended to be original.",
        "evidence": "The meeting will take place next week and everyone should attend."
    }
]


for i, case in enumerate(test_cases, start=1):

    result = lexical_support(
        case["proposition"],
        case["evidence"]
    )

    print("=" * 70)
    print(f"TEST {i}: {case['name']}")
    print("-" * 70)

    print("Proposition:")
    print(case["proposition"])

    print("\nEvidence:")
    print(case["evidence"])

    print("\nMatched terms:")
    print(result["matched_terms"])

    print("Missing terms:")
    print(result["missing_terms"])

    print("Overlap ratio:")
    print(result["overlap_ratio"])

print("=" * 70)

TEST 1: Direct support
----------------------------------------------------------------------
Proposition:
The remote control is intended to be original.

Evidence:
The remote control is intended to be original.

Matched terms:
['remote', 'control', 'intended', 'original']
Missing terms:
[]
Overlap ratio:
1.0
TEST 2: Conversational support
----------------------------------------------------------------------
Proposition:
The remote control is intended to be original.

Evidence:
We want it to be original. Trendy, appealing to a wide market.

Matched terms:
['original']
Missing terms:
['remote', 'control', 'intended']
Overlap ratio:
0.25
TEST 3: Contradictory evidence
----------------------------------------------------------------------
Proposition:
The remote control is intended to be original.

Evidence:
We don't want it to be original. We want something similar to existing products.

Matched terms:
['original']
Missing terms:
['remote', 'control', 'intended']
Overlap ratio:
0.25
TES

In [19]:
# ============================================================
# M7.5 — Contextual Evidence Window
# ============================================================
#
# Purpose:
#   Return nearby transcript evidence around a selected
#   evidence document.
#
# Evidence IDs in the current saved data are INTEGER values.
# ============================================================

def get_context_window(evidence_id, window_size=2):
    """
    Return nearby evidence documents around evidence_id.

    Parameters
    ----------
    evidence_id : int
        Integer evidence ID.

    window_size : int
        Number of evidence documents before and after
        the selected document.

    Returns
    -------
    list
        Chronological context window.
    """

    # Find the selected evidence document
    target_index = None

    for i, doc in enumerate(evidence_documents):

        if int(doc["evidence_id"]) == int(evidence_id):
            target_index = i
            break

    if target_index is None:

        available_ids = [
            doc["evidence_id"]
            for doc in evidence_documents[:10]
        ]

        raise ValueError(
            f"Evidence ID {evidence_id} not found.\n"
            f"Example available IDs: {available_ids}"
        )

    # Determine context boundaries
    start_index = max(
        0,
        target_index - window_size
    )

    end_index = min(
        len(evidence_documents),
        target_index + window_size + 1
    )

    # Return chronological context
    return evidence_documents[start_index:end_index]


# ------------------------------------------------------------
# Test with the actual evidence ID
# ------------------------------------------------------------

test_evidence_id = 15

context = get_context_window(
    test_evidence_id,
    window_size=2
)

print("=" * 70)
print(f"CONTEXT WINDOW AROUND EVIDENCE {test_evidence_id}")
print("=" * 70)

for doc in context:

    print(
        f"Evidence {doc['evidence_id']} | "
        f"{doc['speaker']} | "
        f"{doc['start']:.3f} → {doc['end']:.3f}"
    )

    print(f"  {doc['text']}")
    print()

CONTEXT WINDOW AROUND EVIDENCE 15
Evidence 13 | SPEAKER_02 | 112.185 → 115.748
  And we've got 25 minutes to do that as far as I can understand.

Evidence 14 | SPEAKER_02 | 117.699 → 120.921
  Now, we're developing a remote control, which you probably already know.

Evidence 15 | SPEAKER_02 | 122.503 → 132.290
  We want it to be original, something that people haven't thought of. It's not out in the shops. Trendy, appealing to a wide market, but,

Evidence 16 | SPEAKER_02 | 133.291 → 138.174
  you know, not a hunk of metal. And user-friendly, grannies to kids,

Evidence 17 | SPEAKER_02 | 139.375 → 141.016
  maybe even pooches, should be able to use it.



In [20]:
# ============================================================
# M7.6 — Contextual NLI Verification Test
# ============================================================
#
# Purpose:
#   Compare NLI performance using:
#
#       A. Single evidence utterance
#       B. Expanded contextual evidence
#
# This is a diagnostic experiment.
# It does NOT yet define the final verification rule.
# ============================================================


# ------------------------------------------------------------
# Proposition to verify
# ------------------------------------------------------------

proposition = (
    "The remote control is intended to be original."
)


# ------------------------------------------------------------
# Single evidence
# ------------------------------------------------------------

single_evidence = (
    "We want it to be original. "
    "something that people haven't thought of. "
    "It's not out in the shops. "
    "Trendy, appealing to a wide market, but,"
)


# ------------------------------------------------------------
# Contextual evidence
# ------------------------------------------------------------

context_evidence = " ".join(
    doc["text"]
    for doc in context
)


# ------------------------------------------------------------
# Run NLI on single evidence
# ------------------------------------------------------------

single_result = run_nli(
    single_evidence,
    proposition
)


# ------------------------------------------------------------
# Run NLI on contextual evidence
# ------------------------------------------------------------

context_result = run_nli(
    context_evidence,
    proposition
)


# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print("=" * 70)
print("PROPOSITION")
print("=" * 70)
print(proposition)


print("\n" + "=" * 70)
print("SINGLE EVIDENCE")
print("=" * 70)
print(single_evidence)

print("\nNLI RESULT:")
print(single_result)


print("\n" + "=" * 70)
print("CONTEXTUAL EVIDENCE")
print("=" * 70)
print(context_evidence)

print("\nNLI RESULT:")
print(context_result)

PROPOSITION
The remote control is intended to be original.

SINGLE EVIDENCE
We want it to be original. something that people haven't thought of. It's not out in the shops. Trendy, appealing to a wide market, but,

NLI RESULT:
{'predicted_label': 'NEUTRAL', 'confidence': 0.9994945526123047, 'contradiction_probability': 0.0003343396238051355, 'entailment_probability': 0.0001710433280095458, 'neutral_probability': 0.9994945526123047}

CONTEXTUAL EVIDENCE
And we've got 25 minutes to do that as far as I can understand. Now, we're developing a remote control, which you probably already know. We want it to be original, something that people haven't thought of. It's not out in the shops. Trendy, appealing to a wide market, but, you know, not a hunk of metal. And user-friendly, grannies to kids, maybe even pooches, should be able to use it.

NLI RESULT:
{'predicted_label': 'ENTAILMENT', 'confidence': 0.9787161946296692, 'contradiction_probability': 0.0013938040938228369, 'entailment_probability

In [21]:
# ============================================================
# M7.7 — Contextual NLI Controlled Evaluation
# ============================================================
#
# Purpose:
#   Evaluate DeBERTa NLI on three controlled cases:
#
#       1. Supporting evidence
#       2. Contradictory evidence
#       3. Unrelated evidence
#
# This helps us understand whether NLI can safely
# participate in the final verification pipeline.
#
# NOTE:
#   We will NOT use these results to create arbitrary
#   thresholds yet.
# ============================================================


controlled_cases = [

    {
        "name": "SUPPORTING",
        "evidence": (
            "Now, we're developing a remote control, "
            "which you probably already know. "
            "We want it to be original, "
            "something that people haven't thought of."
        ),
        "proposition": (
            "The remote control is intended to be original."
        )
    },

    {
        "name": "CONTRADICTORY",
        "evidence": (
            "Now, we're developing a remote control, "
            "which you probably already know. "
            "We don't want it to be original. "
            "We want something similar to existing products."
        ),
        "proposition": (
            "The remote control is intended to be original."
        )
    },

    {
        "name": "UNRELATED",
        "evidence": (
            "The meeting will take place next week. "
            "Everyone should attend the next meeting."
        ),
        "proposition": (
            "The remote control is intended to be original."
        )
    }
]


print("=" * 70)
print("CONTEXTUAL NLI CONTROLLED EVALUATION")
print("=" * 70)


for i, case in enumerate(controlled_cases, start=1):

    result = run_nli(
        case["evidence"],
        case["proposition"]
    )

    print("\n" + "=" * 70)
    print(f"TEST {i}: {case['name']}")
    print("-" * 70)

    print("Evidence:")
    print(case["evidence"])

    print("\nProposition:")
    print(case["proposition"])

    print("\nNLI Result:")
    print(
        f"Predicted label : {result['predicted_label']}"
    )

    print(
        f"Contradiction   : "
        f"{result['contradiction_probability']:.4f}"
    )

    print(
        f"Entailment      : "
        f"{result['entailment_probability']:.4f}"
    )

    print(
        f"Neutral         : "
        f"{result['neutral_probability']:.4f}"
    )

print("\n" + "=" * 70)
print("Evaluation complete.")
print("=" * 70)

CONTEXTUAL NLI CONTROLLED EVALUATION

TEST 1: SUPPORTING
----------------------------------------------------------------------
Evidence:
Now, we're developing a remote control, which you probably already know. We want it to be original, something that people haven't thought of.

Proposition:
The remote control is intended to be original.

NLI Result:
Predicted label : ENTAILMENT
Contradiction   : 0.0086
Entailment      : 0.9643
Neutral         : 0.0270

TEST 2: CONTRADICTORY
----------------------------------------------------------------------
Evidence:
Now, we're developing a remote control, which you probably already know. We don't want it to be original. We want something similar to existing products.

Proposition:
The remote control is intended to be original.

NLI Result:
Predicted label : CONTRADICTION
Contradiction   : 0.9972
Entailment      : 0.0006
Neutral         : 0.0021

TEST 3: UNRELATED
----------------------------------------------------------------------
Evidence:
The

In [22]:
# ============================================================
# M7.8 — Real Meeting Attribute-Level NLI Evaluation
# ============================================================
#
# Purpose:
#   Test contextual NLI on real attributes extracted from
#   Claim 2 of the meeting.
#
# Claim 2:
#   "The team is developing a remote control intended to be
#    original, trendy, appealing to a wide market, and
#    user-friendly for a broad range of users."
#
# We evaluate each attribute separately using its known
# source evidence.
#
# This is an evaluation step only.
# It does NOT modify any saved project files.
# ============================================================


# ------------------------------------------------------------
# Claim 2 attributes
# ------------------------------------------------------------

claim_2_attributes = [
    {
        "attribute": "product_development",
        "proposition": "The team is developing a remote control.",
        "source_ids": [14]
    },
    {
        "attribute": "originality",
        "proposition": "The remote control is intended to be original.",
        "source_ids": [15]
    },
    {
        "attribute": "trendiness",
        "proposition": "The remote control is intended to be trendy.",
        "source_ids": [15]
    },
    {
        "attribute": "market_appeal",
        "proposition": "The remote control is intended to appeal to a wide market.",
        "source_ids": [15]
    },
    {
        "attribute": "user_friendliness",
        "proposition": "The remote control is intended to be user-friendly.",
        "source_ids": [16]
    },
    {
        "attribute": "broad_usability",
        "proposition": "The remote control should be usable by a broad range of users.",
        "source_ids": [16, 17]
    }
]


# ------------------------------------------------------------
# Build contextual evidence for each attribute
# ------------------------------------------------------------

print("=" * 80)
print("REAL MEETING ATTRIBUTE-LEVEL NLI EVALUATION")
print("=" * 80)


attribute_results = []


for item in claim_2_attributes:

    # Use the first source evidence as the center of the
    # contextual window.
    source_id = item["source_ids"][0]

    context_docs = get_context_window(
        source_id,
        window_size=2
    )

    context_text = " ".join(
        doc["text"]
        for doc in context_docs
    )

    # Run contextual NLI
    nli_result = run_nli(
        context_text,
        item["proposition"]
    )

    result = {
        "attribute": item["attribute"],
        "proposition": item["proposition"],
        "source_ids": item["source_ids"],
        "nli_label": nli_result["predicted_label"],
        "entailment_probability": nli_result["entailment_probability"],
        "contradiction_probability": nli_result["contradiction_probability"],
        "neutral_probability": nli_result["neutral_probability"]
    }

    attribute_results.append(result)

    print("\n" + "-" * 80)
    print(f"ATTRIBUTE: {item['attribute']}")
    print("-" * 80)

    print("Proposition:")
    print(item["proposition"])

    print("\nSource evidence IDs:")
    print(item["source_ids"])

    print("\nContext:")
    print(context_text)

    print("\nNLI:")
    print(f"Label         : {nli_result['predicted_label']}")
    print(
        f"Entailment    : "
        f"{nli_result['entailment_probability']:.4f}"
    )
    print(
        f"Contradiction : "
        f"{nli_result['contradiction_probability']:.4f}"
    )
    print(
        f"Neutral       : "
        f"{nli_result['neutral_probability']:.4f}"
    )


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("ATTRIBUTE-LEVEL SUMMARY")
print("=" * 80)

for result in attribute_results:

    print(
        f"{result['attribute']:22s} | "
        f"{result['nli_label']:13s} | "
        f"Entailment = "
        f"{result['entailment_probability']:.4f}"
    )

REAL MEETING ATTRIBUTE-LEVEL NLI EVALUATION

--------------------------------------------------------------------------------
ATTRIBUTE: product_development
--------------------------------------------------------------------------------
Proposition:
The team is developing a remote control.

Source evidence IDs:
[14]

Context:
talk about the project plan, discuss our own ideas and everything. And we've got 25 minutes to do that as far as I can understand. Now, we're developing a remote control, which you probably already know. We want it to be original, something that people haven't thought of. It's not out in the shops. Trendy, appealing to a wide market, but, you know, not a hunk of metal. And user-friendly, grannies to kids,

NLI:
Label         : ENTAILMENT
Entailment    : 0.9799
Contradiction : 0.0013
Neutral       : 0.0188

--------------------------------------------------------------------------------
ATTRIBUTE: originality
-------------------------------------------------------

In [23]:
# ============================================================
# M7.9 — Multi-Source Evidence NLI
# ============================================================
#
# Purpose:
#   Test whether combining all known source utterances
#   improves NLI for an attribute.
#
# This is especially important for attributes whose meaning
# is distributed across multiple conversational utterances.
#
# No files are modified.
# ============================================================


# ------------------------------------------------------------
# Attribute under investigation
# ------------------------------------------------------------

proposition = (
    "The remote control should be usable by a broad range "
    "of users."
)

source_ids = [16, 17]


# ------------------------------------------------------------
# Retrieve the exact source evidence
# ------------------------------------------------------------

source_documents = []

for source_id in source_ids:

    for doc in evidence_documents:

        if int(doc["evidence_id"]) == int(source_id):
            source_documents.append(doc)
            break


# ------------------------------------------------------------
# Build evidence from the exact source utterances
# ------------------------------------------------------------

exact_source_text = " ".join(
    doc["text"]
    for doc in source_documents
)


# ------------------------------------------------------------
# Run NLI on exact source evidence
# ------------------------------------------------------------

exact_result = run_nli(
    exact_source_text,
    proposition
)


# ------------------------------------------------------------
# Also create a slightly larger context around ALL sources
# ------------------------------------------------------------

all_context_documents = []

for source_id in source_ids:

    context_docs = get_context_window(
        source_id,
        window_size=2
    )

    for doc in context_docs:

        if doc["evidence_id"] not in [
            d["evidence_id"] for d in all_context_documents
        ]:
            all_context_documents.append(doc)


# Sort chronologically
all_context_documents = sorted(
    all_context_documents,
    key=lambda x: x["start"]
)


combined_context_text = " ".join(
    doc["text"]
    for doc in all_context_documents
)


# ------------------------------------------------------------
# Run NLI on combined context
# ------------------------------------------------------------

combined_result = run_nli(
    combined_context_text,
    proposition
)


# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print("=" * 80)
print("MULTI-SOURCE EVIDENCE NLI TEST")
print("=" * 80)

print("\nPROPOSITION:")
print(proposition)


print("\n" + "-" * 80)
print("EXACT SOURCE EVIDENCE")
print("-" * 80)

for doc in source_documents:

    print(
        f"Evidence {doc['evidence_id']} | "
        f"{doc['speaker']} | "
        f"{doc['start']:.3f} → {doc['end']:.3f}"
    )

    print(doc["text"])
    print()


print("NLI RESULT:")
print(exact_result)


print("\n" + "-" * 80)
print("COMBINED CONTEXT")
print("-" * 80)

for doc in all_context_documents:

    print(
        f"Evidence {doc['evidence_id']} | "
        f"{doc['speaker']} | "
        f"{doc['start']:.3f} → {doc['end']:.3f}"
    )

    print(doc["text"])
    print()


print("NLI RESULT:")
print(combined_result)

MULTI-SOURCE EVIDENCE NLI TEST

PROPOSITION:
The remote control should be usable by a broad range of users.

--------------------------------------------------------------------------------
EXACT SOURCE EVIDENCE
--------------------------------------------------------------------------------
Evidence 16 | SPEAKER_02 | 133.291 → 138.174
you know, not a hunk of metal. And user-friendly, grannies to kids,

Evidence 17 | SPEAKER_02 | 139.375 → 141.016
maybe even pooches, should be able to use it.

NLI RESULT:
{'predicted_label': 'NEUTRAL', 'confidence': 0.9989136457443237, 'contradiction_probability': 0.00027211118140257895, 'entailment_probability': 0.0008143104496411979, 'neutral_probability': 0.9989136457443237}

--------------------------------------------------------------------------------
COMBINED CONTEXT
--------------------------------------------------------------------------------
Evidence 14 | SPEAKER_02 | 117.699 → 120.921
Now, we're developing a remote control, which you prob

In [24]:
# ============================================================
# M7.10 — BGE Semantic Evidence Support Test
# ============================================================
#
# Purpose:
#   Test whether BGE semantic similarity can recognize
#   conversational/paraphrased evidence that NLI failed
#   to recognize.
#
# Example:
#
# Proposition:
#   "The remote control should be usable by a broad range
#    of users."
#
# Evidence:
#   "grannies to kids, maybe even pooches, should be able
#    to use it."
#
# BGE similarity is treated as a retrieval/support signal,
# NOT as proof of truth.
#
# No files are modified.
# ============================================================


# ------------------------------------------------------------
# Proposition
# ------------------------------------------------------------

proposition = (
    "The remote control should be usable by a broad range "
    "of users."
)


# ------------------------------------------------------------
# Exact evidence documents
# ------------------------------------------------------------

source_ids = [16, 17]

source_documents = []

for source_id in source_ids:

    for doc in evidence_documents:

        if int(doc["evidence_id"]) == int(source_id):

            source_documents.append(doc)
            break


# ------------------------------------------------------------
# Combine the source evidence
# ------------------------------------------------------------

source_text = " ".join(
    doc["text"]
    for doc in source_documents
)


# ------------------------------------------------------------
# Generate proposition embedding
# ------------------------------------------------------------

proposition_embedding = bge_model.encode(
    [proposition],
    normalize_embeddings=True,
    convert_to_numpy=True
)


# ------------------------------------------------------------
# Generate evidence embedding
# ------------------------------------------------------------

evidence_embedding = bge_model.encode(
    [source_text],
    normalize_embeddings=True,
    convert_to_numpy=True
)


# ------------------------------------------------------------
# Cosine similarity
#
# Because both embeddings are normalized,
# dot product = cosine similarity.
# ------------------------------------------------------------

similarity = float(
    proposition_embedding[0] @ evidence_embedding[0]
)


# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print("=" * 80)
print("BGE SEMANTIC EVIDENCE SUPPORT TEST")
print("=" * 80)

print("\nPROPOSITION:")
print(proposition)

print("\nEVIDENCE:")
print(source_text)

print("\nBGE COSINE SIMILARITY:")
print(f"{similarity:.4f}")

print("\nInterpretation:")
print(
    "This score is a semantic similarity signal only. "
    "It is NOT being used as proof or as a verification "
    "threshold at this stage."
)

BGE SEMANTIC EVIDENCE SUPPORT TEST

PROPOSITION:
The remote control should be usable by a broad range of users.

EVIDENCE:
you know, not a hunk of metal. And user-friendly, grannies to kids, maybe even pooches, should be able to use it.

BGE COSINE SIMILARITY:
0.5906

Interpretation:
This score is a semantic similarity signal only. It is NOT being used as proof or as a verification threshold at this stage.


In [25]:
# ============================================================
# M7.11 — Speaker & Timestamp Consistency Test
# ============================================================
#
# Purpose:
#   Test deterministic metadata consistency between a
#   generated MoM claim and its supporting evidence.
#
# These checks do NOT require another ML model.
#
# Speaker:
#   Exact comparison with evidence speaker.
#
# Timestamp:
#   Evidence timestamps are taken directly from the
#   speaker-attributed transcript.
#
# This is a diagnostic step only.
# ============================================================


def check_speaker_consistency(claim_speaker, evidence_speaker):
    """
    Check whether the claimed speaker matches the evidence speaker.
    """

    if claim_speaker is None:
        return {
            "applicable": False,
            "passed": True,
            "reason": "No speaker specified in claim."
        }

    if evidence_speaker is None:
        return {
            "applicable": False,
            "passed": True,
            "reason": "Evidence speaker unavailable."
        }

    passed = (
        str(claim_speaker).strip().upper()
        ==
        str(evidence_speaker).strip().upper()
    )

    return {
        "applicable": True,
        "passed": passed,
        "reason": (
            "Speaker matches evidence."
            if passed
            else "Speaker does not match evidence."
        )
    }


def check_timestamp_consistency(
    evidence_start,
    evidence_end,
    expected_start=None,
    expected_end=None,
    tolerance_seconds=2.0
):
    """
    Check timestamp consistency.

    If no expected timestamp is provided, the evidence timestamp
    itself is considered valid and is simply returned.

    If expected timestamps are available, allow a small tolerance.
    """

    # No expected timestamp means we use the evidence timestamp
    if expected_start is None and expected_end is None:
        return {
            "applicable": False,
            "passed": True,
            "reason": "Evidence timestamp accepted directly.",
            "evidence_start": evidence_start,
            "evidence_end": evidence_end
        }

    start_ok = True
    end_ok = True

    if expected_start is not None:
        start_ok = abs(
            float(evidence_start) - float(expected_start)
        ) <= tolerance_seconds

    if expected_end is not None:
        end_ok = abs(
            float(evidence_end) - float(expected_end)
        ) <= tolerance_seconds

    passed = start_ok and end_ok

    return {
        "applicable": True,
        "passed": passed,
        "reason": (
            "Timestamp is consistent."
            if passed
            else "Timestamp is inconsistent."
        ),
        "evidence_start": evidence_start,
        "evidence_end": evidence_end
    }


# ------------------------------------------------------------
# Test 1 — Correct speaker
# ------------------------------------------------------------

evidence = next(
    doc for doc in evidence_documents
    if int(doc["evidence_id"]) == 15
)

correct_speaker_result = check_speaker_consistency(
    "SPEAKER_02",
    evidence["speaker"]
)

correct_time_result = check_timestamp_consistency(
    evidence["start"],
    evidence["end"]
)


# ------------------------------------------------------------
# Test 2 — Incorrect speaker
# ------------------------------------------------------------

wrong_speaker_result = check_speaker_consistency(
    "SPEAKER_03",
    evidence["speaker"]
)


# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print("=" * 80)
print("SPEAKER & TIMESTAMP CONSISTENCY TEST")
print("=" * 80)

print("\nEvidence:")
print(f"Evidence ID : {evidence['evidence_id']}")
print(f"Speaker     : {evidence['speaker']}")
print(
    f"Timestamp   : "
    f"{evidence['start']:.3f} → {evidence['end']:.3f}"
)
print(f"Text        : {evidence['text']}")


print("\n" + "-" * 80)
print("TEST 1 — CORRECT SPEAKER")
print("-" * 80)

print("Claim speaker   : SPEAKER_02")
print("Evidence speaker:", evidence["speaker"])
print("Result          :", correct_speaker_result)


print("\n" + "-" * 80)
print("TEST 2 — INCORRECT SPEAKER")
print("-" * 80)

print("Claim speaker   : SPEAKER_03")
print("Evidence speaker:", evidence["speaker"])
print("Result          :", wrong_speaker_result)


print("\n" + "-" * 80)
print("TIMESTAMP")
print("-" * 80)

print("Result:", correct_time_result)

SPEAKER & TIMESTAMP CONSISTENCY TEST

Evidence:
Evidence ID : 15
Speaker     : SPEAKER_02
Timestamp   : 122.503 → 132.290
Text        : We want it to be original, something that people haven't thought of. It's not out in the shops. Trendy, appealing to a wide market, but,

--------------------------------------------------------------------------------
TEST 1 — CORRECT SPEAKER
--------------------------------------------------------------------------------
Claim speaker   : SPEAKER_02
Evidence speaker: SPEAKER_02
Result          : {'applicable': True, 'passed': True, 'reason': 'Speaker matches evidence.'}

--------------------------------------------------------------------------------
TEST 2 — INCORRECT SPEAKER
--------------------------------------------------------------------------------
Claim speaker   : SPEAKER_03
Evidence speaker: SPEAKER_02
Result          : {'applicable': True, 'passed': False, 'reason': 'Speaker does not match evidence.'}

------------------------------------

In [26]:
# ============================================================
# M7.12 — Attribute Verification Result Structure
# ============================================================
#
# Purpose:
#   Define a standardized structure for storing verification
#   results for every MoM attribute.
#
# This structure will later be used by:
#
#   M7  → Evidence Verification
#   M8  → Verified MoM Generation
#   M9  → Dashboard
#   M11 → Evaluation
#
# No model inference is performed here.
# No existing files are modified.
# ============================================================

from dataclasses import dataclass, asdict
from typing import Optional, List


@dataclass
class AttributeVerificationResult:
    """
    Stores the verification result of one MoM attribute.
    """

    attribute: str

    proposition: str

    evidence_ids: List[int]

    evidence_start: Optional[float]

    evidence_end: Optional[float]

    evidence_speakers: List[str]

    bge_similarity: Optional[float]

    nli_label: Optional[str]

    nli_entailment: Optional[float]

    nli_contradiction: Optional[float]

    nli_neutral: Optional[float]

    lexical_overlap: Optional[float]

    speaker_check: Optional[bool]

    timestamp_check: Optional[bool]

    content_supported: Optional[bool]

    final_status: str

    verification_reason: str


# ------------------------------------------------------------
# Create a small example
# ------------------------------------------------------------

example_result = AttributeVerificationResult(

    attribute="originality",

    proposition=(
        "The remote control is intended to be original."
    ),

    evidence_ids=[15],

    evidence_start=122.503,

    evidence_end=132.290,

    evidence_speakers=["SPEAKER_02"],

    bge_similarity=0.0,

    nli_label="ENTAILMENT",

    nli_entailment=0.9787,

    nli_contradiction=0.0014,

    nli_neutral=0.0199,

    lexical_overlap=0.25,

    speaker_check=True,

    timestamp_check=True,

    content_supported=True,

    final_status="VERIFIED",

    verification_reason=(
        "Evidence context supports the proposition."
    )
)


# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print("=" * 80)
print("ATTRIBUTE VERIFICATION RESULT STRUCTURE")
print("=" * 80)

print()

for key, value in asdict(example_result).items():
    print(f"{key:25s}: {value}")

ATTRIBUTE VERIFICATION RESULT STRUCTURE

attribute                : originality
proposition              : The remote control is intended to be original.
evidence_ids             : [15]
evidence_start           : 122.503
evidence_end             : 132.29
evidence_speakers        : ['SPEAKER_02']
bge_similarity           : 0.0
nli_label                : ENTAILMENT
nli_entailment           : 0.9787
nli_contradiction        : 0.0014
nli_neutral              : 0.0199
lexical_overlap          : 0.25
speaker_check            : True
timestamp_check          : True
content_supported        : True
final_status             : VERIFIED
verification_reason      : Evidence context supports the proposition.


In [27]:
# ============================================================
# M7.13 — Integrated Attribute Evidence Analysis
# ============================================================
#
# Purpose:
#   Combine the existing verification components for ONE
#   attribute:
#
#       1. BGE + FAISS retrieval
#       2. Context expansion
#       3. Contextual NLI
#       4. Lexical support
#       5. Speaker consistency
#       6. Evidence timestamps
#
# IMPORTANT:
#   This cell intentionally DOES NOT make the final
#   VERIFIED / FLAGGED decision.
#
#   It produces all relevant signals first.
#   We will use real results to design the final
#   aggregation rule.
#
# No existing files are modified.
# ============================================================


def analyze_attribute_evidence(
    proposition,
    claim_speaker=None,
    top_k=10,
    context_window=2
):
    """
    Analyze evidence for one MoM attribute.

    Parameters
    ----------
    proposition : str
        Attribute proposition to verify.

    claim_speaker : str or None
        Speaker specified by the generated MoM claim.

    top_k : int
        Number of BGE evidence candidates to retrieve.

    context_window : int
        Number of neighboring utterances on each side.

    Returns
    -------
    dict
        Retrieval and verification signals.
    """

    # --------------------------------------------------------
    # 1. Retrieve candidate evidence using BGE + FAISS
    # --------------------------------------------------------

    retrieved = retrieve_evidence(
        proposition,
        top_k=top_k
    )

    if len(retrieved) == 0:
        return {
            "proposition": proposition,
            "claim_speaker": claim_speaker,
            "retrieved_candidates": [],
            "message": "No evidence candidates retrieved."
        }


    # --------------------------------------------------------
    # 2. Analyze each retrieved candidate
    # --------------------------------------------------------

    analyzed_candidates = []


    for candidate in retrieved:

        evidence_id = int(candidate["evidence_id"])

        # Find the actual evidence document
        evidence_doc = next(
            (
                doc for doc in evidence_documents
                if int(doc["evidence_id"]) == evidence_id
            ),
            None
        )

        if evidence_doc is None:
            continue


        # ----------------------------------------------------
        # Context expansion
        # ----------------------------------------------------

        context_docs = get_context_window(
            evidence_id,
            window_size=context_window
        )

        context_text = " ".join(
            doc["text"]
            for doc in context_docs
        )


        # ----------------------------------------------------
        # Contextual NLI
        # ----------------------------------------------------

        nli_result = run_nli(
            context_text,
            proposition
        )


        # ----------------------------------------------------
        # Lexical support
        # ----------------------------------------------------

        lexical_result = lexical_support(
            proposition,
            context_text
        )


        # ----------------------------------------------------
        # Speaker consistency
        # ----------------------------------------------------

        speaker_result = check_speaker_consistency(
            claim_speaker,
            evidence_doc["speaker"]
        )


        # ----------------------------------------------------
        # Store all signals
        # ----------------------------------------------------

        analyzed_candidates.append({

            "evidence_id": evidence_id,

            "speaker": evidence_doc["speaker"],

            "start": evidence_doc["start"],

            "end": evidence_doc["end"],

            "text": evidence_doc["text"],

            "retrieval_similarity": candidate[
                "retrieval_similarity"
            ],

            "nli_label": nli_result[
                "predicted_label"
            ],

            "nli_entailment": nli_result[
                "entailment_probability"
            ],

            "nli_contradiction": nli_result[
                "contradiction_probability"
            ],

            "nli_neutral": nli_result[
                "neutral_probability"
            ],

            "lexical_overlap": lexical_result[
                "overlap_ratio"
            ],

            "matched_terms": lexical_result[
                "matched_terms"
            ],

            "missing_terms": lexical_result[
                "missing_terms"
            ],

            "speaker_check": speaker_result,

            "context_evidence_ids": [
                int(doc["evidence_id"])
                for doc in context_docs
            ]
        })


    # --------------------------------------------------------
    # 3. Return complete analysis
    # --------------------------------------------------------

    return {

        "proposition": proposition,

        "claim_speaker": claim_speaker,

        "retrieved_candidates": analyzed_candidates
    }


# ============================================================
# Test on the originality attribute
# ============================================================

test_proposition = (
    "The remote control is intended to be original."
)

analysis_result = analyze_attribute_evidence(
    proposition=test_proposition,
    claim_speaker="SPEAKER_02",
    top_k=10,
    context_window=2
)


# ------------------------------------------------------------
# Display compact results
# ------------------------------------------------------------

print("=" * 90)
print("INTEGRATED ATTRIBUTE EVIDENCE ANALYSIS")
print("=" * 90)

print("\nProposition:")
print(analysis_result["proposition"])

print("\nClaim speaker:")
print(analysis_result["claim_speaker"])

print("\n" + "-" * 90)
print("RETRIEVED EVIDENCE CANDIDATES")
print("-" * 90)


for rank, candidate in enumerate(
    analysis_result["retrieved_candidates"],
    start=1
):

    print(
        f"\nRank {rank} | "
        f"Evidence {candidate['evidence_id']}"
    )

    print(
        f"Speaker: {candidate['speaker']} | "
        f"Time: "
        f"{candidate['start']:.3f} → "
        f"{candidate['end']:.3f}"
    )

    print(
        f"BGE similarity: "
        f"{candidate['retrieval_similarity']:.4f}"
    )

    print(
        f"NLI: {candidate['nli_label']} | "
        f"Entailment: "
        f"{candidate['nli_entailment']:.4f} | "
        f"Contradiction: "
        f"{candidate['nli_contradiction']:.4f} | "
        f"Neutral: "
        f"{candidate['nli_neutral']:.4f}"
    )

    print(
        f"Lexical overlap: "
        f"{candidate['lexical_overlap']:.4f}"
    )

    print(
        f"Matched terms: "
        f"{candidate['matched_terms']}"
    )

    print(
        f"Speaker check: "
        f"{candidate['speaker_check']}"
    )

    print(
        f"Context IDs: "
        f"{candidate['context_evidence_ids']}"
    )

    print(
        f"Evidence text: "
        f"{candidate['text']}"
    )


print("\n" + "=" * 90)
print("Analysis complete — no final verification decision made.")
print("=" * 90)

INTEGRATED ATTRIBUTE EVIDENCE ANALYSIS

Proposition:
The remote control is intended to be original.

Claim speaker:
SPEAKER_02

------------------------------------------------------------------------------------------
RETRIEVED EVIDENCE CANDIDATES
------------------------------------------------------------------------------------------

Rank 1 | Evidence 105
Speaker: SPEAKER_01 | Time: 707.248 → 710.889
BGE similarity: 0.7917
NLI: NEUTRAL | Entailment: 0.0013 | Contradiction: 0.0053 | Neutral: 0.9934
Lexical overlap: 0.5000
Matched terms: ['remote', 'control']
Speaker check: {'applicable': True, 'passed': False, 'reason': 'Speaker does not match evidence.'}
Context IDs: [103, 104, 105, 106, 107]
Evidence text: remote controls. You want to integrate everything into one.

Rank 2 | Evidence 14
Speaker: SPEAKER_02 | Time: 117.699 → 120.921
BGE similarity: 0.7810
NLI: ENTAILMENT | Entailment: 0.9812 | Contradiction: 0.0012 | Neutral: 0.0177
Lexical overlap: 0.7500
Matched terms: ['remote'

In [28]:
# ============================================================
# M7.14 — Evidence Candidate Selection
# ============================================================
#
# Purpose:
#   Select plausible supporting evidence from the integrated
#   evidence analysis.
#
# Selection principles:
#
#   1. Contradiction is not supporting evidence.
#   2. Neutral evidence is not treated as support.
#   3. Entailment is the strongest semantic signal.
#   4. Lexical overlap provides additional support.
#   5. Speaker mismatch is rejected when a speaker is claimed.
#   6. BGE similarity is used for retrieval, not proof.
#
# IMPORTANT:
#   No arbitrary numerical weighting is used yet.
#   No final VERIFIED / FLAGGED decision is made.
# ============================================================


def select_supporting_candidates(
    analysis_result,
    min_lexical_overlap=0.0
):
    """
    Select plausible supporting evidence candidates.

    Parameters
    ----------
    analysis_result : dict
        Output from analyze_attribute_evidence().

    min_lexical_overlap : float
        Optional minimum lexical overlap.
        Default is 0 because lexical overlap is only
        a supporting signal and not mandatory for
        semantic entailment.

    Returns
    -------
    list
        Supporting evidence candidates.
    """

    candidates = analysis_result.get(
        "retrieved_candidates",
        []
    )

    supporting = []
    rejected = []


    for candidate in candidates:

        nli_label = candidate["nli_label"]

        speaker_check = candidate["speaker_check"]

        lexical_overlap = candidate["lexical_overlap"]


        # ----------------------------------------------------
        # Reject explicit contradiction
        # ----------------------------------------------------

        if nli_label == "CONTRADICTION":

            rejected.append({
                "evidence_id": candidate["evidence_id"],
                "reason": "NLI contradiction"
            })

            continue


        # ----------------------------------------------------
        # Reject neutral evidence
        # ----------------------------------------------------

        if nli_label == "NEUTRAL":

            rejected.append({
                "evidence_id": candidate["evidence_id"],
                "reason": "NLI neutral"
            })

            continue


        # ----------------------------------------------------
        # Require NLI entailment
        # ----------------------------------------------------

        if nli_label != "ENTAILMENT":

            rejected.append({
                "evidence_id": candidate["evidence_id"],
                "reason": "No semantic support"
            })

            continue


        # ----------------------------------------------------
        # Speaker consistency
        # ----------------------------------------------------

        if (
            speaker_check["applicable"]
            and not speaker_check["passed"]
        ):

            rejected.append({
                "evidence_id": candidate["evidence_id"],
                "reason": "Speaker mismatch"
            })

            continue


        # ----------------------------------------------------
        # Optional lexical condition
        # ----------------------------------------------------

        if lexical_overlap < min_lexical_overlap:

            rejected.append({
                "evidence_id": candidate["evidence_id"],
                "reason": "Insufficient lexical support"
            })

            continue


        # ----------------------------------------------------
        # Candidate accepted
        # ----------------------------------------------------

        supporting.append(candidate)


    # --------------------------------------------------------
    # Sort supporting candidates
    #
    # Stronger NLI support first.
    # BGE similarity is used as a secondary ordering signal.
    # --------------------------------------------------------

    supporting = sorted(
        supporting,
        key=lambda x: (
            x["nli_entailment"],
            x["retrieval_similarity"]
        ),
        reverse=True
    )


    return {
        "supporting_candidates": supporting,
        "rejected_candidates": rejected
    }


# ============================================================
# Test using the originality attribute
# ============================================================

selection_result = select_supporting_candidates(
    analysis_result
)


# ------------------------------------------------------------
# Display supporting candidates
# ------------------------------------------------------------

print("=" * 90)
print("SUPPORTING EVIDENCE CANDIDATES")
print("=" * 90)

for rank, candidate in enumerate(
    selection_result["supporting_candidates"],
    start=1
):

    print(
        f"\nRank {rank} | "
        f"Evidence {candidate['evidence_id']}"
    )

    print(
        f"BGE similarity : "
        f"{candidate['retrieval_similarity']:.4f}"
    )

    print(
        f"NLI entailment : "
        f"{candidate['nli_entailment']:.4f}"
    )

    print(
        f"Lexical overlap: "
        f"{candidate['lexical_overlap']:.4f}"
    )

    print(
        f"Speaker check  : "
        f"{candidate['speaker_check']}"
    )

    print(
        f"Time           : "
        f"{candidate['start']:.3f} → "
        f"{candidate['end']:.3f}"
    )


# ------------------------------------------------------------
# Display rejected candidates
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("REJECTED EVIDENCE CANDIDATES")
print("=" * 90)

for item in selection_result["rejected_candidates"]:

    print(
        f"Evidence {item['evidence_id']} "
        f"→ {item['reason']}"
    )


print("\n" + "=" * 90)
print("Candidate selection complete.")
print("No final VERIFIED / FLAGGED decision made.")
print("=" * 90)

SUPPORTING EVIDENCE CANDIDATES

Rank 1 | Evidence 14
BGE similarity : 0.7810
NLI entailment : 0.9812
Lexical overlap: 0.7500
Speaker check  : {'applicable': True, 'passed': True, 'reason': 'Speaker matches evidence.'}
Time           : 117.699 → 120.921

Rank 2 | Evidence 15
BGE similarity : 0.6708
NLI entailment : 0.9787
Lexical overlap: 0.7500
Speaker check  : {'applicable': True, 'passed': True, 'reason': 'Speaker matches evidence.'}
Time           : 122.503 → 132.290

REJECTED EVIDENCE CANDIDATES
Evidence 105 → NLI neutral
Evidence 107 → NLI neutral
Evidence 145 → NLI neutral
Evidence 102 → NLI neutral
Evidence 135 → NLI neutral
Evidence 117 → NLI neutral
Evidence 112 → NLI neutral
Evidence 134 → NLI neutral

Candidate selection complete.
No final VERIFIED / FLAGGED decision made.


In [29]:
# ============================================================
# M7.15 — Best Supporting Evidence Context
# ============================================================
#
# Purpose:
#   Select the strongest supporting evidence context for
#   one MoM attribute.
#
# The selected unit consists of:
#
#   - Primary evidence utterance
#   - Its surrounding context
#   - NLI support
#   - Lexical support
#   - Speaker consistency
#
# The primary utterance provides the citation timestamp.
# The context provides conversational meaning.
#
# No final VERIFIED / FLAGGED decision is made yet.
# ============================================================


def select_best_evidence_context(selection_result):
    """
    Select the strongest supporting evidence candidate.

    Ranking priority:
        1. NLI entailment
        2. BGE retrieval similarity
        3. Lexical overlap

    Returns
    -------
    dict or None
        Best evidence context.
    """

    candidates = selection_result.get(
        "supporting_candidates",
        []
    )

    if not candidates:
        return None


    # --------------------------------------------------------
    # Candidates are already ordered primarily by NLI
    # entailment and secondarily by BGE similarity.
    # --------------------------------------------------------

    best_candidate = candidates[0]


    # --------------------------------------------------------
    # Retrieve the complete context
    # --------------------------------------------------------

    context_docs = get_context_window(
        best_candidate["evidence_id"],
        window_size=2
    )


    # --------------------------------------------------------
    # Context timestamps
    # --------------------------------------------------------

    context_start = min(
        doc["start"]
        for doc in context_docs
    )

    context_end = max(
        doc["end"]
        for doc in context_docs
    )


    # --------------------------------------------------------
    # Return structured evidence context
    # --------------------------------------------------------

    return {

        "primary_evidence_id":
            best_candidate["evidence_id"],

        "primary_start":
            best_candidate["start"],

        "primary_end":
            best_candidate["end"],

        "primary_speaker":
            best_candidate["speaker"],

        "primary_text":
            best_candidate["text"],

        "context_evidence_ids": [
            int(doc["evidence_id"])
            for doc in context_docs
        ],

        "context_start":
            context_start,

        "context_end":
            context_end,

        "context_speakers": list(
            dict.fromkeys(
                doc["speaker"]
                for doc in context_docs
            )
        ),

        "context_text": " ".join(
            doc["text"]
            for doc in context_docs
        ),

        "bge_similarity":
            best_candidate["retrieval_similarity"],

        "nli_label":
            best_candidate["nli_label"],

        "nli_entailment":
            best_candidate["nli_entailment"],

        "nli_contradiction":
            best_candidate["nli_contradiction"],

        "nli_neutral":
            best_candidate["nli_neutral"],

        "lexical_overlap":
            best_candidate["lexical_overlap"],

        "matched_terms":
            best_candidate["matched_terms"],

        "speaker_check":
            best_candidate["speaker_check"]
    }


# ============================================================
# Test using the originality analysis
# ============================================================

best_evidence = select_best_evidence_context(
    selection_result
)


print("=" * 90)
print("BEST SUPPORTING EVIDENCE CONTEXT")
print("=" * 90)


if best_evidence is None:

    print("\nNo supporting evidence was found.")

else:

    print(
        f"\nPrimary Evidence ID : "
        f"{best_evidence['primary_evidence_id']}"
    )

    print(
        f"Primary Timestamp   : "
        f"{best_evidence['primary_start']:.3f} → "
        f"{best_evidence['primary_end']:.3f}"
    )

    print(
        f"Primary Speaker     : "
        f"{best_evidence['primary_speaker']}"
    )

    print(
        f"\nContext Evidence IDs: "
        f"{best_evidence['context_evidence_ids']}"
    )

    print(
        f"Context Timestamp   : "
        f"{best_evidence['context_start']:.3f} → "
        f"{best_evidence['context_end']:.3f}"
    )

    print(
        f"Context Speakers    : "
        f"{best_evidence['context_speakers']}"
    )

    print(
        f"\nBGE Similarity      : "
        f"{best_evidence['bge_similarity']:.4f}"
    )

    print(
        f"NLI Label           : "
        f"{best_evidence['nli_label']}"
    )

    print(
        f"NLI Entailment      : "
        f"{best_evidence['nli_entailment']:.4f}"
    )

    print(
        f"Lexical Overlap     : "
        f"{best_evidence['lexical_overlap']:.4f}"
    )

    print(
        f"Speaker Check       : "
        f"{best_evidence['speaker_check']}"
    )

    print("\nPrimary Evidence Text:")
    print(best_evidence["primary_text"])

    print("\nFull Context:")
    print(best_evidence["context_text"])


print("\n" + "=" * 90)
print("Best evidence context selection complete.")
print("=" * 90)

BEST SUPPORTING EVIDENCE CONTEXT

Primary Evidence ID : 14
Primary Timestamp   : 117.699 → 120.921
Primary Speaker     : SPEAKER_02

Context Evidence IDs: [12, 13, 14, 15, 16]
Context Timestamp   : 106.340 → 138.174
Context Speakers    : ['SPEAKER_02']

BGE Similarity      : 0.7810
NLI Label           : ENTAILMENT
NLI Entailment      : 0.9812
Lexical Overlap     : 0.7500
Speaker Check       : {'applicable': True, 'passed': True, 'reason': 'Speaker matches evidence.'}

Primary Evidence Text:
Now, we're developing a remote control, which you probably already know.

Full Context:
talk about the project plan, discuss our own ideas and everything. And we've got 25 minutes to do that as far as I can understand. Now, we're developing a remote control, which you probably already know. We want it to be original, something that people haven't thought of. It's not out in the shops. Trendy, appealing to a wide market, but, you know, not a hunk of metal. And user-friendly, grannies to kids,

Best e

In [30]:
# ============================================================
# M7.16 — Direct Evidence Identification
# ============================================================
#
# Purpose:
#   Within a selected contextual evidence window, identify
#   the individual utterance that is most directly related
#   to the proposition.
#
# Important distinction:
#
#   Context window
#       → used for understanding the conversation
#
#   Direct evidence utterance
#       → used for the primary citation/timestamp
#
# This prevents a contextual NLI match from producing a
# misleading primary timestamp.
#
# No existing files are modified.
# ============================================================


def identify_direct_evidence(
    proposition,
    context_docs
):
    """
    Identify the most directly relevant utterance inside
    a contextual evidence window.

    Uses BGE semantic similarity between the proposition
    and each individual utterance.

    Returns
    -------
    dict or None
        Direct evidence document and similarity.
    """

    if not context_docs:
        return None


    # --------------------------------------------------------
    # Extract individual evidence texts
    # --------------------------------------------------------

    texts = [
        doc["text"]
        for doc in context_docs
    ]


    # --------------------------------------------------------
    # Generate proposition embedding
    # --------------------------------------------------------

    proposition_embedding = bge_model.encode(
        [proposition],
        normalize_embeddings=True,
        convert_to_numpy=True
    )


    # --------------------------------------------------------
    # Generate individual evidence embeddings
    # --------------------------------------------------------

    evidence_embeddings = bge_model.encode(
        texts,
        normalize_embeddings=True,
        convert_to_numpy=True
    )


    # --------------------------------------------------------
    # Calculate cosine similarities
    # --------------------------------------------------------

    similarities = (
        evidence_embeddings
        @ proposition_embedding[0]
    )


    # --------------------------------------------------------
    # Find most relevant utterance
    # --------------------------------------------------------

    best_index = int(
        similarities.argmax()
    )

    best_doc = context_docs[best_index]

    best_similarity = float(
        similarities[best_index]
    )


    return {
        "evidence_id": int(
            best_doc["evidence_id"]
        ),

        "speaker": best_doc["speaker"],

        "start": best_doc["start"],

        "end": best_doc["end"],

        "text": best_doc["text"],

        "similarity": best_similarity
    }


# ============================================================
# Test using the selected originality context
# ============================================================

proposition = (
    "The remote control is intended to be original."
)


direct_evidence = identify_direct_evidence(
    proposition,
    context
)


print("=" * 90)
print("DIRECT EVIDENCE IDENTIFICATION")
print("=" * 90)


if direct_evidence is None:

    print("\nNo direct evidence identified.")

else:

    print(
        f"\nEvidence ID : "
        f"{direct_evidence['evidence_id']}"
    )

    print(
        f"Speaker     : "
        f"{direct_evidence['speaker']}"
    )

    print(
        f"Timestamp   : "
        f"{direct_evidence['start']:.3f} → "
        f"{direct_evidence['end']:.3f}"
    )

    print(
        f"BGE Similarity: "
        f"{direct_evidence['similarity']:.4f}"
    )

    print("\nEvidence Text:")
    print(direct_evidence["text"])


print("\n" + "=" * 90)
print("Direct evidence identification complete.")
print("=" * 90)

DIRECT EVIDENCE IDENTIFICATION

Evidence ID : 14
Speaker     : SPEAKER_02
Timestamp   : 117.699 → 120.921
BGE Similarity: 0.7810

Evidence Text:
Now, we're developing a remote control, which you probably already know.

Direct evidence identification complete.


In [31]:
# ============================================================
# M7.17 — Direct Evidence Signal Diagnostic
# ============================================================
#
# Purpose:
#   Compare multiple evidence signals for every utterance
#   inside the selected context window.
#
# Signals:
#   1. BGE semantic similarity
#   2. Lexical overlap
#   3. NLI entailment
#   4. NLI label
#
# This is diagnostic only.
# No final ranking formula is introduced yet.
# ============================================================


def diagnose_context_evidence(
    proposition,
    context_docs
):
    """
    Calculate evidence-support signals for each utterance
    inside a contextual evidence window.
    """

    if not context_docs:
        return []


    # --------------------------------------------------------
    # BGE embeddings for proposition and utterances
    # --------------------------------------------------------

    proposition_embedding = bge_model.encode(
        [proposition],
        normalize_embeddings=True,
        convert_to_numpy=True
    )

    evidence_texts = [
        doc["text"]
        for doc in context_docs
    ]

    evidence_embeddings = bge_model.encode(
        evidence_texts,
        normalize_embeddings=True,
        convert_to_numpy=True
    )

    similarities = (
        evidence_embeddings
        @ proposition_embedding[0]
    )


    # --------------------------------------------------------
    # Analyze every utterance
    # --------------------------------------------------------

    results = []


    for i, doc in enumerate(context_docs):

        text = doc["text"]


        # Lexical support
        lexical_result = lexical_support(
            proposition,
            text
        )


        # NLI
        nli_result = run_nli(
            text,
            proposition
        )


        results.append({

            "evidence_id":
                int(doc["evidence_id"]),

            "speaker":
                doc["speaker"],

            "start":
                doc["start"],

            "end":
                doc["end"],

            "text":
                text,

            "bge_similarity":
                float(similarities[i]),

            "lexical_overlap":
                lexical_result["overlap_ratio"],

            "matched_terms":
                lexical_result["matched_terms"],

            "nli_label":
                nli_result["predicted_label"],

            "nli_entailment":
                nli_result["entailment_probability"],

            "nli_contradiction":
                nli_result["contradiction_probability"],

            "nli_neutral":
                nli_result["neutral_probability"]
        })


    return results


# ============================================================
# Run diagnostic on originality
# ============================================================

proposition = (
    "The remote control is intended to be original."
)

diagnostic_results = diagnose_context_evidence(
    proposition,
    context
)


# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print("=" * 100)
print("DIRECT EVIDENCE SIGNAL DIAGNOSTIC")
print("=" * 100)

print("\nProposition:")
print(proposition)


for result in diagnostic_results:

    print("\n" + "-" * 100)

    print(
        f"Evidence {result['evidence_id']} | "
        f"{result['speaker']} | "
        f"{result['start']:.3f} → "
        f"{result['end']:.3f}"
    )

    print(
        f"BGE similarity     : "
        f"{result['bge_similarity']:.4f}"
    )

    print(
        f"Lexical overlap    : "
        f"{result['lexical_overlap']:.4f}"
    )

    print(
        f"Matched terms      : "
        f"{result['matched_terms']}"
    )

    print(
        f"NLI label          : "
        f"{result['nli_label']}"
    )

    print(
        f"NLI entailment     : "
        f"{result['nli_entailment']:.4f}"
    )

    print(
        f"NLI contradiction  : "
        f"{result['nli_contradiction']:.4f}"
    )

    print(
        f"NLI neutral        : "
        f"{result['nli_neutral']:.4f}"
    )

    print("\nText:")
    print(result["text"])


print("\n" + "=" * 100)
print("Diagnostic complete.")
print("=" * 100)

DIRECT EVIDENCE SIGNAL DIAGNOSTIC

Proposition:
The remote control is intended to be original.

----------------------------------------------------------------------------------------------------
Evidence 13 | SPEAKER_02 | 112.185 → 115.748
BGE similarity     : 0.4592
Lexical overlap    : 0.0000
Matched terms      : []
NLI label          : NEUTRAL
NLI entailment     : 0.0002
NLI contradiction  : 0.0009
NLI neutral        : 0.9989

Text:
And we've got 25 minutes to do that as far as I can understand.

----------------------------------------------------------------------------------------------------
Evidence 14 | SPEAKER_02 | 117.699 → 120.921
BGE similarity     : 0.7810
Lexical overlap    : 0.5000
Matched terms      : ['remote', 'control']
NLI label          : NEUTRAL
NLI entailment     : 0.0012
NLI contradiction  : 0.0112
NLI neutral        : 0.9876

Text:
Now, we're developing a remote control, which you probably already know.

------------------------------------------------------

In [32]:
# ============================================================
# M7.18 — Evidence Group Representation
# ============================================================
#
# Purpose:
#   Represent supporting evidence as a GROUP of utterances
#   rather than forcing every attribute to have one evidence
#   sentence.
#
# Three levels are maintained:
#
#   1. Supporting evidence IDs
#      → utterances directly contributing to the claim
#
#   2. Context evidence IDs
#      → surrounding utterances used for interpretation
#
#   3. Display interval
#      → timestamp range shown to the user
#
# This structure will later be used by:
#
#   M7 → Verification
#   M8 → Verified MoM
#   M9 → Dashboard
# ============================================================


def build_evidence_group(
    supporting_evidence_ids,
    context_evidence_ids
):
    """
    Build a structured evidence group.

    Parameters
    ----------
    supporting_evidence_ids : list[int]
        Evidence utterances directly supporting the attribute.

    context_evidence_ids : list[int]
        Surrounding evidence used to interpret the support.

    Returns
    -------
    dict
        Structured evidence group.
    """

    # --------------------------------------------------------
    # Retrieve supporting documents
    # --------------------------------------------------------

    supporting_docs = [
        doc
        for doc in evidence_documents
        if int(doc["evidence_id"])
        in [int(x) for x in supporting_evidence_ids]
    ]


    # --------------------------------------------------------
    # Retrieve context documents
    # --------------------------------------------------------

    context_docs = [
        doc
        for doc in evidence_documents
        if int(doc["evidence_id"])
        in [int(x) for x in context_evidence_ids]
    ]


    # --------------------------------------------------------
    # Sort chronologically
    # --------------------------------------------------------

    supporting_docs = sorted(
        supporting_docs,
        key=lambda x: x["start"]
    )

    context_docs = sorted(
        context_docs,
        key=lambda x: x["start"]
    )


    # --------------------------------------------------------
    # Determine display interval
    # --------------------------------------------------------

    all_display_docs = (
        supporting_docs
        if supporting_docs
        else context_docs
    )

    if all_display_docs:

        display_start = min(
            doc["start"]
            for doc in all_display_docs
        )

        display_end = max(
            doc["end"]
            for doc in all_display_docs
        )

    else:

        display_start = None
        display_end = None


    # --------------------------------------------------------
    # Build result
    # --------------------------------------------------------

    return {

        "supporting_evidence_ids": [
            int(doc["evidence_id"])
            for doc in supporting_docs
        ],

        "supporting_evidence": [
            {
                "evidence_id": int(doc["evidence_id"]),
                "speaker": doc["speaker"],
                "start": doc["start"],
                "end": doc["end"],
                "text": doc["text"]
            }
            for doc in supporting_docs
        ],

        "context_evidence_ids": [
            int(doc["evidence_id"])
            for doc in context_docs
        ],

        "context_start": display_start,

        "context_end": display_end,

        "context_text": " ".join(
            doc["text"]
            for doc in context_docs
        ),

        "speakers": list(
            dict.fromkeys(
                doc["speaker"]
                for doc in context_docs
            )
        )
    }


# ============================================================
# Test using the originality example
# ============================================================

evidence_group = build_evidence_group(
    supporting_evidence_ids=[14, 15],
    context_evidence_ids=[13, 14, 15, 16, 17]
)


print("=" * 90)
print("EVIDENCE GROUP")
print("=" * 90)

print(
    "\nSupporting evidence IDs:",
    evidence_group["supporting_evidence_ids"]
)

print(
    "Context evidence IDs:",
    evidence_group["context_evidence_ids"]
)

print(
    "Display interval:",
    f"{evidence_group['context_start']:.3f}"
    f" → "
    f"{evidence_group['context_end']:.3f}"
)

print(
    "Speakers:",
    evidence_group["speakers"]
)


print("\nSupporting evidence:")

for item in evidence_group["supporting_evidence"]:

    print(
        f"\nEvidence {item['evidence_id']} | "
        f"{item['speaker']} | "
        f"{item['start']:.3f} → "
        f"{item['end']:.3f}"
    )

    print(item["text"])


print("\nContext text:")
print(evidence_group["context_text"])


print("\n" + "=" * 90)
print("Evidence group construction complete.")
print("=" * 90)

EVIDENCE GROUP

Supporting evidence IDs: [14, 15]
Context evidence IDs: [13, 14, 15, 16, 17]
Display interval: 117.699 → 132.290
Speakers: ['SPEAKER_02']

Supporting evidence:

Evidence 14 | SPEAKER_02 | 117.699 → 120.921
Now, we're developing a remote control, which you probably already know.

Evidence 15 | SPEAKER_02 | 122.503 → 132.290
We want it to be original, something that people haven't thought of. It's not out in the shops. Trendy, appealing to a wide market, but,

Context text:
And we've got 25 minutes to do that as far as I can understand. Now, we're developing a remote control, which you probably already know. We want it to be original, something that people haven't thought of. It's not out in the shops. Trendy, appealing to a wide market, but, you know, not a hunk of metal. And user-friendly, grannies to kids, maybe even pooches, should be able to use it.

Evidence group construction complete.


In [33]:
# ============================================================
# M7 - Reusable Evidence Group Builder
# ============================================================
# Purpose:
#   Build a structured evidence group for one verified
#   proposition/attribute.
#
# The group distinguishes:
#   1. Supporting evidence -> directly contributes to the claim
#   2. Context evidence    -> surrounding utterances for interpretation
#   3. Display interval    -> timestamp range shown to the user
# ============================================================

def create_evidence_group(
    supporting_evidence_ids,
    context_evidence_ids
):
    """
    Create a structured evidence group from evidence IDs.

    Parameters
    ----------
    supporting_evidence_ids : list
        Evidence IDs that directly support the proposition.

    context_evidence_ids : list
        Evidence IDs used as surrounding conversational context.

    Returns
    -------
    dict
        Structured evidence group.
    """

    supporting_ids = [int(x) for x in supporting_evidence_ids]
    context_ids = [int(x) for x in context_evidence_ids]

    # Lookup evidence documents
    evidence_lookup = {
        int(doc["evidence_id"]): doc
        for doc in evidence_documents
    }

    # Retrieve supporting evidence
    supporting_docs = [
        evidence_lookup[eid]
        for eid in supporting_ids
        if eid in evidence_lookup
    ]

    # Retrieve context evidence
    context_docs = [
        evidence_lookup[eid]
        for eid in context_ids
        if eid in evidence_lookup
    ]

    # Sort chronologically
    supporting_docs = sorted(
        supporting_docs,
        key=lambda x: x["start"]
    )

    context_docs = sorted(
        context_docs,
        key=lambda x: x["start"]
    )

    # Display interval should cover the supporting evidence.
    # If no supporting evidence exists, fall back to context.
    display_docs = supporting_docs if supporting_docs else context_docs

    if display_docs:
        display_start = min(
            doc["start"] for doc in display_docs
        )
        display_end = max(
            doc["end"] for doc in display_docs
        )
    else:
        display_start = None
        display_end = None

    # Unique speakers from supporting evidence
    speakers = sorted(
        set(
            doc["speaker"]
            for doc in supporting_docs
            if doc.get("speaker") is not None
        )
    )

    # Combined context text
    context_text = " ".join(
        doc["text"].strip()
        for doc in context_docs
    )

    return {
        "supporting_evidence_ids": [
            int(doc["evidence_id"])
            for doc in supporting_docs
        ],

        "supporting_evidence": supporting_docs,

        "context_evidence_ids": [
            int(doc["evidence_id"])
            for doc in context_docs
        ],

        "context_start": display_start,
        "context_end": display_end,

        "context_text": context_text,

        "speakers": speakers
    }


# ------------------------------------------------------------
# Test the reusable function with the originality example
# ------------------------------------------------------------

test_group = create_evidence_group(
    supporting_evidence_ids=[14, 15],
    context_evidence_ids=[13, 14, 15, 16, 17]
)

print("=" * 90)
print("REUSABLE EVIDENCE GROUP TEST")
print("=" * 90)

print(f"Supporting IDs : {test_group['supporting_evidence_ids']}")
print(f"Context IDs    : {test_group['context_evidence_ids']}")
print(
    f"Display interval: "
    f"{test_group['context_start']:.3f} → "
    f"{test_group['context_end']:.3f}"
)
print(f"Speakers       : {test_group['speakers']}")

print("\nEvidence group function is ready.")

REUSABLE EVIDENCE GROUP TEST
Supporting IDs : [14, 15]
Context IDs    : [13, 14, 15, 16, 17]
Display interval: 117.699 → 132.290
Speakers       : ['SPEAKER_02']

Evidence group function is ready.


In [34]:
# ============================================================
# M7 - Attribute-Level Evidence Verification
# ============================================================
# Purpose:
#   Verify one MoM proposition using:
#   BGE retrieval + contextual NLI + lexical support
#   + speaker consistency.
#
# Important:
#   BGE similarity is used for retrieval, NOT as proof.
#   NLI is evaluated on conversational context because a
#   proposition may require information from multiple utterances.
# ============================================================

def verify_attribute(
    attribute,
    proposition,
    claim_speaker=None,
    top_k=10,
    context_window=2
):
    """
    Verify one atomic proposition.

    Parameters
    ----------
    attribute : str
        Attribute name, e.g. "originality".

    proposition : str
        Atomic proposition to verify.

    claim_speaker : str or None
        Expected speaker if the proposition contains a
        speaker attribution.

    top_k : int
        Number of BGE candidates to inspect.

    context_window : int
        Number of neighboring utterances on each side.

    Returns
    -------
    dict
        Verification result.
    """

    # --------------------------------------------------------
    # Step 1: Retrieve candidate evidence using BGE
    # --------------------------------------------------------

    analysis = analyze_attribute_evidence(
        proposition=proposition,
        claim_speaker=claim_speaker,
        top_k=top_k,
        context_window=context_window
    )

    # --------------------------------------------------------
    # Step 2: Select candidates supported by contextual NLI
    # --------------------------------------------------------

    supporting_candidates = select_supporting_candidates(
        analysis,
        min_lexical_overlap=0.0
    )

    # --------------------------------------------------------
    # Step 3: No supporting evidence found
    # --------------------------------------------------------

    if not supporting_candidates:

        return {
            "attribute": attribute,
            "proposition": proposition,
            "status": "FLAGGED",
            "reason": "No supporting evidence found.",
            "supporting_evidence_ids": [],
            "context_evidence_ids": [],
            "display_start": None,
            "display_end": None,
            "speakers": [],
            "candidates_checked": len(analysis)
        }

    # --------------------------------------------------------
    # Step 4: Keep supporting candidates
    # --------------------------------------------------------

    supporting_ids = [
        int(item["evidence_id"])
        for item in supporting_candidates
    ]

    # --------------------------------------------------------
    # Step 5: Build context around all supporting evidence
    # --------------------------------------------------------

    context_ids = set()

    for item in supporting_candidates:

        item_context = item.get("context_ids", [])

        for eid in item_context:
            context_ids.add(int(eid))

    context_ids = sorted(context_ids)

    # --------------------------------------------------------
    # Step 6: Build final evidence group
    # --------------------------------------------------------

    evidence_group = create_evidence_group(
        supporting_evidence_ids=supporting_ids,
        context_evidence_ids=context_ids
    )

    # --------------------------------------------------------
    # Step 7: Speaker consistency
    # --------------------------------------------------------

    speaker_check = True

    if claim_speaker is not None:

        evidence_speakers = evidence_group["speakers"]

        if evidence_speakers:

            speaker_check = (
                claim_speaker in evidence_speakers
            )

    # --------------------------------------------------------
    # Step 8: Final status
    # --------------------------------------------------------

    if speaker_check:
        status = "VERIFIED"
        reason = (
            "The proposition is supported by retrieved "
            "evidence and contextual NLI, with speaker "
            "consistency satisfied."
        )
    else:
        status = "FLAGGED"
        reason = (
            "The proposition has supporting textual evidence, "
            "but the claimed speaker does not match the "
            "supporting evidence."
        )

    # --------------------------------------------------------
    # Step 9: Return structured verification result
    # --------------------------------------------------------

    return {
        "attribute": attribute,
        "proposition": proposition,

        "status": status,
        "reason": reason,

        "supporting_evidence_ids":
            evidence_group["supporting_evidence_ids"],

        "context_evidence_ids":
            evidence_group["context_evidence_ids"],

        "display_start":
            evidence_group["context_start"],

        "display_end":
            evidence_group["context_end"],

        "speakers":
            evidence_group["speakers"],

        "speaker_check":
            speaker_check,

        "candidates_checked":
            len(analysis),

        "supporting_candidates":
            supporting_candidates
    }


# ============================================================
# Test with the originality attribute
# ============================================================

test_verification = verify_attribute(
    attribute="originality",
    proposition="The remote control is intended to be original.",
    claim_speaker="SPEAKER_02",
    top_k=10,
    context_window=2
)

print("=" * 90)
print("ATTRIBUTE VERIFICATION TEST")
print("=" * 90)

print(f"Attribute       : {test_verification['attribute']}")
print(f"Proposition     : {test_verification['proposition']}")
print(f"Status          : {test_verification['status']}")
print(f"Speaker check   : {test_verification['speaker_check']}")
print(f"Supporting IDs  : {test_verification['supporting_evidence_ids']}")
print(f"Context IDs     : {test_verification['context_evidence_ids']}")
print(
    f"Display interval: "
    f"{test_verification['display_start']:.3f} → "
    f"{test_verification['display_end']:.3f}"
)
print(f"Speakers        : {test_verification['speakers']}")
print(f"Reason          : {test_verification['reason']}")

print("\nSupporting evidence:")
for item in test_verification["supporting_candidates"]:
    print(
        f"  ID {item['evidence_id']} | "
        f"BGE={item['retrieval_similarity']:.4f} | "
        f"NLI={item['nli_label']} "
        f"{item['nli_entailment']:.4f}"
    )

print("\nAttribute verification test complete.")

TypeError: string indices must be integers, not 'str'

In [35]:
# ============================================================
# M7 - Inspect Supporting Candidate Output
# ============================================================
# Purpose:
#   Check the exact structure returned by
#   select_supporting_candidates().
#
# This is a lightweight diagnostic and does NOT rerun
# WhisperX, Pyannote, BGE model loading, or FAISS creation.
# ============================================================

test_analysis = analyze_attribute_evidence(
    proposition="The remote control is intended to be original.",
    claim_speaker="SPEAKER_02",
    top_k=10,
    context_window=2
)

test_supporting = select_supporting_candidates(
    test_analysis,
    min_lexical_overlap=0.0
)

print("=" * 90)
print("SUPPORTING CANDIDATE STRUCTURE")
print("=" * 90)

print("Python type:")
print(type(test_supporting))

print("\nObject:")
print(test_supporting)

print("\nKeys / length:")
try:
    print(len(test_supporting))
except Exception:
    print("Length not available")

if isinstance(test_supporting, dict):
    print("\nDictionary keys:")
    print(list(test_supporting.keys()))

print("\nDiagnostic complete.")

SUPPORTING CANDIDATE STRUCTURE
Python type:
<class 'dict'>

Object:
{'supporting_candidates': [{'evidence_id': 14, 'speaker': 'SPEAKER_02', 'start': 117.699, 'end': 120.921, 'text': "Now, we're developing a remote control, which you probably already know.", 'retrieval_similarity': 0.7810472249984741, 'nli_label': 'ENTAILMENT', 'nli_entailment': 0.9811810255050659, 'nli_contradiction': 0.001153470017015934, 'nli_neutral': 0.017665470018982887, 'lexical_overlap': 0.75, 'matched_terms': ['remote', 'control', 'original'], 'missing_terms': ['intended'], 'speaker_check': {'applicable': True, 'passed': True, 'reason': 'Speaker matches evidence.'}, 'context_evidence_ids': [12, 13, 14, 15, 16]}, {'evidence_id': 15, 'speaker': 'SPEAKER_02', 'start': 122.503, 'end': 132.29, 'text': "We want it to be original, something that people haven't thought of. It's not out in the shops. Trendy, appealing to a wide market, but,", 'retrieval_similarity': 0.6708011627197266, 'nli_label': 'ENTAILMENT', 'nli_en

In [36]:
# ============================================================
# M7 - Corrected Attribute-Level Evidence Verification
# ============================================================
# Purpose:
#   Verify one atomic proposition using:
#   BGE retrieval + contextual NLI + speaker consistency
#   + structured evidence grouping.
#
# No model reloading is required.
# ============================================================

def verify_attribute(
    attribute,
    proposition,
    claim_speaker=None,
    top_k=10,
    context_window=2
):
    """
    Verify one atomic proposition.

    Returns a structured verification result containing:
    - verification status
    - supporting evidence
    - context evidence
    - timestamp interval
    - speaker consistency
    """

    # --------------------------------------------------------
    # 1. Retrieve and analyze candidate evidence
    # --------------------------------------------------------

    analysis = analyze_attribute_evidence(
        proposition=proposition,
        claim_speaker=claim_speaker,
        top_k=top_k,
        context_window=context_window
    )

    # --------------------------------------------------------
    # 2. Select candidates supported by contextual NLI
    # --------------------------------------------------------

    supporting_result = select_supporting_candidates(
        analysis,
        min_lexical_overlap=0.0
    )

    # IMPORTANT:
    # select_supporting_candidates() returns a dictionary.
    supporting_candidates = supporting_result[
        "supporting_candidates"
    ]

    # --------------------------------------------------------
    # 3. No supporting evidence
    # --------------------------------------------------------

    if not supporting_candidates:

        return {
            "attribute": attribute,
            "proposition": proposition,
            "status": "FLAGGED",
            "reason": "No supporting evidence found.",
            "supporting_evidence_ids": [],
            "context_evidence_ids": [],
            "display_start": None,
            "display_end": None,
            "speakers": [],
            "speaker_check": False,
            "candidates_checked": len(analysis),
            "supporting_candidates": [],
            "rejected_candidates":
                supporting_result["rejected_candidates"]
        }

    # --------------------------------------------------------
    # 4. Extract supporting evidence IDs
    # --------------------------------------------------------

    supporting_ids = [
        int(item["evidence_id"])
        for item in supporting_candidates
    ]

    # --------------------------------------------------------
    # 5. Collect context IDs from all supporting candidates
    # --------------------------------------------------------

    context_ids = set()

    for item in supporting_candidates:

        for eid in item.get(
            "context_evidence_ids",
            []
        ):
            context_ids.add(int(eid))

    context_ids = sorted(context_ids)

    # --------------------------------------------------------
    # 6. Build evidence group
    # --------------------------------------------------------

    evidence_group = create_evidence_group(
        supporting_evidence_ids=supporting_ids,
        context_evidence_ids=context_ids
    )

    # --------------------------------------------------------
    # 7. Speaker consistency
    # --------------------------------------------------------

    speaker_check = True

    if claim_speaker is not None:

        evidence_speakers = evidence_group["speakers"]

        if evidence_speakers:

            speaker_check = (
                claim_speaker in evidence_speakers
            )

    # --------------------------------------------------------
    # 8. Determine final verification status
    # --------------------------------------------------------

    if speaker_check:

        status = "VERIFIED"

        reason = (
            "The proposition is supported by contextual "
            "NLI evidence and the claimed speaker is "
            "consistent with the supporting evidence."
        )

    else:

        status = "FLAGGED"

        reason = (
            "Supporting evidence was found, but the "
            "claimed speaker does not match the "
            "supporting evidence."
        )

    # --------------------------------------------------------
    # 9. Return complete result
    # --------------------------------------------------------

    return {
        "attribute": attribute,
        "proposition": proposition,

        "status": status,
        "reason": reason,

        "supporting_evidence_ids":
            evidence_group["supporting_evidence_ids"],

        "context_evidence_ids":
            evidence_group["context_evidence_ids"],

        "display_start":
            evidence_group["context_start"],

        "display_end":
            evidence_group["context_end"],

        "speakers":
            evidence_group["speakers"],

        "speaker_check":
            speaker_check,

        "candidates_checked":
            len(analysis),

        "supporting_candidates":
            supporting_candidates,

        "rejected_candidates":
            supporting_result["rejected_candidates"]
    }


# ============================================================
# Test with the originality attribute
# ============================================================

test_verification = verify_attribute(
    attribute="originality",
    proposition="The remote control is intended to be original.",
    claim_speaker="SPEAKER_02",
    top_k=10,
    context_window=2
)

print("=" * 90)
print("ATTRIBUTE VERIFICATION TEST")
print("=" * 90)

print(
    f"Attribute       : "
    f"{test_verification['attribute']}"
)

print(
    f"Proposition     : "
    f"{test_verification['proposition']}"
)

print(
    f"Status          : "
    f"{test_verification['status']}"
)

print(
    f"Speaker check   : "
    f"{test_verification['speaker_check']}"
)

print(
    f"Supporting IDs  : "
    f"{test_verification['supporting_evidence_ids']}"
)

print(
    f"Context IDs     : "
    f"{test_verification['context_evidence_ids']}"
)

print(
    f"Display interval: "
    f"{test_verification['display_start']:.3f} → "
    f"{test_verification['display_end']:.3f}"
)

print(
    f"Speakers        : "
    f"{test_verification['speakers']}"
)

print(
    f"Reason          : "
    f"{test_verification['reason']}"
)

print("\nSupporting evidence:")

for item in test_verification[
    "supporting_candidates"
]:

    print(
        f"  ID {item['evidence_id']} | "
        f"BGE={item['retrieval_similarity']:.4f} | "
        f"NLI={item['nli_label']} "
        f"{item['nli_entailment']:.4f}"
    )

print("\nAttribute verification test complete.")

ATTRIBUTE VERIFICATION TEST
Attribute       : originality
Proposition     : The remote control is intended to be original.
Status          : VERIFIED
Speaker check   : True
Supporting IDs  : [14, 15]
Context IDs     : [12, 13, 14, 15, 16, 17]
Display interval: 117.699 → 132.290
Speakers        : ['SPEAKER_02']
Reason          : The proposition is supported by contextual NLI evidence and the claimed speaker is consistent with the supporting evidence.

Supporting evidence:
  ID 14 | BGE=0.7810 | NLI=ENTAILMENT 0.9812
  ID 15 | BGE=0.6708 | NLI=ENTAILMENT 0.9787

Attribute verification test complete.


In [37]:
# ============================================================
# M7 - Speaker Consistency Negative Test
# ============================================================
# Purpose:
#   Verify that the system flags a proposition when the
#   claimed speaker is incorrect.
#
# Expected:
#   The evidence should still be found, but the speaker
#   consistency check should fail.
# ============================================================

wrong_speaker_test = verify_attribute(
    attribute="originality",
    proposition="The remote control is intended to be original.",
    claim_speaker="SPEAKER_03",   # Deliberately incorrect
    top_k=10,
    context_window=2
)

print("=" * 90)
print("SPEAKER CONSISTENCY NEGATIVE TEST")
print("=" * 90)

print(
    f"Attribute       : "
    f"{wrong_speaker_test['attribute']}"
)

print(
    f"Proposition     : "
    f"{wrong_speaker_test['proposition']}"
)

print(
    f"Claimed speaker : SPEAKER_03"
)

print(
    f"Evidence speaker: "
    f"{wrong_speaker_test['speakers']}"
)

print(
    f"Speaker check   : "
    f"{wrong_speaker_test['speaker_check']}"
)

print(
    f"Status          : "
    f"{wrong_speaker_test['status']}"
)

print(
    f"Supporting IDs  : "
    f"{wrong_speaker_test['supporting_evidence_ids']}"
)

print(
    f"Reason          : "
    f"{wrong_speaker_test['reason']}"
)

print("\nSpeaker consistency negative test complete.")

SPEAKER CONSISTENCY NEGATIVE TEST
Attribute       : originality
Proposition     : The remote control is intended to be original.
Claimed speaker : SPEAKER_03
Evidence speaker: []
Speaker check   : False
Status          : FLAGGED
Supporting IDs  : []
Reason          : No supporting evidence found.

Speaker consistency negative test complete.


In [2]:
# ============================================================
# M7 Resume Check - Git Status
# ============================================================

%cd /content/drive/MyDrive/MTechIndProj/MoM_Project

!git status
!git log --oneline -3

/content/drive/MyDrive/MTechIndProj/MoM_Project
Refresh index: 100% (10/10), done.
On branch main
Your branch is up to date with 'origin/main'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   04_mom_generation/04_mom_generation.ipynb

no changes added to commit (use "git add" and/or "git commit -a")
32586f5 (HEAD -> main, origin/main) Implement attribute-level evidence verification
3ce72bd Implement VAD and ASR diarization pipeline
443c4da Initial clean project commit


In [3]:
# ============================================================
# M7 - Resume Session: Paths and Saved Data
# ============================================================
# Purpose:
#   Restore the project paths and load the saved data required
#   for the evidence-verification stage.
#
# This does NOT run:
#   - VAD
#   - WhisperX
#   - Pyannote
#   - BART
# ============================================================

import os
import json

# ------------------------------------------------------------
# Project paths
# ------------------------------------------------------------

PROJECT_DIR = "/content/drive/MyDrive/MTechIndProj/MoM_Project"
DATA_DIR = os.path.join(PROJECT_DIR, "data")
TRANSCRIPTS_DIR = os.path.join(DATA_DIR, "transcripts")

# Saved files
SPEAKER_TRANSCRIPT_PATH = os.path.join(
    TRANSCRIPTS_DIR,
    "ES2004a_speaker_transcript.json"
)

MOM_CLAIMS_PATH = os.path.join(
    TRANSCRIPTS_DIR,
    "ES2004a_structured_mom_claims.json"
)

MOM_CANDIDATES_PATH = os.path.join(
    TRANSCRIPTS_DIR,
    "ES2004a_mom_candidates_rule_based.json"
)

# ------------------------------------------------------------
# Load saved speaker transcript
# ------------------------------------------------------------

with open(
    SPEAKER_TRANSCRIPT_PATH,
    "r",
    encoding="utf-8"
) as f:
    speaker_transcript_data = json.load(f)

# ------------------------------------------------------------
# Load saved structured MoM claims
# ------------------------------------------------------------

with open(
    MOM_CLAIMS_PATH,
    "r",
    encoding="utf-8"
) as f:
    mom_claims = json.load(f)

# ------------------------------------------------------------
# Load saved rule-based candidates
# ------------------------------------------------------------

with open(
    MOM_CANDIDATES_PATH,
    "r",
    encoding="utf-8"
) as f:
    mom_candidates = json.load(f)

# ------------------------------------------------------------
# Basic verification
# ------------------------------------------------------------

print("=" * 90)
print("M7 RESUME - DATA LOADED")
print("=" * 90)

print(f"Project directory : {PROJECT_DIR}")
print(f"Speaker transcript: {len(speaker_transcript_data['utterances'])} utterances")
print(f"MoM claims        : {len(mom_claims)}")
print(f"MoM candidates    : {len(mom_candidates)}")

print("\nSaved files verified:")
print(f"✓ {os.path.basename(SPEAKER_TRANSCRIPT_PATH)}")
print(f"✓ {os.path.basename(MOM_CLAIMS_PATH)}")
print(f"✓ {os.path.basename(MOM_CANDIDATES_PATH)}")

print("\nM7 resume data loading complete.")

M7 RESUME - DATA LOADED
Project directory : /content/drive/MyDrive/MTechIndProj/MoM_Project
Speaker transcript: 148 utterances
MoM claims        : 6
MoM candidates    : 5

Saved files verified:
✓ ES2004a_speaker_transcript.json
✓ ES2004a_structured_mom_claims.json
✓ ES2004a_mom_candidates_rule_based.json

M7 resume data loading complete.


In [4]:
# ============================================================
# M7 - Verify Saved MoM Data Before Resume
# ============================================================
# Purpose:
#   Inspect the currently saved claim/candidate files.
#   We will NOT modify anything.
# ============================================================

print("=" * 90)
print("CURRENT SAVED M7 DATA")
print("=" * 90)

print(f"\nStructured MoM claims: {len(mom_claims)}")

for i, claim in enumerate(mom_claims, start=1):
    print(f"\nClaim {i}:")
    print(claim)

print("\n" + "-" * 90)

print(f"Rule-based candidates: {len(mom_candidates)}")

for i, candidate in enumerate(mom_candidates, start=1):
    print(f"\nCandidate {i}:")
    print(candidate)

print("\n" + "=" * 90)
print("DIAGNOSTIC ONLY - NO FILES MODIFIED")
print("=" * 90)

CURRENT SAVED M7 DATA

Structured MoM claims: 6

Claim 1:
meeting_id

Claim 2:
source

Claim 3:
candidate_count

Claim 4:
topic_group_count

Claim 5:
claim_count

Claim 6:
claims

------------------------------------------------------------------------------------------
Rule-based candidates: 5

Candidate 1:
meeting_id

Candidate 2:
method

Candidate 3:
original_utterances

Candidate 4:
candidate_count

Candidate 5:
candidates

DIAGNOSTIC ONLY - NO FILES MODIFIED


In [5]:
# ============================================================
# M7 - Correct Loaded MoM Data Structures
# ============================================================
# Purpose:
#   Extract the actual claims and candidates from their
#   saved JSON containers.
#
# No models are loaded and no files are modified.
# ============================================================

# Extract actual lists from the JSON containers
mom_claims_list = mom_claims["claims"]
mom_candidates_list = mom_candidates["candidates"]

print("=" * 90)
print("M7 DATA STRUCTURE CORRECTED")
print("=" * 90)

print(f"Actual structured MoM claims : {len(mom_claims_list)}")
print(f"Actual rule-based candidates: {len(mom_candidates_list)}")

print("\nSaved metadata:")
print(f"Claim count from file      : {mom_claims['claim_count']}")
print(f"Candidate count from file  : {mom_candidates['candidate_count']}")
print(f"Topic group count          : {mom_claims['topic_group_count']}")

print("\nData structure:")
print(f"mom_claims_list      → {type(mom_claims_list).__name__}")
print(f"mom_candidates_list  → {type(mom_candidates_list).__name__}")

print("\nM7 data structure is ready.")

M7 DATA STRUCTURE CORRECTED
Actual structured MoM claims : 12
Actual rule-based candidates: 50

Saved metadata:
Claim count from file      : 12
Candidate count from file  : 50
Topic group count          : 11

Data structure:
mom_claims_list      → list
mom_candidates_list  → list

M7 data structure is ready.


In [6]:
# ============================================================
# M7 - Resume Session: BGE + FAISS Evidence Retrieval
# ============================================================
# Purpose:
#   Rebuild the evidence-document collection, BGE embeddings,
#   and FAISS index required for evidence retrieval.
#
# This is safe to rerun after a Colab runtime restart.
#
# It does NOT rerun:
#   - VAD
#   - WhisperX
#   - Pyannote diarization
#   - Speaker-word alignment
#   - BART
# ============================================================

import numpy as np
import torch
import faiss

from sentence_transformers import SentenceTransformer

# ------------------------------------------------------------
# Device
# ------------------------------------------------------------

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("=" * 90)
print("M7 - BGE + FAISS RESUME")
print("=" * 90)

print(f"Device: {DEVICE}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# ------------------------------------------------------------
# Create evidence documents from saved speaker transcript
# ------------------------------------------------------------

utterances = speaker_transcript_data["utterances"]

evidence_documents = []

for i, utterance in enumerate(utterances, start=1):

    evidence_documents.append({
        "evidence_id": i,
        "speaker": utterance["speaker"],
        "start": float(utterance["start"]),
        "end": float(utterance["end"]),
        "duration": float(utterance["end"]) - float(utterance["start"]),
        "text": utterance["text"].strip()
    })

print(f"\nEvidence documents: {len(evidence_documents)}")

# ------------------------------------------------------------
# Load BGE embedding model
# ------------------------------------------------------------

BGE_MODEL_NAME = "BAAI/bge-small-en-v1.5"

print(f"\nLoading BGE model: {BGE_MODEL_NAME}")

bge_model = SentenceTransformer(
    BGE_MODEL_NAME,
    device=DEVICE
)

print("BGE model loaded successfully.")

# ------------------------------------------------------------
# Generate normalized embeddings
# ------------------------------------------------------------

evidence_texts = [
    doc["text"]
    for doc in evidence_documents
]

evidence_embeddings = bge_model.encode(
    evidence_texts,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=True
).astype("float32")

print(
    f"\nEmbedding shape: "
    f"{evidence_embeddings.shape}"
)

# ------------------------------------------------------------
# Build FAISS cosine-similarity index
# ------------------------------------------------------------
# Because embeddings are normalized, inner product is
# equivalent to cosine similarity.

embedding_dimension = evidence_embeddings.shape[1]

faiss_index = faiss.IndexFlatIP(
    embedding_dimension
)

faiss_index.add(evidence_embeddings)

print(f"FAISS vectors: {faiss_index.ntotal}")
print(f"FAISS dimension: {embedding_dimension}")

# ------------------------------------------------------------
# Retrieval function
# ------------------------------------------------------------

def retrieve_evidence(
    query_text,
    top_k=10
):
    """
    Retrieve the most semantically similar evidence
    utterances using BGE + FAISS.
    """

    query_embedding = bge_model.encode(
        [query_text],
        normalize_embeddings=True,
        convert_to_numpy=True
    ).astype("float32")

    similarities, indices = faiss_index.search(
        query_embedding,
        top_k
    )

    results = []

    for rank, (similarity, index) in enumerate(
        zip(similarities[0], indices[0]),
        start=1
    ):

        if index < 0:
            continue

        doc = evidence_documents[index]

        results.append({
            "retrieval_rank": rank,
            "evidence_id": doc["evidence_id"],
            "speaker": doc["speaker"],
            "start": doc["start"],
            "end": doc["end"],
            "text": doc["text"],
            "retrieval_similarity": float(similarity)
        })

    return results


# ------------------------------------------------------------
# Lightweight retrieval test
# ------------------------------------------------------------

test_query = (
    "The remote control is intended to be original."
)

test_results = retrieve_evidence(
    test_query,
    top_k=5
)

print("\n" + "=" * 90)
print("RETRIEVAL TEST")
print("=" * 90)

for result in test_results:

    print(
        f"Rank {result['retrieval_rank']} | "
        f"ID {result['evidence_id']} | "
        f"Similarity {result['retrieval_similarity']:.4f} | "
        f"{result['speaker']}"
    )

    print(
        f"  {result['start']:.3f} → "
        f"{result['end']:.3f}"
    )

    print(
        f"  {result['text']}"
    )

print("\nBGE + FAISS resume setup complete.")

ModuleNotFoundError: No module named 'faiss'

In [7]:
# ============================================================
# M7 - Restore FAISS Dependency
# ============================================================
# This installs FAISS for the current Colab runtime only.
# It does not modify the project files.
# ============================================================

!pip install -q faiss-cpu

print("FAISS installation complete.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 46.3 MB/s eta 0:00:00
FAISS installation complete.


In [8]:
# ============================================================
# M7 - Resume Session: BGE + FAISS Evidence Retrieval
# ============================================================
# Purpose:
#   Rebuild the evidence-document collection, BGE embeddings,
#   and FAISS index required for evidence retrieval.
#
# This is safe to rerun after a Colab runtime restart.
#
# It does NOT rerun:
#   - VAD
#   - WhisperX
#   - Pyannote diarization
#   - Speaker-word alignment
#   - BART
# ============================================================

import numpy as np
import torch
import faiss

from sentence_transformers import SentenceTransformer

# ------------------------------------------------------------
# Device
# ------------------------------------------------------------

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("=" * 90)
print("M7 - BGE + FAISS RESUME")
print("=" * 90)

print(f"Device: {DEVICE}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# ------------------------------------------------------------
# Create evidence documents from saved speaker transcript
# ------------------------------------------------------------

utterances = speaker_transcript_data["utterances"]

evidence_documents = []

for i, utterance in enumerate(utterances, start=1):

    evidence_documents.append({
        "evidence_id": i,
        "speaker": utterance["speaker"],
        "start": float(utterance["start"]),
        "end": float(utterance["end"]),
        "duration": float(utterance["end"]) - float(utterance["start"]),
        "text": utterance["text"].strip()
    })

print(f"\nEvidence documents: {len(evidence_documents)}")

# ------------------------------------------------------------
# Load BGE embedding model
# ------------------------------------------------------------

BGE_MODEL_NAME = "BAAI/bge-small-en-v1.5"

print(f"\nLoading BGE model: {BGE_MODEL_NAME}")

bge_model = SentenceTransformer(
    BGE_MODEL_NAME,
    device=DEVICE
)

print("BGE model loaded successfully.")

# ------------------------------------------------------------
# Generate normalized embeddings
# ------------------------------------------------------------

evidence_texts = [
    doc["text"]
    for doc in evidence_documents
]

evidence_embeddings = bge_model.encode(
    evidence_texts,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=True
).astype("float32")

print(
    f"\nEmbedding shape: "
    f"{evidence_embeddings.shape}"
)

# ------------------------------------------------------------
# Build FAISS cosine-similarity index
# ------------------------------------------------------------
# Because embeddings are normalized, inner product is
# equivalent to cosine similarity.

embedding_dimension = evidence_embeddings.shape[1]

faiss_index = faiss.IndexFlatIP(
    embedding_dimension
)

faiss_index.add(evidence_embeddings)

print(f"FAISS vectors: {faiss_index.ntotal}")
print(f"FAISS dimension: {embedding_dimension}")

# ------------------------------------------------------------
# Retrieval function
# ------------------------------------------------------------

def retrieve_evidence(
    query_text,
    top_k=10
):
    """
    Retrieve the most semantically similar evidence
    utterances using BGE + FAISS.
    """

    query_embedding = bge_model.encode(
        [query_text],
        normalize_embeddings=True,
        convert_to_numpy=True
    ).astype("float32")

    similarities, indices = faiss_index.search(
        query_embedding,
        top_k
    )

    results = []

    for rank, (similarity, index) in enumerate(
        zip(similarities[0], indices[0]),
        start=1
    ):

        if index < 0:
            continue

        doc = evidence_documents[index]

        results.append({
            "retrieval_rank": rank,
            "evidence_id": doc["evidence_id"],
            "speaker": doc["speaker"],
            "start": doc["start"],
            "end": doc["end"],
            "text": doc["text"],
            "retrieval_similarity": float(similarity)
        })

    return results


# ------------------------------------------------------------
# Lightweight retrieval test
# ------------------------------------------------------------

test_query = (
    "The remote control is intended to be original."
)

test_results = retrieve_evidence(
    test_query,
    top_k=5
)

print("\n" + "=" * 90)
print("RETRIEVAL TEST")
print("=" * 90)

for result in test_results:

    print(
        f"Rank {result['retrieval_rank']} | "
        f"ID {result['evidence_id']} | "
        f"Similarity {result['retrieval_similarity']:.4f} | "
        f"{result['speaker']}"
    )

    print(
        f"  {result['start']:.3f} → "
        f"{result['end']:.3f}"
    )

    print(
        f"  {result['text']}"
    )

print("\nBGE + FAISS resume setup complete.")

M7 - BGE + FAISS RESUME
Device: cuda
GPU: Tesla T4

Evidence documents: 148

Loading BGE model: BAAI/bge-small-en-v1.5


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

BGE model loaded successfully.


Batches:   0%|          | 0/5 [00:00<?, ?it/s]


Embedding shape: (148, 384)
FAISS vectors: 148
FAISS dimension: 384

RETRIEVAL TEST
Rank 1 | ID 105 | Similarity 0.7917 | SPEAKER_01
  707.248 → 710.889
  remote controls. You want to integrate everything into one.
Rank 2 | ID 14 | Similarity 0.7810 | SPEAKER_02
  117.699 → 120.921
  Now, we're developing a remote control, which you probably already know.
Rank 3 | ID 107 | Similarity 0.7408 | SPEAKER_03
  713.150 → 729.357
  experience has only been given the remote control with the object I buy, not doing any tampering with it and programming, using it to program TV videos and things, but basically on off volume, up and down, channel one to that basic function. I don't think I could go any further with it.
Rank 4 | ID 145 | Similarity 0.7396 | SPEAKER_02
  1025.075 → 1036.842
  I mean, the sky remote controls and everything. They're kind of molded and look a bit different than the tele-west remote controls. They're silver plastic, which looks a bit smarter. So yeah, I guess that's it

In [9]:
# ============================================================
# M7 - Resume Session: NLI Verification Model
# ============================================================
# Purpose:
#   Restore the DeBERTa NLI model used for contextual evidence
#   verification.
#
# Model:
#   cross-encoder/nli-deberta-v3-small
#
# Labels:
#   0 = CONTRADICTION
#   1 = ENTAILMENT
#   2 = NEUTRAL
#
# This does not modify any project files.
# ============================================================

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# ------------------------------------------------------------
# Device
# ------------------------------------------------------------

NLI_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("=" * 90)
print("M7 - NLI MODEL RESUME")
print("=" * 90)

print(f"Device: {NLI_DEVICE}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# ------------------------------------------------------------
# Load tokenizer
# ------------------------------------------------------------

NLI_MODEL_NAME = "cross-encoder/nli-deberta-v3-small"

print(f"\nLoading NLI model: {NLI_MODEL_NAME}")

nli_tokenizer = AutoTokenizer.from_pretrained(
    NLI_MODEL_NAME
)

# ------------------------------------------------------------
# Load model
# ------------------------------------------------------------

nli_model = AutoModelForSequenceClassification.from_pretrained(
    NLI_MODEL_NAME
)

nli_model = nli_model.to(NLI_DEVICE)
nli_model.eval()

# ------------------------------------------------------------
# Label mapping
# ------------------------------------------------------------

NLI_LABELS = {
    0: "CONTRADICTION",
    1: "ENTAILMENT",
    2: "NEUTRAL"
}

print("\nNLI model loaded successfully.")

print("\nLabel mapping:")
for label_id, label_name in NLI_LABELS.items():
    print(f"  {label_id} → {label_name}")

print("\nM7 NLI model is ready.")

M7 - NLI MODEL RESUME
Device: cuda
GPU: Tesla T4

Loading NLI model: cross-encoder/nli-deberta-v3-small


config.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.35k [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

...

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  568MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]


NLI model loaded successfully.

Label mapping:
  0 → CONTRADICTION
  1 → ENTAILMENT
  2 → NEUTRAL

M7 NLI model is ready.


In [10]:
# ============================================================
# M7 - Restore NLI Inference Function
# ============================================================
# Purpose:
#   Restore the lightweight NLI inference function after a
#   Colab runtime restart.
#
# Model is already loaded in:
#   nli_tokenizer
#   nli_model
# ============================================================

import torch


def run_nli(
    premise,
    hypothesis
):
    """
    Run Natural Language Inference.

    Parameters
    ----------
    premise : str
        Evidence/context from the meeting.

    hypothesis : str
        Proposition generated by the MoM system.

    Returns
    -------
    dict
        NLI label and probabilities.
    """

    inputs = nli_tokenizer(
        premise,
        hypothesis,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    inputs = {
        key: value.to(NLI_DEVICE)
        for key, value in inputs.items()
    }

    with torch.no_grad():

        outputs = nli_model(**inputs)

        probabilities = torch.softmax(
            outputs.logits,
            dim=-1
        )[0]

    probabilities = probabilities.cpu().numpy()

    predicted_label_id = int(
        probabilities.argmax()
    )

    return {
        "label": NLI_LABELS[predicted_label_id],
        "contradiction": float(probabilities[0]),
        "entailment": float(probabilities[1]),
        "neutral": float(probabilities[2])
    }


# ============================================================
# NLI Sanity Test
# ============================================================

test_evidence = (
    "Now, we're developing a remote control, "
    "which you probably already know."
)

test_proposition = (
    "The team is developing a remote control."
)

nli_test = run_nli(
    premise=test_evidence,
    hypothesis=test_proposition
)

print("=" * 90)
print("M7 - NLI SANITY TEST")
print("=" * 90)

print(f"Premise    : {test_evidence}")
print(f"Hypothesis : {test_proposition}")

print("\nResult:")
print(f"Label         : {nli_test['label']}")
print(f"Contradiction : {nli_test['contradiction']:.4f}")
print(f"Entailment    : {nli_test['entailment']:.4f}")
print(f"Neutral       : {nli_test['neutral']:.4f}")

print("\nNLI function restored successfully.")

M7 - NLI SANITY TEST
Premise    : Now, we're developing a remote control, which you probably already know.
Hypothesis : The team is developing a remote control.

Result:
Label         : ENTAILMENT
Contradiction : 0.0005
Entailment    : 0.7902
Neutral       : 0.2093

NLI function restored successfully.


In [11]:
# ============================================================
# M7 - Restore Context Window Function
# ============================================================
# Purpose:
#   Retrieve neighboring meeting utterances around a selected
#   evidence utterance.
#
# Example:
#   Evidence ID 15 with window_size=2
#   → IDs 13, 14, 15, 16, 17
#
# No models are loaded and no files are modified.
# ============================================================

def get_context_window(
    evidence_id,
    window_size=2
):
    """
    Return the evidence utterance together with neighboring
    utterances.

    Parameters
    ----------
    evidence_id : int
        Central evidence utterance.

    window_size : int
        Number of utterances before and after the evidence.

    Returns
    -------
    list
        Evidence documents ordered chronologically.
    """

    evidence_id = int(evidence_id)

    # Locate the evidence position
    evidence_positions = {
        int(doc["evidence_id"]): index
        for index, doc in enumerate(evidence_documents)
    }

    if evidence_id not in evidence_positions:
        return []

    center_index = evidence_positions[evidence_id]

    start_index = max(
        0,
        center_index - window_size
    )

    end_index = min(
        len(evidence_documents),
        center_index + window_size + 1
    )

    context_docs = evidence_documents[
        start_index:end_index
    ]

    return sorted(
        context_docs,
        key=lambda x: x["start"]
    )


# ============================================================
# Test
# ============================================================

test_context = get_context_window(
    evidence_id=15,
    window_size=2
)

print("=" * 90)
print("M7 - CONTEXT WINDOW TEST")
print("=" * 90)

print(
    f"Central evidence ID: 15"
)

print(
    f"Context IDs: "
    f"{[doc['evidence_id'] for doc in test_context]}"
)

print("\nContext:")

for doc in test_context:

    print(
        f"\nID {doc['evidence_id']} | "
        f"{doc['speaker']} | "
        f"{doc['start']:.3f} → {doc['end']:.3f}"
    )

    print(
        f"  {doc['text']}"
    )

print("\nContext window function restored successfully.")

M7 - CONTEXT WINDOW TEST
Central evidence ID: 15
Context IDs: [13, 14, 15, 16, 17]

Context:

ID 13 | SPEAKER_02 | 112.185 → 115.748
  And we've got 25 minutes to do that as far as I can understand.

ID 14 | SPEAKER_02 | 117.699 → 120.921
  Now, we're developing a remote control, which you probably already know.

ID 15 | SPEAKER_02 | 122.503 → 132.290
  We want it to be original, something that people haven't thought of. It's not out in the shops. Trendy, appealing to a wide market, but,

ID 16 | SPEAKER_02 | 133.291 → 138.174
  you know, not a hunk of metal. And user-friendly, grannies to kids,

ID 17 | SPEAKER_02 | 139.375 → 141.016
  maybe even pooches, should be able to use it.

Context window function restored successfully.


In [12]:
# ============================================================
# M7 - Restore Lexical Support Function
# ============================================================
# Purpose:
#   Measure simple lexical overlap between a proposition and
#   evidence/context text.
#
# Important:
#   Lexical overlap is ONLY a supporting signal.
#   It is NOT used as proof of factual correctness.
# ============================================================

import re


def lexical_support(
    proposition,
    evidence_text
):
    """
    Calculate lexical overlap between a proposition and
    evidence text.

    Returns
    -------
    dict
        overlap ratio, matched terms, and missing terms.
    """

    # --------------------------------------------------------
    # Normalize text
    # --------------------------------------------------------

    proposition_words = re.findall(
        r"\b[a-zA-Z]+\b",
        proposition.lower()
    )

    evidence_words = re.findall(
        r"\b[a-zA-Z]+\b",
        evidence_text.lower()
    )

    # Remove common stopwords
    stopwords = {
        "the", "a", "an", "is", "are", "was", "were",
        "be", "been", "being", "to", "of", "and", "or",
        "in", "on", "for", "with", "by", "from", "that",
        "this", "it", "its", "as", "at", "but", "should",
        "would", "could", "can", "will", "may", "might",
        "has", "have", "had", "do", "does", "did",
        "they", "them", "their", "we", "our", "you",
        "your", "he", "she", "his", "her", "i", "me",
        "my", "us"
    }

    proposition_terms = [
        word
        for word in proposition_words
        if word not in stopwords
    ]

    evidence_term_set = set(
        word
        for word in evidence_words
        if word not in stopwords
    )

    # --------------------------------------------------------
    # Calculate matched and missing terms
    # --------------------------------------------------------

    matched_terms = sorted(
        set(proposition_terms) &
        evidence_term_set
    )

    missing_terms = sorted(
        set(proposition_terms) -
        evidence_term_set
    )

    # --------------------------------------------------------
    # Calculate overlap ratio
    # --------------------------------------------------------

    if proposition_terms:

        overlap_ratio = (
            len(matched_terms) /
            len(set(proposition_terms))
        )

    else:

        overlap_ratio = 0.0

    return {
        "lexical_overlap": float(overlap_ratio),
        "matched_terms": matched_terms,
        "missing_terms": missing_terms
    }


# ============================================================
# Test with the originality evidence group
# ============================================================

test_proposition = (
    "The remote control is intended to be original."
)

test_evidence_text = (
    "Now, we're developing a remote control, "
    "which you probably already know. "
    "We want it to be original, something that "
    "people haven't thought of. It's not out in "
    "the shops. Trendy, appealing to a wide market."
)

lexical_test = lexical_support(
    proposition=test_proposition,
    evidence_text=test_evidence_text
)

print("=" * 90)
print("M7 - LEXICAL SUPPORT TEST")
print("=" * 90)

print(f"Proposition: {test_proposition}")

print(
    f"\nLexical overlap: "
    f"{lexical_test['lexical_overlap']:.4f}"
)

print(
    f"Matched terms: "
    f"{lexical_test['matched_terms']}"
)

print(
    f"Missing terms: "
    f"{lexical_test['missing_terms']}"
)

print("\nLexical support function restored successfully.")

M7 - LEXICAL SUPPORT TEST
Proposition: The remote control is intended to be original.

Lexical overlap: 0.7500
Matched terms: ['control', 'original', 'remote']
Missing terms: ['intended']

Lexical support function restored successfully.


In [13]:
# ============================================================
# M7 - Restore Attribute Evidence Analysis
# ============================================================
# Purpose:
#   Analyze retrieved evidence for one atomic proposition.
#
# Pipeline:
#   Proposition
#       ↓
#   BGE retrieval
#       ↓
#   Context expansion
#       ↓
#   Contextual NLI
#       ↓
#   Lexical support
#       ↓
#   Speaker consistency
#
# BGE similarity is retrieval-only and is NOT treated as proof.
# ============================================================


def analyze_attribute_evidence(
    proposition,
    claim_speaker=None,
    top_k=10,
    context_window=2
):
    """
    Analyze BGE-retrieved evidence for an atomic proposition.

    Parameters
    ----------
    proposition : str
        Atomic proposition to verify.

    claim_speaker : str or None
        Expected speaker, if speaker attribution is applicable.

    top_k : int
        Number of BGE candidates to inspect.

    context_window : int
        Number of neighboring utterances on each side.

    Returns
    -------
    list
        Candidate evidence analysis results.
    """

    # --------------------------------------------------------
    # 1. Retrieve evidence using BGE + FAISS
    # --------------------------------------------------------

    retrieval_results = retrieve_evidence(
        query_text=proposition,
        top_k=top_k
    )

    analysis_results = []

    # --------------------------------------------------------
    # 2. Analyze every retrieved candidate
    # --------------------------------------------------------

    for candidate in retrieval_results:

        evidence_id = int(
            candidate["evidence_id"]
        )

        # ----------------------------------------------------
        # Get conversational context
        # ----------------------------------------------------

        context_docs = get_context_window(
            evidence_id=evidence_id,
            window_size=context_window
        )

        context_ids = [
            int(doc["evidence_id"])
            for doc in context_docs
        ]

        context_text = " ".join(
            doc["text"].strip()
            for doc in context_docs
        )

        # ----------------------------------------------------
        # Contextual NLI
        # ----------------------------------------------------
        # Premise = meeting context
        # Hypothesis = generated proposition
        # ----------------------------------------------------

        nli_result = run_nli(
            premise=context_text,
            hypothesis=proposition
        )

        # ----------------------------------------------------
        # Lexical support
        # ----------------------------------------------------

        lexical_result = lexical_support(
            proposition=proposition,
            evidence_text=context_text
        )

        # ----------------------------------------------------
        # Speaker consistency
        # ----------------------------------------------------

        evidence_speaker = candidate["speaker"]

        if claim_speaker is None:

            speaker_result = {
                "applicable": False,
                "passed": True,
                "reason": "No speaker attribution specified."
            }

        elif evidence_speaker == claim_speaker:

            speaker_result = {
                "applicable": True,
                "passed": True,
                "reason": "Speaker matches evidence."
            }

        else:

            speaker_result = {
                "applicable": True,
                "passed": False,
                "reason": "Speaker does not match evidence."
            }

        # ----------------------------------------------------
        # Store complete candidate analysis
        # ----------------------------------------------------

        analysis_results.append({

            "evidence_id": evidence_id,

            "speaker": evidence_speaker,

            "start": candidate["start"],

            "end": candidate["end"],

            "text": candidate["text"],

            "retrieval_similarity":
                candidate["retrieval_similarity"],

            "nli_label":
                nli_result["label"],

            "nli_entailment":
                nli_result["entailment"],

            "nli_contradiction":
                nli_result["contradiction"],

            "nli_neutral":
                nli_result["neutral"],

            "lexical_overlap":
                lexical_result["lexical_overlap"],

            "matched_terms":
                lexical_result["matched_terms"],

            "missing_terms":
                lexical_result["missing_terms"],

            "speaker_check":
                speaker_result,

            "context_evidence_ids":
                context_ids,

            "context_text":
                context_text
        })

    return analysis_results


# ============================================================
# Lightweight test
# ============================================================

test_analysis = analyze_attribute_evidence(
    proposition="The remote control is intended to be original.",
    claim_speaker="SPEAKER_02",
    top_k=10,
    context_window=2
)

print("=" * 90)
print("M7 - ATTRIBUTE EVIDENCE ANALYSIS TEST")
print("=" * 90)

print(
    f"Candidates analyzed: "
    f"{len(test_analysis)}"
)

print("\nTop candidates:")

for item in test_analysis[:5]:

    print(
        f"\nID {item['evidence_id']} | "
        f"BGE={item['retrieval_similarity']:.4f} | "
        f"NLI={item['nli_label']} "
        f"{item['nli_entailment']:.4f}"
    )

    print(
        f"Speaker: {item['speaker']} | "
        f"Speaker check: "
        f"{item['speaker_check']['passed']}"
    )

    print(
        f"Context IDs: "
        f"{item['context_evidence_ids']}"
    )

print("\nAttribute evidence analysis function restored successfully.")

M7 - ATTRIBUTE EVIDENCE ANALYSIS TEST
Candidates analyzed: 10

Top candidates:

ID 105 | BGE=0.7917 | NLI=NEUTRAL 0.0013
Speaker: SPEAKER_01 | Speaker check: False
Context IDs: [103, 104, 105, 106, 107]

ID 14 | BGE=0.7810 | NLI=ENTAILMENT 0.9812
Speaker: SPEAKER_02 | Speaker check: True
Context IDs: [12, 13, 14, 15, 16]

ID 107 | BGE=0.7408 | NLI=NEUTRAL 0.0017
Speaker: SPEAKER_03 | Speaker check: False
Context IDs: [105, 106, 107, 108, 109]

ID 145 | BGE=0.7396 | NLI=NEUTRAL 0.0004
Speaker: SPEAKER_02 | Speaker check: True
Context IDs: [143, 144, 145, 146, 147]

ID 102 | BGE=0.7126 | NLI=NEUTRAL 0.0013
Speaker: SPEAKER_02 | Speaker check: True
Context IDs: [100, 101, 102, 103, 104]

Attribute evidence analysis function restored successfully.


In [14]:
# ============================================================
# M7 - Restore Supporting Candidate Selection
# ============================================================
# Purpose:
#   Select evidence candidates that can support an atomic
#   proposition after BGE retrieval and contextual NLI.
#
# Selection rules:
#   1. Reject CONTRADICTION
#   2. Reject NEUTRAL
#   3. Require ENTAILMENT
#   4. Reject speaker mismatch when speaker attribution
#      is applicable
#
# BGE similarity is used for ranking only.
# ============================================================


def select_supporting_candidates(
    analysis_result,
    min_lexical_overlap=0.0
):
    """
    Select candidates that provide potential support
    for the proposition.

    Parameters
    ----------
    analysis_result : list
        Output from analyze_attribute_evidence().

    min_lexical_overlap : float
        Optional minimum lexical overlap.

    Returns
    -------
    dict
        supporting_candidates and rejected_candidates.
    """

    supporting_candidates = []
    rejected_candidates = []

    # --------------------------------------------------------
    # Evaluate every retrieved candidate
    # --------------------------------------------------------

    for item in analysis_result:

        evidence_id = item["evidence_id"]

        nli_label = item["nli_label"]

        speaker_check = item["speaker_check"]

        lexical_overlap = item["lexical_overlap"]

        # ----------------------------------------------------
        # Reject contradiction
        # ----------------------------------------------------

        if nli_label == "CONTRADICTION":

            rejected_candidates.append({
                "evidence_id": evidence_id,
                "reason": "NLI contradiction"
            })

            continue

        # ----------------------------------------------------
        # Reject neutral
        # ----------------------------------------------------

        if nli_label == "NEUTRAL":

            rejected_candidates.append({
                "evidence_id": evidence_id,
                "reason": "NLI neutral"
            })

            continue

        # ----------------------------------------------------
        # Require entailment
        # ----------------------------------------------------

        if nli_label != "ENTAILMENT":

            rejected_candidates.append({
                "evidence_id": evidence_id,
                "reason": "NLI did not establish entailment"
            })

            continue

        # ----------------------------------------------------
        # Speaker consistency
        # ----------------------------------------------------

        if speaker_check["applicable"]:

            if not speaker_check["passed"]:

                rejected_candidates.append({
                    "evidence_id": evidence_id,
                    "reason": "Speaker mismatch"
                })

                continue

        # ----------------------------------------------------
        # Optional lexical-support threshold
        # ----------------------------------------------------

        if lexical_overlap < min_lexical_overlap:

            rejected_candidates.append({
                "evidence_id": evidence_id,
                "reason": "Insufficient lexical support"
            })

            continue

        # ----------------------------------------------------
        # Candidate passed all selection rules
        # ----------------------------------------------------

        supporting_candidates.append(item)

    # --------------------------------------------------------
    # Rank supporting candidates
    # --------------------------------------------------------
    # NLI entailment is the primary ranking signal.
    # BGE similarity is secondary.
    # --------------------------------------------------------

    supporting_candidates = sorted(
        supporting_candidates,
        key=lambda x: (
            x["nli_entailment"],
            x["retrieval_similarity"]
        ),
        reverse=True
    )

    return {
        "supporting_candidates": supporting_candidates,
        "rejected_candidates": rejected_candidates
    }


# ============================================================
# Test
# ============================================================

test_selection = select_supporting_candidates(
    test_analysis,
    min_lexical_overlap=0.0
)

print("=" * 90)
print("M7 - SUPPORTING CANDIDATE SELECTION TEST")
print("=" * 90)

print(
    f"Supporting candidates: "
    f"{len(test_selection['supporting_candidates'])}"
)

print(
    f"Rejected candidates: "
    f"{len(test_selection['rejected_candidates'])}"
)

print("\nSupporting candidates:")

for item in test_selection[
    "supporting_candidates"
]:

    print(
        f"  ID {item['evidence_id']} | "
        f"BGE={item['retrieval_similarity']:.4f} | "
        f"NLI={item['nli_label']} "
        f"{item['nli_entailment']:.4f} | "
        f"Speaker={item['speaker']}"
    )

print("\nRejected candidates:")

for item in test_selection[
    "rejected_candidates"
]:

    print(
        f"  ID {item['evidence_id']} | "
        f"{item['reason']}"
    )

print("\nCandidate selection function restored successfully.")

M7 - SUPPORTING CANDIDATE SELECTION TEST
Supporting candidates: 2
Rejected candidates: 8

Supporting candidates:
  ID 14 | BGE=0.7810 | NLI=ENTAILMENT 0.9812 | Speaker=SPEAKER_02
  ID 15 | BGE=0.6708 | NLI=ENTAILMENT 0.9787 | Speaker=SPEAKER_02

Rejected candidates:
  ID 105 | NLI neutral
  ID 107 | NLI neutral
  ID 145 | NLI neutral
  ID 102 | NLI neutral
  ID 135 | NLI neutral
  ID 117 | NLI neutral
  ID 112 | NLI neutral
  ID 134 | NLI neutral

Candidate selection function restored successfully.


In [15]:
# ============================================================
# M7 - Restore Evidence Group Builder
# ============================================================
# Purpose:
#   Combine multiple supporting utterances and their
#   surrounding context into one structured evidence group.
#
# Important distinction:
#   Supporting evidence = directly contributes to proposition
#   Context evidence   = surrounding conversation
# ============================================================


def create_evidence_group(
    supporting_evidence_ids,
    context_evidence_ids
):
    """
    Build a structured evidence group.

    Parameters
    ----------
    supporting_evidence_ids : list
        Evidence IDs that directly support the proposition.

    context_evidence_ids : list
        Evidence IDs providing conversational context.

    Returns
    -------
    dict
        Structured evidence group.
    """

    supporting_ids = [
        int(x) for x in supporting_evidence_ids
    ]

    context_ids = [
        int(x) for x in context_evidence_ids
    ]

    # --------------------------------------------------------
    # Evidence lookup
    # --------------------------------------------------------

    evidence_lookup = {
        int(doc["evidence_id"]): doc
        for doc in evidence_documents
    }

    # --------------------------------------------------------
    # Retrieve supporting documents
    # --------------------------------------------------------

    supporting_docs = [
        evidence_lookup[eid]
        for eid in supporting_ids
        if eid in evidence_lookup
    ]

    # --------------------------------------------------------
    # Retrieve context documents
    # --------------------------------------------------------

    context_docs = [
        evidence_lookup[eid]
        for eid in context_ids
        if eid in evidence_lookup
    ]

    # --------------------------------------------------------
    # Sort chronologically
    # --------------------------------------------------------

    supporting_docs = sorted(
        supporting_docs,
        key=lambda x: x["start"]
    )

    context_docs = sorted(
        context_docs,
        key=lambda x: x["start"]
    )

    # --------------------------------------------------------
    # Display interval
    # --------------------------------------------------------
    # The displayed interval covers the supporting evidence.
    # Context remains available separately for verification.
    # --------------------------------------------------------

    display_docs = (
        supporting_docs
        if supporting_docs
        else context_docs
    )

    if display_docs:

        display_start = min(
            doc["start"]
            for doc in display_docs
        )

        display_end = max(
            doc["end"]
            for doc in display_docs
        )

    else:

        display_start = None
        display_end = None

    # --------------------------------------------------------
    # Unique speakers
    # --------------------------------------------------------

    speakers = sorted(
        set(
            doc["speaker"]
            for doc in supporting_docs
            if doc.get("speaker") is not None
        )
    )

    # --------------------------------------------------------
    # Combined context text
    # --------------------------------------------------------

    context_text = " ".join(
        doc["text"].strip()
        for doc in context_docs
    )

    return {

        "supporting_evidence_ids": [
            int(doc["evidence_id"])
            for doc in supporting_docs
        ],

        "supporting_evidence": supporting_docs,

        "context_evidence_ids": [
            int(doc["evidence_id"])
            for doc in context_docs
        ],

        "context_start": display_start,

        "context_end": display_end,

        "context_text": context_text,

        "speakers": speakers
    }


# ============================================================
# Test
# ============================================================

test_group = create_evidence_group(
    supporting_evidence_ids=[14, 15],
    context_evidence_ids=[12, 13, 14, 15, 16, 17]
)

print("=" * 90)
print("M7 - EVIDENCE GROUP RESTORATION TEST")
print("=" * 90)

print(
    f"Supporting IDs : "
    f"{test_group['supporting_evidence_ids']}"
)

print(
    f"Context IDs    : "
    f"{test_group['context_evidence_ids']}"
)

print(
    f"Display interval: "
    f"{test_group['context_start']:.3f} → "
    f"{test_group['context_end']:.3f}"
)

print(
    f"Speakers       : "
    f"{test_group['speakers']}"
)

print("\nEvidence group builder restored successfully.")

M7 - EVIDENCE GROUP RESTORATION TEST
Supporting IDs : [14, 15]
Context IDs    : [12, 13, 14, 15, 16, 17]
Display interval: 117.699 → 132.290
Speakers       : ['SPEAKER_02']

Evidence group builder restored successfully.


In [16]:
# ============================================================
# M7 - Restore Complete Attribute Verification
# ============================================================
# Purpose:
#   Combine retrieval, contextual NLI, candidate selection,
#   evidence grouping, and speaker consistency into one
#   reusable verification function.
# ============================================================


def verify_attribute(
    attribute,
    proposition,
    claim_speaker=None,
    top_k=10,
    context_window=2
):
    """
    Verify one atomic MoM proposition.

    Parameters
    ----------
    attribute : str
        Attribute being verified.

    proposition : str
        Atomic proposition.

    claim_speaker : str or None
        Expected speaker, if applicable.

    top_k : int
        Number of BGE candidates to inspect.

    context_window : int
        Number of neighboring utterances on each side.

    Returns
    -------
    dict
        Complete verification result.
    """

    # --------------------------------------------------------
    # 1. Analyze retrieved evidence
    # --------------------------------------------------------

    analysis = analyze_attribute_evidence(
        proposition=proposition,
        claim_speaker=claim_speaker,
        top_k=top_k,
        context_window=context_window
    )

    # --------------------------------------------------------
    # 2. Select supporting candidates
    # --------------------------------------------------------

    selection = select_supporting_candidates(
        analysis,
        min_lexical_overlap=0.0
    )

    supporting_candidates = selection[
        "supporting_candidates"
    ]

    rejected_candidates = selection[
        "rejected_candidates"
    ]

    # --------------------------------------------------------
    # 3. No supporting evidence
    # --------------------------------------------------------

    if not supporting_candidates:

        return {
            "attribute": attribute,
            "proposition": proposition,
            "status": "FLAGGED",
            "reason": "No supporting evidence found.",

            "supporting_evidence_ids": [],
            "context_evidence_ids": [],

            "display_start": None,
            "display_end": None,

            "speakers": [],

            "speaker_check": False,

            "candidates_checked": len(analysis),

            "supporting_candidates": [],
            "rejected_candidates": rejected_candidates
        }

    # --------------------------------------------------------
    # 4. Supporting evidence IDs
    # --------------------------------------------------------

    supporting_ids = [
        int(item["evidence_id"])
        for item in supporting_candidates
    ]

    # --------------------------------------------------------
    # 5. Collect context IDs
    # --------------------------------------------------------

    context_ids = set()

    for item in supporting_candidates:

        for evidence_id in item.get(
            "context_evidence_ids",
            []
        ):
            context_ids.add(int(evidence_id))

    context_ids = sorted(context_ids)

    # --------------------------------------------------------
    # 6. Build evidence group
    # --------------------------------------------------------

    evidence_group = create_evidence_group(
        supporting_evidence_ids=supporting_ids,
        context_evidence_ids=context_ids
    )

    # --------------------------------------------------------
    # 7. Speaker consistency
    # --------------------------------------------------------

    speaker_check = True

    if claim_speaker is not None:

        evidence_speakers = evidence_group[
            "speakers"
        ]

        if evidence_speakers:

            speaker_check = (
                claim_speaker
                in evidence_speakers
            )

    # --------------------------------------------------------
    # 8. Final verification status
    # --------------------------------------------------------

    if speaker_check:

        status = "VERIFIED"

        reason = (
            "The proposition is supported by contextual "
            "NLI evidence and the claimed speaker is "
            "consistent with the supporting evidence."
        )

    else:

        status = "FLAGGED"

        reason = (
            "Supporting evidence was found, but the "
            "claimed speaker does not match the "
            "supporting evidence."
        )

    # --------------------------------------------------------
    # 9. Return complete result
    # --------------------------------------------------------

    return {

        "attribute": attribute,

        "proposition": proposition,

        "status": status,

        "reason": reason,

        "supporting_evidence_ids":
            evidence_group[
                "supporting_evidence_ids"
            ],

        "supporting_evidence":
            evidence_group[
                "supporting_evidence"
            ],

        "context_evidence_ids":
            evidence_group[
                "context_evidence_ids"
            ],

        "display_start":
            evidence_group[
                "context_start"
            ],

        "display_end":
            evidence_group[
                "context_end"
            ],

        "context_text":
            evidence_group[
                "context_text"
            ],

        "speakers":
            evidence_group[
                "speakers"
            ],

        "speaker_check":
            speaker_check,

        "candidates_checked":
            len(analysis),

        "supporting_candidates":
            supporting_candidates,

        "rejected_candidates":
            rejected_candidates
    }


# ============================================================
# End-to-End Test
# ============================================================

test_verification = verify_attribute(
    attribute="originality",
    proposition=(
        "The remote control is intended to be original."
    ),
    claim_speaker="SPEAKER_02",
    top_k=10,
    context_window=2
)

print("=" * 90)
print("M7 - COMPLETE ATTRIBUTE VERIFICATION TEST")
print("=" * 90)

print(
    f"Attribute       : "
    f"{test_verification['attribute']}"
)

print(
    f"Proposition     : "
    f"{test_verification['proposition']}"
)

print(
    f"Status          : "
    f"{test_verification['status']}"
)

print(
    f"Speaker check   : "
    f"{test_verification['speaker_check']}"
)

print(
    f"Supporting IDs  : "
    f"{test_verification['supporting_evidence_ids']}"
)

print(
    f"Context IDs     : "
    f"{test_verification['context_evidence_ids']}"
)

print(
    f"Display interval: "
    f"{test_verification['display_start']:.3f} → "
    f"{test_verification['display_end']:.3f}"
)

print(
    f"Speakers        : "
    f"{test_verification['speakers']}"
)

print(
    f"Candidates checked: "
    f"{test_verification['candidates_checked']}"
)

print(
    f"\nReason:\n"
    f"{test_verification['reason']}"
)

print("\nSupporting evidence:")

for item in test_verification[
    "supporting_candidates"
]:

    print(
        f"  ID {item['evidence_id']} | "
        f"BGE={item['retrieval_similarity']:.4f} | "
        f"NLI={item['nli_label']} "
        f"{item['nli_entailment']:.4f} | "
        f"Speaker={item['speaker']}"
    )

print("\n" + "=" * 90)
print("COMPLETE ATTRIBUTE VERIFICATION TEST FINISHED")
print("=" * 90)

M7 - COMPLETE ATTRIBUTE VERIFICATION TEST
Attribute       : originality
Proposition     : The remote control is intended to be original.
Status          : VERIFIED
Speaker check   : True
Supporting IDs  : [14, 15]
Context IDs     : [12, 13, 14, 15, 16, 17]
Display interval: 117.699 → 132.290
Speakers        : ['SPEAKER_02']
Candidates checked: 10

Reason:
The proposition is supported by contextual NLI evidence and the claimed speaker is consistent with the supporting evidence.

Supporting evidence:
  ID 14 | BGE=0.7810 | NLI=ENTAILMENT 0.9812 | Speaker=SPEAKER_02
  ID 15 | BGE=0.6708 | NLI=ENTAILMENT 0.9787 | Speaker=SPEAKER_02

COMPLETE ATTRIBUTE VERIFICATION TEST FINISHED


In [17]:
# ============================================================
# M7 - Improved Speaker Consistency Verification
# ============================================================
# Purpose:
#   Separate content evidence from speaker attribution.
#
# Important:
#   We should NOT discard valid textual evidence merely because
#   the claimed speaker is incorrect.
#
# Desired behavior:
#
#   Correct speaker:
#       Content = PASS
#       Speaker = PASS
#       Final   = VERIFIED
#
#   Wrong speaker:
#       Content = PASS
#       Speaker = FAIL
#       Final   = FLAGGED
# ============================================================


def check_speaker_consistency(
    claim_speaker,
    evidence_speakers
):
    """
    Check whether the claimed speaker is consistent with
    the speakers present in the supporting evidence.
    """

    # --------------------------------------------------------
    # Speaker attribution not applicable
    # --------------------------------------------------------

    if claim_speaker is None:

        return {
            "applicable": False,
            "passed": True,
            "reason": "No speaker attribution specified."
        }

    # --------------------------------------------------------
    # No evidence speakers available
    # --------------------------------------------------------

    if not evidence_speakers:

        return {
            "applicable": True,
            "passed": False,
            "reason": "No evidence speaker available."
        }

    # --------------------------------------------------------
    # Speaker match
    # --------------------------------------------------------

    if claim_speaker in evidence_speakers:

        return {
            "applicable": True,
            "passed": True,
            "reason": "Claimed speaker matches supporting evidence."
        }

    # --------------------------------------------------------
    # Speaker mismatch
    # --------------------------------------------------------

    return {
        "applicable": True,
        "passed": False,
        "reason": (
            f"Claimed speaker '{claim_speaker}' does not "
            f"match supporting evidence speakers "
            f"{evidence_speakers}."
        )
    }


# ============================================================
# Test 1 - Correct speaker
# ============================================================

correct_speaker_result = check_speaker_consistency(
    claim_speaker="SPEAKER_02",
    evidence_speakers=["SPEAKER_02"]
)

# ============================================================
# Test 2 - Incorrect speaker
# ============================================================

wrong_speaker_result = check_speaker_consistency(
    claim_speaker="SPEAKER_03",
    evidence_speakers=["SPEAKER_02"]
)

# ============================================================
# Display results
# ============================================================

print("=" * 90)
print("SPEAKER CONSISTENCY TEST")
print("=" * 90)

print("\nCorrect speaker:")
print(correct_speaker_result)

print("\nWrong speaker:")
print(wrong_speaker_result)

print("\nSpeaker consistency function ready.")

SPEAKER CONSISTENCY TEST

Correct speaker:
{'applicable': True, 'passed': True, 'reason': 'Claimed speaker matches supporting evidence.'}

Wrong speaker:
{'applicable': True, 'passed': False, 'reason': "Claimed speaker 'SPEAKER_03' does not match supporting evidence speakers ['SPEAKER_02']."}

Speaker consistency function ready.


In [18]:
# ============================================================
# M7 - Timestamp Consistency Verification
# ============================================================
# Purpose:
#   Check whether evidence timestamps are consistent with
#   an expected temporal reference.
#
# Important:
#   Timestamps are only checked when the claim provides an
#   expected temporal reference.
#
# A tolerance is used because ASR/diarization timestamps
# are approximate.
# ============================================================


def check_timestamp_consistency(
    evidence_start,
    evidence_end,
    expected_start=None,
    expected_end=None,
    tolerance_seconds=2.0
):
    """
    Check temporal consistency between evidence and an
    expected timestamp interval.

    Parameters
    ----------
    evidence_start : float
        Start timestamp of supporting evidence.

    evidence_end : float
        End timestamp of supporting evidence.

    expected_start : float or None
        Expected start timestamp.

    expected_end : float or None
        Expected end timestamp.

    tolerance_seconds : float
        Allowed timestamp difference.

    Returns
    -------
    dict
        Timestamp consistency result.
    """

    # --------------------------------------------------------
    # Timestamp check not applicable
    # --------------------------------------------------------

    if expected_start is None and expected_end is None:

        return {
            "applicable": False,
            "passed": True,
            "reason": (
                "No expected timestamp specified; "
                "timestamp consistency is not applicable."
            )
        }

    # --------------------------------------------------------
    # Validate evidence timestamps
    # --------------------------------------------------------

    if evidence_start is None or evidence_end is None:

        return {
            "applicable": True,
            "passed": False,
            "reason": "Evidence timestamps are unavailable."
        }

    # --------------------------------------------------------
    # Compare start timestamp
    # --------------------------------------------------------

    start_match = True

    if expected_start is not None:

        start_match = (
            abs(
                float(evidence_start)
                - float(expected_start)
            )
            <= tolerance_seconds
        )

    # --------------------------------------------------------
    # Compare end timestamp
    # --------------------------------------------------------

    end_match = True

    if expected_end is not None:

        end_match = (
            abs(
                float(evidence_end)
                - float(expected_end)
            )
            <= tolerance_seconds
        )

    # --------------------------------------------------------
    # Final timestamp result
    # --------------------------------------------------------

    passed = start_match and end_match

    if passed:

        reason = (
            "Evidence timestamps are consistent with "
            "the expected temporal reference."
        )

    else:

        reason = (
            "Evidence timestamps are inconsistent with "
            "the expected temporal reference."
        )

    return {
        "applicable": True,
        "passed": passed,
        "reason": reason,
        "evidence_start": float(evidence_start),
        "evidence_end": float(evidence_end),
        "expected_start": (
            float(expected_start)
            if expected_start is not None
            else None
        ),
        "expected_end": (
            float(expected_end)
            if expected_end is not None
            else None
        ),
        "tolerance_seconds": tolerance_seconds
    }


# ============================================================
# Test 1 - No expected timestamp
# ============================================================

timestamp_test_none = check_timestamp_consistency(
    evidence_start=117.699,
    evidence_end=132.290
)

# ============================================================
# Test 2 - Matching timestamp
# ============================================================

timestamp_test_match = check_timestamp_consistency(
    evidence_start=117.699,
    evidence_end=132.290,
    expected_start=117.699,
    expected_end=132.290
)

# ============================================================
# Test 3 - Deliberately incorrect timestamp
# ============================================================

timestamp_test_wrong = check_timestamp_consistency(
    evidence_start=117.699,
    evidence_end=132.290,
    expected_start=500.0,
    expected_end=510.0
)

# ============================================================
# Display results
# ============================================================

print("=" * 90)
print("TIMESTAMP CONSISTENCY TEST")
print("=" * 90)

print("\nTest 1 - No expected timestamp:")
print(timestamp_test_none)

print("\nTest 2 - Matching timestamp:")
print(timestamp_test_match)

print("\nTest 3 - Incorrect timestamp:")
print(timestamp_test_wrong)

print("\nTimestamp consistency function ready.")

TIMESTAMP CONSISTENCY TEST

Test 1 - No expected timestamp:
{'applicable': False, 'passed': True, 'reason': 'No expected timestamp specified; timestamp consistency is not applicable.'}

Test 2 - Matching timestamp:
{'applicable': True, 'passed': True, 'reason': 'Evidence timestamps are consistent with the expected temporal reference.', 'evidence_start': 117.699, 'evidence_end': 132.29, 'expected_start': 117.699, 'expected_end': 132.29, 'tolerance_seconds': 2.0}

Test 3 - Incorrect timestamp:
{'applicable': True, 'passed': False, 'reason': 'Evidence timestamps are inconsistent with the expected temporal reference.', 'evidence_start': 117.699, 'evidence_end': 132.29, 'expected_start': 500.0, 'expected_end': 510.0, 'tolerance_seconds': 2.0}

Timestamp consistency function ready.


In [19]:
# ============================================================
# M7 - EVENT / DECISION CONSISTENCY
# ============================================================

def check_event_type_consistency(claim_type, evidence_texts):
    """
    Check whether the evidence supports the claimed event type.

    Supported claim types:
        ACTION
        DECISION
        DISCUSSION
        INFORMATION

    This is a lightweight rule-based consistency check.
    It is intended as a verification signal, not as a standalone
    classifier.
    """

    if claim_type is None:
        return {
            "applicable": False,
            "passed": True,
            "reason": "No claim type specified; event type consistency is not applicable."
        }

    claim_type = claim_type.upper().strip()

    if not evidence_texts:
        return {
            "applicable": True,
            "passed": False,
            "reason": "No evidence text available for event type verification."
        }

    combined_text = " ".join(evidence_texts).lower()

    # --------------------------------------------------------
    # Type-specific indicative patterns
    # --------------------------------------------------------

    action_terms = [
        "will", "i'll", "we'll", "shall",
        "need to", "have to", "going to",
        "before the next meeting",
        "by next meeting"
    ]

    decision_terms = [
        "decided", "decision", "agreed",
        "we will use", "we chose",
        "we choose", "let's use",
        "final decision"
    ]

    discussion_terms = [
        "discuss", "discussion", "talked",
        "consider", "considering",
        "question", "idea", "suggest"
    ]

    information_terms = [
        "is", "are", "was", "were",
        "means", "includes", "consists",
        "the meeting", "the project"
    ]

    term_groups = {
        "ACTION": action_terms,
        "DECISION": decision_terms,
        "DISCUSSION": discussion_terms,
        "INFORMATION": information_terms
    }

    matched_terms = [
        term for term in term_groups[claim_type]
        if term in combined_text
    ]

    # --------------------------------------------------------
    # Basic decision rule
    # --------------------------------------------------------

    passed = len(matched_terms) > 0

    if passed:
        reason = (
            f"Evidence contains indicators consistent with "
            f"the claimed event type '{claim_type}'."
        )
    else:
        reason = (
            f"Evidence does not contain clear indicators "
            f"supporting the claimed event type '{claim_type}'."
        )

    return {
        "applicable": True,
        "passed": passed,
        "claim_type": claim_type,
        "matched_terms": matched_terms,
        "reason": reason
    }


# ============================================================
# TEST EVENT / DECISION CONSISTENCY
# ============================================================

print("=" * 90)
print("EVENT / DECISION CONSISTENCY TEST")
print("=" * 90)

# Test 1 - ACTION
test_1 = check_event_type_consistency(
    "ACTION",
    ["Participants planned to work on their individual tasks before the next meeting."]
)

print("\nTest 1 - ACTION:")
print(test_1)


# Test 2 - DECISION
test_2 = check_event_type_consistency(
    "DECISION",
    ["The team decided to use the LCD display for the product."]
)

print("\nTest 2 - DECISION:")
print(test_2)


# Test 3 - DISCUSSION
test_3 = check_event_type_consistency(
    "DISCUSSION",
    ["The participants discussed different options for the product design."]
)

print("\nTest 3 - DISCUSSION:")
print(test_3)


# Test 4 - Incorrect event type
test_4 = check_event_type_consistency(
    "DECISION",
    ["The participants discussed different options for the product design."]
)

print("\nTest 4 - Incorrect DECISION:")
print(test_4)

EVENT / DECISION CONSISTENCY TEST

Test 1 - ACTION:
{'applicable': True, 'passed': True, 'claim_type': 'ACTION', 'matched_terms': ['before the next meeting'], 'reason': "Evidence contains indicators consistent with the claimed event type 'ACTION'."}

Test 2 - DECISION:
{'applicable': True, 'passed': True, 'claim_type': 'DECISION', 'matched_terms': ['decided'], 'reason': "Evidence contains indicators consistent with the claimed event type 'DECISION'."}

Test 3 - DISCUSSION:
{'applicable': True, 'passed': True, 'claim_type': 'DISCUSSION', 'matched_terms': ['discuss'], 'reason': "Evidence contains indicators consistent with the claimed event type 'DISCUSSION'."}

Test 4 - Incorrect DECISION:
{'applicable': True, 'passed': False, 'claim_type': 'DECISION', 'matched_terms': [], 'reason': "Evidence does not contain clear indicators supporting the claimed event type 'DECISION'."}


In [20]:
# ============================================================
# M7 - SPEAKER CONSISTENCY INTEGRATION HELPER
# ============================================================

def evaluate_speaker_consistency_for_evidence(
    claim_speaker,
    supporting_evidence
):
    """
    Evaluate whether the claimed speaker is consistent with
    the speakers present in the supporting evidence.

    This check is kept separate from evidence retrieval so that
    a speaker mismatch can be explicitly reported rather than
    being interpreted as 'no evidence found'.
    """

    # --------------------------------------------------------
    # No claimed speaker
    # --------------------------------------------------------
    if claim_speaker is None:
        return {
            "applicable": False,
            "passed": True,
            "reason": "No claim speaker specified; speaker consistency is not applicable."
        }

    # --------------------------------------------------------
    # No supporting evidence
    # --------------------------------------------------------
    if not supporting_evidence:
        return {
            "applicable": True,
            "passed": False,
            "reason": "No supporting evidence available for speaker consistency check.",
            "claim_speaker": claim_speaker,
            "evidence_speakers": []
        }

    # --------------------------------------------------------
    # Extract speakers from evidence
    # --------------------------------------------------------
    evidence_speakers = []

    for item in supporting_evidence:
        speaker = item.get("speaker")

        if speaker is not None and speaker not in evidence_speakers:
            evidence_speakers.append(speaker)

    # --------------------------------------------------------
    # No speaker information in evidence
    # --------------------------------------------------------
    if not evidence_speakers:
        return {
            "applicable": True,
            "passed": False,
            "reason": "Supporting evidence does not contain speaker information.",
            "claim_speaker": claim_speaker,
            "evidence_speakers": []
        }

    # --------------------------------------------------------
    # Speaker consistency check
    # --------------------------------------------------------
    passed = claim_speaker in evidence_speakers

    if passed:
        reason = (
            f"Claimed speaker '{claim_speaker}' is consistent "
            f"with the supporting evidence."
        )
    else:
        reason = (
            f"Claimed speaker '{claim_speaker}' does not match "
            f"the supporting evidence speakers: {evidence_speakers}."
        )

    return {
        "applicable": True,
        "passed": passed,
        "claim_speaker": claim_speaker,
        "evidence_speakers": evidence_speakers,
        "reason": reason
    }


# ============================================================
# TEST
# ============================================================

print("=" * 90)
print("SPEAKER CONSISTENCY INTEGRATION TEST")
print("=" * 90)

# Example supporting evidence
test_evidence = [
    {
        "evidence_id": 14,
        "speaker": "SPEAKER_02",
        "text": "The remote control should be original."
    },
    {
        "evidence_id": 15,
        "speaker": "SPEAKER_02",
        "text": "It should be trendy and appeal to a wide market."
    }
]

# ------------------------------------------------------------
# Test 1 - Correct speaker
# ------------------------------------------------------------

test_1 = evaluate_speaker_consistency_for_evidence(
    "SPEAKER_02",
    test_evidence
)

print("\nTest 1 - Correct speaker:")
print(test_1)


# ------------------------------------------------------------
# Test 2 - Incorrect speaker
# ------------------------------------------------------------

test_2 = evaluate_speaker_consistency_for_evidence(
    "SPEAKER_03",
    test_evidence
)

print("\nTest 2 - Incorrect speaker:")
print(test_2)


# ------------------------------------------------------------
# Test 3 - No claimed speaker
# ------------------------------------------------------------

test_3 = evaluate_speaker_consistency_for_evidence(
    None,
    test_evidence
)

print("\nTest 3 - No claimed speaker:")
print(test_3)

SPEAKER CONSISTENCY INTEGRATION TEST

Test 1 - Correct speaker:
{'applicable': True, 'passed': True, 'claim_speaker': 'SPEAKER_02', 'evidence_speakers': ['SPEAKER_02'], 'reason': "Claimed speaker 'SPEAKER_02' is consistent with the supporting evidence."}

Test 2 - Incorrect speaker:
{'applicable': True, 'passed': False, 'claim_speaker': 'SPEAKER_03', 'evidence_speakers': ['SPEAKER_02'], 'reason': "Claimed speaker 'SPEAKER_03' does not match the supporting evidence speakers: ['SPEAKER_02']."}

Test 3 - No claimed speaker:
{'applicable': False, 'passed': True, 'reason': 'No claim speaker specified; speaker consistency is not applicable.'}


In [32]:
# ============================================================
# M7 - FINAL MULTI-ATTRIBUTE VERIFICATION INTEGRATION
# ============================================================

def verify_mom_claim(
    claim,
    supporting_evidence,
    content_proposition=None,
    expected_start=None,
    expected_end=None,
    timestamp_tolerance=2.0
):
    """
    Multi-attribute verification of a structured MoM claim.

    Verification attributes:
        1. Content consistency
        2. Speaker consistency
        3. Timestamp consistency
        4. Event / decision consistency

    Content consistency is evaluated using the existing
    verify_attribute() pipeline.

    Overall:
        VERIFIED -> all applicable checks pass
        FLAGGED   -> one or more applicable checks fail
    """

    # --------------------------------------------------------
    # Extract claim information
    # --------------------------------------------------------

    claim_text = claim.get("claim", "")
    claim_speaker = claim.get("speaker")
    claim_type = claim.get("type")

    # --------------------------------------------------------
    # Evidence information
    # --------------------------------------------------------

    evidence_texts = [
        item.get("text", "")
        for item in supporting_evidence
        if item.get("text")
    ]

    evidence_speakers = [
        item.get("speaker")
        for item in supporting_evidence
        if item.get("speaker")
    ]

    evidence_speakers = list(dict.fromkeys(evidence_speakers))

    # ========================================================
    # 1. CONTENT CONSISTENCY
    # ========================================================

    if content_proposition is None:
        content_proposition = claim_text

    content_verification = verify_attribute(
        attribute="content",
        proposition=content_proposition,
        claim_speaker=claim_speaker,
        top_k=10,
        context_window=2
    )

    content_check = {
        "applicable": True,
        "passed": content_verification["status"] == "VERIFIED",
        "reason": content_verification["reason"],
        "supporting_evidence_ids":
            content_verification["supporting_evidence_ids"],
        "context_evidence_ids":
            content_verification["context_evidence_ids"]
    }

    # --------------------------------------------------------
    # Use verified content evidence for downstream checks
    # --------------------------------------------------------

    verified_evidence_ids = (
        content_verification["supporting_evidence_ids"]
    )

    # If content verification found supporting evidence,
    # use those evidence items where possible.
    verified_evidence = [
        item
        for item in supporting_evidence
        if item.get("evidence_id") in verified_evidence_ids
    ]

    # Fall back to supplied evidence if IDs are not present.
    if not verified_evidence:
        verified_evidence = supporting_evidence

    # ========================================================
    # 2. SPEAKER CONSISTENCY
    # ========================================================

    speaker_check = evaluate_speaker_consistency_for_evidence(
        claim_speaker,
        verified_evidence
    )

    # ========================================================
    # 3. TIMESTAMP CONSISTENCY
    # ========================================================

    evidence_start = None
    evidence_end = None

    if verified_evidence:

        starts = [
            item.get("start")
            for item in verified_evidence
            if item.get("start") is not None
        ]

        ends = [
            item.get("end")
            for item in verified_evidence
            if item.get("end") is not None
        ]

        if starts:
            evidence_start = min(starts)

        if ends:
            evidence_end = max(ends)

    timestamp_check = check_timestamp_consistency(
        evidence_start,
        evidence_end,
        expected_start,
        expected_end,
        timestamp_tolerance
    )

    # ========================================================
    # 4. EVENT / DECISION CONSISTENCY
    # ========================================================

    event_check = check_event_type_consistency(
        claim_type,
        evidence_texts
    )

    # ========================================================
    # COMBINE CHECKS
    # ========================================================

    checks = {
        "content_consistency": content_check,
        "speaker_consistency": speaker_check,
        "timestamp_consistency": timestamp_check,
        "event_type_consistency": event_check
    }

    failed_checks = [
        name
        for name, result in checks.items()
        if result["applicable"] and not result["passed"]
    ]

    overall_status = (
        "VERIFIED"
        if len(failed_checks) == 0
        else
        "FLAGGED"
    )

    # ========================================================
    # OVERALL REASON
    # ========================================================

    if overall_status == "VERIFIED":

        overall_reason = (
            "The claim passed all applicable "
            "multi-attribute consistency checks."
        )

    else:

        overall_reason = (
            "The claim failed one or more consistency checks: "
            + ", ".join(failed_checks)
        )

    # ========================================================
    # FINAL RESULT
    # ========================================================

    return {
        "claim": claim_text,
        "content_proposition": content_proposition,
        "claim_speaker": claim_speaker,
        "claim_type": claim_type,

        "status": overall_status,
        "reason": overall_reason,

        "checks": checks,
        "failed_checks": failed_checks,

        "evidence_ids": [
            item.get("evidence_id")
            for item in verified_evidence
            if item.get("evidence_id") is not None
        ],

        "evidence_speakers": evidence_speakers,

        "evidence_start": evidence_start,
        "evidence_end": evidence_end
    }


print("=" * 90)
print("FINAL M7 MULTI-ATTRIBUTE VERIFICATION FUNCTION READY")
print("=" * 90)

print("\nContent verification:")
print("  Uses existing verify_attribute() pipeline")

print("\nAdditional checks:")
print("  Speaker consistency")
print("  Timestamp consistency")
print("  Event / decision consistency")

print("\nOverall output:")
print("  VERIFIED / FLAGGED")

FINAL M7 MULTI-ATTRIBUTE VERIFICATION FUNCTION READY

Content verification:
  Uses existing verify_attribute() pipeline

Additional checks:
  Speaker consistency
  Timestamp consistency
  Event / decision consistency

Overall output:
  VERIFIED / FLAGGED


In [23]:
# ============================================================
# M7 - IMPROVED EVENT / DECISION CONSISTENCY
# ============================================================

def check_event_type_consistency(claim_type, evidence_texts):
    """
    Check whether the retrieved evidence is compatible with
    the claimed event type.

    Supported types:
        ACTION
        DECISION
        DISCUSSION
        INFORMATION

    INFORMATION is treated as the default descriptive/factual
    category. It should not require a special trigger word.
    Instead, it is considered consistent when the evidence does
    not contain strong indicators of another event type.
    """

    if claim_type is None:
        return {
            "applicable": False,
            "passed": True,
            "reason": (
                "No claim type specified; event type consistency "
                "is not applicable."
            )
        }

    claim_type = claim_type.upper().strip()

    if not evidence_texts:
        return {
            "applicable": True,
            "passed": False,
            "claim_type": claim_type,
            "matched_terms": [],
            "reason": (
                "No evidence text available for event type verification."
            )
        }

    combined_text = " ".join(evidence_texts).lower()

    # --------------------------------------------------------
    # Event-type indicators
    # --------------------------------------------------------

    action_terms = [
        "will",
        "i'll",
        "we'll",
        "shall",
        "need to",
        "have to",
        "going to",
        "before the next meeting",
        "by next meeting",
        "by tomorrow",
        "by friday"
    ]

    decision_terms = [
        "decided",
        "decision",
        "agreed",
        "we will use",
        "we chose",
        "we choose",
        "let's use",
        "final decision"
    ]

    discussion_terms = [
        "discuss",
        "discussion",
        "talked",
        "consider",
        "considering",
        "question",
        "idea",
        "suggest"
    ]

    # --------------------------------------------------------
    # INFORMATION
    #
    # Information claims are descriptive/factual statements.
    # They do not require a specific trigger word.
    #
    # We only flag INFORMATION when the evidence contains
    # strong action/decision indicators that make the type
    # inconsistent.
    # --------------------------------------------------------

    if claim_type == "INFORMATION":

        strong_action_matches = [
            term for term in action_terms
            if term in combined_text
        ]

        strong_decision_matches = [
            term for term in decision_terms
            if term in combined_text
        ]

        # If evidence strongly indicates an action or decision,
        # INFORMATION may be an incorrect event type.
        if strong_action_matches or strong_decision_matches:

            matched_terms = (
                strong_action_matches +
                strong_decision_matches
            )

            return {
                "applicable": True,
                "passed": False,
                "claim_type": claim_type,
                "matched_terms": matched_terms,
                "reason": (
                    "Evidence contains strong action/decision "
                    "indicators that are inconsistent with the "
                    "claimed INFORMATION type."
                )
            }

        return {
            "applicable": True,
            "passed": True,
            "claim_type": claim_type,
            "matched_terms": [],
            "reason": (
                "Evidence is compatible with the claimed "
                "INFORMATION event type."
            )
        }

    # --------------------------------------------------------
    # ACTION
    # --------------------------------------------------------

    if claim_type == "ACTION":

        matched_terms = [
            term for term in action_terms
            if term in combined_text
        ]

    # --------------------------------------------------------
    # DECISION
    # --------------------------------------------------------

    elif claim_type == "DECISION":

        matched_terms = [
            term for term in decision_terms
            if term in combined_text
        ]

    # --------------------------------------------------------
    # DISCUSSION
    # --------------------------------------------------------

    elif claim_type == "DISCUSSION":

        matched_terms = [
            term for term in discussion_terms
            if term in combined_text
        ]

    # --------------------------------------------------------
    # Unknown type
    # --------------------------------------------------------

    else:

        return {
            "applicable": False,
            "passed": True,
            "claim_type": claim_type,
            "matched_terms": [],
            "reason": (
                f"Unsupported event type '{claim_type}'; "
                "event type consistency is not evaluated."
            )
        }

    # --------------------------------------------------------
    # Result for ACTION / DECISION / DISCUSSION
    # --------------------------------------------------------

    passed = len(matched_terms) > 0

    if passed:
        reason = (
            f"Evidence contains indicators consistent with "
            f"the claimed event type '{claim_type}'."
        )
    else:
        reason = (
            f"Evidence does not contain clear indicators "
            f"supporting the claimed event type '{claim_type}'."
        )

    return {
        "applicable": True,
        "passed": passed,
        "claim_type": claim_type,
        "matched_terms": matched_terms,
        "reason": reason
    }


# ============================================================
# TEST THE IMPROVED FUNCTION
# ============================================================

print("=" * 90)
print("IMPROVED EVENT / DECISION CONSISTENCY TEST")
print("=" * 90)

# Test 1 - INFORMATION
test_1 = check_event_type_consistency(
    "INFORMATION",
    [
        "The remote control should be original.",
        "It should also be trendy and appeal to a wide market."
    ]
)

print("\nTest 1 - INFORMATION:")
print(test_1)


# Test 2 - ACTION
test_2 = check_event_type_consistency(
    "ACTION",
    [
        "Participants planned to work on their individual tasks "
        "before the next meeting."
    ]
)

print("\nTest 2 - ACTION:")
print(test_2)


# Test 3 - DECISION
test_3 = check_event_type_consistency(
    "DECISION",
    [
        "The team decided to use the LCD display for the product."
    ]
)

print("\nTest 3 - DECISION:")
print(test_3)


# Test 4 - DISCUSSION
test_4 = check_event_type_consistency(
    "DISCUSSION",
    [
        "The participants discussed different options "
        "for the product design."
    ]
)

print("\nTest 4 - DISCUSSION:")
print(test_4)


# Test 5 - Incorrect DECISION
test_5 = check_event_type_consistency(
    "DECISION",
    [
        "The participants discussed different options "
        "for the product design."
    ]
)

print("\nTest 5 - Incorrect DECISION:")
print(test_5)

IMPROVED EVENT / DECISION CONSISTENCY TEST

Test 1 - INFORMATION:
{'applicable': True, 'passed': True, 'claim_type': 'INFORMATION', 'matched_terms': [], 'reason': 'Evidence is compatible with the claimed INFORMATION event type.'}

Test 2 - ACTION:
{'applicable': True, 'passed': True, 'claim_type': 'ACTION', 'matched_terms': ['before the next meeting'], 'reason': "Evidence contains indicators consistent with the claimed event type 'ACTION'."}

Test 3 - DECISION:
{'applicable': True, 'passed': True, 'claim_type': 'DECISION', 'matched_terms': ['decided'], 'reason': "Evidence contains indicators consistent with the claimed event type 'DECISION'."}

Test 4 - DISCUSSION:
{'applicable': True, 'passed': True, 'claim_type': 'DISCUSSION', 'matched_terms': ['discuss'], 'reason': "Evidence contains indicators consistent with the claimed event type 'DISCUSSION'."}

Test 5 - Incorrect DECISION:
{'applicable': True, 'passed': False, 'claim_type': 'DECISION', 'matched_terms': [], 'reason': "Evidence d

In [24]:
# ============================================================
# M7 - REAL CLAIM 2 VERIFICATION (UPDATED)
# ============================================================

verification_result = verify_mom_claim(
    claim=test_claim,
    supporting_evidence=test_supporting_evidence
)

print("=" * 90)
print("REAL CLAIM 2 - MULTI-ATTRIBUTE VERIFICATION")
print("=" * 90)

print("\nClaim:")
print(verification_result["claim"])

print("\nClaim Speaker:")
print(verification_result["claim_speaker"])

print("\nClaim Type:")
print(verification_result["claim_type"])

print("\nOverall Status:")
print(verification_result["status"])

print("\nOverall Reason:")
print(verification_result["reason"])

print("\n--- Individual Verification Checks ---")

for check_name, result in verification_result["checks"].items():
    print(f"\n{check_name}:")
    print(f"  Applicable: {result['applicable']}")
    print(f"  Passed:     {result['passed']}")
    print(f"  Reason:     {result['reason']}")

print("\nEvidence IDs:")
print(verification_result["evidence_ids"])

print("\nEvidence Speakers:")
print(verification_result["evidence_speakers"])

print("\nEvidence Interval:")
print(
    f"{verification_result['evidence_start']:.3f}s"
    f" -> "
    f"{verification_result['evidence_end']:.3f}s"
)

REAL CLAIM 2 - MULTI-ATTRIBUTE VERIFICATION

Claim:
The remote control is intended to be original.

Claim Speaker:
SPEAKER_02

Claim Type:
INFORMATION

Overall Status:
VERIFIED

Overall Reason:
The claim passed all applicable multi-attribute consistency checks.

--- Individual Verification Checks ---

content_consistency:
  Applicable: True
  Passed:     True
  Reason:     Supporting evidence was retrieved and passed the evidence-selection stage.

speaker_consistency:
  Applicable: True
  Passed:     True
  Reason:     Claimed speaker 'SPEAKER_02' is consistent with the supporting evidence.

timestamp_consistency:
  Applicable: False
  Passed:     True
  Reason:     No expected timestamp specified; timestamp consistency is not applicable.

event_type_consistency:
  Applicable: True
  Passed:     True
  Reason:     Evidence is compatible with the claimed INFORMATION event type.

Evidence IDs:
[14, 15]

Evidence Speakers:
['SPEAKER_02']

Evidence Interval:
117.699s -> 132.290s


In [25]:
# ============================================================
# M7 - NEGATIVE TEST: INCORRECT SPEAKER
# ============================================================

wrong_speaker_claim = {
    "claim": "The remote control is intended to be original.",
    "speaker": "SPEAKER_03",   # Deliberately incorrect
    "type": "INFORMATION"
}

wrong_speaker_result = verify_mom_claim(
    claim=wrong_speaker_claim,
    supporting_evidence=test_supporting_evidence
)

print("=" * 90)
print("M7 NEGATIVE TEST - INCORRECT SPEAKER")
print("=" * 90)

print("\nClaim:")
print(wrong_speaker_result["claim"])

print("\nClaimed Speaker:")
print(wrong_speaker_result["claim_speaker"])

print("\nEvidence Speakers:")
print(wrong_speaker_result["evidence_speakers"])

print("\nOverall Status:")
print(wrong_speaker_result["status"])

print("\nOverall Reason:")
print(wrong_speaker_result["reason"])

print("\n--- Individual Checks ---")

for check_name, result in wrong_speaker_result["checks"].items():
    print(f"\n{check_name}:")
    print(f"  Applicable: {result['applicable']}")
    print(f"  Passed:     {result['passed']}")
    print(f"  Reason:     {result['reason']}")

print("\nFailed Checks:")
print(wrong_speaker_result["failed_checks"])

M7 NEGATIVE TEST - INCORRECT SPEAKER

Claim:
The remote control is intended to be original.

Claimed Speaker:
SPEAKER_03

Evidence Speakers:
['SPEAKER_02']

Overall Status:
FLAGGED

Overall Reason:
The claim failed one or more consistency checks: speaker_consistency

--- Individual Checks ---

content_consistency:
  Applicable: True
  Passed:     True
  Reason:     Supporting evidence was retrieved and passed the evidence-selection stage.

speaker_consistency:
  Applicable: True
  Passed:     False
  Reason:     Claimed speaker 'SPEAKER_03' does not match the supporting evidence speakers: ['SPEAKER_02'].

timestamp_consistency:
  Applicable: False
  Passed:     True
  Reason:     No expected timestamp specified; timestamp consistency is not applicable.

event_type_consistency:
  Applicable: True
  Passed:     True
  Reason:     Evidence is compatible with the claimed INFORMATION event type.

Failed Checks:
['speaker_consistency']


In [26]:
# ============================================================
# M7 NEGATIVE TEST - INCORRECT EVENT TYPE
# ============================================================

wrong_event_claim = {
    "claim": "The team decided to use the LCD display.",
    "speaker": "SPEAKER_03",
    "type": "DECISION"
}

# Evidence deliberately represents a discussion,
# not a confirmed decision.
wrong_event_evidence = [
    {
        "evidence_id": 72,
        "speaker": "SPEAKER_03",
        "start": 779.877,
        "end": 796.000,
        "text": "We could have an LCD display with menus similar to a mobile phone."
    },
    {
        "evidence_id": 73,
        "speaker": "SPEAKER_03",
        "start": 796.000,
        "end": 812.063,
        "text": "This would make browsing and navigation easier."
    }
]

wrong_event_result = verify_mom_claim(
    claim=wrong_event_claim,
    supporting_evidence=wrong_event_evidence
)

print("=" * 90)
print("M7 NEGATIVE TEST - INCORRECT EVENT TYPE")
print("=" * 90)

print("\nClaim:")
print(wrong_event_result["claim"])

print("\nClaim Type:")
print(wrong_event_result["claim_type"])

print("\nOverall Status:")
print(wrong_event_result["status"])

print("\nOverall Reason:")
print(wrong_event_result["reason"])

print("\n--- Individual Checks ---")

for check_name, result in wrong_event_result["checks"].items():
    print(f"\n{check_name}:")
    print(f"  Applicable: {result['applicable']}")
    print(f"  Passed:     {result['passed']}")
    print(f"  Reason:     {result['reason']}")

print("\nFailed Checks:")
print(wrong_event_result["failed_checks"])

M7 NEGATIVE TEST - INCORRECT EVENT TYPE

Claim:
The team decided to use the LCD display.

Claim Type:
DECISION

Overall Status:
FLAGGED

Overall Reason:
The claim failed one or more consistency checks: event_type_consistency

--- Individual Checks ---

content_consistency:
  Applicable: True
  Passed:     True
  Reason:     Supporting evidence was retrieved and passed the evidence-selection stage.

speaker_consistency:
  Applicable: True
  Passed:     True
  Reason:     Claimed speaker 'SPEAKER_03' is consistent with the supporting evidence.

timestamp_consistency:
  Applicable: False
  Passed:     True
  Reason:     No expected timestamp specified; timestamp consistency is not applicable.

event_type_consistency:
  Applicable: True
  Passed:     False
  Reason:     Evidence does not contain clear indicators supporting the claimed event type 'DECISION'.

Failed Checks:
['event_type_consistency']


In [27]:
# ============================================================
# M7 - NEGATIVE CONTENT TEST
# ============================================================

wrong_content_proposition = (
    "The remote control is intended to be inexpensive."
)

print("=" * 90)
print("M7 NEGATIVE TEST - INCORRECT CONTENT")
print("=" * 90)

print("\nIncorrect proposition:")
print(wrong_content_proposition)

# Run the existing attribute-level evidence analysis
content_analysis = analyze_attribute_evidence(
    proposition=wrong_content_proposition,
    claim_speaker="SPEAKER_02",
    top_k=10,
    context_window=2
)

print("\nCandidates checked:")
print(len(content_analysis["candidates"]))

print("\nTop evidence candidates:")

for candidate in content_analysis["candidates"][:5]:

    print("\nEvidence ID:", candidate.get("evidence_id"))
    print("Speaker:", candidate.get("speaker"))
    print("Similarity:", candidate.get("similarity"))
    print("NLI entailment:", candidate.get("nli_entailment"))
    print("NLI contradiction:", candidate.get("nli_contradiction"))
    print("NLI neutral:", candidate.get("nli_neutral"))
    print("Text:", candidate.get("text"))

M7 NEGATIVE TEST - INCORRECT CONTENT

Incorrect proposition:
The remote control is intended to be inexpensive.

Candidates checked:


TypeError: list indices must be integers or slices, not str

In [28]:
# ============================================================
# M7 - INSPECT EXISTING CONTENT ANALYSIS RESULT
# ============================================================

print("=" * 90)
print("CONTENT ANALYSIS RESULT STRUCTURE")
print("=" * 90)

print("\nReturn type:")
print(type(content_analysis))

print("\nNumber of returned items:")
print(len(content_analysis))

print("\nFirst returned item:")
print(content_analysis[0] if len(content_analysis) > 0 else "No items returned")

CONTENT ANALYSIS RESULT STRUCTURE

Return type:
<class 'list'>

Number of returned items:
10

First returned item:
{'evidence_id': 107, 'speaker': 'SPEAKER_03', 'start': 713.15, 'end': 729.357, 'text': "experience has only been given the remote control with the object I buy, not doing any tampering with it and programming, using it to program TV videos and things, but basically on off volume, up and down, channel one to that basic function. I don't think I could go any further with it.", 'retrieval_similarity': 0.7908881902694702, 'nli_label': 'NEUTRAL', 'nli_entailment': 0.0007824167259968817, 'nli_contradiction': 0.006045519839972258, 'nli_neutral': 0.9931720495223999, 'lexical_overlap': 0.5, 'matched_terms': ['control', 'remote'], 'missing_terms': ['inexpensive', 'intended'], 'speaker_check': {'applicable': True, 'passed': False, 'reason': 'Speaker does not match evidence.'}, 'context_evidence_ids': [105, 106, 107, 108, 109], 'context_text': "remote controls. You want to integrate e

In [29]:
# ============================================================
# M7 - NEGATIVE CONTENT TEST
# EXISTING EVIDENCE SELECTION
# ============================================================

print("=" * 90)
print("M7 NEGATIVE CONTENT TEST - EVIDENCE SELECTION")
print("=" * 90)

# Select evidence candidates using the existing
# NLI + lexical + speaker filtering pipeline.
content_selection = select_supporting_candidates(
    content_analysis,
    min_lexical_overlap=0.0
)

supporting_candidates = content_selection["supporting_candidates"]
rejected_candidates = content_selection["rejected_candidates"]

print("\nIncorrect proposition:")
print(wrong_content_proposition)

print("\nSupporting candidates:")
print(len(supporting_candidates))

for candidate in supporting_candidates:
    print(
        f"  ID {candidate['evidence_id']} | "
        f"NLI: {candidate['nli_label']} | "
        f"Entailment: {candidate['nli_entailment']:.4f} | "
        f"Similarity: {candidate['retrieval_similarity']:.4f}"
    )

print("\nRejected candidates:")
print(len(rejected_candidates))

print("\nReason for rejection of top candidates:")

for candidate in rejected_candidates[:5]:
    print(
        f"\n  ID {candidate['evidence_id']}"
    )
    print(
        f"  NLI label: {candidate['nli_label']}"
    )
    print(
        f"  Entailment: {candidate['nli_entailment']:.4f}"
    )
    print(
        f"  Lexical overlap: {candidate['lexical_overlap']:.4f}"
    )
    print(
        f"  Speaker check: {candidate['speaker_check']}"
    )

M7 NEGATIVE CONTENT TEST - EVIDENCE SELECTION

Incorrect proposition:
The remote control is intended to be inexpensive.

Supporting candidates:
0

Rejected candidates:
10

Reason for rejection of top candidates:

  ID 107


KeyError: 'nli_label'

In [30]:
# ============================================================
# M7 - NEGATIVE CONTENT VERIFICATION
# ============================================================

content_verification = verify_attribute(
    attribute="incorrect_content",
    proposition=wrong_content_proposition,
    claim_speaker="SPEAKER_02",
    top_k=10,
    context_window=2
)

print("=" * 90)
print("M7 NEGATIVE TEST - CONTENT VERIFICATION")
print("=" * 90)

print("\nIncorrect proposition:")
print(wrong_content_proposition)

print("\nVerification result:")
print(content_verification)

print("\nStatus:")
print(content_verification.get("status"))

print("\nReason:")
print(content_verification.get("reason"))

M7 NEGATIVE TEST - CONTENT VERIFICATION

Incorrect proposition:
The remote control is intended to be inexpensive.

Verification result:
{'attribute': 'incorrect_content', 'proposition': 'The remote control is intended to be inexpensive.', 'status': 'FLAGGED', 'reason': 'No supporting evidence found.', 'supporting_evidence_ids': [], 'context_evidence_ids': [], 'display_start': None, 'display_end': None, 'speakers': [], 'speaker_check': False, 'candidates_checked': 10, 'supporting_candidates': [], 'rejected_candidates': [{'evidence_id': 107, 'reason': 'NLI neutral'}, {'evidence_id': 105, 'reason': 'NLI neutral'}, {'evidence_id': 145, 'reason': 'NLI neutral'}, {'evidence_id': 102, 'reason': 'NLI neutral'}, {'evidence_id': 14, 'reason': 'NLI neutral'}, {'evidence_id': 117, 'reason': 'NLI neutral'}, {'evidence_id': 135, 'reason': 'NLI neutral'}, {'evidence_id': 112, 'reason': 'NLI neutral'}, {'evidence_id': 133, 'reason': 'NLI neutral'}, {'evidence_id': 111, 'reason': 'NLI neutral'}]}

Stat

In [31]:
# ============================================================
# M7 - INSPECT CORRECT CONTENT VERIFICATION
# ============================================================

correct_content_proposition = (
    "The remote control is intended to be original."
)

correct_content_verification = verify_attribute(
    attribute="originality",
    proposition=correct_content_proposition,
    claim_speaker="SPEAKER_02",
    top_k=10,
    context_window=2
)

print("=" * 90)
print("M7 - CORRECT CONTENT VERIFICATION")
print("=" * 90)

print("\nProposition:")
print(correct_content_proposition)

print("\nStatus:")
print(correct_content_verification["status"])

print("\nReason:")
print(correct_content_verification["reason"])

print("\nSupporting Evidence IDs:")
print(correct_content_verification["supporting_evidence_ids"])

print("\nContext Evidence IDs:")
print(correct_content_verification["context_evidence_ids"])

print("\nDisplay Interval:")
print(
    correct_content_verification["display_start"],
    "->",
    correct_content_verification["display_end"]
)

print("\nSpeakers:")
print(correct_content_verification["speakers"])

print("\nCandidates Checked:")
print(correct_content_verification["candidates_checked"])

M7 - CORRECT CONTENT VERIFICATION

Proposition:
The remote control is intended to be original.

Status:
VERIFIED

Reason:
The proposition is supported by contextual NLI evidence and the claimed speaker is consistent with the supporting evidence.

Supporting Evidence IDs:
[14, 15]

Context Evidence IDs:
[12, 13, 14, 15, 16, 17]

Display Interval:
117.699 -> 132.29

Speakers:
['SPEAKER_02']

Candidates Checked:
10


In [33]:
# ============================================================
# M7 - FINAL POSITIVE END-TO-END TEST
# ============================================================

final_positive_result = verify_mom_claim(
    claim=test_claim,
    supporting_evidence=test_supporting_evidence,
    content_proposition="The remote control is intended to be original."
)

print("=" * 90)
print("M7 FINAL POSITIVE END-TO-END TEST")
print("=" * 90)

print("\nClaim:")
print(final_positive_result["claim"])

print("\nContent Proposition:")
print(final_positive_result["content_proposition"])

print("\nClaim Speaker:")
print(final_positive_result["claim_speaker"])

print("\nClaim Type:")
print(final_positive_result["claim_type"])

print("\nOverall Status:")
print(final_positive_result["status"])

print("\nOverall Reason:")
print(final_positive_result["reason"])

print("\n--- Individual Checks ---")

for check_name, result in final_positive_result["checks"].items():

    print(f"\n{check_name}:")
    print(f"  Applicable: {result['applicable']}")
    print(f"  Passed:     {result['passed']}")
    print(f"  Reason:     {result['reason']}")

print("\nEvidence IDs:")
print(final_positive_result["evidence_ids"])

print("\nEvidence Speakers:")
print(final_positive_result["evidence_speakers"])

print("\nEvidence Interval:")
print(
    f"{final_positive_result['evidence_start']:.3f}s"
    f" -> "
    f"{final_positive_result['evidence_end']:.3f}s"
)

print("\nFailed Checks:")
print(final_positive_result["failed_checks"])

M7 FINAL POSITIVE END-TO-END TEST

Claim:
The remote control is intended to be original.

Content Proposition:
The remote control is intended to be original.

Claim Speaker:
SPEAKER_02

Claim Type:
INFORMATION

Overall Status:
VERIFIED

Overall Reason:
The claim passed all applicable multi-attribute consistency checks.

--- Individual Checks ---

content_consistency:
  Applicable: True
  Passed:     True
  Reason:     The proposition is supported by contextual NLI evidence and the claimed speaker is consistent with the supporting evidence.

speaker_consistency:
  Applicable: True
  Passed:     True
  Reason:     Claimed speaker 'SPEAKER_02' is consistent with the supporting evidence.

timestamp_consistency:
  Applicable: False
  Passed:     True
  Reason:     No expected timestamp specified; timestamp consistency is not applicable.

event_type_consistency:
  Applicable: True
  Passed:     True
  Reason:     Evidence is compatible with the claimed INFORMATION event type.

Evidence IDs:


In [34]:
# ============================================================
# M7 - FINAL NEGATIVE END-TO-END TESTS
# ============================================================

print("=" * 90)
print("M7 FINAL NEGATIVE END-TO-END TESTS")
print("=" * 90)


# ============================================================
# TEST 1 - INCORRECT CONTENT
# ============================================================

wrong_content_claim = {
    "claim": "The remote control is intended to be inexpensive.",
    "speaker": "SPEAKER_02",
    "type": "INFORMATION"
}

wrong_content_result = verify_mom_claim(
    claim=wrong_content_claim,
    supporting_evidence=test_supporting_evidence,
    content_proposition=(
        "The remote control is intended to be inexpensive."
    )
)

print("\n" + "-" * 90)
print("TEST 1 - INCORRECT CONTENT")
print("-" * 90)

print("Status:", wrong_content_result["status"])
print("Reason:", wrong_content_result["reason"])
print("Failed Checks:", wrong_content_result["failed_checks"])

for name, result in wrong_content_result["checks"].items():
    print(
        f"{name}: "
        f"applicable={result['applicable']}, "
        f"passed={result['passed']}"
    )


# ============================================================
# TEST 2 - INCORRECT SPEAKER
# ============================================================

wrong_speaker_claim = {
    "claim": "The remote control is intended to be original.",
    "speaker": "SPEAKER_03",
    "type": "INFORMATION"
}

wrong_speaker_result = verify_mom_claim(
    claim=wrong_speaker_claim,
    supporting_evidence=test_supporting_evidence,
    content_proposition=(
        "The remote control is intended to be original."
    )
)

print("\n" + "-" * 90)
print("TEST 2 - INCORRECT SPEAKER")
print("-" * 90)

print("Status:", wrong_speaker_result["status"])
print("Reason:", wrong_speaker_result["reason"])
print("Failed Checks:", wrong_speaker_result["failed_checks"])

for name, result in wrong_speaker_result["checks"].items():
    print(
        f"{name}: "
        f"applicable={result['applicable']}, "
        f"passed={result['passed']}"
    )


# ============================================================
# TEST 3 - INCORRECT EVENT TYPE
# ============================================================

wrong_event_claim = {
    "claim": "The team decided to use the LCD display.",
    "speaker": "SPEAKER_03",
    "type": "DECISION"
}

wrong_event_result = verify_mom_claim(
    claim=wrong_event_claim,
    supporting_evidence=wrong_event_evidence,
    content_proposition=(
        "The team decided to use the LCD display."
    )
)

print("\n" + "-" * 90)
print("TEST 3 - INCORRECT EVENT TYPE")
print("-" * 90)

print("Status:", wrong_event_result["status"])
print("Reason:", wrong_event_result["reason"])
print("Failed Checks:", wrong_event_result["failed_checks"])

for name, result in wrong_event_result["checks"].items():
    print(
        f"{name}: "
        f"applicable={result['applicable']}, "
        f"passed={result['passed']}"
    )


# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n" + "=" * 90)
print("M7 NEGATIVE TEST SUMMARY")
print("=" * 90)

print(
    "Incorrect Content  :",
    wrong_content_result["status"],
    "| Failed:",
    wrong_content_result["failed_checks"]
)

print(
    "Incorrect Speaker  :",
    wrong_speaker_result["status"],
    "| Failed:",
    wrong_speaker_result["failed_checks"]
)

print(
    "Incorrect Event    :",
    wrong_event_result["status"],
    "| Failed:",
    wrong_event_result["failed_checks"]
)

M7 FINAL NEGATIVE END-TO-END TESTS

------------------------------------------------------------------------------------------
TEST 1 - INCORRECT CONTENT
------------------------------------------------------------------------------------------
Status: FLAGGED
Reason: The claim failed one or more consistency checks: content_consistency
Failed Checks: ['content_consistency']
content_consistency: applicable=True, passed=False
speaker_consistency: applicable=True, passed=True
timestamp_consistency: applicable=False, passed=True
event_type_consistency: applicable=True, passed=True

------------------------------------------------------------------------------------------
TEST 2 - INCORRECT SPEAKER
------------------------------------------------------------------------------------------
Status: FLAGGED
Reason: The claim failed one or more consistency checks: content_consistency, speaker_consistency
Failed Checks: ['content_consistency', 'speaker_consistency']
content_consistency: applicabl

In [2]:
# ============================================================
# RESUME M7 - RESTORE EVALUATION ENVIRONMENT
# ============================================================

import os
import json
import numpy as np
import pandas as pd

PROJECT_DIR = "/content/drive/MyDrive/MTechIndProj/MoM_Project"
os.chdir(PROJECT_DIR)

print("=" * 90)
print("M7 RESUME - PROJECT ENVIRONMENT")
print("=" * 90)

print("\nProject directory:")
print(os.getcwd())

print("\nRequired project folders:")
for folder in [
    "data",
    "data/transcripts",
    "data/vad",
    "outputs",
    "evaluation_results",
    "models"
]:
    print(f"{folder}: {'OK' if os.path.exists(folder) else 'MISSING'}")

print("\nM7 notebook:")
print(
    "04_mom_generation/04_mom_generation.ipynb:",
    "OK"
    if os.path.exists("04_mom_generation/04_mom_generation.ipynb")
    else "MISSING"
)

print("\nGit checkpoint:")
os.system("git log -1 --oneline")

print("\nEnvironment restoration check complete.")

M7 RESUME - PROJECT ENVIRONMENT

Project directory:
/content/drive/MyDrive/MTechIndProj/MoM_Project

Required project folders:
data: OK
data/transcripts: OK
data/vad: OK
outputs: OK
evaluation_results: OK
models: OK

M7 notebook:
04_mom_generation/04_mom_generation.ipynb: OK

Git checkpoint:

Environment restoration check complete.


In [4]:
# ============================================================
# RESUME M7 - LOAD STRUCTURED MoM CLAIMS
# ============================================================

import os
import json

PROJECT_DIR = "/content/drive/MyDrive/MTechIndProj/MoM_Project"

claims_path = os.path.join(
    PROJECT_DIR,
    "data",
    "transcripts",
    "ES2004a_structured_mom_claims.json"
)

print("=" * 90)
print("M7 - LOADING STRUCTURED MoM CLAIMS")
print("=" * 90)

print("\nFile:")
print(claims_path)

# Check file exists
if not os.path.exists(claims_path):
    raise FileNotFoundError(
        f"Structured claims file not found:\n{claims_path}"
    )

# Load claims
with open(claims_path, "r", encoding="utf-8") as f:
    structured_claims = json.load(f)

print("\nFile loaded successfully.")

print("\nNumber of structured claims:")
print(len(structured_claims))

print("\nFirst claim:")
print(structured_claims[0])

print("\nM7 structured claims restoration complete.")

M7 - LOADING STRUCTURED MoM CLAIMS

File:
/content/drive/MyDrive/MTechIndProj/MoM_Project/data/transcripts/ES2004a_structured_mom_claims.json

File loaded successfully.

Number of structured claims:
6

First claim:


KeyError: 0

In [5]:
# ============================================================
# M7 - INSPECT STRUCTURED CLAIMS FILE
# ============================================================

print("=" * 90)
print("STRUCTURED MoM CLAIMS - FILE STRUCTURE")
print("=" * 90)

print("\nTop-level type:")
print(type(structured_claims))

print("\nTop-level keys:")
print(list(structured_claims.keys()))

print("\nNumber of top-level keys:")
print(len(structured_claims))

print("\nTop-level contents summary:")

for key, value in structured_claims.items():

    print(f"\nKey: {key}")
    print(f"Type: {type(value)}")

    if isinstance(value, list):
        print(f"Number of items: {len(value)}")

        if len(value) > 0:
            print("First item:")
            print(value[0])

    elif isinstance(value, dict):
        print("Dictionary keys:")
        print(list(value.keys()))

    else:
        print("Value:")
        print(value)

STRUCTURED MoM CLAIMS - FILE STRUCTURE

Top-level type:
<class 'dict'>

Top-level keys:
['meeting_id', 'source', 'candidate_count', 'topic_group_count', 'claim_count', 'claims']

Number of top-level keys:
6

Top-level contents summary:

Key: meeting_id
Type: <class 'str'>
Value:
ES2004a

Key: source
Type: <class 'str'>
Value:
Structured MoM claim generation

Key: candidate_count
Type: <class 'int'>
Value:
42

Key: topic_group_count
Type: <class 'int'>
Value:
11

Key: claim_count
Type: <class 'int'>
Value:
12

Key: claims
Type: <class 'list'>
Number of items: 12
First item:
{'claim_id': 1, 'topic': 'meeting_introduction_and_agenda', 'claim_text': 'The meeting was introduced as the first meeting, with an agenda covering team interaction, tool training, the project plan, and discussion of ideas.', 'source_candidate_ids': [5, 6, 7, 8], 'speaker': 'SPEAKER_02', 'start': 84.658, 'end': 109.823, 'event_type': 'INFORMATION'}


In [6]:
# ============================================================
# M7 - OVERVIEW OF ALL 12 STRUCTURED CLAIMS
# ============================================================

claims = structured_claims["claims"]

print("=" * 110)
print("M7 - STRUCTURED MoM CLAIM OVERVIEW")
print("=" * 110)

print(
    f"{'ID':<5}"
    f"{'Speaker':<16}"
    f"{'Event Type':<14}"
    f"{'Start':<12}"
    f"{'End':<12}"
    f"Claim"
)

print("-" * 110)

for claim in claims:

    claim_id = claim.get("claim_id")
    speaker = claim.get("speaker")
    event_type = claim.get("event_type")
    start = claim.get("start")
    end = claim.get("end")
    text = claim.get("claim_text")

    print(
        f"{claim_id:<5}"
        f"{str(speaker):<16}"
        f"{str(event_type):<14}"
        f"{str(start):<12}"
        f"{str(end):<12}"
        f"{text}"
    )

print("\nTotal claims:", len(claims))

M7 - STRUCTURED MoM CLAIM OVERVIEW
ID   Speaker         Event Type    Start       End         Claim
--------------------------------------------------------------------------------------------------------------
1    SPEAKER_02      INFORMATION   84.658      109.823     The meeting was introduced as the first meeting, with an agenda covering team interaction, tool training, the project plan, and discussion of ideas.
2    SPEAKER_02      INFORMATION   117.699     141.016     The team is developing a remote control intended to be original, trendy, appealing to a wide market, and user-friendly for a broad range of users.
3    SPEAKER_02      INFORMATION   142.933     167.74      The design process includes functional design, conceptual design, and detailed design, with individual work addressing product requirements and implementation.
4    MULTIPLE        DISCUSSION    563.145     623.419     The discussed selling price was 25 euros, while production costs were estimated at 12.50, with di

In [7]:
# ============================================================
# M7 - ADAPTER FOR STRUCTURED CLAIMS
# ============================================================

def adapt_structured_claim(claim):
    """
    Convert the saved structured MoM claim format into the
    format expected by verify_mom_claim().

    Original fields:
        claim_text
        speaker
        event_type
        start
        end

    The original claim dictionary is preserved separately.
    """

    adapted_claim = {
        "claim": claim.get("claim_text", ""),
        "speaker": claim.get("speaker"),
        "type": claim.get("event_type")
    }

    expected_start = claim.get("start")
    expected_end = claim.get("end")

    return {
        "original_claim": claim,
        "claim": adapted_claim,
        "expected_start": expected_start,
        "expected_end": expected_end
    }


# ============================================================
# TEST ADAPTER ON CLAIM 1
# ============================================================

print("=" * 90)
print("M7 - CLAIM ADAPTER TEST")
print("=" * 90)

adapted_test = adapt_structured_claim(claims[0])

print("\nOriginal claim:")
print(adapted_test["original_claim"])

print("\nAdapted claim:")
print(adapted_test["claim"])

print("\nExpected start:")
print(adapted_test["expected_start"])

print("\nExpected end:")
print(adapted_test["expected_end"])

M7 - CLAIM ADAPTER TEST

Original claim:
{'claim_id': 1, 'topic': 'meeting_introduction_and_agenda', 'claim_text': 'The meeting was introduced as the first meeting, with an agenda covering team interaction, tool training, the project plan, and discussion of ideas.', 'source_candidate_ids': [5, 6, 7, 8], 'speaker': 'SPEAKER_02', 'start': 84.658, 'end': 109.823, 'event_type': 'INFORMATION'}

Adapted claim:
{'claim': 'The meeting was introduced as the first meeting, with an agenda covering team interaction, tool training, the project plan, and discussion of ideas.', 'speaker': 'SPEAKER_02', 'type': 'INFORMATION'}

Expected start:
84.658

Expected end:
109.823


In [8]:
# ============================================================
# M7 - INSPECT SOURCE CANDIDATES FOR CLAIMS
# ============================================================

print("=" * 90)
print("M7 - SOURCE CANDIDATE INSPECTION")
print("=" * 90)

# Check transcript structure
print("\nSpeaker transcript type:")
print(type(speaker_transcript))

if isinstance(speaker_transcript, dict):
    print("\nSpeaker transcript keys:")
    print(list(speaker_transcript.keys()))

# ------------------------------------------------------------
# Inspect possible transcript list
# ------------------------------------------------------------

if isinstance(speaker_transcript, list):

    transcript_items = speaker_transcript

elif isinstance(speaker_transcript, dict):

    # Common structure used by our saved transcript
    if "utterances" in speaker_transcript:
        transcript_items = speaker_transcript["utterances"]

    elif "segments" in speaker_transcript:
        transcript_items = speaker_transcript["segments"]

    else:
        transcript_items = []

else:
    transcript_items = []


print("\nTranscript item count:")
print(len(transcript_items))

# ------------------------------------------------------------
# Inspect Claim 1 source IDs
# ------------------------------------------------------------

claim_1 = claims[0]

print("\nClaim 1 source candidate IDs:")
print(claim_1["source_candidate_ids"])

print("\nMatching transcript candidates:")

for item in transcript_items:

    evidence_id = (
        item.get("evidence_id")
        if isinstance(item, dict)
        else None
    )

    if evidence_id in claim_1["source_candidate_ids"]:

        print("\nEvidence ID:", evidence_id)
        print("Speaker:", item.get("speaker"))
        print("Start:", item.get("start"))
        print("End:", item.get("end"))
        print("Text:", item.get("text"))

M7 - SOURCE CANDIDATE INSPECTION

Speaker transcript type:


NameError: name 'speaker_transcript' is not defined

In [9]:
# ============================================================
# M7 - LOAD SAVED SPEAKER TRANSCRIPT
# ============================================================

import os
import json

PROJECT_DIR = "/content/drive/MyDrive/MTechIndProj/MoM_Project"

speaker_transcript_path = os.path.join(
    PROJECT_DIR,
    "data",
    "transcripts",
    "ES2004a_speaker_transcript.json"
)

print("=" * 90)
print("M7 - LOADING SAVED SPEAKER TRANSCRIPT")
print("=" * 90)

print("\nFile path:")
print(speaker_transcript_path)

print("\nFile exists:", os.path.exists(speaker_transcript_path))

with open(speaker_transcript_path, "r", encoding="utf-8") as f:
    speaker_transcript = json.load(f)

print("\nLoaded successfully.")
print("Type:", type(speaker_transcript))

if isinstance(speaker_transcript, dict):
    print("\nKeys:")
    print(list(speaker_transcript.keys()))

elif isinstance(speaker_transcript, list):
    print("\nList length:")
    print(len(speaker_transcript))

M7 - LOADING SAVED SPEAKER TRANSCRIPT

File path:
/content/drive/MyDrive/MTechIndProj/MoM_Project/data/transcripts/ES2004a_speaker_transcript.json

File exists: True

Loaded successfully.
Type: <class 'dict'>

Keys:
['meeting_id', 'source', 'gap_threshold_seconds', 'num_speakers', 'speakers', 'num_utterances', 'utterances']


In [10]:
# ============================================================
# M7 - INSPECT SPEAKER TRANSCRIPT UTTERANCE STRUCTURE
# ============================================================

print("=" * 90)
print("M7 - SPEAKER TRANSCRIPT UTTERANCE INSPECTION")
print("=" * 90)

utterances = speaker_transcript["utterances"]

print("\nNumber of utterances:", len(utterances))

print("\nFirst 3 utterances:")
for i, item in enumerate(utterances[:3]):
    print(f"\n--- Utterance {i} ---")
    print("Type:", type(item))
    print("Content:", item)

print("\n" + "=" * 90)
print("CHECKING FOR EVIDENCE IDs")
print("=" * 90)

for key in utterances[0].keys():
    print("Field:", key)

M7 - SPEAKER TRANSCRIPT UTTERANCE INSPECTION

Number of utterances: 148

First 3 utterances:

--- Utterance 0 ---
Type: <class 'dict'>
Content: {'speaker': 'SPEAKER_02', 'start': 10.998, 'end': 14.521, 'duration': 3.5230000000000015, 'text': "Are we, we're not like the dim lights, so we can see that a bit better."}

--- Utterance 1 ---
Type: <class 'dict'>
Content: {'speaker': 'SPEAKER_01', 'start': 17.943, 'end': 18.163, 'duration': 0.21999999999999886, 'text': 'Yeah.'}

--- Utterance 2 ---
Type: <class 'dict'>
Content: {'speaker': 'SPEAKER_02', 'start': 18.944, 'end': 20.945, 'duration': 2.0010000000000012, 'text': "Okay, that's fine."}

CHECKING FOR EVIDENCE IDs
Field: speaker
Field: start
Field: end
Field: duration
Field: text


In [11]:
# ============================================================
# M7 - LOAD MOM-WORTHY CANDIDATES
# ============================================================

import os
import json

candidate_path = os.path.join(
    PROJECT_DIR,
    "data",
    "transcripts",
    "ES2004a_mom_worthy_candidates.json"
)

print("=" * 90)
print("M7 - LOADING MOM-WORTHY CANDIDATES")
print("=" * 90)

print("\nFile path:")
print(candidate_path)

print("\nFile exists:", os.path.exists(candidate_path))

with open(candidate_path, "r", encoding="utf-8") as f:
    mom_candidates = json.load(f)

print("\nLoaded successfully.")
print("Type:", type(mom_candidates))

if isinstance(mom_candidates, dict):
    print("\nTop-level keys:")
    print(list(mom_candidates.keys()))

elif isinstance(mom_candidates, list):
    print("\nNumber of candidates:")
    print(len(mom_candidates))

print("\n" + "=" * 90)
print("FIRST 3 CANDIDATES")
print("=" * 90)

candidates_list = (
    mom_candidates["candidates"]
    if isinstance(mom_candidates, dict) and "candidates" in mom_candidates
    else mom_candidates
)

for i, candidate in enumerate(candidates_list[:3]):
    print(f"\n--- Candidate {i} ---")
    print(candidate)

M7 - LOADING MOM-WORTHY CANDIDATES

File path:
/content/drive/MyDrive/MTechIndProj/MoM_Project/data/transcripts/ES2004a_mom_worthy_candidates.json

File exists: True

Loaded successfully.
Type: <class 'dict'>

Top-level keys:
['meeting_id', 'source', 'original_candidate_count', 'excluded_candidate_count', 'mom_worthy_candidate_count', 'excluded_candidate_ids', 'candidates']

FIRST 3 CANDIDATES

--- Candidate 0 ---
{'candidate_id': 5, 'source_utterance_id': 9, 'speaker': 'SPEAKER_02', 'start': 84.658, 'end': 93.127, 'text': "I'm Sarah, the project manager, and this is our first meeting, surprisingly enough. Okay, this is our agenda.", 'event_type': 'INFORMATION', 'mom_worthy': True}

--- Candidate 1 ---
{'candidate_id': 6, 'source_utterance_id': 10, 'speaker': 'SPEAKER_02', 'start': 95.79, 'end': 101.315, 'text': 'We will do some stuff, get to know each other a bit better, feel more comfortable with each other.', 'event_type': 'ACTION', 'mom_worthy': True}

--- Candidate 2 ---
{'candida

In [12]:
# ============================================================
# M7 - BUILD CANDIDATE EVIDENCE LOOKUP
# ============================================================

print("=" * 90)
print("M7 - BUILDING CANDIDATE EVIDENCE LOOKUP")
print("=" * 90)

# Candidate list from the saved MoM-worthy candidate file
candidates_list = mom_candidates["candidates"]

# Map candidate_id -> candidate information
candidate_evidence = {
    candidate["candidate_id"]: candidate
    for candidate in candidates_list
}

print("\nNumber of MoM-worthy candidates:", len(candidate_evidence))

print("\nCandidate IDs:")
print(sorted(candidate_evidence.keys()))

# ------------------------------------------------------------
# Verify Claim 1 source candidates
# ------------------------------------------------------------

claim_1 = structured_claims["claims"][0]

print("\n" + "=" * 90)
print("CLAIM 1 SOURCE EVIDENCE")
print("=" * 90)

print("\nClaim:")
print(claim_1["claim_text"])

print("\nSource candidate IDs:")
print(claim_1["source_candidate_ids"])

for candidate_id in claim_1["source_candidate_ids"]:

    candidate = candidate_evidence.get(candidate_id)

    print("\n" + "-" * 70)

    if candidate is None:
        print("Candidate ID:", candidate_id)
        print("ERROR: Candidate not found")
        continue

    print("Candidate ID:", candidate["candidate_id"])
    print("Source utterance ID:", candidate["source_utterance_id"])
    print("Speaker:", candidate["speaker"])
    print("Start:", candidate["start"])
    print("End:", candidate["end"])
    print("Event type:", candidate["event_type"])
    print("Text:", candidate["text"])

M7 - BUILDING CANDIDATE EVIDENCE LOOKUP

Number of MoM-worthy candidates: 42

Candidate IDs:
[5, 6, 7, 8, 10, 11, 12, 13, 14, 15, 49, 50, 52, 54, 55, 56, 57, 59, 60, 61, 62, 63, 64, 66, 67, 68, 69, 72, 73, 76, 79, 85, 88, 89, 90, 91, 92, 93, 96, 97, 98, 99]

CLAIM 1 SOURCE EVIDENCE

Claim:
The meeting was introduced as the first meeting, with an agenda covering team interaction, tool training, the project plan, and discussion of ideas.

Source candidate IDs:
[5, 6, 7, 8]

----------------------------------------------------------------------
Candidate ID: 5
Source utterance ID: 9
Speaker: SPEAKER_02
Start: 84.658
End: 93.127
Event type: INFORMATION
Text: I'm Sarah, the project manager, and this is our first meeting, surprisingly enough. Okay, this is our agenda.

----------------------------------------------------------------------
Candidate ID: 6
Source utterance ID: 10
Speaker: SPEAKER_02
Start: 95.79
End: 101.315
Event type: ACTION
Text: We will do some stuff, get to know each othe

In [13]:
# ============================================================
# M7 - CONSTRUCT SUPPORTING EVIDENCE FOR CLAIM 1
# ============================================================

print("=" * 90)
print("M7 - CONSTRUCTING CLAIM 1 EVIDENCE")
print("=" * 90)

claim_1 = structured_claims["claims"][0]

# Retrieve the actual candidate records
supporting_evidence_claim_1 = []

for candidate_id in claim_1["source_candidate_ids"]:

    candidate = candidate_evidence.get(candidate_id)

    if candidate is None:
        print(f"WARNING: Candidate {candidate_id} not found")
        continue

    evidence_item = {
        "evidence_id": candidate["candidate_id"],
        "speaker": candidate["speaker"],
        "start": candidate["start"],
        "end": candidate["end"],
        "text": candidate["text"],
        "event_type": candidate["event_type"],
        "source_utterance_id": candidate["source_utterance_id"]
    }

    supporting_evidence_claim_1.append(evidence_item)

print("\nNumber of evidence items:",
      len(supporting_evidence_claim_1))

print("\nEvidence IDs:")
print([
    item["evidence_id"]
    for item in supporting_evidence_claim_1
])

print("\nEvidence speakers:")
print([
    item["speaker"]
    for item in supporting_evidence_claim_1
])

print("\nEvidence interval:")
print(
    min(item["start"] for item in supporting_evidence_claim_1),
    "→",
    max(item["end"] for item in supporting_evidence_claim_1)
)

print("\nEvidence text:")

for item in supporting_evidence_claim_1:
    print(
        f"\n[{item['evidence_id']}] "
        f"{item['start']:.3f}–{item['end']:.3f} "
        f"{item['speaker']}"
    )
    print(item["text"])

M7 - CONSTRUCTING CLAIM 1 EVIDENCE

Number of evidence items: 4

Evidence IDs:
[5, 6, 7, 8]

Evidence speakers:
['SPEAKER_02', 'SPEAKER_02', 'SPEAKER_02', 'SPEAKER_02']

Evidence interval:
84.658 → 109.823

Evidence text:

[5] 84.658–93.127 SPEAKER_02
I'm Sarah, the project manager, and this is our first meeting, surprisingly enough. Okay, this is our agenda.

[6] 95.790–101.315 SPEAKER_02
We will do some stuff, get to know each other a bit better, feel more comfortable with each other.

[7] 102.536–105.239 SPEAKER_02
Then we'll go do tool training,

[8] 106.340–109.823 SPEAKER_02
talk about the project plan, discuss our own ideas and everything.


In [14]:
# ============================================================
# M7 - END-TO-END VERIFICATION OF CLAIM 1
# ============================================================

print("=" * 90)
print("M7 - END-TO-END CLAIM 1 VERIFICATION")
print("=" * 90)

claim_1 = structured_claims["claims"][0]

result_claim_1 = verify_mom_claim(
    claim={
        "claim": claim_1["claim_text"],
        "speaker": claim_1["speaker"],
        "type": claim_1["event_type"]
    },
    supporting_evidence=supporting_evidence_claim_1,
    content_proposition=claim_1["claim_text"],
    expected_start=claim_1["start"],
    expected_end=claim_1["end"],
    timestamp_tolerance=2.0
)

print("\nClaim:")
print(result_claim_1["claim"])

print("\nOverall Status:")
print(result_claim_1["status"])

print("\nReason:")
print(result_claim_1["reason"])

print("\n" + "-" * 70)
print("ATTRIBUTE CHECKS")
print("-" * 70)

for check_name, check_result in result_claim_1["checks"].items():

    print(f"\n{check_name}:")
    print(check_result)

print("\n" + "-" * 70)
print("FAILED CHECKS")
print("-" * 70)

print(result_claim_1["failed_checks"])

print("\n" + "-" * 70)
print("FINAL EVIDENCE")
print("-" * 70)

print("Evidence IDs:", result_claim_1["evidence_ids"])
print("Speakers:", result_claim_1["evidence_speakers"])
print(
    "Evidence interval:",
    result_claim_1["evidence_start"],
    "→",
    result_claim_1["evidence_end"]
)

M7 - END-TO-END CLAIM 1 VERIFICATION


NameError: name 'verify_mom_claim' is not defined

In [15]:
# ============================================================
# M7 - CHECK VERIFICATION FUNCTIONS IN CURRENT RUNTIME
# ============================================================

print("=" * 90)
print("M7 - VERIFICATION FUNCTION CHECK")
print("=" * 90)

functions_to_check = [
    "run_nli",
    "get_context_window",
    "analyze_attribute_evidence",
    "select_supporting_candidates",
    "create_evidence_group",
    "check_speaker_consistency",
    "evaluate_speaker_consistency_for_evidence",
    "check_timestamp_consistency",
    "check_event_type_consistency",
    "verify_attribute",
    "verify_mom_claim"
]

print()

for function_name in functions_to_check:
    exists = function_name in globals()
    print(f"{function_name:<45} : {'AVAILABLE' if exists else 'MISSING'}")

M7 - VERIFICATION FUNCTION CHECK

run_nli                                       : MISSING
get_context_window                            : MISSING
analyze_attribute_evidence                    : MISSING
select_supporting_candidates                  : MISSING
create_evidence_group                         : MISSING
check_speaker_consistency                     : MISSING
evaluate_speaker_consistency_for_evidence     : MISSING
check_timestamp_consistency                   : MISSING
check_event_type_consistency                  : MISSING
verify_attribute                              : MISSING
verify_mom_claim                              : MISSING


In [16]:
# ============================================================
# M7 - LOCATE SAVED VERIFICATION FUNCTION DEFINITIONS
# ============================================================

import json
import re

m7_notebook_path = os.path.join(
    PROJECT_DIR,
    "04_mom_generation",
    "04_mom_generation.ipynb"
)

print("=" * 90)
print("M7 - LOCATING SAVED VERIFICATION FUNCTIONS")
print("=" * 90)

print("\nNotebook exists:", os.path.exists(m7_notebook_path))

with open(m7_notebook_path, "r", encoding="utf-8") as f:
    m7_notebook = json.load(f)

print("Notebook loaded.")
print("Number of cells:", len(m7_notebook["cells"]))

target_functions = [
    "def run_nli",
    "def get_context_window",
    "def analyze_attribute_evidence",
    "def select_supporting_candidates",
    "def create_evidence_group",
    "def check_speaker_consistency",
    "def evaluate_speaker_consistency_for_evidence",
    "def check_timestamp_consistency",
    "def check_event_type_consistency",
    "def verify_attribute",
    "def verify_mom_claim"
]

print("\n" + "=" * 90)
print("FUNCTION LOCATIONS")
print("=" * 90)

for function_name in target_functions:

    matches = []

    for cell_index, cell in enumerate(m7_notebook["cells"]):

        if cell.get("cell_type") != "code":
            continue

        source = "".join(cell.get("source", []))

        if function_name in source:
            matches.append(cell_index)

    print(f"\n{function_name}:")
    print("  Cell(s):", matches)

M7 - LOCATING SAVED VERIFICATION FUNCTIONS

Notebook exists: True
Notebook loaded.
Number of cells: 141

FUNCTION LOCATIONS

def run_nli:
  Cell(s): [48, 55, 63, 104]

def get_context_window:
  Cell(s): [77, 105]

def analyze_attribute_evidence:
  Cell(s): [85, 107]

def select_supporting_candidates:
  Cell(s): [86, 108]

def create_evidence_group:
  Cell(s): [91, 109]

def check_speaker_consistency:
  Cell(s): [83, 111]

def evaluate_speaker_consistency_for_evidence:
  Cell(s): [114]

def check_timestamp_consistency:
  Cell(s): [83, 112]

def check_event_type_consistency:
  Cell(s): [113, 116]

def verify_attribute:
  Cell(s): [92, 94, 110]

def verify_mom_claim:
  Cell(s): [115]


In [18]:
# ============================================================
# M7 - INSPECT SAVED FUNCTION CELLS
# ============================================================

print("=" * 90)
print("M7 - SAVED FUNCTION CELL INSPECTION")
print("=" * 90)

target_cells = list(range(104, 117))

for cell_index in target_cells:

    cell = m7_notebook["cells"][cell_index]

    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))

    print("\n" + "=" * 90)
    print(f"CELL {cell_index}")
    print("=" * 90)

    # Print only the first 12 lines so we can identify
    # imports, function names, and dependencies.
    lines = source.splitlines()

    for line in lines[:12]:
        print(line)

    if len(lines) > 12:
        print(f"... ({len(lines) - 12} more lines)")

M7 - SAVED FUNCTION CELL INSPECTION

CELL 104
# ============================================================
# M7 - Restore NLI Inference Function
# ============================================================
# Purpose:
#   Restore the lightweight NLI inference function after a
#   Colab runtime restart.
#
# Model is already loaded in:
#   nli_tokenizer
#   nli_model
# ============================================================

... (92 more lines)

CELL 105
# ============================================================
# M7 - Restore Context Window Function
# ============================================================
# Purpose:
#   Retrieve neighboring meeting utterances around a selected
#   evidence utterance.
#
# Example:
#   Evidence ID 15 with window_size=2
#   → IDs 13, 14, 15, 16, 17
#
# No models are loaded and no files are modified.
... (94 more lines)

CELL 106
# ============================================================
# M7 - Restore Lexical Support Function
# ======

In [20]:
# ============================================================
# M7 - RESTORE TESTED VERIFICATION FUNCTIONS
# ============================================================

print("=" * 90)
print("M7 - RESTORING SAVED VERIFICATION FUNCTIONS")
print("=" * 90)

# Latest tested M7 cells.
# Cell 116 is preferred over the earlier event-type implementation
# in cell 113.

restore_cells = [
    104, 105, 106, 107, 108, 109,
    110, 111, 112, 114, 115, 116
]

for cell_index in restore_cells:

    cell = m7_notebook["cells"][cell_index]

    if cell.get("cell_type") != "code":
        print(f"Skipping cell {cell_index}: not a code cell")
        continue

    source = "".join(cell.get("source", []))

    print(f"\nRestoring cell {cell_index}...")

    # Execute the exact code saved in the notebook.
    exec(compile(source, f"<M7 cell {cell_index}>", "exec"))

print("\n" + "=" * 90)
print("M7 FUNCTION RESTORATION COMPLETE")
print("=" * 90)

# Verify that the required functions now exist.
functions_to_check = [
    "run_nli",
    "get_context_window",
    "analyze_attribute_evidence",
    "select_supporting_candidates",
    "create_evidence_group",
    "check_speaker_consistency",
    "evaluate_speaker_consistency_for_evidence",
    "check_timestamp_consistency",
    "check_event_type_consistency",
    "verify_attribute",
    "verify_mom_claim"
]

print("\nFunction availability:")

for function_name in functions_to_check:
    print(
        f"{function_name:<45} : "
        f"{'AVAILABLE' if function_name in globals() else 'MISSING'}"
    )

M7 - RESTORING SAVED VERIFICATION FUNCTIONS

Restoring cell 104...


NameError: name 'nli_tokenizer' is not defined

In [21]:
# ============================================================
# M7 - RESTORE NLI MODEL AND TOKENIZER
# ============================================================

from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

print("=" * 90)
print("M7 - RESTORING NLI MODEL")
print("=" * 90)

NLI_MODEL_NAME = "cross-encoder/nli-deberta-v3-small"

print("\nModel:", NLI_MODEL_NAME)

# Load tokenizer
nli_tokenizer = AutoTokenizer.from_pretrained(NLI_MODEL_NAME)

# Load model
nli_model = AutoModelForSequenceClassification.from_pretrained(
    NLI_MODEL_NAME
)

# Select device
nli_device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

nli_model = nli_model.to(nli_device)
nli_model.eval()

print("\nNLI model loaded successfully.")
print("Device:", nli_device)

print("\nModel labels:")
print(nli_model.config.id2label)

print("\nTokenizer:", type(nli_tokenizer).__name__)
print("Model:", type(nli_model).__name__)

M7 - RESTORING NLI MODEL

Model: cross-encoder/nli-deberta-v3-small


config.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.35k [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

...

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  568MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]


NLI model loaded successfully.
Device: cuda

Model labels:
{0: 'contradiction', 1: 'entailment', 2: 'neutral'}

Tokenizer: DebertaV2Tokenizer
Model: DebertaV2ForSequenceClassification


In [22]:
# ============================================================
# M7 - RESTORE TESTED VERIFICATION FUNCTIONS
# ============================================================

print("=" * 90)
print("M7 - RESTORING SAVED VERIFICATION FUNCTIONS")
print("=" * 90)

restore_cells = [
    104, 105, 106, 107, 108, 109,
    110, 111, 112, 114, 115, 116
]

for cell_index in restore_cells:

    cell = m7_notebook["cells"][cell_index]

    if cell.get("cell_type") != "code":
        print(f"Skipping cell {cell_index}: not a code cell")
        continue

    source = "".join(cell.get("source", []))

    print(f"\nRestoring cell {cell_index}...")

    exec(compile(source, f"<M7 cell {cell_index}>", "exec"))

print("\n" + "=" * 90)
print("M7 FUNCTION RESTORATION COMPLETE")
print("=" * 90)

functions_to_check = [
    "run_nli",
    "get_context_window",
    "analyze_attribute_evidence",
    "select_supporting_candidates",
    "create_evidence_group",
    "check_speaker_consistency",
    "evaluate_speaker_consistency_for_evidence",
    "check_timestamp_consistency",
    "check_event_type_consistency",
    "verify_attribute",
    "verify_mom_claim"
]

print("\nFunction availability:")

for function_name in functions_to_check:
    print(
        f"{function_name:<45} : "
        f"{'AVAILABLE' if function_name in globals() else 'MISSING'}"
    )

M7 - RESTORING SAVED VERIFICATION FUNCTIONS

Restoring cell 104...


NameError: name 'NLI_DEVICE' is not defined

In [23]:
# ============================================================
# M7 - RESTORE NLI DEVICE VARIABLE
# ============================================================

NLI_DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 90)
print("M7 - NLI DEVICE RESTORED")
print("=" * 90)

print("NLI_DEVICE:", NLI_DEVICE)

M7 - NLI DEVICE RESTORED
NLI_DEVICE: cuda


In [24]:
# ============================================================
# M7 - RESTORE TESTED VERIFICATION FUNCTIONS
# ============================================================

print("=" * 90)
print("M7 - RESTORING SAVED VERIFICATION FUNCTIONS")
print("=" * 90)

restore_cells = [
    104, 105, 106, 107, 108, 109,
    110, 111, 112, 114, 115, 116
]

for cell_index in restore_cells:

    cell = m7_notebook["cells"][cell_index]

    if cell.get("cell_type") != "code":
        print(f"Skipping cell {cell_index}: not a code cell")
        continue

    source = "".join(cell.get("source", []))

    print(f"\nRestoring cell {cell_index}...")

    exec(compile(source, f"<M7 cell {cell_index}>", "exec"))

print("\n" + "=" * 90)
print("M7 FUNCTION RESTORATION COMPLETE")
print("=" * 90)

functions_to_check = [
    "run_nli",
    "get_context_window",
    "analyze_attribute_evidence",
    "select_supporting_candidates",
    "create_evidence_group",
    "check_speaker_consistency",
    "evaluate_speaker_consistency_for_evidence",
    "check_timestamp_consistency",
    "check_event_type_consistency",
    "verify_attribute",
    "verify_mom_claim"
]

print("\nFunction availability:")

for function_name in functions_to_check:
    print(
        f"{function_name:<45} : "
        f"{'AVAILABLE' if function_name in globals() else 'MISSING'}"
    )

M7 - RESTORING SAVED VERIFICATION FUNCTIONS

Restoring cell 104...


NameError: name 'NLI_LABELS' is not defined

In [25]:
# ============================================================
# M7 - RESTORE NLI LABEL MAPPING
# ============================================================

NLI_LABELS = {
    0: "contradiction",
    1: "entailment",
    2: "neutral"
}

print("=" * 90)
print("M7 - NLI LABEL MAPPING RESTORED")
print("=" * 90)

print("NLI_LABELS:", NLI_LABELS)

M7 - NLI LABEL MAPPING RESTORED
NLI_LABELS: {0: 'contradiction', 1: 'entailment', 2: 'neutral'}


In [26]:
# ============================================================
# M7 - RESTORE TESTED VERIFICATION FUNCTIONS
# ============================================================

print("=" * 90)
print("M7 - RESTORING SAVED VERIFICATION FUNCTIONS")
print("=" * 90)

restore_cells = [
    104, 105, 106, 107, 108, 109,
    110, 111, 112, 114, 115, 116
]

for cell_index in restore_cells:

    cell = m7_notebook["cells"][cell_index]

    if cell.get("cell_type") != "code":
        print(f"Skipping cell {cell_index}: not a code cell")
        continue

    source = "".join(cell.get("source", []))

    print(f"\nRestoring cell {cell_index}...")

    exec(compile(source, f"<M7 cell {cell_index}>", "exec"))

print("\n" + "=" * 90)
print("M7 FUNCTION RESTORATION COMPLETE")
print("=" * 90)

functions_to_check = [
    "run_nli",
    "get_context_window",
    "analyze_attribute_evidence",
    "select_supporting_candidates",
    "create_evidence_group",
    "check_speaker_consistency",
    "evaluate_speaker_consistency_for_evidence",
    "check_timestamp_consistency",
    "check_event_type_consistency",
    "verify_attribute",
    "verify_mom_claim"
]

print("\nFunction availability:")

for function_name in functions_to_check:
    print(
        f"{function_name:<45} : "
        f"{'AVAILABLE' if function_name in globals() else 'MISSING'}"
    )

M7 - RESTORING SAVED VERIFICATION FUNCTIONS

Restoring cell 104...
M7 - NLI SANITY TEST
Premise    : Now, we're developing a remote control, which you probably already know.
Hypothesis : The team is developing a remote control.

Result:
Label         : entailment
Contradiction : 0.0005
Entailment    : 0.7902
Neutral       : 0.2093

NLI function restored successfully.

Restoring cell 105...


NameError: name 'evidence_documents' is not defined

In [27]:
# ============================================================
# M7 - LOCATE EVIDENCE DOCUMENTS SETUP
# ============================================================

print("=" * 90)
print("M7 - LOCATING EVIDENCE DOCUMENTS SETUP")
print("=" * 90)

# Search the saved notebook for where evidence_documents
# is created or assigned.

matches = []

for cell_index, cell in enumerate(m7_notebook["cells"]):

    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))

    if "evidence_documents" in source:
        matches.append(cell_index)

print("\nCells containing 'evidence_documents':")
print(matches)

print("\n" + "=" * 90)
print("RELEVANT CODE")
print("=" * 90)

for cell_index in matches:

    source = "".join(
        m7_notebook["cells"][cell_index].get("source", [])
    )

    print("\n" + "-" * 90)
    print(f"CELL {cell_index}")
    print("-" * 90)

    # Show lines containing evidence_documents and
    # a few surrounding lines.
    lines = source.splitlines()

    for i, line in enumerate(lines):
        if "evidence_documents" in line:
            start = max(0, i - 5)
            end = min(len(lines), i + 8)

            for j in range(start, end):
                print(f"{j + 1:4}: {lines[j]}")

            print()

M7 - LOCATING EVIDENCE DOCUMENTS SETUP

Cells containing 'evidence_documents':
[27, 29, 30, 32, 33, 35, 67, 69, 71, 73, 77, 81, 82, 83, 85, 90, 91, 100, 102, 105, 109]

RELEVANT CODE

------------------------------------------------------------------------------------------
CELL 27
------------------------------------------------------------------------------------------
  14: #
  15: # These IDs will later be used to link verified MoM claims
  16: # back to their supporting transcript evidence.
  17: # ============================================================
  18: 
  19: evidence_documents = []
  20: 
  21: for idx, utterance in enumerate(speaker_utterances, start=1):
  22: 
  23:     evidence_documents.append({
  24:         "evidence_id": idx,
  25:         "speaker": utterance["speaker"],
  26:         "start": utterance["start"],

  18: 
  19: evidence_documents = []
  20: 
  21: for idx, utterance in enumerate(speaker_utterances, start=1):
  22: 
  23:     evidence_documents.

In [28]:
# ============================================================
# M7 - RESTORE EVIDENCE DOCUMENTS
# ============================================================

print("=" * 90)
print("M7 - RESTORING EVIDENCE DOCUMENTS")
print("=" * 90)

# Use the already-loaded speaker transcript
utterances = speaker_transcript["utterances"]

# Recreate evidence documents exactly as defined
# in the saved M7 notebook.
evidence_documents = []

for i, utterance in enumerate(utterances, start=1):

    evidence_documents.append({
        "evidence_id": i,
        "speaker": utterance["speaker"],
        "start": float(utterance["start"]),
        "end": float(utterance["end"]),
        "duration": float(utterance["end"]) - float(utterance["start"]),
        "text": utterance["text"].strip()
    })

print("\nEvidence documents restored:", len(evidence_documents))

print("\nFirst evidence document:")
print(evidence_documents[0])

print("\nClaim 1 evidence check:")

for evidence_id in [5, 6, 7, 8]:

    evidence = next(
        doc for doc in evidence_documents
        if doc["evidence_id"] == evidence_id
    )

    print(
        f"\nEvidence {evidence_id}: "
        f"{evidence['start']:.3f}–{evidence['end']:.3f} "
        f"{evidence['speaker']}"
    )
    print(evidence["text"])

M7 - RESTORING EVIDENCE DOCUMENTS

Evidence documents restored: 148

First evidence document:
{'evidence_id': 1, 'speaker': 'SPEAKER_02', 'start': 10.998, 'end': 14.521, 'duration': 3.5230000000000015, 'text': "Are we, we're not like the dim lights, so we can see that a bit better."}

Claim 1 evidence check:

Evidence 5: 25.128–26.509 SPEAKER_03
So we've got both of these clipped on.

Evidence 6: 29.091–32.073 SPEAKER_03
Is she gonna answer me? Yeah, I've got both of them.

Evidence 7: 50.605–51.346 SPEAKER_03
I'm just gonna

Evidence 8: 80.574–83.017 SPEAKER_02
Okay, hello everybody.


In [29]:
# ============================================================
# M7 - CORRECT CANDIDATE → EVIDENCE ID MAPPING
# ============================================================

print("=" * 90)
print("M7 - CORRECTING CANDIDATE / EVIDENCE ID MAPPING")
print("=" * 90)

claim_1 = structured_claims["claims"][0]

correct_evidence_claim_1 = []

for candidate_id in claim_1["source_candidate_ids"]:

    candidate = candidate_evidence[candidate_id]

    # candidate_id is NOT the M7 evidence_id.
    # source_utterance_id corresponds to the original
    # speaker-transcript utterance number.
    evidence_id = candidate["source_utterance_id"]

    evidence = next(
        doc for doc in evidence_documents
        if int(doc["evidence_id"]) == int(evidence_id)
    )

    correct_evidence_claim_1.append(evidence.copy())

print("\nClaim 1 candidate IDs:")
print(claim_1["source_candidate_ids"])

print("\nMapped M7 evidence IDs:")
print([
    item["evidence_id"]
    for item in correct_evidence_claim_1
])

print("\nMapped source utterance IDs:")
print([
    candidate_evidence[cid]["source_utterance_id"]
    for cid in claim_1["source_candidate_ids"]
])

print("\nEvidence interval:")
print(
    min(item["start"] for item in correct_evidence_claim_1),
    "→",
    max(item["end"] for item in correct_evidence_claim_1)
)

print("\nMapped evidence:")

for item in correct_evidence_claim_1:
    print(
        f"\nEvidence ID {item['evidence_id']}: "
        f"{item['start']:.3f}–{item['end']:.3f} "
        f"{item['speaker']}"
    )
    print(item["text"])

M7 - CORRECTING CANDIDATE / EVIDENCE ID MAPPING

Claim 1 candidate IDs:
[5, 6, 7, 8]

Mapped M7 evidence IDs:
[9, 10, 11, 12]

Mapped source utterance IDs:
[9, 10, 11, 12]

Evidence interval:
84.658 → 109.823

Mapped evidence:

Evidence ID 9: 84.658–93.127 SPEAKER_02
I'm Sarah, the project manager, and this is our first meeting, surprisingly enough. Okay, this is our agenda.

Evidence ID 10: 95.790–101.315 SPEAKER_02
We will do some stuff, get to know each other a bit better, feel more comfortable with each other.

Evidence ID 11: 102.536–105.239 SPEAKER_02
Then we'll go do tool training,

Evidence ID 12: 106.340–109.823 SPEAKER_02
talk about the project plan, discuss our own ideas and everything.


In [30]:
# ============================================================
# M7 - RESTORE REMAINING VERIFICATION FUNCTIONS
# ============================================================

print("=" * 90)
print("M7 - RESTORING REMAINING VERIFICATION FUNCTIONS")
print("=" * 90)

restore_cells = [
    105, 106, 107, 108, 109,
    110, 111, 112, 114, 115, 116
]

for cell_index in restore_cells:

    cell = m7_notebook["cells"][cell_index]

    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))

    print(f"\nRestoring cell {cell_index}...")

    exec(compile(source, f"<M7 cell {cell_index}>", "exec"))

print("\n" + "=" * 90)
print("M7 FUNCTION RESTORATION CHECK")
print("=" * 90)

functions_to_check = [
    "run_nli",
    "get_context_window",
    "analyze_attribute_evidence",
    "select_supporting_candidates",
    "create_evidence_group",
    "check_speaker_consistency",
    "evaluate_speaker_consistency_for_evidence",
    "check_timestamp_consistency",
    "check_event_type_consistency",
    "verify_attribute",
    "verify_mom_claim"
]

for function_name in functions_to_check:
    print(
        f"{function_name:<45} : "
        f"{'AVAILABLE' if function_name in globals() else 'MISSING'}"
    )

M7 - RESTORING REMAINING VERIFICATION FUNCTIONS

Restoring cell 105...
M7 - CONTEXT WINDOW TEST
Central evidence ID: 15
Context IDs: [13, 14, 15, 16, 17]

Context:

ID 13 | SPEAKER_02 | 112.185 → 115.748
  And we've got 25 minutes to do that as far as I can understand.

ID 14 | SPEAKER_02 | 117.699 → 120.921
  Now, we're developing a remote control, which you probably already know.

ID 15 | SPEAKER_02 | 122.503 → 132.290
  We want it to be original, something that people haven't thought of. It's not out in the shops. Trendy, appealing to a wide market, but,

ID 16 | SPEAKER_02 | 133.291 → 138.174
  you know, not a hunk of metal. And user-friendly, grannies to kids,

ID 17 | SPEAKER_02 | 139.375 → 141.016
  maybe even pooches, should be able to use it.

Context window function restored successfully.

Restoring cell 106...
M7 - LEXICAL SUPPORT TEST
Proposition: The remote control is intended to be original.

Lexical overlap: 0.7500
Matched terms: ['control', 'original', 'remote']
Missing

NameError: name 'retrieve_evidence' is not defined

In [31]:
# ============================================================
# M7 - LOCATE SAVED RETRIEVE_EVIDENCE FUNCTION
# ============================================================

print("=" * 90)
print("M7 - LOCATING RETRIEVE_EVIDENCE FUNCTION")
print("=" * 90)

matches = []

for cell_index, cell in enumerate(m7_notebook["cells"]):

    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))

    if "def retrieve_evidence" in source:
        matches.append(cell_index)

print("\nCells containing retrieve_evidence definition:")
print(matches)

print("\n" + "=" * 90)
print("FUNCTION DEFINITIONS")
print("=" * 90)

for cell_index in matches:

    source = "".join(
        m7_notebook["cells"][cell_index].get("source", [])
    )

    print("\n" + "-" * 90)
    print(f"CELL {cell_index}")
    print("-" * 90)

    lines = source.splitlines()

    for i, line in enumerate(lines):

        if "def retrieve_evidence" in line:

            start = max(0, i - 5)
            end = min(len(lines), i + 45)

            for j in range(start, end):
                print(f"{j + 1:4}: {lines[j]}")

            break

M7 - LOCATING RETRIEVE_EVIDENCE FUNCTION

Cells containing retrieve_evidence definition:
[33, 35, 67, 69, 71, 100, 102]

FUNCTION DEFINITIONS

------------------------------------------------------------------------------------------
CELL 33
------------------------------------------------------------------------------------------
   8: # Retrieval similarity identifies candidate evidence.
   9: # It does NOT by itself prove that the evidence supports
  10: # the claim. Verification will be performed in the next stage.
  11: # ============================================================
  12: 
  13: def retrieve_evidence(claim_text, top_k=5):
  14: 
  15:     # Generate normalized BGE embedding for the claim
  16:     claim_embedding = bge_model.encode(
  17:         [claim_text],
  18:         normalize_embeddings=True,
  19:         convert_to_numpy=True
  20:     ).astype("float32")
  21: 
  22:     # Search FAISS index
  23:     similarities, indices = faiss_index.search(
  24:    

In [32]:
# ============================================================
# M7 - CHECK BGE + FAISS DEPENDENCIES
# ============================================================

print("=" * 90)
print("M7 - CHECKING BGE + FAISS DEPENDENCIES")
print("=" * 90)

dependencies = [
    "bge_model",
    "faiss_index",
    "evidence_documents",
    "evidence_embeddings"
]

for name in dependencies:
    print(
        f"{name:<25} : "
        f"{'AVAILABLE' if name in globals() else 'MISSING'}"
    )

if "faiss_index" in globals():
    print("\nFAISS vectors:", faiss_index.ntotal)

if "evidence_documents" in globals():
    print("Evidence documents:", len(evidence_documents))

if "evidence_embeddings" in globals():
    print("Embedding shape:", evidence_embeddings.shape)

M7 - CHECKING BGE + FAISS DEPENDENCIES
bge_model                 : MISSING
faiss_index               : MISSING
evidence_documents        : AVAILABLE
evidence_embeddings       : MISSING
Evidence documents: 148


In [33]:
# ============================================================
# M7 - RESTORE BGE + FAISS RETRIEVAL
# ============================================================

import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

print("=" * 90)
print("M7 - RESTORING BGE + FAISS RETRIEVAL")
print("=" * 90)

# ------------------------------------------------------------
# 1. Load BGE model
# ------------------------------------------------------------

BGE_MODEL_NAME = "BAAI/bge-small-en-v1.5"

print("\nLoading BGE model:")
print(BGE_MODEL_NAME)

bge_model = SentenceTransformer(BGE_MODEL_NAME)

print("BGE model loaded successfully.")

# ------------------------------------------------------------
# 2. Generate evidence embeddings
# ------------------------------------------------------------

evidence_texts = [
    doc["text"]
    for doc in evidence_documents
]

print("\nNumber of evidence documents:", len(evidence_texts))
print("Generating normalized embeddings...")

evidence_embeddings = bge_model.encode(
    evidence_texts,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=True
).astype("float32")

print("\nEmbedding shape:", evidence_embeddings.shape)
print("Embedding dtype:", evidence_embeddings.dtype)

# ------------------------------------------------------------
# 3. Build FAISS index
# ------------------------------------------------------------

embedding_dimension = evidence_embeddings.shape[1]

faiss_index = faiss.IndexFlatIP(embedding_dimension)

faiss_index.add(evidence_embeddings)

print("\nFAISS index created.")
print("Embedding dimension:", embedding_dimension)
print("FAISS vectors:", faiss_index.ntotal)

# ------------------------------------------------------------
# 4. Validate
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("M7 - BGE + FAISS RESTORATION CHECK")
print("=" * 90)

print("Evidence documents :", len(evidence_documents))
print("Embedding shape    :", evidence_embeddings.shape)
print("FAISS vectors      :", faiss_index.ntotal)
print("Embedding dimension:", embedding_dimension)

print(
    "\nIndex successfully built:",
    faiss_index.ntotal == len(evidence_documents)
)

ModuleNotFoundError: No module named 'faiss'

In [34]:
# ============================================================
# M7 - INSTALL MISSING FAISS DEPENDENCY
# ============================================================

!pip install -q faiss-cpu

print("FAISS installation completed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 58.9 MB/s eta 0:00:00
FAISS installation completed.


In [35]:
# ============================================================
# M7 - VERIFY FAISS
# ============================================================

import faiss

print("=" * 90)
print("M7 - FAISS VERIFICATION")
print("=" * 90)

print("FAISS version:", faiss.__version__)
print("FAISS import : SUCCESS")

M7 - FAISS VERIFICATION
FAISS version: 1.15.0
FAISS import : SUCCESS


In [36]:
# ============================================================
# M7 - RESTORE BGE EMBEDDING MODEL
# ============================================================

from sentence_transformers import SentenceTransformer

print("=" * 90)
print("M7 - LOADING BGE MODEL")
print("=" * 90)

BGE_MODEL_NAME = "BAAI/bge-small-en-v1.5"

bge_model = SentenceTransformer(BGE_MODEL_NAME)

print("\nBGE model loaded successfully.")
print("Model:", BGE_MODEL_NAME)

M7 - LOADING BGE MODEL


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


BGE model loaded successfully.
Model: BAAI/bge-small-en-v1.5


In [2]:
# ============================================================
# STEP 1 - CHECK PERSISTENT PROJECT ARTIFACTS
# ============================================================

from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive/MyDrive/MTechIndProj/MoM_Project"
)

print("=" * 90)
print("RESTART-SAFE PIPELINE - PERSISTENT ARTIFACT CHECK")
print("=" * 90)

folders_to_check = [
    "data/raw",
    "data/vad",
    "data/transcripts",
    "data/embeddings",
    "data/faiss",
    "data/evidence",
    "outputs/mom",
    "outputs/evidence",
    "outputs/verified",
    "evaluation_results",
    "models",
]

for folder in folders_to_check:
    path = PROJECT_DIR / folder

    if path.exists():
        files = list(path.iterdir())
        print(f"\n✓ {folder:<25} {len(files)} item(s)")

        for f in files[:10]:
            print(f"    └── {f.name}")

        if len(files) > 10:
            print(f"    └── ... {len(files) - 10} more")
    else:
        print(f"\n✗ {folder:<25} MISSING")

print("\n" + "=" * 90)
print("CHECK COMPLETE")
print("=" * 90)

RESTART-SAFE PIPELINE - PERSISTENT ARTIFACT CHECK

✓ data/raw                  2 item(s)
    └── ami
    └── processed

✓ data/vad                  1 item(s)
    └── ES2004a_silero_vad.json

✓ data/transcripts          17 item(s)
    └── ES2004a_whisper_small.json
    └── ES2004b_whisper_small.json
    └── ES2004c_whisper_small.json
    └── ES2004d_whisper_small.json
    └── ES2005a_whisper_small.json
    └── ES2005b_whisper_small.json
    └── ES2005c_whisper_small.json
    └── ES2006a_whisper_small.json
    └── ES2006b_whisper_small.json
    └── ES2008a_whisper_small.json
    └── ... 7 more

✗ data/embeddings           MISSING

✗ data/faiss                MISSING

✗ data/evidence             MISSING

✓ outputs/mom               10 item(s)
    └── ES2004a_bart_baseline.json
    └── ES2004b_bart_baseline.json
    └── ES2004c_bart_baseline.json
    └── ES2004d_bart_baseline.json
    └── ES2005a_bart_baseline.json
    └── ES2005b_bart_baseline.json
    └── ES2005c_bart_baseline.json
    └

In [3]:
# ============================================================
# STEP 2 - CREATE RESTART-SAFE ARTIFACT DIRECTORIES
# ============================================================

from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive/MyDrive/MTechIndProj/MoM_Project"
)

persistent_dirs = [
    PROJECT_DIR / "data" / "embeddings",
    PROJECT_DIR / "data" / "faiss",
    PROJECT_DIR / "data" / "evidence",
    PROJECT_DIR / "outputs" / "verified",
]

print("=" * 90)
print("CREATING PERSISTENT ARTIFACT DIRECTORIES")
print("=" * 90)

for path in persistent_dirs:
    path.mkdir(parents=True, exist_ok=True)
    print(f"✓ {path}")

print("\nAll persistent directories are ready.")

CREATING PERSISTENT ARTIFACT DIRECTORIES
✓ /content/drive/MyDrive/MTechIndProj/MoM_Project/data/embeddings
✓ /content/drive/MyDrive/MTechIndProj/MoM_Project/data/faiss
✓ /content/drive/MyDrive/MTechIndProj/MoM_Project/data/evidence
✓ /content/drive/MyDrive/MTechIndProj/MoM_Project/outputs/verified

All persistent directories are ready.


In [4]:
# ============================================================
# STEP 3 - SAVE M7 EVIDENCE DOCUMENTS PERMANENTLY
# ============================================================

import json
from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive/MyDrive/MTechIndProj/MoM_Project"
)

EVIDENCE_DIR = PROJECT_DIR / "data" / "evidence"
EVIDENCE_DIR.mkdir(parents=True, exist_ok=True)

EVIDENCE_FILE = (
    EVIDENCE_DIR / "ES2004a_evidence_documents.json"
)

print("=" * 90)
print("SAVING M7 EVIDENCE DOCUMENTS")
print("=" * 90)

# ------------------------------------------------------------
# Validate evidence documents
# ------------------------------------------------------------

print("Evidence documents in memory:", len(evidence_documents))

if len(evidence_documents) == 0:
    raise ValueError("evidence_documents is empty.")

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

with open(EVIDENCE_FILE, "w", encoding="utf-8") as f:
    json.dump(
        evidence_documents,
        f,
        indent=2,
        ensure_ascii=False
    )

print("\n✓ Evidence documents saved")
print("File:", EVIDENCE_FILE)
print("Documents:", len(evidence_documents))

# ------------------------------------------------------------
# Verify file
# ------------------------------------------------------------

print("\nFile exists:", EVIDENCE_FILE.exists())
print("File size:", round(EVIDENCE_FILE.stat().st_size / 1024, 2), "KB")

print("\n" + "=" * 90)
print("STEP 3 COMPLETE")
print("=" * 90)

SAVING M7 EVIDENCE DOCUMENTS


NameError: name 'evidence_documents' is not defined

In [5]:
# ============================================================
# STEP 3A - REBUILD M7 EVIDENCE DOCUMENTS FROM SAVED DATA
# ============================================================

import json
from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive/MyDrive/MTechIndProj/MoM_Project"
)

TRANSCRIPT_FILE = (
    PROJECT_DIR
    / "data"
    / "transcripts"
    / "ES2004a_speaker_transcript.json"
)

EVIDENCE_DIR = PROJECT_DIR / "data" / "evidence"
EVIDENCE_DIR.mkdir(parents=True, exist_ok=True)

EVIDENCE_FILE = (
    EVIDENCE_DIR / "ES2004a_evidence_documents.json"
)

print("=" * 90)
print("STEP 3A - RESTORING M7 EVIDENCE DOCUMENTS")
print("=" * 90)

# ------------------------------------------------------------
# Check saved source
# ------------------------------------------------------------

if not TRANSCRIPT_FILE.exists():
    raise FileNotFoundError(
        f"Speaker transcript not found:\n{TRANSCRIPT_FILE}"
    )

print("\n✓ Speaker transcript found:")
print(TRANSCRIPT_FILE)

# ------------------------------------------------------------
# Load speaker transcript
# ------------------------------------------------------------

with open(TRANSCRIPT_FILE, "r", encoding="utf-8") as f:
    speaker_transcript_data = json.load(f)

utterances = speaker_transcript_data["utterances"]

print("\nSpeaker utterances loaded:", len(utterances))

# ------------------------------------------------------------
# Reconstruct M7 evidence documents
# ------------------------------------------------------------

evidence_documents = []

for idx, utterance in enumerate(utterances, start=1):

    evidence_documents.append({
        "evidence_id": idx,
        "speaker": utterance["speaker"],
        "start": utterance["start"],
        "end": utterance["end"],
        "duration": utterance.get(
            "duration",
            utterance["end"] - utterance["start"]
        ),
        "text": utterance["text"]
    })

print("Evidence documents reconstructed:", len(evidence_documents))

# ------------------------------------------------------------
# Save immediately to Google Drive
# ------------------------------------------------------------

with open(EVIDENCE_FILE, "w", encoding="utf-8") as f:
    json.dump(
        evidence_documents,
        f,
        indent=2,
        ensure_ascii=False
    )

print("\n✓ Evidence documents permanently saved")
print("File:", EVIDENCE_FILE)
print("File size:",
      round(EVIDENCE_FILE.stat().st_size / 1024, 2),
      "KB")

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("VALIDATION")
print("=" * 90)

print("Expected documents : 148")
print("Actual documents   :", len(evidence_documents))
print("Count correct      :", len(evidence_documents) == 148)

print("\nFirst evidence:")
print(evidence_documents[0])

print("\nLast evidence:")
print(evidence_documents[-1])

print("\n" + "=" * 90)
print("STEP 3A COMPLETE")
print("=" * 90)

STEP 3A - RESTORING M7 EVIDENCE DOCUMENTS

✓ Speaker transcript found:
/content/drive/MyDrive/MTechIndProj/MoM_Project/data/transcripts/ES2004a_speaker_transcript.json

Speaker utterances loaded: 148
Evidence documents reconstructed: 148

✓ Evidence documents permanently saved
File: /content/drive/MyDrive/MTechIndProj/MoM_Project/data/evidence/ES2004a_evidence_documents.json
File size: 32.94 KB

VALIDATION
Expected documents : 148
Actual documents   : 148
Count correct      : True

First evidence:
{'evidence_id': 1, 'speaker': 'SPEAKER_02', 'start': 10.998, 'end': 14.521, 'duration': 3.5230000000000015, 'text': "Are we, we're not like the dim lights, so we can see that a bit better."}

Last evidence:
{'evidence_id': 148, 'speaker': 'SPEAKER_03', 'start': 1046.428, 'end': 1048.43, 'duration': 2.0019999999999527, 'text': 'Do we go back to our room? Yeah.'}

STEP 3A COMPLETE


In [6]:
# ============================================================
# STEP 4 - CHECK PERSISTENT BGE MODEL CACHE
# ============================================================

from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive/MyDrive/MTechIndProj/MoM_Project"
)

MODEL_DIR = PROJECT_DIR / "models" / "bge"

print("=" * 90)
print("STEP 4 - CHECKING BGE MODEL CACHE")
print("=" * 90)

if MODEL_DIR.exists():
    files = list(MODEL_DIR.rglob("*"))

    print("\n✓ BGE model directory exists:")
    print(MODEL_DIR)

    print("\nItems found:", len(files))

    for f in files[:20]:
        if f.is_file():
            print("   └──", f.relative_to(MODEL_DIR))

    if len(files) > 20:
        print("   └── ...")
else:
    print("\n✗ Persistent BGE model directory does not exist yet.")

print("\n" + "=" * 90)
print("CHECK COMPLETE")
print("=" * 90)

STEP 4 - CHECKING BGE MODEL CACHE

✗ Persistent BGE model directory does not exist yet.

CHECK COMPLETE


In [7]:
# ============================================================
# STEP 5 - SAVE BGE MODEL TO GOOGLE DRIVE
# ============================================================

from pathlib import Path
from sentence_transformers import SentenceTransformer

PROJECT_DIR = Path(
    "/content/drive/MyDrive/MTechIndProj/MoM_Project"
)

BGE_MODEL_DIR = PROJECT_DIR / "models" / "bge"
BGE_MODEL_DIR.mkdir(parents=True, exist_ok=True)

BGE_MODEL_NAME = "BAAI/bge-small-en-v1.5"

print("=" * 90)
print("STEP 5 - DOWNLOADING BGE MODEL TO GOOGLE DRIVE")
print("=" * 90)

print("\nModel:", BGE_MODEL_NAME)
print("Persistent location:", BGE_MODEL_DIR)

bge_model = SentenceTransformer(
    BGE_MODEL_NAME
)

print("\n✓ BGE model downloaded/loaded.")

# Save model to Drive
bge_model.save(str(BGE_MODEL_DIR))

print("✓ BGE model saved permanently to Google Drive.")

# Verify
saved_files = list(BGE_MODEL_DIR.rglob("*"))

print("\nSaved model files:", len(saved_files))

for f in saved_files[:15]:
    if f.is_file():
        print("   └──", f.relative_to(BGE_MODEL_DIR))

print("\n" + "=" * 90)
print("STEP 5 COMPLETE")
print("=" * 90)

STEP 5 - DOWNLOADING BGE MODEL TO GOOGLE DRIVE

Model: BAAI/bge-small-en-v1.5
Persistent location: /content/drive/MyDrive/MTechIndProj/MoM_Project/models/bge


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


✓ BGE model downloaded/loaded.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✓ BGE model saved permanently to Google Drive.

Saved model files: 11
   └── config_sentence_transformers.json
   └── config.json
   └── model.safetensors
   └── tokenizer_config.json
   └── tokenizer.json
   └── sentence_bert_config.json
   └── modules.json
   └── README.md
   └── 1_Pooling/config.json

STEP 5 COMPLETE


In [8]:
# ============================================================
# STEP 6 - GENERATE & SAVE BGE EVIDENCE EMBEDDINGS
# ============================================================

import json
import numpy as np
from pathlib import Path
from sentence_transformers import SentenceTransformer

PROJECT_DIR = Path(
    "/content/drive/MyDrive/MTechIndProj/MoM_Project"
)

EVIDENCE_FILE = (
    PROJECT_DIR
    / "data"
    / "evidence"
    / "ES2004a_evidence_documents.json"
)

EMBEDDING_DIR = (
    PROJECT_DIR / "data" / "embeddings"
)

EMBEDDING_DIR.mkdir(parents=True, exist_ok=True)

EMBEDDING_FILE = (
    EMBEDDING_DIR
    / "ES2004a_bge_embeddings.npy"
)

BGE_MODEL_DIR = (
    PROJECT_DIR / "models" / "bge"
)

print("=" * 90)
print("STEP 6 - GENERATING PERSISTENT BGE EMBEDDINGS")
print("=" * 90)

# ------------------------------------------------------------
# 1. Load saved evidence documents
# ------------------------------------------------------------

with open(EVIDENCE_FILE, "r", encoding="utf-8") as f:
    evidence_documents = json.load(f)

print("\nEvidence documents loaded:", len(evidence_documents))

if len(evidence_documents) != 148:
    raise ValueError(
        f"Expected 148 evidence documents, "
        f"found {len(evidence_documents)}"
    )

# ------------------------------------------------------------
# 2. Load BGE from persistent Drive location
# ------------------------------------------------------------

print("\nLoading BGE from persistent Drive model...")

bge_model = SentenceTransformer(
    str(BGE_MODEL_DIR)
)

print("✓ BGE model loaded.")

# ------------------------------------------------------------
# 3. Generate embeddings
# ------------------------------------------------------------

evidence_texts = [
    doc["text"]
    for doc in evidence_documents
]

print("\nGenerating embeddings for 148 evidence documents...")

evidence_embeddings = bge_model.encode(
    evidence_texts,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=True
).astype("float32")

print("\nEmbedding shape:", evidence_embeddings.shape)

# ------------------------------------------------------------
# 4. Save embeddings permanently
# ------------------------------------------------------------

np.save(
    EMBEDDING_FILE,
    evidence_embeddings
)

print("\n✓ Embeddings saved permanently.")
print("File:", EMBEDDING_FILE)

print(
    "File size:",
    round(EMBEDDING_FILE.stat().st_size / (1024 * 1024), 2),
    "MB"
)

# ------------------------------------------------------------
# 5. Verify
# ------------------------------------------------------------

loaded_embeddings = np.load(EMBEDDING_FILE)

print("\n" + "=" * 90)
print("VALIDATION")
print("=" * 90)

print("Saved shape   :", loaded_embeddings.shape)
print("Expected rows :", len(evidence_documents))
print(
    "Rows correct  :",
    loaded_embeddings.shape[0] == len(evidence_documents)
)

print(
    "Dimension     :",
    loaded_embeddings.shape[1]
)

print(
    "Finite values :",
    np.isfinite(loaded_embeddings).all()
)

print("\n" + "=" * 90)
print("STEP 6 COMPLETE")
print("=" * 90)

STEP 6 - GENERATING PERSISTENT BGE EMBEDDINGS

Evidence documents loaded: 148

Loading BGE from persistent Drive model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

✓ BGE model loaded.

Generating embeddings for 148 evidence documents...


Batches:   0%|          | 0/5 [00:00<?, ?it/s]


Embedding shape: (148, 384)

✓ Embeddings saved permanently.
File: /content/drive/MyDrive/MTechIndProj/MoM_Project/data/embeddings/ES2004a_bge_embeddings.npy
File size: 0.22 MB

VALIDATION
Saved shape   : (148, 384)
Expected rows : 148
Rows correct  : True
Dimension     : 384
Finite values : True

STEP 6 COMPLETE


In [11]:
# ============================================================
# STEP 7 - BUILD & SAVE FAISS INDEX PERMANENTLY
# ============================================================

import numpy as np
import faiss
from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive/MyDrive/MTechIndProj/MoM_Project"
)

EMBEDDING_FILE = (
    PROJECT_DIR
    / "data"
    / "embeddings"
    / "ES2004a_bge_embeddings.npy"
)

FAISS_DIR = PROJECT_DIR / "data" / "faiss"
FAISS_DIR.mkdir(parents=True, exist_ok=True)

FAISS_FILE = (
    FAISS_DIR / "ES2004a_faiss.index"
)

print("=" * 90)
print("STEP 7 - BUILDING PERSISTENT FAISS INDEX")
print("=" * 90)

# ------------------------------------------------------------
# 1. Load saved embeddings
# ------------------------------------------------------------

evidence_embeddings = np.load(EMBEDDING_FILE)

print("\nEmbeddings loaded:")
print("Shape:", evidence_embeddings.shape)
print("Dtype:", evidence_embeddings.dtype)

# ------------------------------------------------------------
# 2. Build cosine-similarity FAISS index
# ------------------------------------------------------------

dimension = evidence_embeddings.shape[1]

faiss_index = faiss.IndexFlatIP(dimension)

faiss_index.add(evidence_embeddings)

print("\n✓ FAISS index built")
print("Dimension:", dimension)
print("Vectors :", faiss_index.ntotal)

# ------------------------------------------------------------
# 3. Save index permanently
# ------------------------------------------------------------

faiss.write_index(
    faiss_index,
    str(FAISS_FILE)
)

print("\n✓ FAISS index saved permanently.")
print("File:", FAISS_FILE)

print(
    "File size:",
    round(FAISS_FILE.stat().st_size / 1024, 2),
    "KB"
)

# ------------------------------------------------------------
# 4. Reload and validate
# ------------------------------------------------------------

reloaded_index = faiss.read_index(
    str(FAISS_FILE)
)

print("\n" + "=" * 90)
print("VALIDATION")
print("=" * 90)

print("Expected vectors :", evidence_embeddings.shape[0])
print("FAISS vectors    :", reloaded_index.ntotal)
print("Dimension        :", reloaded_index.d)

print(
    "Vector count correct:",
    reloaded_index.ntotal == evidence_embeddings.shape[0]
)

print(
    "Dimension correct   :",
    reloaded_index.d == evidence_embeddings.shape[1]
)

print("\n" + "=" * 90)
print("STEP 7 COMPLETE")
print("=" * 90)

STEP 7 - BUILDING PERSISTENT FAISS INDEX

Embeddings loaded:
Shape: (148, 384)
Dtype: float32

✓ FAISS index built
Dimension: 384
Vectors : 148

✓ FAISS index saved permanently.
File: /content/drive/MyDrive/MTechIndProj/MoM_Project/data/faiss/ES2004a_faiss.index
File size: 222.04 KB

VALIDATION
Expected vectors : 148
FAISS vectors    : 148
Dimension        : 384
Vector count correct: True
Dimension correct   : True

STEP 7 COMPLETE


In [10]:
!pip install -q faiss-cpu

print("FAISS installation completed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 79.6 MB/s eta 0:00:00
FAISS installation completed.


In [12]:
# ============================================================
# STEP 8 - RESTART-SAFE M7 LOADER
# ============================================================

import json
import numpy as np
import faiss
from pathlib import Path
from sentence_transformers import SentenceTransformer

print("=" * 90)
print("RESTART-SAFE M7 LOADER")
print("=" * 90)

# ------------------------------------------------------------
# Project paths
# ------------------------------------------------------------

PROJECT_DIR = Path(
    "/content/drive/MyDrive/MTechIndProj/MoM_Project"
)

BGE_MODEL_DIR = PROJECT_DIR / "models" / "bge"

EVIDENCE_FILE = (
    PROJECT_DIR
    / "data"
    / "evidence"
    / "ES2004a_evidence_documents.json"
)

EMBEDDING_FILE = (
    PROJECT_DIR
    / "data"
    / "embeddings"
    / "ES2004a_bge_embeddings.npy"
)

FAISS_FILE = (
    PROJECT_DIR
    / "data"
    / "faiss"
    / "ES2004a_faiss.index"
)

# ------------------------------------------------------------
# 1. Load evidence documents
# ------------------------------------------------------------

with open(EVIDENCE_FILE, "r", encoding="utf-8") as f:
    evidence_documents = json.load(f)

print(f"✓ Evidence documents loaded: {len(evidence_documents)}")

# ------------------------------------------------------------
# 2. Load BGE model
# ------------------------------------------------------------

bge_model = SentenceTransformer(
    str(BGE_MODEL_DIR)
)

print("✓ BGE model loaded from Drive")

# ------------------------------------------------------------
# 3. Load embeddings
# ------------------------------------------------------------

evidence_embeddings = np.load(
    EMBEDDING_FILE
)

print(
    f"✓ BGE embeddings loaded: "
    f"{evidence_embeddings.shape}"
)

# ------------------------------------------------------------
# 4. Load FAISS index
# ------------------------------------------------------------

faiss_index = faiss.read_index(
    str(FAISS_FILE)
)

print(
    f"✓ FAISS index loaded: "
    f"{faiss_index.ntotal} vectors"
)

# ------------------------------------------------------------
# 5. Validation
# ------------------------------------------------------------

assert len(evidence_documents) == evidence_embeddings.shape[0]
assert faiss_index.ntotal == len(evidence_documents)
assert faiss_index.d == evidence_embeddings.shape[1]

print("\n" + "=" * 90)
print("M7 RETRIEVAL LAYER READY")
print("=" * 90)

print("Evidence documents :", len(evidence_documents))
print("Embedding shape    :", evidence_embeddings.shape)
print("FAISS vectors      :", faiss_index.ntotal)
print("FAISS dimension    :", faiss_index.d)

print("\n✓ Everything loaded from persistent storage.")
print("✓ No embedding generation required.")
print("✓ No FAISS rebuilding required.")

print("=" * 90)

RESTART-SAFE M7 LOADER
✓ Evidence documents loaded: 148


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

✓ BGE model loaded from Drive
✓ BGE embeddings loaded: (148, 384)
✓ FAISS index loaded: 148 vectors

M7 RETRIEVAL LAYER READY
Evidence documents : 148
Embedding shape    : (148, 384)
FAISS vectors      : 148
FAISS dimension    : 384

✓ Everything loaded from persistent storage.
✓ No embedding generation required.
✓ No FAISS rebuilding required.


In [13]:
# ============================================================
# STEP 9 - LOAD STRUCTURED MOM CLAIMS
# ============================================================

import json
from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive/MyDrive/MTechIndProj/MoM_Project"
)

CLAIMS_FILE = (
    PROJECT_DIR
    / "data"
    / "transcripts"
    / "ES2004a_structured_mom_claims.json"
)

print("=" * 90)
print("STEP 9 - LOADING STRUCTURED MOM CLAIMS")
print("=" * 90)

if not CLAIMS_FILE.exists():
    raise FileNotFoundError(
        f"Structured claims file not found:\n{CLAIMS_FILE}"
    )

with open(CLAIMS_FILE, "r", encoding="utf-8") as f:
    structured_claims = json.load(f)

claims = structured_claims["claims"]

print("\n✓ Structured claims loaded")
print("Meeting ID     :", structured_claims["meeting_id"])
print("Candidate count:", structured_claims["candidate_count"])
print("Topic groups   :", structured_claims["topic_group_count"])
print("Claim count    :", structured_claims["claim_count"])
print("Loaded claims  :", len(claims))

# Quick validation
assert len(claims) == structured_claims["claim_count"]

print("\nFirst claim:")
print(claims[0])

print("\n" + "=" * 90)
print("STEP 9 COMPLETE")
print("=" * 90)

STEP 9 - LOADING STRUCTURED MOM CLAIMS

✓ Structured claims loaded
Meeting ID     : ES2004a
Candidate count: 42
Topic groups   : 11
Claim count    : 12
Loaded claims  : 12

First claim:
{'claim_id': 1, 'topic': 'meeting_introduction_and_agenda', 'claim_text': 'The meeting was introduced as the first meeting, with an agenda covering team interaction, tool training, the project plan, and discussion of ideas.', 'source_candidate_ids': [5, 6, 7, 8], 'speaker': 'SPEAKER_02', 'start': 84.658, 'end': 109.823, 'event_type': 'INFORMATION'}

STEP 9 COMPLETE


In [14]:
# ============================================================
# STEP 10 - ADAPT SAVED CLAIMS FOR M7 VERIFICATION
# ============================================================

def adapt_structured_claim(claim):
    """
    Convert the saved structured MoM claim format
    into the format expected by the M7 verifier.

    MULTIPLE speaker claims are preserved explicitly.
    """

    return {
        "claim_id": claim["claim_id"],
        "topic": claim["topic"],
        "claim": claim["claim_text"],
        "speaker": claim["speaker"],
        "type": claim["event_type"],
        "expected_start": claim["start"],
        "expected_end": claim["end"],
        "source_candidate_ids": claim["source_candidate_ids"]
    }


# ------------------------------------------------------------
# Adapt all 12 claims
# ------------------------------------------------------------

adapted_claims = [
    adapt_structured_claim(claim)
    for claim in claims
]

print("=" * 90)
print("STEP 10 - CLAIM ADAPTATION")
print("=" * 90)

print("Original claims :", len(claims))
print("Adapted claims  :", len(adapted_claims))

print("\nExample - Claim 1:")
print(adapted_claims[0])

print("\nSpeaker distribution:")

from collections import Counter

speaker_counts = Counter(
    claim["speaker"]
    for claim in adapted_claims
)

for speaker, count in speaker_counts.items():
    print(f"  {speaker:<15}: {count}")

print("\nEvent-type distribution:")

event_counts = Counter(
    claim["type"]
    for claim in adapted_claims
)

for event_type, count in event_counts.items():
    print(f"  {event_type:<15}: {count}")

print("\n" + "=" * 90)
print("STEP 10 COMPLETE")
print("=" * 90)

STEP 10 - CLAIM ADAPTATION
Original claims : 12
Adapted claims  : 12

Example - Claim 1:
{'claim_id': 1, 'topic': 'meeting_introduction_and_agenda', 'claim': 'The meeting was introduced as the first meeting, with an agenda covering team interaction, tool training, the project plan, and discussion of ideas.', 'speaker': 'SPEAKER_02', 'type': 'INFORMATION', 'expected_start': 84.658, 'expected_end': 109.823, 'source_candidate_ids': [5, 6, 7, 8]}

Speaker distribution:
  SPEAKER_02     : 4
  MULTIPLE       : 6
  SPEAKER_03     : 2

Event-type distribution:
  INFORMATION    : 3
  DISCUSSION     : 8
  ACTION         : 1

STEP 10 COMPLETE


In [15]:
# ============================================================
# STEP 11 - RESTORE NLI MODEL FOR M7 VERIFICATION
# ============================================================

from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

print("=" * 90)
print("STEP 11 - LOADING NLI MODEL")
print("=" * 90)

NLI_MODEL_NAME = "cross-encoder/nli-deberta-v3-small"

NLI_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("\nModel :", NLI_MODEL_NAME)
print("Device:", NLI_DEVICE)

nli_tokenizer = AutoTokenizer.from_pretrained(
    NLI_MODEL_NAME
)

nli_model = AutoModelForSequenceClassification.from_pretrained(
    NLI_MODEL_NAME
)

nli_model = nli_model.to(NLI_DEVICE)
nli_model.eval()

NLI_LABELS = {
    0: "contradiction",
    1: "entailment",
    2: "neutral"
}

print("\n✓ NLI tokenizer loaded")
print("✓ NLI model loaded")
print("✓ Model set to evaluation mode")

print("\nLabel mapping:")
for idx, label in NLI_LABELS.items():
    print(f"  {idx}: {label}")

print("\n" + "=" * 90)
print("STEP 11 COMPLETE")
print("=" * 90)

STEP 11 - LOADING NLI MODEL

Model : cross-encoder/nli-deberta-v3-small
Device: cuda


config.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.35k [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

...

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  568MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]


✓ NLI tokenizer loaded
✓ NLI model loaded
✓ Model set to evaluation mode

Label mapping:
  0: contradiction
  1: entailment
  2: neutral

STEP 11 COMPLETE


In [16]:
# ============================================================
# STEP 12 - LOCATE FINAL M7 VERIFICATION FUNCTIONS
# ============================================================

import json
from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive/MyDrive/MTechIndProj/MoM_Project"
)

NOTEBOOK_FILE = (
    PROJECT_DIR
    / "04_mom_generation"
    / "04_mom_generation.ipynb"
)

print("=" * 90)
print("STEP 12 - LOCATING FINAL M7 VERIFICATION FUNCTIONS")
print("=" * 90)

with open(NOTEBOOK_FILE, "r", encoding="utf-8") as f:
    notebook = json.load(f)

keywords = [
    "def run_nli",
    "def get_context_window",
    "def lexical",
    "def retrieve_evidence",
    "def analyze_attribute_evidence",
    "def select_supporting_candidates",
    "def create_evidence_group",
    "def check_speaker_consistency",
    "def check_timestamp_consistency",
    "def check_event_type_consistency",
    "def verify_attribute",
    "def verify_mom_claim"
]

for keyword in keywords:
    matches = []

    for cell_idx, cell in enumerate(notebook["cells"]):
        source = "".join(cell.get("source", []))

        if keyword in source:
            matches.append(cell_idx)

    print(f"\n{keyword}")
    print("  Found in cells:", matches)

print("\n" + "=" * 90)
print("STEP 12 COMPLETE")
print("=" * 90)

STEP 12 - LOCATING FINAL M7 VERIFICATION FUNCTIONS

def run_nli
  Found in cells: [48, 55, 63, 104, 140]

def get_context_window
  Found in cells: [77, 105, 140]

def lexical
  Found in cells: [75, 106]

def retrieve_evidence
  Found in cells: [33, 35, 67, 69, 71, 100, 102, 153]

def analyze_attribute_evidence
  Found in cells: [85, 107, 140]

def select_supporting_candidates
  Found in cells: [86, 108, 140]

def create_evidence_group
  Found in cells: [91, 109, 140]

def check_speaker_consistency
  Found in cells: [83, 111, 140]

def check_timestamp_consistency
  Found in cells: [83, 112, 140]

def check_event_type_consistency
  Found in cells: [113, 116, 140]

def verify_attribute
  Found in cells: [92, 94, 110, 140]

def verify_mom_claim
  Found in cells: [115, 140]

STEP 12 COMPLETE


In [18]:
# ============================================================
# STEP 13 - RESTORE FINAL CONSOLIDATED M7 VERIFIER
# ============================================================

import json
import os
from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive/MyDrive/MTechIndProj/MoM_Project"
)

NOTEBOOK_FILE = (
    PROJECT_DIR
    / "04_mom_generation"
    / "04_mom_generation.ipynb"
)

print("=" * 90)
print("STEP 13 - RESTORING FINAL M7 VERIFIER")
print("=" * 90)

# ------------------------------------------------------------
# Load the FINAL consolidated M7 cell (cell 140)
# ------------------------------------------------------------

with open(NOTEBOOK_FILE, "r", encoding="utf-8") as f:
    notebook = json.load(f)

cell_source = "".join(
    notebook["cells"][140]["source"]
)

print("\n✓ Final M7 cell located: cell 140")
print("Source length:", len(cell_source), "characters")

# ------------------------------------------------------------
# Execute the saved final implementation
# ------------------------------------------------------------

exec(cell_source)

print("\n" + "=" * 90)
print("STEP 13 COMPLETE")
print("=" * 90)
print("✓ Final M7 verification functions restored")

STEP 13 - RESTORING FINAL M7 VERIFIER

✓ Final M7 cell located: cell 140
Source length: 1482 characters
M7 - LOCATING SAVED VERIFICATION FUNCTIONS

Notebook exists: True
Notebook loaded.
Number of cells: 175

FUNCTION LOCATIONS

def run_nli:
  Cell(s): [48, 55, 63, 104, 140, 172]

def get_context_window:
  Cell(s): [77, 105, 140, 172]

def analyze_attribute_evidence:
  Cell(s): [85, 107, 140, 172]

def select_supporting_candidates:
  Cell(s): [86, 108, 140, 172]

def create_evidence_group:
  Cell(s): [91, 109, 140, 172]

def check_speaker_consistency:
  Cell(s): [83, 111, 140, 172]

def evaluate_speaker_consistency_for_evidence:
  Cell(s): [114, 140]

def check_timestamp_consistency:
  Cell(s): [83, 112, 140, 172]

def check_event_type_consistency:
  Cell(s): [113, 116, 140, 172]

def verify_attribute:
  Cell(s): [92, 94, 110, 140, 172]

def verify_mom_claim:
  Cell(s): [115, 140, 172]

STEP 13 COMPLETE
✓ Final M7 verification functions restored


In [19]:
# ============================================================
# STEP 14 - END-TO-END TEST ON CLAIM 1
# ============================================================

print("=" * 90)
print("STEP 14 - TESTING CLAIM 1")
print("=" * 90)

claim = adapted_claims[0]

print("\nClaim ID :", claim["claim_id"])
print("Topic    :", claim["topic"])
print("Speaker  :", claim["speaker"])
print("Type     :", claim["type"])
print("Claim    :", claim["claim"])

# ------------------------------------------------------------
# IMPORTANT:
# source_candidate_ids are candidate IDs, NOT evidence IDs.
# For Claim 1:
# candidate IDs [5,6,7,8]
# map to speaker-transcript evidence IDs [9,10,11,12]
#
# We therefore use the actual M7 evidence IDs.
# ------------------------------------------------------------

supporting_evidence = [
    doc
    for doc in evidence_documents
    if doc["evidence_id"] in [9, 10, 11, 12]
]

print("\nSupporting evidence IDs:",
      [doc["evidence_id"] for doc in supporting_evidence])

print("\nEvidence interval:")
print(
    min(doc["start"] for doc in supporting_evidence),
    "→",
    max(doc["end"] for doc in supporting_evidence)
)

# ------------------------------------------------------------
# Run final M7 verifier
# ------------------------------------------------------------

result = verify_mom_claim(
    claim=claim,
    supporting_evidence=supporting_evidence,
    content_proposition=claim["claim"],
    expected_start=claim["expected_start"],
    expected_end=claim["expected_end"]
)

print("\n" + "=" * 90)
print("CLAIM 1 VERIFICATION RESULT")
print("=" * 90)

print("Status :", result["status"])
print("Reason :", result["reason"])

print("\nFailed checks:")
print(result["failed_checks"])

print("\nEvidence IDs:")
print(result["evidence_ids"])

print("\nEvidence speakers:")
print(result["evidence_speakers"])

print("\nChecks:")

for check_name, check_result in result["checks"].items():
    print(f"\n{check_name}:")
    print(check_result)

print("\n" + "=" * 90)
print("STEP 14 COMPLETE")
print("=" * 90)

STEP 14 - TESTING CLAIM 1

Claim ID : 1
Topic    : meeting_introduction_and_agenda
Speaker  : SPEAKER_02
Type     : INFORMATION
Claim    : The meeting was introduced as the first meeting, with an agenda covering team interaction, tool training, the project plan, and discussion of ideas.

Supporting evidence IDs: [9, 10, 11, 12]

Evidence interval:
84.658 → 109.823


NameError: name 'verify_mom_claim' is not defined

In [20]:
# ============================================================
# STEP 15 - RESTORE verify_mom_claim()
# ============================================================

import json
from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive/MyDrive/MTechIndProj/MoM_Project"
)

NOTEBOOK_FILE = (
    PROJECT_DIR
    / "04_mom_generation"
    / "04_mom_generation.ipynb"
)

print("=" * 90)
print("STEP 15 - RESTORING verify_mom_claim()")
print("=" * 90)

# ------------------------------------------------------------
# Load notebook
# ------------------------------------------------------------

with open(NOTEBOOK_FILE, "r", encoding="utf-8") as f:
    notebook = json.load(f)

# ------------------------------------------------------------
# Cell 115 contains the actual final verifier
# ------------------------------------------------------------

cell_source = "".join(
    notebook["cells"][115]["source"]
)

print("\n✓ Loaded saved implementation from cell 115")
print("Source length:", len(cell_source), "characters")

# ------------------------------------------------------------
# Restore function
# ------------------------------------------------------------

exec(cell_source)

# ------------------------------------------------------------
# Verify that the function now exists
# ------------------------------------------------------------

print("\nverify_mom_claim available:",
      "verify_mom_claim" in globals())

if "verify_mom_claim" not in globals():
    raise RuntimeError(
        "verify_mom_claim was not restored."
    )

print("\n" + "=" * 90)
print("STEP 15 COMPLETE")
print("=" * 90)

STEP 15 - RESTORING verify_mom_claim()

✓ Loaded saved implementation from cell 115
Source length: 6581 characters
FINAL M7 MULTI-ATTRIBUTE VERIFICATION FUNCTION READY

Content verification:
  Uses existing verify_attribute() pipeline

Additional checks:
  Speaker consistency
  Timestamp consistency
  Event / decision consistency

Overall output:
  VERIFIED / FLAGGED

verify_mom_claim available: True

STEP 15 COMPLETE


In [21]:
# ============================================================
# STEP 16 - VERIFY CLAIM 1
# ============================================================

print("=" * 90)
print("STEP 16 - CLAIM 1 END-TO-END VERIFICATION")
print("=" * 90)

claim = adapted_claims[0]

# Actual M7 evidence IDs corresponding to Claim 1
claim1_evidence_ids = [9, 10, 11, 12]

supporting_evidence = [
    doc
    for doc in evidence_documents
    if doc["evidence_id"] in claim1_evidence_ids
]

print("\nClaim ID :", claim["claim_id"])
print("Topic    :", claim["topic"])
print("Speaker  :", claim["speaker"])
print("Type     :", claim["type"])

print("\nClaim:")
print(claim["claim"])

print("\nEvidence IDs:",
      [doc["evidence_id"] for doc in supporting_evidence])

# ------------------------------------------------------------
# Run verifier
# ------------------------------------------------------------

result = verify_mom_claim(
    claim=claim,
    supporting_evidence=supporting_evidence,
    content_proposition=claim["claim"],
    expected_start=claim["expected_start"],
    expected_end=claim["expected_end"]
)

# ------------------------------------------------------------
# Display result
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("CLAIM 1 RESULT")
print("=" * 90)

print("\nSTATUS :", result["status"])
print("REASON :", result["reason"])

print("\nFailed checks:")
print(result["failed_checks"])

print("\nEvidence IDs:")
print(result["evidence_ids"])

print("\nEvidence speakers:")
print(result["evidence_speakers"])

print("\nEvidence interval:")
print(
    result["evidence_start"],
    "→",
    result["evidence_end"]
)

print("\nCHECK RESULTS:")

for name, check in result["checks"].items():
    print(f"\n{name}:")
    print(check)

print("\n" + "=" * 90)
print("STEP 16 COMPLETE")
print("=" * 90)

STEP 16 - CLAIM 1 END-TO-END VERIFICATION

Claim ID : 1
Topic    : meeting_introduction_and_agenda
Speaker  : SPEAKER_02
Type     : INFORMATION

Claim:
The meeting was introduced as the first meeting, with an agenda covering team interaction, tool training, the project plan, and discussion of ideas.

Evidence IDs: [9, 10, 11, 12]


NameError: name 'verify_attribute' is not defined

In [22]:
# ============================================================
# STEP 17 - RESTORE M7 VERIFICATION DEPENDENCY CHAIN
# ============================================================

import json
from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive/MyDrive/MTechIndProj/MoM_Project"
)

NOTEBOOK_FILE = (
    PROJECT_DIR
    / "04_mom_generation"
    / "04_mom_generation.ipynb"
)

print("=" * 90)
print("STEP 17 - RESTORING M7 VERIFICATION DEPENDENCIES")
print("=" * 90)

# ------------------------------------------------------------
# Load notebook
# ------------------------------------------------------------

with open(NOTEBOOK_FILE, "r", encoding="utf-8") as f:
    notebook = json.load(f)

# ------------------------------------------------------------
# Restore actual implementation cells
#
# retrieve_evidence          → cell 102
# analyze_attribute_evidence → cell 107
# verify_attribute           → cell 110
# ------------------------------------------------------------

restore_cells = {
    102: "retrieve_evidence",
    107: "analyze_attribute_evidence",
    110: "verify_attribute",
}

for cell_idx, function_name in restore_cells.items():

    source = "".join(
        notebook["cells"][cell_idx]["source"]
    )

    print(
        f"\nRestoring {function_name}() "
        f"from cell {cell_idx}..."
    )

    exec(source)

    print(
        f"✓ {function_name} available:",
        function_name in globals()
    )

# ------------------------------------------------------------
# Final dependency check
# ------------------------------------------------------------

required_functions = [
    "retrieve_evidence",
    "analyze_attribute_evidence",
    "verify_attribute",
    "verify_mom_claim",
]

print("\n" + "=" * 90)
print("DEPENDENCY CHECK")
print("=" * 90)

for name in required_functions:
    print(
        f"{name:<35}:",
        "AVAILABLE" if name in globals() else "MISSING"
    )

missing = [
    name
    for name in required_functions
    if name not in globals()
]

if missing:
    raise RuntimeError(
        f"Missing M7 functions: {missing}"
    )

print("\n✓ Complete M7 verification dependency chain restored.")

print("=" * 90)
print("STEP 17 COMPLETE")
print("=" * 90)

STEP 17 - RESTORING M7 VERIFICATION DEPENDENCIES

Restoring retrieve_evidence() from cell 102...
M7 - BGE + FAISS RESUME
Device: cuda
GPU: Tesla T4

Evidence documents: 148

Loading BGE model: BAAI/bge-small-en-v1.5


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BGE model loaded successfully.


Batches:   0%|          | 0/5 [00:00<?, ?it/s]


Embedding shape: (148, 384)
FAISS vectors: 148
FAISS dimension: 384

RETRIEVAL TEST
Rank 1 | ID 105 | Similarity 0.7917 | SPEAKER_01
  707.248 → 710.889
  remote controls. You want to integrate everything into one.
Rank 2 | ID 14 | Similarity 0.7810 | SPEAKER_02
  117.699 → 120.921
  Now, we're developing a remote control, which you probably already know.
Rank 3 | ID 107 | Similarity 0.7408 | SPEAKER_03
  713.150 → 729.357
  experience has only been given the remote control with the object I buy, not doing any tampering with it and programming, using it to program TV videos and things, but basically on off volume, up and down, channel one to that basic function. I don't think I could go any further with it.
Rank 4 | ID 145 | Similarity 0.7396 | SPEAKER_02
  1025.075 → 1036.842
  I mean, the sky remote controls and everything. They're kind of molded and look a bit different than the tele-west remote controls. They're silver plastic, which looks a bit smarter. So yeah, I guess that's it

NameError: name 'get_context_window' is not defined

In [23]:
# ============================================================
# FIX - RESTORE get_context_window()
# ============================================================

import json
from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive/MyDrive/MTechIndProj/MoM_Project"
)

NOTEBOOK_FILE = (
    PROJECT_DIR
    / "04_mom_generation"
    / "04_mom_generation.ipynb"
)

with open(NOTEBOOK_FILE, "r", encoding="utf-8") as f:
    notebook = json.load(f)

# Cell 105 contains the saved implementation
source = "".join(
    notebook["cells"][105]["source"]
)

exec(source)

print("=" * 90)
print("CONTEXT WINDOW RESTORED")
print("=" * 90)
print("get_context_window available:",
      "get_context_window" in globals())

M7 - CONTEXT WINDOW TEST
Central evidence ID: 15
Context IDs: [13, 14, 15, 16, 17]

Context:

ID 13 | SPEAKER_02 | 112.185 → 115.748
  And we've got 25 minutes to do that as far as I can understand.

ID 14 | SPEAKER_02 | 117.699 → 120.921
  Now, we're developing a remote control, which you probably already know.

ID 15 | SPEAKER_02 | 122.503 → 132.290
  We want it to be original, something that people haven't thought of. It's not out in the shops. Trendy, appealing to a wide market, but,

ID 16 | SPEAKER_02 | 133.291 → 138.174
  you know, not a hunk of metal. And user-friendly, grannies to kids,

ID 17 | SPEAKER_02 | 139.375 → 141.016
  maybe even pooches, should be able to use it.

Context window function restored successfully.
CONTEXT WINDOW RESTORED
get_context_window available: True


In [24]:
# ============================================================
# STEP 17B - RESTORE REMAINING M7 VERIFICATION FUNCTIONS
# ============================================================

import json
from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive/MyDrive/MTechIndProj/MoM_Project"
)

NOTEBOOK_FILE = (
    PROJECT_DIR
    / "04_mom_generation"
    / "04_mom_generation.ipynb"
)

print("=" * 90)
print("STEP 17B - RESTORING REMAINING M7 FUNCTIONS")
print("=" * 90)

with open(NOTEBOOK_FILE, "r", encoding="utf-8") as f:
    notebook = json.load(f)

# ------------------------------------------------------------
# Restore functions in dependency order
# ------------------------------------------------------------

restore_cells = [
    (106, "lexical_support"),
    (107, "analyze_attribute_evidence"),
    (108, "select_supporting_candidates"),
    (109, "create_evidence_group"),
    (111, "check_speaker_consistency"),
    (112, "check_timestamp_consistency"),
    (113, "check_event_type_consistency"),
    (114, "evaluate_speaker_consistency_for_evidence"),
    (110, "verify_attribute"),
    (115, "verify_mom_claim"),
]

for cell_idx, function_name in restore_cells:

    source = "".join(
        notebook["cells"][cell_idx]["source"]
    )

    print(f"\nRestoring {function_name}() from cell {cell_idx}...")

    exec(source)

    print(
        f"✓ {function_name} available:",
        function_name in globals()
    )

# ------------------------------------------------------------
# Final availability check
# ------------------------------------------------------------

required_functions = [
    "get_context_window",
    "retrieve_evidence",
    "analyze_attribute_evidence",
    "select_supporting_candidates",
    "create_evidence_group",
    "check_speaker_consistency",
    "check_timestamp_consistency",
    "check_event_type_consistency",
    "evaluate_speaker_consistency_for_evidence",
    "verify_attribute",
    "verify_mom_claim",
]

print("\n" + "=" * 90)
print("FINAL M7 FUNCTION CHECK")
print("=" * 90)

missing = []

for name in required_functions:
    available = name in globals()

    print(
        f"{name:<45}:",
        "AVAILABLE" if available else "MISSING"
    )

    if not available:
        missing.append(name)

if missing:
    raise RuntimeError(
        f"Missing M7 functions: {missing}"
    )

print("\n✓ Complete M7 verification chain is available.")
print("=" * 90)
print("STEP 17B COMPLETE")
print("=" * 90)

STEP 17B - RESTORING REMAINING M7 FUNCTIONS

Restoring lexical_support() from cell 106...
M7 - LEXICAL SUPPORT TEST
Proposition: The remote control is intended to be original.

Lexical overlap: 0.7500
Matched terms: ['control', 'original', 'remote']
Missing terms: ['intended']

Lexical support function restored successfully.
✓ lexical_support available: True

Restoring analyze_attribute_evidence() from cell 107...


NameError: name 'run_nli' is not defined

In [25]:
# ============================================================
# STEP 17C - RESTORE run_nli()
# ============================================================

import json
from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive/MyDrive/MTechIndProj/MoM_Project"
)

NOTEBOOK_FILE = (
    PROJECT_DIR
    / "04_mom_generation"
    / "04_mom_generation.ipynb"
)

print("=" * 90)
print("STEP 17C - RESTORING run_nli()")
print("=" * 90)

with open(NOTEBOOK_FILE, "r", encoding="utf-8") as f:
    notebook = json.load(f)

# Cell 104 contains the saved M7 run_nli implementation
source = "".join(
    notebook["cells"][104]["source"]
)

exec(source)

print("\nrun_nli available:",
      "run_nli" in globals())

if "run_nli" not in globals():
    raise RuntimeError("run_nli was not restored.")

print("\n" + "=" * 90)
print("STEP 17C COMPLETE")
print("=" * 90)

STEP 17C - RESTORING run_nli()
M7 - NLI SANITY TEST
Premise    : Now, we're developing a remote control, which you probably already know.
Hypothesis : The team is developing a remote control.

Result:
Label         : entailment
Contradiction : 0.0005
Entailment    : 0.7902
Neutral       : 0.2093

NLI function restored successfully.

run_nli available: True

STEP 17C COMPLETE


In [26]:
# ============================================================
# STEP 17D - RESTORE analyze_attribute_evidence()
# ============================================================

import json
from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive/MyDrive/MTechIndProj/MoM_Project"
)

NOTEBOOK_FILE = (
    PROJECT_DIR
    / "04_mom_generation"
    / "04_mom_generation.ipynb"
)

print("=" * 90)
print("STEP 17D - RESTORING analyze_attribute_evidence()")
print("=" * 90)

with open(NOTEBOOK_FILE, "r", encoding="utf-8") as f:
    notebook = json.load(f)

# Cell 107 contains the saved implementation
source = "".join(
    notebook["cells"][107]["source"]
)

exec(source)

print("\nanalyze_attribute_evidence available:",
      "analyze_attribute_evidence" in globals())

if "analyze_attribute_evidence" not in globals():
    raise RuntimeError(
        "analyze_attribute_evidence was not restored."
    )

print("\n" + "=" * 90)
print("STEP 17D COMPLETE")
print("=" * 90)

STEP 17D - RESTORING analyze_attribute_evidence()
M7 - ATTRIBUTE EVIDENCE ANALYSIS TEST
Candidates analyzed: 10

Top candidates:

ID 105 | BGE=0.7917 | NLI=neutral 0.0013
Speaker: SPEAKER_01 | Speaker check: False
Context IDs: [103, 104, 105, 106, 107]

ID 14 | BGE=0.7810 | NLI=entailment 0.9812
Speaker: SPEAKER_02 | Speaker check: True
Context IDs: [12, 13, 14, 15, 16]

ID 107 | BGE=0.7408 | NLI=neutral 0.0017
Speaker: SPEAKER_03 | Speaker check: False
Context IDs: [105, 106, 107, 108, 109]

ID 145 | BGE=0.7396 | NLI=neutral 0.0004
Speaker: SPEAKER_02 | Speaker check: True
Context IDs: [143, 144, 145, 146, 147]

ID 102 | BGE=0.7126 | NLI=neutral 0.0013
Speaker: SPEAKER_02 | Speaker check: True
Context IDs: [100, 101, 102, 103, 104]

Attribute evidence analysis function restored successfully.

analyze_attribute_evidence available: True

STEP 17D COMPLETE


In [27]:
# ============================================================
# STEP 17E - RESTORE REMAINING M7 VERIFICATION FUNCTIONS
# ============================================================

import json
from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive/MyDrive/MTechIndProj/MoM_Project"
)

NOTEBOOK_FILE = (
    PROJECT_DIR
    / "04_mom_generation"
    / "04_mom_generation.ipynb"
)

print("=" * 90)
print("STEP 17E - RESTORING REMAINING M7 FUNCTIONS")
print("=" * 90)

with open(NOTEBOOK_FILE, "r", encoding="utf-8") as f:
    notebook = json.load(f)

# These cells contain the actual function implementations.
# Dependencies already restored:
#   run_nli()
#   get_context_window()
#   lexical_support()
#   retrieve_evidence()
#   analyze_attribute_evidence()

restore_cells = [
    (108, "select_supporting_candidates"),
    (109, "create_evidence_group"),
    (111, "check_speaker_consistency"),
    (112, "check_timestamp_consistency"),
    (113, "check_event_type_consistency"),
    (114, "evaluate_speaker_consistency_for_evidence"),
    (110, "verify_attribute"),
    (115, "verify_mom_claim"),
]

for cell_idx, function_name in restore_cells:

    source = "".join(
        notebook["cells"][cell_idx]["source"]
    )

    print(f"\nRestoring {function_name}() from cell {cell_idx}...")

    exec(source)

    print(
        f"✓ {function_name} available:",
        function_name in globals()
    )

# ------------------------------------------------------------
# Final check
# ------------------------------------------------------------

required_functions = [
    "run_nli",
    "get_context_window",
    "lexical_support",
    "retrieve_evidence",
    "analyze_attribute_evidence",
    "select_supporting_candidates",
    "create_evidence_group",
    "check_speaker_consistency",
    "check_timestamp_consistency",
    "check_event_type_consistency",
    "evaluate_speaker_consistency_for_evidence",
    "verify_attribute",
    "verify_mom_claim",
]

print("\n" + "=" * 90)
print("FINAL M7 FUNCTION CHECK")
print("=" * 90)

missing = []

for name in required_functions:
    status = name in globals()

    print(
        f"{name:<45}:",
        "AVAILABLE" if status else "MISSING"
    )

    if not status:
        missing.append(name)

if missing:
    raise RuntimeError(
        f"Missing M7 functions: {missing}"
    )

print("\n✓ COMPLETE M7 VERIFICATION CHAIN RESTORED")
print("=" * 90)
print("STEP 17E COMPLETE")
print("=" * 90)

STEP 17E - RESTORING REMAINING M7 FUNCTIONS

Restoring select_supporting_candidates() from cell 108...
M7 - SUPPORTING CANDIDATE SELECTION TEST
Supporting candidates: 0
Rejected candidates: 10

Supporting candidates:

Rejected candidates:
  ID 105 | NLI did not establish entailment
  ID 14 | NLI did not establish entailment
  ID 107 | NLI did not establish entailment
  ID 145 | NLI did not establish entailment
  ID 102 | NLI did not establish entailment
  ID 135 | NLI did not establish entailment
  ID 117 | NLI did not establish entailment
  ID 112 | NLI did not establish entailment
  ID 15 | NLI did not establish entailment
  ID 134 | NLI did not establish entailment

Candidate selection function restored successfully.
✓ select_supporting_candidates available: True

Restoring create_evidence_group() from cell 109...
M7 - EVIDENCE GROUP RESTORATION TEST
Supporting IDs : [14, 15]
Context IDs    : [12, 13, 14, 15, 16, 17]
Display interval: 117.699 → 132.290
Speakers       : ['SPEAKER_02'

TypeError: unsupported format string passed to NoneType.__format__

In [28]:
# ============================================================
# STEP 17F - CHECK M7 FUNCTION AVAILABILITY
# ============================================================

print("=" * 90)
print("STEP 17F - CHECKING M7 FUNCTIONS")
print("=" * 90)

required_functions = [
    "run_nli",
    "get_context_window",
    "lexical_support",
    "retrieve_evidence",
    "analyze_attribute_evidence",
    "select_supporting_candidates",
    "create_evidence_group",
    "check_speaker_consistency",
    "check_timestamp_consistency",
    "check_event_type_consistency",
    "evaluate_speaker_consistency_for_evidence",
    "verify_attribute",
    "verify_mom_claim",
]

for name in required_functions:
    print(
        f"{name:<45}:",
        "AVAILABLE" if callable(globals().get(name))
        else "MISSING"
    )

print("\n" + "=" * 90)
print("STEP 17F COMPLETE")
print("=" * 90)

STEP 17F - CHECKING M7 FUNCTIONS
run_nli                                      : AVAILABLE
get_context_window                           : AVAILABLE
lexical_support                              : AVAILABLE
retrieve_evidence                            : AVAILABLE
analyze_attribute_evidence                   : AVAILABLE
select_supporting_candidates                 : AVAILABLE
create_evidence_group                        : AVAILABLE
check_speaker_consistency                    : AVAILABLE
check_timestamp_consistency                  : AVAILABLE
check_event_type_consistency                 : AVAILABLE
evaluate_speaker_consistency_for_evidence    : AVAILABLE
verify_attribute                             : AVAILABLE
verify_mom_claim                             : AVAILABLE

STEP 17F COMPLETE


In [29]:
# ============================================================
# STEP 18 - FINAL END-TO-END TEST: CLAIM 1
# ============================================================

print("=" * 90)
print("STEP 18 - FINAL CLAIM 1 VERIFICATION")
print("=" * 90)

claim = adapted_claims[0]

# Correct mapping:
# Claim 1 candidate IDs [5,6,7,8]
# correspond to M7 evidence IDs [9,10,11,12]

claim1_evidence_ids = [9, 10, 11, 12]

supporting_evidence = [
    doc
    for doc in evidence_documents
    if doc["evidence_id"] in claim1_evidence_ids
]

print("\nClaim ID :", claim["claim_id"])
print("Topic    :", claim["topic"])
print("Speaker  :", claim["speaker"])
print("Type     :", claim["type"])

print("\nClaim:")
print(claim["claim"])

print("\nEvidence IDs:",
      [doc["evidence_id"] for doc in supporting_evidence])

# ------------------------------------------------------------
# Verify
# ------------------------------------------------------------

result = verify_mom_claim(
    claim=claim,
    supporting_evidence=supporting_evidence,
    content_proposition=claim["claim"],
    expected_start=claim["expected_start"],
    expected_end=claim["expected_end"]
)

# ------------------------------------------------------------
# Display concise result
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("CLAIM 1 VERIFICATION RESULT")
print("=" * 90)

print("STATUS :", result["status"])
print("REASON :", result["reason"])

print("\nFailed checks:")
print(result["failed_checks"])

print("\nEvidence IDs:")
print(result["evidence_ids"])

print("\nEvidence speakers:")
print(result["evidence_speakers"])

print("\nEvidence interval:")
print(
    result["evidence_start"],
    "→",
    result["evidence_end"]
)

print("\nIndividual checks:")

for name, check in result["checks"].items():
    status = check.get("passed", "N/A")
    print(f"  {name:<30}: {status}")

print("\n" + "=" * 90)
print("STEP 18 COMPLETE")
print("=" * 90)

STEP 18 - FINAL CLAIM 1 VERIFICATION

Claim ID : 1
Topic    : meeting_introduction_and_agenda
Speaker  : SPEAKER_02
Type     : INFORMATION

Claim:
The meeting was introduced as the first meeting, with an agenda covering team interaction, tool training, the project plan, and discussion of ideas.

Evidence IDs: [9, 10, 11, 12]

CLAIM 1 VERIFICATION RESULT
STATUS : FLAGGED
REASON : The claim failed one or more consistency checks: content_consistency

Failed checks:
['content_consistency']

Evidence IDs:
[9, 10, 11, 12]

Evidence speakers:
['SPEAKER_02']

Evidence interval:
84.658 → 109.823

Individual checks:
  content_consistency           : False
  speaker_consistency           : True
  timestamp_consistency         : True
  event_type_consistency        : True

STEP 18 COMPLETE


In [30]:
# ============================================================
# STEP 19 - CLAIM 1 ATTRIBUTE-LEVEL CONTENT TEST
# ============================================================

print("=" * 90)
print("STEP 19 - TESTING CLAIM 1 ATOMIC CONTENT ATTRIBUTES")
print("=" * 90)

claim1_attributes = [
    {
        "attribute": "first_meeting",
        "proposition": "This was the first meeting.",
    },
    {
        "attribute": "team_interaction",
        "proposition": "The meeting agenda included team interaction.",
    },
    {
        "attribute": "tool_training",
        "proposition": "The meeting agenda included tool training.",
    },
    {
        "attribute": "project_plan",
        "proposition": "The meeting agenda included the project plan.",
    },
    {
        "attribute": "discussion_of_ideas",
        "proposition": "The meeting agenda included discussion of ideas.",
    },
]

print("\nTesting", len(claim1_attributes), "atomic propositions...\n")

attribute_results = []

for item in claim1_attributes:

    result = verify_attribute(
        attribute=item["attribute"],
        proposition=item["proposition"],
        claim_speaker="SPEAKER_02",
        top_k=10,
        context_window=2
    )

    attribute_results.append({
        "attribute": item["attribute"],
        "proposition": item["proposition"],
        "status": result["status"],
        "supporting_ids": result.get(
            "supporting_evidence_ids", []
        ),
        "context_ids": result.get(
            "context_evidence_ids", []
        ),
        "display_start": result.get(
            "display_start"
        ),
        "display_end": result.get(
            "display_end"
        ),
        "reason": result.get("reason", "")
    })

    print("-" * 90)
    print("Attribute :", item["attribute"])
    print("Proposition:", item["proposition"])
    print("Status    :", result["status"])
    print("Supporting:", result.get(
        "supporting_evidence_ids", []
    ))
    print("Context   :", result.get(
        "context_evidence_ids", []
    ))
    print("Reason    :", result.get("reason", ""))

print("\n" + "=" * 90)
print("ATTRIBUTE SUMMARY")
print("=" * 90)

for result in attribute_results:
    print(
        f"{result['attribute']:<25} "
        f"→ {result['status']:<8} "
        f"Evidence: {result['supporting_ids']}"
    )

verified_count = sum(
    r["status"] == "VERIFIED"
    for r in attribute_results
)

print("\nVerified attributes:",
      verified_count,
      "/",
      len(attribute_results))

print("\n" + "=" * 90)
print("STEP 19 COMPLETE")
print("=" * 90)

STEP 19 - TESTING CLAIM 1 ATOMIC CONTENT ATTRIBUTES

Testing 5 atomic propositions...

------------------------------------------------------------------------------------------
Attribute : first_meeting
Proposition: This was the first meeting.
Status    : FLAGGED
Supporting: []
Context   : []
Reason    : No supporting evidence found.
------------------------------------------------------------------------------------------
Attribute : team_interaction
Proposition: The meeting agenda included team interaction.
Status    : FLAGGED
Supporting: []
Context   : []
Reason    : No supporting evidence found.
------------------------------------------------------------------------------------------
Attribute : tool_training
Proposition: The meeting agenda included tool training.
Status    : FLAGGED
Supporting: []
Context   : []
Reason    : No supporting evidence found.
------------------------------------------------------------------------------------------
Attribute : project_plan
Proposition

In [31]:
# ============================================================
# STEP 20 - DIAGNOSE FIRST_MEETING RETRIEVAL + NLI
# ============================================================

proposition = "This was the first meeting."

print("=" * 90)
print("STEP 20 - DIAGNOSING FIRST_MEETING")
print("=" * 90)

print("\nProposition:")
print(proposition)

# ------------------------------------------------------------
# 1. Retrieve top candidates
# ------------------------------------------------------------

candidates = retrieve_evidence(
    proposition,
    top_k=10
)

print("\n" + "-" * 90)
print("TOP RETRIEVED EVIDENCE")
print("-" * 90)

for rank, candidate in enumerate(candidates, start=1):

    evidence_id = candidate["evidence_id"]

    print(
        f"\nRank {rank} | "
        f"ID {evidence_id} | "
        f"BGE={candidate['retrieval_similarity']:.4f}"
    )

    print(
        f"Speaker: {candidate['speaker']} | "
        f"{candidate['start']:.3f} → {candidate['end']:.3f}"
    )

    print("Text:", candidate["text"])

    # --------------------------------------------------------
    # Context
    # --------------------------------------------------------

    context = get_context_window(
        evidence_id,
        window_size=2
    )

    context_text = " ".join(
        item["text"]
        for item in context
    )

    # --------------------------------------------------------
    # NLI on context
    # --------------------------------------------------------

    nli_result = run_nli(
        context_text,
        proposition
    )

    print(
        "NLI:",
        nli_result["label"],
        "|",
        f"Contradiction={nli_result['contradiction']:.4f}",
        f"| Entailment={nli_result['entailment']:.4f}",
        f"| Neutral={nli_result['neutral']:.4f}"
    )

    # --------------------------------------------------------
    # Lexical support
    # --------------------------------------------------------

    lexical_result = lexical_support(
        proposition,
        context_text
    )

    print(
        "Lexical overlap:",
        f"{lexical_result['overlap_ratio']:.4f}"
    )

    print(
        "Matched:",
        lexical_result["matched_terms"]
    )

    print(
        "Missing:",
        lexical_result["missing_terms"]
    )

print("\n" + "=" * 90)
print("STEP 20 COMPLETE")
print("=" * 90)

STEP 20 - DIAGNOSING FIRST_MEETING

Proposition:
This was the first meeting.

------------------------------------------------------------------------------------------
TOP RETRIEVED EVIDENCE
------------------------------------------------------------------------------------------

Rank 1 | ID 9 | BGE=0.7503
Speaker: SPEAKER_02 | 84.658 → 93.127
Text: I'm Sarah, the project manager, and this is our first meeting, surprisingly enough. Okay, this is our agenda.
NLI: entailment | Contradiction=0.0002 | Entailment=0.9807 | Neutral=0.0192


KeyError: 'overlap_ratio'

In [32]:
# ============================================================
# STEP 21 - INSPECT LEXICAL SUPPORT OUTPUT
# ============================================================

proposition = "This was the first meeting."

context = get_context_window(
    9,
    window_size=2
)

context_text = " ".join(
    item["text"]
    for item in context
)

lexical_result = lexical_support(
    proposition,
    context_text
)

print("=" * 90)
print("STEP 21 - LEXICAL SUPPORT OUTPUT")
print("=" * 90)

print("\nProposition:")
print(proposition)

print("\nContext:")
print(context_text)

print("\nLexical-support result:")
print(lexical_result)

print("\nReturned keys:")
print(list(lexical_result.keys()))

print("\n" + "=" * 90)
print("STEP 21 COMPLETE")
print("=" * 90)

STEP 21 - LEXICAL SUPPORT OUTPUT

Proposition:
This was the first meeting.

Context:
I'm just gonna Okay, hello everybody. I'm Sarah, the project manager, and this is our first meeting, surprisingly enough. Okay, this is our agenda. We will do some stuff, get to know each other a bit better, feel more comfortable with each other. Then we'll go do tool training,

Lexical-support result:
{'lexical_overlap': 1.0, 'matched_terms': ['first', 'meeting'], 'missing_terms': []}

Returned keys:
['lexical_overlap', 'matched_terms', 'missing_terms']

STEP 21 COMPLETE


In [33]:
# ============================================================
# STEP 22 - INSPECT FIRST_MEETING CANDIDATE SELECTION
# ============================================================

proposition = "This was the first meeting."

analysis_result = analyze_attribute_evidence(
    proposition=proposition,
    claim_speaker="SPEAKER_02",
    top_k=10,
    context_window=2
)

print("=" * 90)
print("STEP 22 - FIRST_MEETING CANDIDATE ANALYSIS")
print("=" * 90)

print("\nNumber of candidates checked:",
      len(analysis_result.get("candidates", [])))

print("\nAnalysis result keys:")
print(list(analysis_result.keys()))

print("\n" + "-" * 90)
print("FULL ANALYSIS RESULT")
print("-" * 90)

import pprint
pprint.pprint(analysis_result, depth=4)

print("\n" + "=" * 90)
print("STEP 22 COMPLETE")
print("=" * 90)

STEP 22 - FIRST_MEETING CANDIDATE ANALYSIS


AttributeError: 'list' object has no attribute 'get'

In [34]:
# ============================================================
# STEP 22 - INSPECT FIRST_MEETING CANDIDATE ANALYSIS
# Corrected for list output
# ============================================================

proposition = "This was the first meeting."

analysis_result = analyze_attribute_evidence(
    proposition=proposition,
    claim_speaker="SPEAKER_02",
    top_k=10,
    context_window=2
)

print("=" * 90)
print("STEP 22 - FIRST_MEETING CANDIDATE ANALYSIS")
print("=" * 90)

print("\nReturn type:")
print(type(analysis_result))

print("\nNumber of candidates:")
print(len(analysis_result))

print("\n" + "-" * 90)
print("CANDIDATES")
print("-" * 90)

for rank, candidate in enumerate(analysis_result, start=1):

    print(f"\nRank {rank}")

    if isinstance(candidate, dict):

        print("ID:",
              candidate.get("evidence_id"))

        print("BGE similarity:",
              candidate.get("similarity"))

        print("Speaker:",
              candidate.get("speaker"))

        print("NLI:",
              candidate.get("nli_label"))

        print("NLI probabilities:",
              candidate.get("nli_probs"))

        print("Lexical support:",
              candidate.get("lexical_support"))

        print("Speaker check:",
              candidate.get("speaker_check"))

        print("Text:",
              candidate.get("text"))

    else:
        print(candidate)

print("\n" + "=" * 90)
print("STEP 22 COMPLETE")
print("=" * 90)

STEP 22 - FIRST_MEETING CANDIDATE ANALYSIS

Return type:
<class 'list'>

Number of candidates:
10

------------------------------------------------------------------------------------------
CANDIDATES
------------------------------------------------------------------------------------------

Rank 1
ID: 9
BGE similarity: None
Speaker: SPEAKER_02
NLI: entailment
NLI probabilities: None
Lexical support: None
Speaker check: {'applicable': True, 'passed': True, 'reason': 'Speaker matches evidence.'}
Text: I'm Sarah, the project manager, and this is our first meeting, surprisingly enough. Okay, this is our agenda.

Rank 2
ID: 91
BGE similarity: None
Speaker: SPEAKER_03
NLI: neutral
NLI probabilities: None
Lexical support: None
Speaker check: {'applicable': True, 'passed': False, 'reason': 'Speaker does not match evidence.'}
Text: In our first year?

Rank 3
ID: 35
BGE similarity: None
Speaker: SPEAKER_03
NLI: neutral
NLI probabilities: None
Lexical support: None
Speaker check: {'applicable': 

In [35]:
# ============================================================
# STEP 23 - INSPECT ANALYZE_ATTRIBUTE_EVIDENCE FUNCTION
# ============================================================

import inspect

print("=" * 90)
print("STEP 23 - CURRENT analyze_attribute_evidence()")
print("=" * 90)

print(inspect.getsource(analyze_attribute_evidence))

print("=" * 90)
print("STEP 23 COMPLETE")
print("=" * 90)

STEP 23 - CURRENT analyze_attribute_evidence()


OSError: could not get source code

In [36]:
# ============================================================
# STEP 23 - FIND analyze_attribute_evidence() CELL
# ============================================================

import json
import re

notebook_path = (
    "/content/drive/MyDrive/MTechIndProj/"
    "MoM_Project/04_mom_generation/04_mom_generation.ipynb"
)

with open(notebook_path, "r", encoding="utf-8") as f:
    nb = json.load(f)

print("=" * 90)
print("STEP 23 - SEARCHING NOTEBOOK FOR analyze_attribute_evidence()")
print("=" * 90)

found = False

for i, cell in enumerate(nb["cells"]):

    source = "".join(cell.get("source", []))

    if "def analyze_attribute_evidence" in source:

        found = True

        print(f"\nFunction found in notebook cell: {i}")
        print("-" * 90)
        print(source)
        print("-" * 90)

if not found:
    print("\nFunction definition was not found.")

print("\n" + "=" * 90)
print("STEP 23 COMPLETE")
print("=" * 90)

STEP 23 - SEARCHING NOTEBOOK FOR analyze_attribute_evidence()

Function found in notebook cell: 85
------------------------------------------------------------------------------------------
# ============================================================
# M7.13 — Integrated Attribute Evidence Analysis
# ============================================================
#
# Purpose:
#   Combine the existing verification components for ONE
#   attribute:
#
#       1. BGE + FAISS retrieval
#       2. Context expansion
#       3. Contextual NLI
#       4. Lexical support
#       5. Speaker consistency
#       6. Evidence timestamps
#
# IMPORTANT:
#   This cell intentionally DOES NOT make the final
#   VERIFIED / FLAGGED decision.
#
#   It produces all relevant signals first.
#   We will use real results to design the final
#   aggregation rule.
#
# No existing files are modified.
# ============================================================


def analyze_attribute_evidence(
    proposition,
    

In [37]:
# ============================================================
# STEP 24 - INSPECT select_supporting_candidates()
# ============================================================

import json

notebook_path = (
    "/content/drive/MyDrive/MTechIndProj/"
    "MoM_Project/04_mom_generation/04_mom_generation.ipynb"
)

with open(notebook_path, "r", encoding="utf-8") as f:
    nb = json.load(f)

print("=" * 90)
print("STEP 24 - SEARCHING FOR select_supporting_candidates()")
print("=" * 90)

for i, cell in enumerate(nb["cells"]):

    source = "".join(cell.get("source", []))

    if "def select_supporting_candidates" in source:

        print(f"\nFunction found in notebook cell: {i}")
        print("-" * 90)
        print(source)
        print("-" * 90)

print("\n" + "=" * 90)
print("STEP 24 COMPLETE")
print("=" * 90)

STEP 24 - SEARCHING FOR select_supporting_candidates()

Function found in notebook cell: 86
------------------------------------------------------------------------------------------
# ============================================================
# M7.14 — Evidence Candidate Selection
# ============================================================
#
# Purpose:
#   Select plausible supporting evidence from the integrated
#   evidence analysis.
#
# Selection principles:
#
#   1. Contradiction is not supporting evidence.
#   2. Neutral evidence is not treated as support.
#   3. Entailment is the strongest semantic signal.
#   4. Lexical overlap provides additional support.
#   5. Speaker mismatch is rejected when a speaker is claimed.
#   6. BGE similarity is used for retrieval, not proof.
#
# IMPORTANT:
#   No arbitrary numerical weighting is used yet.
#   No final VERIFIED / FLAGGED decision is made.
# ============================================================


def select_supporting_ca

In [38]:
# ============================================================
# STEP 25 - TEST FIRST_MEETING THROUGH SELECTION
# ============================================================

proposition = "This was the first meeting."

analysis_result = analyze_attribute_evidence(
    proposition=proposition,
    claim_speaker="SPEAKER_02",
    top_k=10,
    context_window=2
)

selection_result = select_supporting_candidates(
    analysis_result,
    min_lexical_overlap=0.0
)

print("=" * 90)
print("STEP 25 - FIRST_MEETING SELECTION TEST")
print("=" * 90)

print("\nSupporting candidates:")
for item in selection_result["supporting_candidates"]:
    print(
        f"ID {item['evidence_id']} | "
        f"BGE={item['retrieval_similarity']:.4f} | "
        f"NLI={item['nli_label']} | "
        f"Entailment={item['nli_entailment']:.4f} | "
        f"Lexical={item['lexical_overlap']:.4f} | "
        f"Speaker={item['speaker']}"
    )

print("\nRejected candidates:")
for item in selection_result["rejected_candidates"]:
    print(
        f"ID {item['evidence_id']} → {item['reason']}"
    )

print("\n" + "-" * 90)

supporting_ids = [
    item["evidence_id"]
    for item in selection_result["supporting_candidates"]
]

print("Supporting evidence IDs:", supporting_ids)

if 9 in supporting_ids:
    print("\n✅ ID 9 WAS ACCEPTED AS SUPPORTING EVIDENCE")
else:
    print("\n❌ ID 9 WAS NOT ACCEPTED")

print("\n" + "=" * 90)
print("STEP 25 COMPLETE")
print("=" * 90)

STEP 25 - FIRST_MEETING SELECTION TEST

Supporting candidates:

Rejected candidates:
ID 9 → NLI did not establish entailment
ID 91 → NLI did not establish entailment
ID 35 → NLI did not establish entailment
ID 12 → NLI did not establish entailment
ID 96 → NLI did not establish entailment
ID 10 → NLI did not establish entailment
ID 47 → NLI did not establish entailment
ID 72 → NLI did not establish entailment
ID 8 → NLI did not establish entailment
ID 139 → NLI did not establish entailment

------------------------------------------------------------------------------------------
Supporting evidence IDs: []

❌ ID 9 WAS NOT ACCEPTED

STEP 25 COMPLETE


In [39]:
# ============================================================
# STEP 26 - FIX NLI LABEL CASE MISMATCH
# ============================================================

def select_supporting_candidates(
    analysis_result,
    min_lexical_overlap=0.0
):
    """
    Select candidates that provide potential support
    for an atomic proposition.

    NLI labels are normalized to uppercase so that
    lowercase labels returned by run_nli() are handled
    correctly.
    """

    supporting_candidates = []
    rejected_candidates = []

    # --------------------------------------------------------
    # Evaluate every retrieved candidate
    # --------------------------------------------------------

    for item in analysis_result:

        evidence_id = item["evidence_id"]

        # Normalize NLI label
        nli_label = str(item["nli_label"]).upper()

        speaker_check = item["speaker_check"]
        lexical_overlap = item["lexical_overlap"]

        # ----------------------------------------------------
        # Reject contradiction
        # ----------------------------------------------------

        if nli_label == "CONTRADICTION":

            rejected_candidates.append({
                "evidence_id": evidence_id,
                "reason": "NLI contradiction"
            })

            continue

        # ----------------------------------------------------
        # Reject neutral
        # ----------------------------------------------------

        if nli_label == "NEUTRAL":

            rejected_candidates.append({
                "evidence_id": evidence_id,
                "reason": "NLI neutral"
            })

            continue

        # ----------------------------------------------------
        # Require entailment
        # ----------------------------------------------------

        if nli_label != "ENTAILMENT":

            rejected_candidates.append({
                "evidence_id": evidence_id,
                "reason": "NLI did not establish entailment"
            })

            continue

        # ----------------------------------------------------
        # Speaker consistency
        # ----------------------------------------------------

        if speaker_check["applicable"]:

            if not speaker_check["passed"]:

                rejected_candidates.append({
                    "evidence_id": evidence_id,
                    "reason": "Speaker mismatch"
                })

                continue

        # ----------------------------------------------------
        # Optional lexical-support threshold
        # ----------------------------------------------------

        if lexical_overlap < min_lexical_overlap:

            rejected_candidates.append({
                "evidence_id": evidence_id,
                "reason": "Insufficient lexical support"
            })

            continue

        # ----------------------------------------------------
        # Candidate passed all selection rules
        # ----------------------------------------------------

        supporting_candidates.append(item)

    # --------------------------------------------------------
    # Rank supporting candidates
    # --------------------------------------------------------
    # NLI entailment = primary
    # BGE similarity = secondary
    # --------------------------------------------------------

    supporting_candidates = sorted(
        supporting_candidates,
        key=lambda x: (
            x["nli_entailment"],
            x["retrieval_similarity"]
        ),
        reverse=True
    )

    return {
        "supporting_candidates": supporting_candidates,
        "rejected_candidates": rejected_candidates
    }


print("=" * 90)
print("STEP 26 COMPLETE")
print("=" * 90)
print("select_supporting_candidates() updated successfully.")
print("NLI labels are now normalized using .upper().")

STEP 26 COMPLETE
select_supporting_candidates() updated successfully.
NLI labels are now normalized using .upper().


In [40]:
# ============================================================
# STEP 27 - RETEST FIRST_MEETING SELECTION
# ============================================================

proposition = "This was the first meeting."

analysis_result = analyze_attribute_evidence(
    proposition=proposition,
    claim_speaker="SPEAKER_02",
    top_k=10,
    context_window=2
)

selection_result = select_supporting_candidates(
    analysis_result,
    min_lexical_overlap=0.0
)

print("=" * 90)
print("STEP 27 - FIRST_MEETING SELECTION RETEST")
print("=" * 90)

print("\nSupporting candidates:")

for item in selection_result["supporting_candidates"]:
    print(
        f"ID {item['evidence_id']} | "
        f"BGE={item['retrieval_similarity']:.4f} | "
        f"NLI={item['nli_label']} | "
        f"Entailment={item['nli_entailment']:.4f} | "
        f"Lexical={item['lexical_overlap']:.4f} | "
        f"Speaker={item['speaker']}"
    )

print("\nRejected candidates:")

for item in selection_result["rejected_candidates"]:
    print(
        f"ID {item['evidence_id']} → {item['reason']}"
    )

supporting_ids = [
    item["evidence_id"]
    for item in selection_result["supporting_candidates"]
]

print("\n" + "-" * 90)
print("Supporting evidence IDs:", supporting_ids)

if 9 in supporting_ids:
    print("✅ SUCCESS: ID 9 is now supporting evidence.")
else:
    print("❌ ID 9 is still not supporting evidence.")

print("\n" + "=" * 90)
print("STEP 27 COMPLETE")
print("=" * 90)

STEP 27 - FIRST_MEETING SELECTION RETEST

Supporting candidates:
ID 10 | BGE=0.6155 | NLI=entailment | Entailment=0.9930 | Lexical=1.0000 | Speaker=SPEAKER_02
ID 8 | BGE=0.5905 | NLI=entailment | Entailment=0.9866 | Lexical=1.0000 | Speaker=SPEAKER_02
ID 9 | BGE=0.7503 | NLI=entailment | Entailment=0.9807 | Lexical=1.0000 | Speaker=SPEAKER_02

Rejected candidates:
ID 91 → NLI neutral
ID 35 → NLI neutral
ID 12 → NLI neutral
ID 96 → NLI contradiction
ID 47 → NLI contradiction
ID 72 → NLI neutral
ID 139 → NLI neutral

------------------------------------------------------------------------------------------
Supporting evidence IDs: [10, 8, 9]
✅ SUCCESS: ID 9 is now supporting evidence.

STEP 27 COMPLETE


In [41]:
# ============================================================
# STEP 28 - VERIFY FIRST_MEETING END-TO-END
# ============================================================

proposition = "This was the first meeting."

result = verify_attribute(
    attribute="first_meeting",
    proposition=proposition,
    claim_speaker="SPEAKER_02",
    top_k=10,
    context_window=2
)

print("=" * 90)
print("STEP 28 - FIRST_MEETING VERIFICATION")
print("=" * 90)

print("\nAttribute:")
print("first_meeting")

print("\nProposition:")
print(proposition)

print("\nStatus:")
print(result["status"])

print("\nReason:")
print(result["reason"])

print("\nSupporting evidence IDs:")
print(result["supporting_evidence_ids"])

print("\nContext evidence IDs:")
print(result["context_evidence_ids"])

print("\nDisplay interval:")
print(
    f"{result['display_start']:.3f} → "
    f"{result['display_end']:.3f}"
)

print("\nSpeakers:")
print(result["speakers"])

print("\nSpeaker check:")
print(result["speaker_check"])

print("\nCandidates checked:")
print(result["candidates_checked"])

print("\n" + "=" * 90)
print("STEP 28 COMPLETE")
print("=" * 90)

STEP 28 - FIRST_MEETING VERIFICATION

Attribute:
first_meeting

Proposition:
This was the first meeting.

Status:
VERIFIED

Reason:
The proposition is supported by contextual NLI evidence and the claimed speaker is consistent with the supporting evidence.

Supporting evidence IDs:
[8, 9, 10]

Context evidence IDs:
[6, 7, 8, 9, 10, 11, 12]

Display interval:
80.574 → 101.315

Speakers:
['SPEAKER_02']

Speaker check:
True

Candidates checked:
10

STEP 28 COMPLETE


In [42]:
# ============================================================
# STEP 29 - VERIFY TEAM INTERACTION
# ============================================================

proposition = (
    "The meeting included activities to help the team "
    "get to know each other and interact."
)

result = verify_attribute(
    attribute="team_interaction",
    proposition=proposition,
    claim_speaker="SPEAKER_02",
    top_k=10,
    context_window=2
)

print("=" * 90)
print("STEP 29 - TEAM INTERACTION VERIFICATION")
print("=" * 90)

print("\nAttribute:")
print("team_interaction")

print("\nProposition:")
print(proposition)

print("\nStatus:")
print(result["status"])

print("\nReason:")
print(result["reason"])

print("\nSupporting evidence IDs:")
print(result["supporting_evidence_ids"])

print("\nContext evidence IDs:")
print(result["context_evidence_ids"])

print("\nDisplay interval:")
print(
    f"{result['display_start']:.3f} → "
    f"{result['display_end']:.3f}"
)

print("\nSpeakers:")
print(result["speakers"])

print("\nSpeaker check:")
print(result["speaker_check"])

print("\nCandidates checked:")
print(result["candidates_checked"])

print("\n" + "=" * 90)
print("STEP 29 COMPLETE")
print("=" * 90)

STEP 29 - TEAM INTERACTION VERIFICATION

Attribute:
team_interaction

Proposition:
The meeting included activities to help the team get to know each other and interact.

Status:
VERIFIED

Reason:
The proposition is supported by contextual NLI evidence and the claimed speaker is consistent with the supporting evidence.

Supporting evidence IDs:
[10]

Context evidence IDs:
[8, 9, 10, 11, 12]

Display interval:
95.790 → 101.315

Speakers:
['SPEAKER_02']

Speaker check:
True

Candidates checked:
10

STEP 29 COMPLETE


In [43]:
# ============================================================
# STEP 30 - VERIFY TOOL TRAINING
# ============================================================

proposition = (
    "The meeting included tool training."
)

result = verify_attribute(
    attribute="tool_training",
    proposition=proposition,
    claim_speaker="SPEAKER_02",
    top_k=10,
    context_window=2
)

print("=" * 90)
print("STEP 30 - TOOL TRAINING VERIFICATION")
print("=" * 90)

print("\nAttribute:")
print("tool_training")

print("\nProposition:")
print(proposition)

print("\nStatus:")
print(result["status"])

print("\nReason:")
print(result["reason"])

print("\nSupporting evidence IDs:")
print(result["supporting_evidence_ids"])

print("\nContext evidence IDs:")
print(result["context_evidence_ids"])

print("\nDisplay interval:")
print(
    f"{result['display_start']:.3f} → "
    f"{result['display_end']:.3f}"
)

print("\nSpeakers:")
print(result["speakers"])

print("\nSpeaker check:")
print(result["speaker_check"])

print("\nCandidates checked:")
print(result["candidates_checked"])

print("\n" + "=" * 90)
print("STEP 30 COMPLETE")
print("=" * 90)

STEP 30 - TOOL TRAINING VERIFICATION

Attribute:
tool_training

Proposition:
The meeting included tool training.

Status:
VERIFIED

Reason:
The proposition is supported by contextual NLI evidence and the claimed speaker is consistent with the supporting evidence.

Supporting evidence IDs:
[9, 10, 11, 12]

Context evidence IDs:
[7, 8, 9, 10, 11, 12, 13, 14]

Display interval:
84.658 → 109.823

Speakers:
['SPEAKER_02']

Speaker check:
True

Candidates checked:
10

STEP 30 COMPLETE


In [44]:
# ============================================================
# STEP 31 - VERIFY PROJECT PLAN
# ============================================================

proposition = (
    "The meeting included discussion of the project plan."
)

result = verify_attribute(
    attribute="project_plan",
    proposition=proposition,
    claim_speaker="SPEAKER_02",
    top_k=10,
    context_window=2
)

print("=" * 90)
print("STEP 31 - PROJECT PLAN VERIFICATION")
print("=" * 90)

print("\nAttribute:")
print("project_plan")

print("\nProposition:")
print(proposition)

print("\nStatus:")
print(result["status"])

print("\nReason:")
print(result["reason"])

print("\nSupporting evidence IDs:")
print(result["supporting_evidence_ids"])

print("\nContext evidence IDs:")
print(result["context_evidence_ids"])

print("\nDisplay interval:")
print(
    f"{result['display_start']:.3f} → "
    f"{result['display_end']:.3f}"
)

print("\nSpeakers:")
print(result["speakers"])

print("\nSpeaker check:")
print(result["speaker_check"])

print("\nCandidates checked:")
print(result["candidates_checked"])

print("\n" + "=" * 90)
print("STEP 31 COMPLETE")
print("=" * 90)

STEP 31 - PROJECT PLAN VERIFICATION

Attribute:
project_plan

Proposition:
The meeting included discussion of the project plan.

Status:
VERIFIED

Reason:
The proposition is supported by contextual NLI evidence and the claimed speaker is consistent with the supporting evidence.

Supporting evidence IDs:
[10]

Context evidence IDs:
[8, 9, 10, 11, 12]

Display interval:
95.790 → 101.315

Speakers:
['SPEAKER_02']

Speaker check:
True

Candidates checked:
10

STEP 31 COMPLETE


In [45]:
# ============================================================
# STEP 32 - VERIFY DISCUSSION OF IDEAS
# ============================================================

proposition = (
    "The meeting included discussion of the participants' own ideas."
)

result = verify_attribute(
    attribute="discussion_of_ideas",
    proposition=proposition,
    claim_speaker="SPEAKER_02",
    top_k=10,
    context_window=2
)

print("=" * 90)
print("STEP 32 - DISCUSSION OF IDEAS VERIFICATION")
print("=" * 90)

print("\nAttribute:")
print("discussion_of_ideas")

print("\nProposition:")
print(proposition)

print("\nStatus:")
print(result["status"])

print("\nReason:")
print(result["reason"])

print("\nSupporting evidence IDs:")
print(result["supporting_evidence_ids"])

print("\nContext evidence IDs:")
print(result["context_evidence_ids"])

print("\nDisplay interval:")
print(
    f"{result['display_start']:.3f} → "
    f"{result['display_end']:.3f}"
)

print("\nSpeakers:")
print(result["speakers"])

print("\nSpeaker check:")
print(result["speaker_check"])

print("\nCandidates checked:")
print(result["candidates_checked"])

print("\n" + "=" * 90)
print("STEP 32 COMPLETE")
print("=" * 90)

STEP 32 - DISCUSSION OF IDEAS VERIFICATION

Attribute:
discussion_of_ideas

Proposition:
The meeting included discussion of the participants' own ideas.

Status:
VERIFIED

Reason:
The proposition is supported by contextual NLI evidence and the claimed speaker is consistent with the supporting evidence.

Supporting evidence IDs:
[10]

Context evidence IDs:
[8, 9, 10, 11, 12]

Display interval:
95.790 → 101.315

Speakers:
['SPEAKER_02']

Speaker check:
True

Candidates checked:
10

STEP 32 COMPLETE


In [46]:
# ============================================================
# STEP 33 - RETEST COMPLETE CLAIM 1
# ============================================================

claim_1 = {
    "claim_id": 1,
    "topic": "meeting intro/agenda",
    "claim_text": (
        "The meeting was introduced as the first meeting, "
        "with an agenda covering team interaction, tool training, "
        "the project plan, and discussion of ideas."
    ),
    "speaker": "SPEAKER_02",
    "start": 84.658,
    "end": 109.823,
    "event_type": "INFORMATION"
}

result = verify_mom_claim(
    claim=claim_1,
    supporting_evidence=[8, 9, 10, 11, 12],
    content_proposition=claim_1["claim_text"],
    expected_start=claim_1["start"],
    expected_end=claim_1["end"]
)

print("=" * 90)
print("STEP 33 - COMPLETE CLAIM 1 VERIFICATION")
print("=" * 90)

print("\nClaim:")
print(claim_1["claim_text"])

print("\nStatus:")
print(result["status"])

print("\nReason:")
print(result["reason"])

print("\nFailed checks:")
print(result["failed_checks"])

print("\nEvidence IDs:")
print(result["evidence_ids"])

print("\nEvidence speakers:")
print(result["evidence_speakers"])

print("\nEvidence interval:")
print(
    f"{result['evidence_start']:.3f} → "
    f"{result['evidence_end']:.3f}"
)

print("\nChecks:")
for check_name, check_result in result["checks"].items():
    print(f"\n{check_name}:")
    print(check_result)

print("\n" + "=" * 90)
print("STEP 33 COMPLETE")
print("=" * 90)

AttributeError: 'int' object has no attribute 'get'

In [47]:
# ============================================================
# STEP 33 - RETEST COMPLETE CLAIM 1
# Corrected: pass evidence dictionaries, not integer IDs
# ============================================================

claim_1 = {
    "claim_id": 1,
    "topic": "meeting intro/agenda",
    "claim_text": (
        "The meeting was introduced as the first meeting, "
        "with an agenda covering team interaction, tool training, "
        "the project plan, and discussion of ideas."
    ),
    "speaker": "SPEAKER_02",
    "start": 84.658,
    "end": 109.823,
    "event_type": "INFORMATION"
}

# ------------------------------------------------------------
# Analyze the complete claim
# ------------------------------------------------------------

analysis_result = analyze_attribute_evidence(
    proposition=claim_1["claim_text"],
    claim_speaker=claim_1["speaker"],
    top_k=10,
    context_window=2
)

# ------------------------------------------------------------
# Select supporting evidence
# ------------------------------------------------------------

selection_result = select_supporting_candidates(
    analysis_result,
    min_lexical_overlap=0.0
)

supporting_evidence = selection_result[
    "supporting_candidates"
]

# ------------------------------------------------------------
# Run final claim-level verification
# ------------------------------------------------------------

result = verify_mom_claim(
    claim=claim_1,
    supporting_evidence=supporting_evidence,
    content_proposition=claim_1["claim_text"],
    expected_start=claim_1["start"],
    expected_end=claim_1["end"]
)

print("=" * 90)
print("STEP 33 - COMPLETE CLAIM 1 VERIFICATION")
print("=" * 90)

print("\nClaim:")
print(claim_1["claim_text"])

print("\nSupporting evidence supplied:")
print([
    item["evidence_id"]
    for item in supporting_evidence
])

print("\nStatus:")
print(result["status"])

print("\nReason:")
print(result["reason"])

print("\nFailed checks:")
print(result["failed_checks"])

print("\nEvidence IDs:")
print(result["evidence_ids"])

print("\nEvidence speakers:")
print(result["evidence_speakers"])

print("\nEvidence interval:")
print(
    f"{result['evidence_start']:.3f} → "
    f"{result['evidence_end']:.3f}"
)

print("\nChecks:")

for check_name, check_result in result["checks"].items():
    print(f"\n{check_name}:")
    print(check_result)

print("\n" + "=" * 90)
print("STEP 33 COMPLETE")
print("=" * 90)

STEP 33 - COMPLETE CLAIM 1 VERIFICATION

Claim:
The meeting was introduced as the first meeting, with an agenda covering team interaction, tool training, the project plan, and discussion of ideas.

Supporting evidence supplied:
[10, 11, 9]

Status:
FLAGGED

Reason:
The claim failed one or more consistency checks: timestamp_consistency

Failed checks:
['timestamp_consistency']

Evidence IDs:
[10, 11, 9]

Evidence speakers:
['SPEAKER_02']

Evidence interval:
84.658 → 105.239

Checks:

content_consistency:
{'applicable': True, 'passed': True, 'reason': 'The proposition is supported by contextual NLI evidence and the claimed speaker is consistent with the supporting evidence.', 'supporting_evidence_ids': [9, 10, 11], 'context_evidence_ids': [7, 8, 9, 10, 11, 12, 13]}

speaker_consistency:
{'applicable': True, 'passed': True, 'claim_speaker': 'SPEAKER_02', 'evidence_speakers': ['SPEAKER_02'], 'reason': "Claimed speaker 'SPEAKER_02' is consistent with the supporting evidence."}

timestamp_co

In [48]:
# ============================================================
# STEP 34 - INSPECT TIMESTAMP CONSISTENCY FUNCTION
# ============================================================

import json

notebook_path = (
    "/content/drive/MyDrive/MTechIndProj/"
    "MoM_Project/04_mom_generation/04_mom_generation.ipynb"
)

with open(notebook_path, "r", encoding="utf-8") as f:
    nb = json.load(f)

print("=" * 90)
print("STEP 34 - SEARCHING FOR check_timestamp_consistency()")
print("=" * 90)

for i, cell in enumerate(nb["cells"]):

    source = "".join(cell.get("source", []))

    if "def check_timestamp_consistency" in source:

        print(f"\nFunction found in notebook cell: {i}")
        print("-" * 90)
        print(source)
        print("-" * 90)

print("\n" + "=" * 90)
print("STEP 34 COMPLETE")
print("=" * 90)

STEP 34 - SEARCHING FOR check_timestamp_consistency()

Function found in notebook cell: 83
------------------------------------------------------------------------------------------
# ============================================================
# M7.11 — Speaker & Timestamp Consistency Test
# ============================================================
#
# Purpose:
#   Test deterministic metadata consistency between a
#   generated MoM claim and its supporting evidence.
#
# These checks do NOT require another ML model.
#
# Speaker:
#   Exact comparison with evidence speaker.
#
# Timestamp:
#   Evidence timestamps are taken directly from the
#   speaker-attributed transcript.
#
# This is a diagnostic step only.
# ============================================================


def check_speaker_consistency(claim_speaker, evidence_speaker):
    """
    Check whether the claimed speaker matches the evidence speaker.
    """

    if claim_speaker is None:
        return {
            "applic

In [49]:
# ============================================================
# STEP 35 - CLAIM 1 WITHOUT INDEPENDENT EXPECTED TIMESTAMP
# ============================================================

claim_1 = {
    "claim_id": 1,
    "topic": "meeting intro/agenda",
    "claim_text": (
        "The meeting was introduced as the first meeting, "
        "with an agenda covering team interaction, tool training, "
        "the project plan, and discussion of ideas."
    ),
    "speaker": "SPEAKER_02",
    "start": 84.658,
    "end": 109.823,
    "event_type": "INFORMATION"
}

# ------------------------------------------------------------
# Analyze complete claim
# ------------------------------------------------------------

analysis_result = analyze_attribute_evidence(
    proposition=claim_1["claim_text"],
    claim_speaker=claim_1["speaker"],
    top_k=10,
    context_window=2
)

# ------------------------------------------------------------
# Select supporting evidence
# ------------------------------------------------------------

selection_result = select_supporting_candidates(
    analysis_result,
    min_lexical_overlap=0.0
)

supporting_evidence = selection_result[
    "supporting_candidates"
]

# ------------------------------------------------------------
# Verify claim
# IMPORTANT:
# No expected_start / expected_end supplied.
# ------------------------------------------------------------

result = verify_mom_claim(
    claim=claim_1,
    supporting_evidence=supporting_evidence,
    content_proposition=claim_1["claim_text"]
)

print("=" * 90)
print("STEP 35 - CLAIM 1 WITHOUT EXPECTED TIMESTAMP")
print("=" * 90)

print("\nClaim:")
print(claim_1["claim_text"])

print("\nSupporting evidence:")
print([
    item["evidence_id"]
    for item in supporting_evidence
])

print("\nStatus:")
print(result["status"])

print("\nReason:")
print(result["reason"])

print("\nFailed checks:")
print(result["failed_checks"])

print("\nEvidence interval:")
print(
    f"{result['evidence_start']:.3f} → "
    f"{result['evidence_end']:.3f}"
)

print("\nChecks:")

for check_name, check_result in result["checks"].items():
    print(f"\n{check_name}:")
    print(check_result)

print("\n" + "=" * 90)
print("STEP 35 COMPLETE")
print("=" * 90)

STEP 35 - CLAIM 1 WITHOUT EXPECTED TIMESTAMP

Claim:
The meeting was introduced as the first meeting, with an agenda covering team interaction, tool training, the project plan, and discussion of ideas.

Supporting evidence:
[10, 11, 9]

Status:
VERIFIED

Reason:
The claim passed all applicable multi-attribute consistency checks.

Failed checks:
[]

Evidence interval:
84.658 → 105.239

Checks:

content_consistency:
{'applicable': True, 'passed': True, 'reason': 'The proposition is supported by contextual NLI evidence and the claimed speaker is consistent with the supporting evidence.', 'supporting_evidence_ids': [9, 10, 11], 'context_evidence_ids': [7, 8, 9, 10, 11, 12, 13]}

speaker_consistency:
{'applicable': True, 'passed': True, 'claim_speaker': 'SPEAKER_02', 'evidence_speakers': ['SPEAKER_02'], 'reason': "Claimed speaker 'SPEAKER_02' is consistent with the supporting evidence."}

timestamp_consistency:
{'applicable': False, 'passed': True, 'reason': 'No expected timestamp specified

In [50]:
# ============================================================
# STEP 36 - SAVE M7 CHECKPOINT IN NOTEBOOK
# ============================================================

import json
from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive/MyDrive/MTechIndProj/MoM_Project"
)

CHECKPOINT_DIR = PROJECT_DIR / "outputs" / "verified"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

checkpoint = {
    "project": "Evidence-Grounded Meeting Minutes Generator",
    "meeting_id": "ES2004a",
    "milestone": "M7 Multi-Attribute Evidence Verification",
    "status": "WORKING",
    "nli_label_normalization": "FIXED",
    "atomic_claim_1_attributes_verified": 5,
    "complete_claim_1_status": "VERIFIED",
    "timestamp_policy": (
        "Citation timestamps are taken from verified evidence. "
        "Expected timestamps are only used when independently available."
    ),
    "last_verified_claim": 1,
    "last_test_step": 35
}

checkpoint_path = (
    CHECKPOINT_DIR /
    "ES2004a_M7_checkpoint.json"
)

with open(checkpoint_path, "w", encoding="utf-8") as f:
    json.dump(checkpoint, f, indent=2)

print("=" * 90)
print("STEP 36 - M7 CHECKPOINT SAVED")
print("=" * 90)

print("\nCheckpoint:")
print(checkpoint_path)

print("\nStatus:")
print("✅ M7 checkpoint saved")

print("\nNext session should resume from:")
print("→ Claim 11 / ACTION verification")
print("→ Then remaining claims")
print("→ Then batch verification")
print("→ Then evaluation")
print("→ Then dashboard")

STEP 36 - M7 CHECKPOINT SAVED

Checkpoint:
/content/drive/MyDrive/MTechIndProj/MoM_Project/outputs/verified/ES2004a_M7_checkpoint.json

Status:
✅ M7 checkpoint saved

Next session should resume from:
→ Claim 11 / ACTION verification
→ Then remaining claims
→ Then batch verification
→ Then evaluation
→ Then dashboard


In [51]:
!git status

fatal: not a git repository (or any of the parent directories): .git
